# End-to-End Farm Crop, Condition and Yield from Capella X-band SAR

**ANRF AISEHack 2.0 — Round 2 · Sokhda, Vadodara · 966 farms · target date 13 Oct 2025**

Everything below is computed **inside this notebook run**, starting from the official
competition data and public Sentinel source imagery. No farm-level feature, crop label,
health score, yield value or submission row is imported from anywhere.

```
competition Capella SLC + boundaries ─┐
                                      ├─► geometry v3.2.1 ─► Jun19 correction ─► structure/evidence ─┐
Sentinel-1 RTC  (pinned HF source) ───┤                                                              ├─► CROP_ID v4 ─► HEALTH v2 ─► YIELD v3.1 ─► submission.csv
Sentinel-2 L2A  (pinned HF source) ───┘                                                              ┘
```

## 1 · Runtime, competition data discovery, and provenance lock

**The competition dataset contains a trap.** The Jun19 product directory holds a stray copy of
the **Jun06** SLC. A "first match" or "largest TIFF" heuristic silently loads the wrong scene.
We resolve every scene by its **acquisition tag**, require the file to sit in its **own matching
product directory**, and fail if a tag resolves ambiguously.

In [ ]:
import os, sys, json, glob, hashlib, shutil, subprocess, time, warnings
from pathlib import Path
import numpy as np
warnings.filterwarnings("ignore")
T_START = time.time()

WORK = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
ROOT = WORK / "round2"                 # canonical modules expect a `round2/` working root
SRC  = WORK / "src"
for d in (ROOT, ROOT / "_diag", SRC):
    d.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for b in iter(lambda: fh.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

def rel(p):
    p = Path(p)
    for base, tag in ((WORK, "work://"), (Path("/kaggle/input"), "input://")):
        try:    return tag + p.resolve().relative_to(base).as_posix()
        except Exception: pass
    return p.name

# ---- competition root ----
def find_competition_root():
    for base in (Path("/kaggle/input"), Path.cwd(), Path.cwd().parent):
        if not base.is_dir():
            continue
        # .../<competition>/Farm_boundaries_shp/Farm_boundaries_shp/Sokhda_Farms.shp
        hits = sorted({p.parent.parent.parent for p in base.glob("**/Farm_boundaries_shp/**/*.shp")})
        hits = [h for h in hits if list(h.glob("CAPELLA_*SLC_HH_*"))]
        if len(hits) == 1:
            return hits[0]
        if len(hits) > 1:
            raise RuntimeError(f"Ambiguous competition root: {hits}")
    raise FileNotFoundError("Competition dataset not found")

COMP = find_competition_root()
print("competition root :", rel(COMP))
print("python           :", sys.version.split()[0], "| numpy", np.__version__)
print("cpu              :", os.cpu_count())
try:
    import psutil; print("RAM available GB : %.1f" % (psutil.virtual_memory().available / 1e9))
except Exception: pass

# ---- PROVENANCE LOCK: exactly four scenes, each in its own product directory ----
TAGS = {"Jun06": "20250606072501_20250606072506", "Jun19": "20250619021410_20250619021415",
        "Aug14": "20250814031124_20250814031129", "Oct13": "20251013022643_20251013022648"}
STAGE = ROOT / "data"
(STAGE / "Farm_boundaries_shp" / "Farm_boundaries_shp").mkdir(parents=True, exist_ok=True)
(STAGE / "Village_Shp" / "Village_Shp").mkdir(parents=True, exist_ok=True)

def link(src, dst):
    dst = Path(dst)
    if dst.exists() or dst.is_symlink():
        return
    try:    os.symlink(src, dst)
    except Exception: shutil.copyfile(src, dst)

all_slc = sorted(COMP.glob("**/CAPELLA_*SLC_HH_*.tif"))
print(f"\\nSLC .tif files present in competition data: {len(all_slc)}  (expected 4 + 1 stray)")
locked = {}
for name, tag in TAGS.items():
    prod = COMP / f"CAPELLA_C14_SM_SLC_HH_{tag}"
    assert prod.is_dir(), f"{name}: product directory missing"
    # the scene must live in the directory bearing its OWN tag
    own = sorted(prod.glob(f"CAPELLA_*SLC_HH_{tag}.tif"))
    assert len(own) == 1, f"{name}: expected exactly 1 self-tagged SLC, found {len(own)}"
    strays = [p for p in prod.glob("CAPELLA_*SLC_HH_*.tif") if p != own[0]]
    d = STAGE / prod.name; d.mkdir(parents=True, exist_ok=True)
    for pat in (f"CAPELLA_*SLC_HH_{tag}.tif", f"CAPELLA_*SLC_HH_{tag}_extended.json",
                f"CAPELLA_*GEO_HH_{tag}*preview.tif", f"CAPELLA_*SLC_HH_{tag}.json"):
        for f in prod.glob(pat):
            link(f, d / f.name)
    locked[name] = dict(tag=tag, slc=own[0].name, bytes=own[0].stat().st_size,
                        strays_excluded=[s.name for s in strays])
    print(f"  {name}  {own[0].name[:58]}  {own[0].stat().st_size/1e6:6.0f} MB"
          + (f"   EXCLUDED STRAY: {strays[0].name[:46]}" if strays else ""))

n_stray = sum(len(v["strays_excluded"]) for v in locked.values())
print(f"\\nprovenance lock: 4 scenes locked by tag, {n_stray} stray file(s) excluded")
assert len({v["slc"] for v in locked.values()}) == 4, "scene filenames must be distinct"

for f in (COMP / "Farm_boundaries_shp/Farm_boundaries_shp").glob("Sokhda_Farms.*"):
    link(f, STAGE / "Farm_boundaries_shp/Farm_boundaries_shp" / f.name)
for f in (COMP / "Village_Shp/Village_Shp").glob("Sokhda_Village.*"):
    link(f, STAGE / "Village_Shp/Village_Shp" / f.name)
print("boundaries staged:", len(list((STAGE / "Farm_boundaries_shp/Farm_boundaries_shp").glob("*"))),
      "farm files,", len(list((STAGE / "Village_Shp/Village_Shp").glob("*"))), "village files")

## 2 · Farm boundaries

966 polygons, `village_id = 22`, median ~0.27 ha. Every later stage samples these polygons.

In [ ]:
import shapefile
from shapely.geometry import shape as gshape
from shapely.ops import transform as shp_transform
from pyproj import Transformer

FARM_SHP = STAGE / "Farm_boundaries_shp/Farm_boundaries_shp/Sokhda_Farms"
sf = shapefile.Reader(str(FARM_SHP))
recs, shps = sf.records(), sf.shapes()
to_m = Transformer.from_crs("EPSG:4326", "EPSG:32643", always_xy=True).transform
areas = np.array([shp_transform(to_m, gshape(s.__geo_interface__).buffer(0)).area
                  for s in shps]) / 1e4
fids = [int(r[0]) for r in recs]
vids = sorted({int(r[3]) for r in recs})

assert len(recs) == 966, f"expected 966 farms, got {len(recs)}"
assert sorted(fids) == list(range(1, 967)), "farm_id must be a complete 1..966 sequence"
assert vids == [22], f"village_id must be 22, got {vids}"
print(f"farms {len(recs)} | farm_id 1..966 complete | village_id {vids}")
print(f"area ha: median {np.median(areas):.3f}  p05 {np.percentile(areas,5):.3f}  "
      f"p95 {np.percentile(areas,95):.3f}  total {areas.sum():.1f}")
print(f"farms below 0.10 ha: {(areas<0.10).sum()} "
      f"-- small fields drive the reliability gating used later")

---
## 3 · Capella SLC → farm backscatter  ·  `[P] S03 · S04 · S05`

### WHY
An SLC pixel is a complex number in radar (range, azimuth) geometry; a farm is a 0.27 ha
lat/lon polygon. Connecting the two correctly is the single largest source of silent error in
this pipeline — and an earlier build got it wrong by **~5.8 km** by trusting the SICD corner
coordinates. Every Capella feature computed before that discovery was an artifact of sampling
the wrong ground.

### METHOD
$$\beta^0=\bigl(|DN|\cdot s\bigr)^2,\qquad \mathrm{dB}=10\log_{10}\beta^0$$

with scale $s$ read from each scene's `_extended.json`. SLC data is single-look: **ENL = 1**, so
per-pixel speckle SD is **5.573 dB**. Pixel spacing is finer than the resolution cell by
**1.80×**, so effective looks are `n_eff = n_valid / 1.80` wherever a standard error is formed —
oversampling is never mistaken for precision.

Geometry projects each polygon through the **RPC model with a shared-height search**, then
validates against the orthorectified `GEO` preview. **No external DEM is used.**

### CODE
Three canonical modules are materialised and executed: the competition forensic audit (S03),
farm centroids (S04, pure geometry), and the v3.2.1 geometry rebuild (S05).

In [ ]:
# ---- S04 farm centroids: pure geometry, no model ----
import csv as _csv
cent_path = ROOT / "_diag/farm_centroids.csv"
tr_c = Transformer.from_crs("EPSG:4326", "EPSG:32643", always_xy=True)
with open(cent_path, "w", newline="", encoding="utf-8") as fh:
    w = _csv.writer(fh); w.writerow(["farm_id", "x", "y", "lon", "lat"])
    for rec, s in zip(recs, shps):
        g = gshape(s.__geo_interface__).buffer(0)
        c = g.centroid
        x, y = tr_c.transform(c.x, c.y)
        w.writerow([int(rec[0]), round(x, 3), round(y, 3), round(c.x, 7), round(c.y, 7)])
print(f"S04 farm centroids -> {rel(cent_path)}  ({sum(1 for _ in open(cent_path))-1} rows)")

In [ ]:
# S03 -- competition-data forensic audit (organizer data only).
_SRC = '"""\nCOMPETITION DATA DEEP DIVE - independent forensic audit of organizer-provided data only.\n\nRules obeyed:\n  - competition-provided data ONLY (no S1/S2/HLS anywhere in this script)\n  - observation is separated from interpretation\n  - no crop names attached to unsupervised groups\n  - production code is READ, never modified\nOutputs: round2/_diag/competition_data_deep_dive.json  +  round2/_diag/data_deep_dive/*.png\n"""\nimport csv, hashlib, json, math, os, sys, warnings\nimport numpy as np\n\nsys.stdout.reconfigure(encoding="utf-8", errors="replace")\nwarnings.filterwarnings("ignore")\nnp.random.seed(0)\nR = "round2/"\nOUT = R + "_diag/data_deep_dive/"\nos.makedirs(OUT, exist_ok=True)\nA = {}                                              # the audit record\nDATES = ["Jun06", "Jun19", "Aug14", "Oct13"]\nTAGS = {"Jun06": "20250606072501_20250606072506", "Jun19": "20250619021410_20250619021415",\n        "Aug14": "20250814031124_20250814031129", "Oct13": "20251013022643_20251013022648"}\n\n\ndef fl(v):\n    try:\n        return float(v)\n    except (TypeError, ValueError):\n        return float("nan")\n\n\ndef stats(v, pref=""):\n    v = np.asarray(v, float)\n    v = v[np.isfinite(v)]\n    if v.size == 0:\n        return {}\n    q = np.percentile(v, [1, 5, 25, 50, 75, 95, 99])\n    mu, sd = v.mean(), v.std()\n    return {pref + "n": int(v.size), pref + "mean": round(float(mu), 4),\n            pref + "median": round(float(q[3]), 4), pref + "sd": round(float(sd), 4),\n            pref + "iqr": round(float(q[4] - q[2]), 4),\n            pref + "mad": round(float(np.median(np.abs(v - q[3]))), 4),\n            pref + "p01": round(float(q[0]), 4), pref + "p05": round(float(q[1]), 4),\n            pref + "p95": round(float(q[5]), 4), pref + "p99": round(float(q[6]), 4),\n            pref + "min": round(float(v.min()), 4), pref + "max": round(float(v.max()), 4),\n            pref + "skew": round(float(((v - mu) ** 3).mean() / (sd ** 3 + 1e-12)), 4),\n            pref + "kurtosis_excess": round(float(((v - mu) ** 4).mean() / (sd ** 4 + 1e-12) - 3), 4)}\n\n\n# ============================================================ 1. INVENTORY\nprint("=" * 70); print("1. INVENTORY"); print("=" * 70)\ninv = []\nfor root, dirs, files in os.walk(R + "data"):\n    for fn in files:\n        p = os.path.join(root, fn)\n        sz = os.path.getsize(p)\n        e = dict(path=p.replace("\\\\", "/"), bytes=sz, mb=round(sz / 1e6, 2),\n                 ext=os.path.splitext(fn)[1].lower())\n        if sz < 60e6:                                # hash only what is cheap\n            h = hashlib.sha256()\n            with open(p, "rb") as fh:\n                for b in iter(lambda: fh.read(1 << 20), b""):\n                    h.update(b)\n            e["sha256_16"] = h.hexdigest()[:16]\n        inv.append(e)\nA["inventory"] = sorted(inv, key=lambda x: -x["bytes"])\nprint("  %d organizer files, %.0f MB total"\n      % (len(inv), sum(x["bytes"] for x in inv) / 1e6))\nfor e in A["inventory"][:6]:\n    print("    %8.1f MB  %s" % (e["mb"], e["path"].split("/")[-1][:72]))\n\n# promised-vs-delivered\npromised = {"capella_slc_x4": True, "farm_boundaries_shp": True, "village_boundaries_shp": True,\n            "round1_crop_classification.csv": False}\nA["promised_vs_delivered"] = dict(\n    round1_crop_classification_csv="ABSENT - the dataset description lists it as file 4 "\n    "(\'Crop type per farm, carried forward from your Round 1 submission\') but NO such file "\n    "exists anywhere in the repository. Verified by recursive search.",\n    village_count="description says \'Digitized field-level polygons across 4 villages\' and "\n    "\'Boundaries for the 4 villages\'; DELIVERED is ONE village (Sokhda). The overview text "\n    "agrees with the delivery (966 farms, 1 village). The dataset description appears to be "\n    "a generic template not updated for this delivery.",\n    village_join_claim="description says the farm shapefile \'does not include a direct "\n    "village reference field. You will need to spatially join\'. FALSE for the delivered "\n    "file: it carries VILLAGE=\'Sokhda\' and ID_1=22 on every record. No spatial join needed.")\n\n# ============================================================ 2. VECTORS\nprint("\\n" + "=" * 70); print("2/8. VECTOR DEEP DIVE"); print("=" * 70)\nimport shapefile\nfrom shapely.geometry import shape as gshape\nfrom shapely.ops import transform as shp_transform\nfrom shapely.strtree import STRtree\nfrom pyproj import Transformer\n\nFS = R + "data/Farm_boundaries_shp/Farm_boundaries_shp/Sokhda_Farms"\nVS = R + "data/Village_Shp/Village_Shp/Sokhda_Village"\nsf = shapefile.Reader(FS)\nrecs, shps = sf.records(), sf.shapes()\nto_m = Transformer.from_crs("EPSG:4326", "EPSG:32643", always_xy=True).transform\nvil = shapefile.Reader(VS)\nvgeo = shp_transform(to_m, gshape(vil.shapes()[0].__geo_interface__).buffer(0))\n\nfid, geo, area, per, npart, nhole, valid_raw = [], [], [], [], [], [], []\nfor rec, s in zip(recs, shps):\n    g = gshape(s.__geo_interface__)\n    valid_raw.append(bool(g.is_valid))\n    if not g.is_valid:\n        g = g.buffer(0)\n    gm = shp_transform(to_m, g)\n    fid.append(int(rec[0])); geo.append(gm)\n    area.append(gm.area); per.append(gm.length)\n    npart.append(len(gm.geoms) if gm.geom_type == "MultiPolygon" else 1)\n    nhole.append(sum(len(p.interiors) for p in\n                     (gm.geoms if gm.geom_type == "MultiPolygon" else [gm])))\nfid = np.array(fid); area = np.array(area); per = np.array(per)\nnpart = np.array(npart); nhole = np.array(nhole)\ncomp = 4 * math.pi * area / np.clip(per ** 2, 1e-9, None)\n\ntree = STRtree(geo)\nov = 0; ov_pairs = []\nfor i, g in enumerate(geo):\n    for j in tree.query(g):\n        if j <= i:\n            continue\n        inter = g.intersection(geo[j]).area\n        if inter > 1.0:\n            ov += 1\n            if len(ov_pairs) < 15:\n                ov_pairs.append([int(fid[i]), int(fid[j]), round(inter, 1)])\nnn = []\nfor i, g in enumerate(geo):\n    d = sorted(g.distance(geo[j]) for j in tree.query(g.buffer(120)) if j != i)\n    nn.append(d[0] if d else np.nan)\nnn = np.array(nn)\n\nA["farm_vectors"] = dict(\n    n_records=len(fid), n_unique_fid=int(len(set(fid.tolist()))),\n    expected_966=bool(len(fid) == 966),\n    fields=[f[0] for f in sf.fields if f[0] != "DeletionFlag"],\n    anomalous_empty_field=("the .dbf carries an UNNAMED numeric field (name=\'\', N,24,15) "\n                           "whose value is None on every record - a malformed/orphan column"),\n    id_field_constant=bool(len(set(r[1] for r in recs)) == 1),\n    village_field_values=sorted({str(r[4]) for r in recs}),\n    id1_values=sorted({int(r[3]) for r in recs}),\n    invalid_geometries_raw=int(len(valid_raw) - sum(valid_raw)),\n    multipart=int((npart > 1).sum()), with_holes=int((nhole > 0).sum()),\n    overlapping_pairs=int(ov), overlap_examples=ov_pairs,\n    total_farm_area_ha=round(float(area.sum() / 1e4), 2),\n    village_area_ha=round(float(vgeo.area / 1e4), 2),\n    farm_area_as_pct_of_village=round(float(100 * area.sum() / vgeo.area), 1),\n    farms_outside_village=int(sum(1 for g in geo if not g.centroid.within(vgeo))),\n    **stats(area / 1e4, "area_ha_"), **{"perimeter_m_" + k.split("_", 1)[1]: v\n                                        for k, v in stats(per, "perimeter_m_").items()},\n    compactness_median=round(float(np.median(comp)), 4),\n    nn_distance_m_median=round(float(np.nanmedian(nn)), 2),\n    nn_distance_m_p95=round(float(np.nanpercentile(nn, 95)), 2),\n    n_touching_neighbour=int(np.nansum(nn < 1.0)))\nprint("  farms=%d unique=%d | area ha median %.3f | multipart %d | overlaps %d"\n      % (len(fid), len(set(fid.tolist())), np.median(area / 1e4), (npart > 1).sum(), ov))\nprint("  farm area covers %.1f%% of the village polygon" % A["farm_vectors"]["farm_area_as_pct_of_village"])\nA["village_vector"] = dict(n=1, fields=[f[0] for f in vil.fields if f[0] != "DeletionFlag"],\n                           ID=int(vil.records()[0][0]), VILLAGE=str(vil.records()[0][1]),\n                           area_ha=round(float(vgeo.area / 1e4), 2))\nprint("  village ID=%d name=%s area=%.1f ha"\n      % (A["village_vector"]["ID"], A["village_vector"]["VILLAGE"], A["village_vector"]["area_ha"]))\n\n# ============================================================ 3. CAPELLA METADATA\nprint("\\n" + "=" * 70); print("3. CAPELLA ACQUISITION AUDIT (metadata + raster headers)"); print("=" * 70)\nimport rasterio\nACQ = {}\nfor d, tag in TAGS.items():\n    folder = R + "data/CAPELLA_C14_SM_SLC_HH_%s" % tag\n    slc = "%s/CAPELLA_C14_SM_SLC_HH_%s.tif" % (folder, tag)\n    ext = "%s/CAPELLA_C14_SM_SLC_HH_%s_extended.json" % (folder, tag)\n    e = dict(folder_files=sorted(os.listdir(folder)), slc_bytes=os.path.getsize(slc))\n    try:\n        J = json.load(open(ext))\n        col = J.get("collect", {})\n        img = col.get("image", {})\n        rad = col.get("radar", {})\n        st = col.get("state", {})\n        cp = img.get("center_pixel", {})\n        e.update(\n            product_type=J.get("product_type"), product_version=J.get("product_version"),\n            software_version=J.get("software_version"), processing_time=J.get("processing_time"),\n            start=col.get("start_timestamp"), stop=col.get("stop_timestamp"),\n            local_datetime=col.get("local_datetime"), mode=col.get("mode"),\n            platform=col.get("platform"), collect_id=col.get("collect_id"),\n            data_type=img.get("data_type"), rows=img.get("rows"), cols=img.get("columns"),\n            scene_length_m=round(float(img.get("length", float("nan"))), 1),\n            scene_width_m=round(float(img.get("width", float("nan"))), 1),\n            pixel_spacing_row_m=img.get("pixel_spacing_row"),\n            pixel_spacing_col_m=img.get("pixel_spacing_column"),\n            range_res_m=img.get("range_resolution"),\n            ground_range_res_m=img.get("ground_range_resolution"),\n            azimuth_res_m=img.get("azimuth_resolution"),\n            azimuth_looks=img.get("azimuth_looks"), range_looks=img.get("range_looks"),\n            enl=img.get("enl"), radiometry=img.get("radiometry"),\n            calibration=img.get("calibration"), scale_factor=img.get("scale_factor"),\n            nesz_peak_db=img.get("nesz_peak"), algorithm=img.get("algorithm"),\n            az_beam_corrected=img.get("azimuth_beam_pattern_corrected"),\n            el_beam_corrected=img.get("elevation_beam_pattern_corrected"),\n            center_incidence_deg=cp.get("incidence_angle"),\n            center_lat=(cp.get("target_position") or [None, None])[1]\n            if isinstance(cp.get("target_position"), list) else None,\n            center_pixel_keys=sorted(cp.keys()) if isinstance(cp, dict) else None,\n            tx_pol=rad.get("transmit_polarization"), rx_pol=rad.get("receive_polarization"),\n            center_frequency_hz=rad.get("center_frequency"), pointing=rad.get("pointing"),\n            prf=(rad.get("prf") or [None])[0] if isinstance(rad.get("prf"), list) else None,\n            orbit_direction=st.get("direction"), state_source=st.get("source"),\n            n_state_vectors=len(st.get("state_vectors", [])),\n            terrain_models=img.get("terrain_models"))\n        e["metadata_keys_top"] = sorted(J.keys())\n    except Exception as ex:\n        e["metadata_error"] = str(ex)[:120]\n    try:\n        with rasterio.open(slc) as s:\n            e.update(raster_shape=[s.height, s.width], raster_dtype=str(s.dtypes[0]),\n                     raster_count=s.count, raster_crs=str(s.crs), raster_nodata=s.nodata,\n                     raster_bounds=[round(float(x), 6) for x in s.bounds] if s.crs else None,\n                     has_rpcs=bool(s.rpcs), tags=dict(list(s.tags().items())[:8]))\n    except Exception as ex:\n        e["raster_error"] = str(ex)[:120]\n    ACQ[d] = e\n    print("  %-6s %s %s %5sx%-5s inc=%s look=%s orbit=%s scale=%.3e nesz=%.2f"\n          % (d, (e.get("start") or "?")[:19], e.get("mode"), e.get("rows"), e.get("cols"),\n             ("%.2f" % e["center_incidence_deg"]) if e.get("center_incidence_deg") else "?",\n             e.get("pointing"), e.get("orbit_direction"),\n             e.get("scale_factor") or float("nan"), e.get("nesz_peak_db") or float("nan")))\nA["acquisitions"] = ACQ\n\n# provenance: does each folder contain ONLY its own acquisition?\nprov = {}\nfor d, tag in TAGS.items():\n    folder = R + "data/CAPELLA_C14_SM_SLC_HH_%s" % tag\n    foreign = [f for f in os.listdir(folder)\n               if any(t in f for t in TAGS.values() if t != tag)]\n    prov[d] = dict(foreign_files=foreign, clean=len(foreign) == 0)\nA["provenance_check"] = dict(per_folder=prov,\n                             all_clean=all(v["clean"] for v in prov.values()),\n                             history=("a stray Jun06 SLC was previously found inside the Jun19 "\n                                      "folder and silently selected by a \'largest file\' "\n                                      "heuristic; it was quarantined to round2/_quarantine and "\n                                      "the heuristic removed repository-wide. This check "\n                                      "re-verifies the CURRENT state independently."))\nprint("  provenance: all folders clean = %s" % A["provenance_check"]["all_clean"])\nqd = R + "_quarantine"\nA["provenance_check"]["quarantine_contents"] = sorted(os.listdir(qd)) if os.path.isdir(qd) else []\n\n# frozen pipeline uses exact-filename matching -> verify the names it resolves\nA["frozen_pipeline_resolution"] = {\n    d: dict(expected="CAPELLA_C14_SM_SLC_HH_%s.tif" % tag,\n            exists=os.path.exists(R + "data/CAPELLA_C14_SM_SLC_HH_%s/CAPELLA_C14_SM_SLC_HH_%s.tif" % (tag, tag)))\n    for d, tag in TAGS.items()}\n\njson.dump(A, open(R + "_diag/competition_data_deep_dive.json", "w"), indent=1, default=str)\nprint("\\n[stage 1 written] round2/_diag/competition_data_deep_dive.json")\n'
Path(SRC/"audit_competition_data.py").write_text(_SRC, encoding="utf-8")
print("wrote src/audit_competition_data.py  ({} lines)".format(_SRC.count(chr(10))))

In [ ]:
env = dict(os.environ, CAPELLA_BASE=str(STAGE), PYTHONWARNINGS="ignore")

def run_stage(mod, produces, label, timeout=5400):
    """Execute a materialised canonical module and assert its declared outputs appear."""
    t0 = time.time()
    p = subprocess.run([sys.executable, str(SRC / mod)], cwd=str(WORK), env=env,
                       capture_output=True, text=True, timeout=timeout)
    dt = time.time() - t0
    if p.returncode != 0:
        print(p.stdout[-2000:]); print(p.stderr[-3000:])
        raise RuntimeError(f"{label} ({mod}) failed rc={p.returncode}")
    for out in produces:
        assert (ROOT / out).is_file(), f"{label}: {out} not produced"
    print(f"  {label:<34} {dt:7.1f}s  -> {', '.join(produces)}")
    return dt, p.stdout

T = {}
T["S03"], _ = run_stage("audit_competition_data.py",
                        ["_diag/competition_data_deep_dive.json"], "S03 competition audit")

In [ ]:
# S05 -- canonical Capella geometry / radiometry v3.2.1 (science unmodified).
# PATCHED FOR NOTEBOOK RUNTIME: scene search pinned to the provenance-locked staging directory, so the stray Jun06 duplicate inside the Jun19 product folder can never be selected
_SRC = '"""\nAISEHACK 2.0 Round 2 - GEOMETRY / RADIOMETRY / FIELD-FEATURE REBUILD  v3.2.1\nRepairs B5, B6, I9-I12 (V3.2 closure); preserves the verified B4 (sub-pixel projection) and I5-I8 from CODEX_AUDIT_GEOMETRY_REBUILD_V31.txt\n\nCOORDINATE CONVENTION (established by round-trip + controlled rasterization tests,\nnot assumed):  rowcol(..., op=IDENTITY) returns CORNER-ORIGIN fractional row/col;\npixel CENTRE of pixel (i,j) lies at (i+0.5, j+0.5).  Masking therefore tests\n(col+0.5, row+0.5).  This reproduces rasterio.features.geometry_mask exactly\n(Jaccard 1.0000 on a controlled unit-grid test).\n\nCapella X-band HH SLC only. Supplied Capella GEO previews used ONLY for SAR\nalignment. No optical, no Sentinel-1, no weather, no leaderboard, no health/yield\nreference is read anywhere in this script.\n\nOutputs\n  round2/farm_features_geometry_v321.csv\n  round2/_geom321/geometry_rebuild_v321_log.json\n  round2/_geom321/run_manifest_<stamp>.json\n  round2/_geom321/missing_farm_dates_v321.json\n  round2/_geom321/topology_report.json\n"""\nimport os, sys, json, glob, csv, math, time, hashlib, platform, argparse\nfrom datetime import datetime, timezone\nimport numpy as np\nimport rasterio\nfrom rasterio.windows import Window\nfrom rasterio.transform import RPCTransformer\nimport shapefile\nfrom matplotlib.path import Path as MplPath\nfrom shapely.geometry import shape as gshape, Polygon, MultiPolygon\nfrom shapely.ops import transform as shp_transform\nfrom pyproj import Transformer\nfrom scipy.stats import spearmanr\n\nIDENTITY = (lambda v: v)          # keep fractional row/col (B4 repair)\nT0 = time.time()\nSTART_ISO = datetime.now(timezone.utc).isoformat()\nOUT = "round2/_geom321"; os.makedirs(OUT, exist_ok=True)\nSEED = 20260805\nCSV_PATH = "round2/farm_features_geometry_v321.csv"\n\nap = argparse.ArgumentParser()\nap.add_argument("--tag", default="run")\nARGS = ap.parse_args()\n\nSCENES = [("Jun06", "20250606072501_20250606072506"),\n          ("Jun19", "20250619021410_20250619021415"),\n          ("Aug14", "20250814031124_20250814031129"),\n          ("Oct13", "20251013022643_20251013022648")]\n\n# geometry is treated as unusable below this true component area\nMIN_COMPONENT_M2 = 25.0        # ~5 m x 5 m; smaller than any real field here\nMIN_FARM_M2      = 50.0\nGATE_PX          = 30          # minimum valid (non-zero-fill) pixels to emit a feature\nHGRID = np.round(np.arange(-30.0, -4.9, 0.5), 2)\nBLOCK_SIZES = [4, 8, 16]\nPRIMARY_BLOCK = 8\nNBOOT = 200\n\nROOTS = [os.environ.get("CAPELLA_BASE") or "round2/data", "round2/data", "."]\nBASE = None\nfor r in ROOTS:\n    if os.path.isdir(r) and glob.glob(os.path.join(r, "**", "CAPELLA_*SLC_HH_*.tif"), recursive=True):\n        BASE = r; break\nassert BASE, "Capella SLC not found"\ndef find(pat):\n    h = sorted(glob.glob(os.path.join(BASE, "**", pat), recursive=True))\n    return h[0] if h else None\ndef slc_path(t):  return find(f"CAPELLA_*SLC_HH_{t}.tif")\ndef meta_path(t): return find(f"CAPELLA_*SLC_HH_{t}_extended.json")\ndef geo_path(t):  return find(f"CAPELLA_*GEO_HH_{t}*preview.tif")\n\n# ---------- D1: FULL hashes and sizes for every consumed file ----------\ndef sha256_full(p):\n    h = hashlib.sha256()\n    with open(p, "rb") as f:\n        for b in iter(lambda: f.read(1 << 22), b""): h.update(b)\n    return h.hexdigest()\nSHP_BASE = find("*Farm*.shp")\nINPUTS = {}\nfor name, tag in SCENES:\n    for kind, p in (("slc", slc_path(tag)), ("meta", meta_path(tag)), ("geo_preview", geo_path(tag))):\n        INPUTS[f"{name}_{kind}"] = dict(path=os.path.relpath(p), bytes=os.path.getsize(p), sha256=sha256_full(p))\nfor ext in (".shp", ".shx", ".dbf", ".prj", ".cpg"):\n    p = SHP_BASE[:-4] + ext\n    if os.path.exists(p):\n        INPUTS["farms" + ext] = dict(path=os.path.relpath(p), bytes=os.path.getsize(p), sha256=sha256_full(p))\nprint("[hash] %d input files hashed in full" % len(INPUTS), flush=True)\n\ndef parse_t(s):\n    s = s.replace("Z", "")\n    if "." in s:\n        head, frac = s.split("."); frac = (frac + "000000000")[:9]\n        return datetime.fromisoformat(head).replace(tzinfo=timezone.utc).timestamp() + int(frac) / 1e9\n    return datetime.fromisoformat(s).replace(tzinfo=timezone.utc).timestamp()\n\n# ================= B1: TRUE TOPOLOGY ==========================================\nsf = shapefile.Reader(SHP_BASE)\n_recs, _shapes = sf.records(), sf.shapes()\norder = np.argsort([int(r[0]) for r in _recs])\nFIDS = [int(_recs[i][0]) for i in order]\nNF = len(FIDS)\nto_m  = Transformer.from_crs("EPSG:4326", "EPSG:32643", always_xy=True).transform\nto_ll = Transformer.from_crs("EPSG:32643", "EPSG:4326", always_xy=True).transform\n\nGEOM = {}          # fid -> dict(components=[(ext_lonlat, [hole_lonlat...])], area_m2, cx, cy, flags)\ntopology = []\nfor k, i in enumerate(order):\n    fid = FIDS[k]\n    g = gshape(_shapes[i].__geo_interface__)\n    if not g.is_valid: g = g.buffer(0)\n    gm = shp_transform(to_m, g)                       # metric CRS for area/centroid\n    parts = list(gm.geoms) if gm.geom_type == "MultiPolygon" else ([gm] if gm.geom_type == "Polygon" else [])\n    kept = [p for p in parts if p.area >= MIN_COMPONENT_M2]\n    dropped = [p.area for p in parts if p.area < MIN_COMPONENT_M2]\n    area_m2 = float(sum(p.area for p in kept))\n    unusable = (area_m2 < MIN_FARM_M2) or (len(kept) == 0)\n    if kept:\n        um = MultiPolygon(kept) if len(kept) > 1 else kept[0]\n        rp = um.representative_point()                # sample point, guaranteed INSIDE (not a centroid)\n        cx, cy = to_ll(rp.x, rp.y)\n        ct = um.centroid                              # I6: true area-weighted centroid\n        tx, ty = to_ll(ct.x, ct.y)\n    else:\n        rp = gm.representative_point() if not gm.is_empty else None\n        cx, cy = (to_ll(rp.x, rp.y) if rp is not None else (np.nan, np.nan))\n        tx, ty = cx, cy\n    comps = []\n    for p in kept:\n        pl = shp_transform(to_ll, p)\n        comps.append((np.asarray(pl.exterior.coords, float),\n                      [np.asarray(r.coords, float) for r in pl.interiors]))\n    GEOM[fid] = dict(components=comps, area_m2=area_m2, cx=cx, cy=cy, tx=tx, ty=ty,\n                     n_components=len(kept), n_holes=sum(len(h) for _, h in comps),\n                     unusable=int(unusable))\n    if len(parts) > 1 or dropped or unusable:\n        topology.append(dict(farm_id=fid, geom_type=gm.geom_type, n_parts_raw=len(parts),\n                             n_components_kept=len(kept), dropped_part_areas_m2=[round(a, 4) for a in dropped],\n                             area_m2=round(area_m2, 4), area_ha=round(area_m2 / 1e4, 6),\n                             unusable=int(unusable)))\nMULTI_IDS   = sorted({t["farm_id"] for t in topology if t["n_parts_raw"] > 1})          # I5: TRUE multipart\nFLAGGED_IDS = sorted({t["farm_id"] for t in topology})                                   # any anomaly\nDROPPED_IDS = sorted({t["farm_id"] for t in topology if t["dropped_part_areas_m2"]})      # lost a component\nUNUSABLE_IDS_L = sorted({t["farm_id"] for t in topology if t["unusable"]})\nN_UNUSABLE = sum(1 for f in FIDS if GEOM[f]["unusable"])\nprint("[topology] multipart/flagged farms: %d | unusable geometry: %d" % (len(MULTI_IDS), N_UNUSABLE), flush=True)\nSET_SEMANTICS = dict(\n    note=("These are DISTINCT SEMANTIC CATEGORIES that OVERLAP by design. "\n          "They are NOT disjoint sets."),\n    definitions=dict(\n        multipart_ids="record parsed to >1 polygon component (true MultiPolygon)",\n        flagged_ids="ANY geometric anomaly: multipart OR dropped component OR unusable",\n        dropped_component_ids="lost >=1 component below the 25 m^2 threshold",\n        unusable_ids="surviving area < 50 m^2, so no pixels are extracted"),\n    expected_relations=[\n        "flagged_ids is the UNION of multipart_ids, dropped_component_ids and unusable_ids",\n        "unusable_ids is a SUBSET of flagged_ids",\n        "multipart_ids and unusable_ids INTERSECT (a multipart record can be all-degenerate)",\n        "dropped_component_ids and multipart_ids INTERSECT but neither contains the other"],\n    measured=dict(\n        multipart_and_unusable=sorted(set(MULTI_IDS) & set(UNUSABLE_IDS_L)),\n        multipart_not_unusable=sorted(set(MULTI_IDS) - set(UNUSABLE_IDS_L)),\n        unusable_not_multipart=sorted(set(UNUSABLE_IDS_L) - set(MULTI_IDS)),\n        dropped_and_multipart=sorted(set(DROPPED_IDS) & set(MULTI_IDS)),\n        flagged_equals_union=bool(set(FLAGGED_IDS) ==\n            (set(MULTI_IDS) | set(DROPPED_IDS) | set(UNUSABLE_IDS_L))),\n        unusable_subset_of_flagged=bool(set(UNUSABLE_IDS_L) <= set(FLAGGED_IDS)),\n        pairwise_intersection_sizes=dict(\n            multipart_x_dropped=len(set(MULTI_IDS) & set(DROPPED_IDS)),\n            multipart_x_unusable=len(set(MULTI_IDS) & set(UNUSABLE_IDS_L)),\n            dropped_x_unusable=len(set(DROPPED_IDS) & set(UNUSABLE_IDS_L)))))\njson.dump(dict(min_component_m2=MIN_COMPONENT_M2, min_farm_m2=MIN_FARM_M2,\n               flagged=topology,\n               multipart_ids=MULTI_IDS,            # exactly the true MultiPolygon records\n               flagged_ids=FLAGGED_IDS,            # multipart OR degenerate OR dropped part\n               dropped_component_ids=DROPPED_IDS,  # lost >=1 component below threshold\n               unusable_ids=UNUSABLE_IDS_L,        # no usable area at all\n               counts=dict(multipart=len(MULTI_IDS), flagged=len(FLAGGED_IDS),\n                           dropped=len(DROPPED_IDS), unusable=len(UNUSABLE_IDS_L)),\n               set_semantics=SET_SEMANTICS),\n          open(f"{OUT}/topology_report.json", "w"), indent=1)\n\n# B1 regression tests\nREG = {}\nfor fid in (15, 58):\n    G = GEOM[fid]\n    REG[fid] = dict(n_components=G["n_components"], area_ha=round(G["area_m2"] / 1e4, 6), unusable=G["unusable"])\nassert GEOM[15]["unusable"] == 1, "REGRESSION FAIL: farm 15 must be flagged unusable"\nassert GEOM[58]["n_components"] == 2 and GEOM[58]["area_m2"] > 500, "REGRESSION FAIL: farm 58 must keep 2 real components"\nassert GEOM[58]["n_components"] == 2, "REGRESSION: farm 58 components bridged/lost"\nprint("[regression] farm15 unusable=%d area=%.6f ha | farm58 comps=%d area=%.6f ha"\n      % (REG[15]["unusable"], REG[15]["area_ha"], REG[58]["n_components"], REG[58]["area_ha"]), flush=True)\n\n# ---- REGRESSION TESTS: fail loudly on any convention/topology regression ----\ndef _regression_tests():\n    T = rasterio.transform.Affine(1.0, 0.0, 0.0, 0.0, -1.0, 10.0)\n    from rasterio.features import geometry_mask as _gm\n    from shapely.geometry import Polygon as _P\n    poly = _P([(2.3, 3.2), (6.7, 3.2), (6.7, 7.8), (2.3, 7.8)])\n    ref = _gm([poly], out_shape=(10, 10), transform=T, invert=True)\n    xs, ys = np.array(poly.exterior.coords).T\n    r, c = rasterio.transform.rowcol(T, xs, ys, op=IDENTITY)\n    # R1: identity op must NOT return integers\n    assert not np.allclose(np.asarray(r, float), np.round(np.asarray(r, float))),         "REGRESSION: projected vertices are integer-only (B4 has returned)"\n    gr_, gc_ = np.mgrid[0:10, 0:10]\n    m = MplPath(np.column_stack([np.asarray(c, float), np.asarray(r, float)])).contains_points(\n        np.column_stack([gc_.ravel() + 0.5, gr_.ravel() + 0.5])).reshape(10, 10)\n    # R2: our convention must reproduce the authoritative rasterizer exactly\n    assert (m == ref).all(), "REGRESSION: mask convention disagrees with rasterio.features.geometry_mask"\n    # R3: SLC (RPC) and GEO (affine) must share the +0.5 centre convention\n    off = []\n    for nm, tg in SCENES[:1]:\n        with rasterio.open(slc_path(tg)) as _s:\n            with RPCTransformer(_s.rpcs, RPC_HEIGHT=-18.5) as _t:\n                lon, lat = _t.xy(20000.0, 2000.0)\n                rr, cc = _t.rowcol(lon, lat, op=IDENTITY)\n                off.append(abs(float(np.atleast_1d(rr)[0]) - 20000.5))\n        with rasterio.open(geo_path(tg)) as _g:\n            X, Y = rasterio.transform.xy(_g.transform, 100, 200, offset="center")\n            rr, cc = rasterio.transform.rowcol(_g.transform, X, Y, op=IDENTITY)\n            off.append(abs(float(np.atleast_1d(rr)[0]) - 100.5))\n    assert max(off) < 1e-2, f"REGRESSION: SLC/GEO centre conventions disagree ({off})"\n    return dict(identity_op_fractional=True, mask_matches_rasterio=True,\n                centre_offset_max_err=float(max(off)))\nREG_CONV = _regression_tests()\nprint("[regression] convention tests PASS  max centre offset err %.2e" % REG_CONV["centre_offset_max_err"], flush=True)\n\nAREA_HA = np.array([GEOM[f]["area_m2"] / 1e4 for f in FIDS])\nCENT  = np.array([[GEOM[f]["cx"], GEOM[f]["cy"]] for f in FIDS])   # representative (sample) point\nCENTC = np.array([[GEOM[f]["tx"], GEOM[f]["ty"]] for f in FIDS])   # true area-weighted centroid\n\n# ---------------- metadata ----------------\nMETA = {}\nfor name, tag in SCENES:\n    j = json.load(open(meta_path(tag))); col = j["collect"]; img = col["image"]; ig = img["image_geometry"]\n    sv = col["state"]["state_vectors"]\n    META[name] = dict(tag=tag, scale=img["scale_factor"], radiometry=img["radiometry"],\n                      inc_c=img["center_pixel"]["incidence_angle"],\n                      t0=parse_t(ig["first_line_time"]), dt=ig["delta_line_time"],\n                      r0=ig["range_to_first_sample"], dr=ig["delta_range_sample"],\n                      tc=parse_t(img["center_pixel"]["center_time"]),\n                      tgt_c=np.asarray(img["center_pixel"]["target_position"], float),\n                      nesz=img["nesz_polynomial"], nesz_peak=img["nesz_peak"],\n                      px_row=ig.get("pixel_spacing_row", img["pixel_spacing_row"]),\n                      px_col=ig.get("pixel_spacing_column", img["pixel_spacing_column"]),\n                      sv_t=np.array([parse_t(s["time"]) for s in sv]),\n                      sv_p=np.array([s["position"] for s in sv], float))\n\nWGS_A, WGS_E2 = 6378137.0, 6.69437999014e-3\ndef llh2ecef(lon, lat, h):\n    la, lo = np.radians(lat), np.radians(lon)\n    N = WGS_A / np.sqrt(1 - WGS_E2 * np.sin(la) ** 2)\n    return np.stack([(N + h) * np.cos(la) * np.cos(lo), (N + h) * np.cos(la) * np.sin(lo),\n                     (N * (1 - WGS_E2) + h) * np.sin(la)], axis=-1)\ndef ellip_normal(lon, lat):\n    la, lo = np.radians(lat), np.radians(lon)\n    return np.stack([np.cos(la) * np.cos(lo), np.cos(la) * np.sin(lo), np.sin(la)], axis=-1)\ndef sat_at(name, t):\n    m = META[name]; t = np.atleast_1d(np.asarray(t, float))\n    return np.stack([np.interp(t, m["sv_t"], m["sv_p"][:, k]) for k in range(3)], axis=-1)\ndef incidence(name, lon, lat, h, row):\n    m = META[name]\n    sat = sat_at(name, m["t0"] + np.asarray(row, float) * m["dt"])\n    tgt = np.atleast_2d(llh2ecef(np.asarray(lon, float), np.asarray(lat, float), float(h)))\n    look = sat - tgt; look /= np.linalg.norm(look, axis=-1, keepdims=True)\n    nrm = np.atleast_2d(ellip_normal(np.asarray(lon, float), np.asarray(lat, float)))\n    out = np.degrees(np.arccos(np.clip(np.sum(look * nrm, axis=-1), -1, 1)))\n    return out if out.size > 1 else float(out.ravel()[0])\ndef nesz_sigma0_db(name, col_idx):\n    """Official Capella: NESZ (SIGMA-ZERO) in dB vs ABSOLUTE slant range in metres."""\n    m = META[name]; c = m["nesz"]["coefficients"]\n    R = m["r0"] + np.asarray(col_idx, float) * m["dr"]\n    return sum(c[k] * R ** k for k in range(len(c)))\n\n# ---------------- topology-safe projection + masking ----------------\ndef project_geom(name, h, geoms):\n    """Project every ring of every component into SLC row/col at height h."""\n    m = META[name]\n    with rasterio.open(slc_path(m["tag"])) as s:\n        with RPCTransformer(s.rpcs, RPC_HEIGHT=float(h)) as tr:\n            out = []\n            for fid in FIDS:\n                comps = []\n                for ext, holes in geoms[fid]["components"]:\n                    rr, cc = tr.rowcol(ext[:, 0].tolist(), ext[:, 1].tolist(), op=IDENTITY)\n                    e = np.column_stack([np.asarray(cc, float), np.asarray(rr, float)])   # (x=col, y=row)\n                    hs = []\n                    for hh in holes:\n                        rr2, cc2 = tr.rowcol(hh[:, 0].tolist(), hh[:, 1].tolist(), op=IDENTITY)\n                        hs.append(np.column_stack([np.asarray(cc2, float), np.asarray(rr2, float)]))\n                    comps.append((e, hs))\n                out.append(comps)\n    return out\n\ndef farm_mask_sets(arr, r0, c0, comps, centre_offset=0.5):\n    """B5: return (geometric_ids, valid_ids, values) with NO 30-pixel gate applied.\n    geometric_ids = every pixel whose CENTRE falls inside the polygon (incl. zero-fill)\n    valid_ids     = subset of those with finite (non-zero-fill) data\n    centre_offset: 0.5 = V3.2/V3.2.1 convention; 0.0 reproduces the V3.1 convention."""\n    if not comps: return set(), set(), None\n    xs = np.concatenate([e[:, 0] for e, _ in comps]); ys = np.concatenate([e[:, 1] for e, _ in comps])\n    cmin = max(0, int(np.floor(xs.min())) - c0); cmax = min(arr.shape[1], int(np.ceil(xs.max())) - c0 + 1)\n    rmin = max(0, int(np.floor(ys.min())) - r0); rmax = min(arr.shape[0], int(np.ceil(ys.max())) - r0 + 1)\n    if rmax <= rmin or cmax <= cmin: return set(), set(), None\n    gr, gc = np.mgrid[rmin:rmax, cmin:cmax]\n    pts = np.column_stack([gc.ravel() + c0 + centre_offset, gr.ravel() + r0 + centre_offset])\n    inside = np.zeros(pts.shape[0], bool)\n    for e, hs in comps:\n        m = MplPath(e).contains_points(pts)\n        for hh in hs:\n            if len(hh) >= 4: m &= ~MplPath(hh).contains_points(pts)\n        inside |= m\n    if not inside.any(): return set(), set(), None\n    rr = gr.ravel()[inside] + r0; cc = gc.ravel()[inside] + c0\n    v = arr[rmin:rmax, cmin:cmax].ravel()[inside]\n    geo_ids = set(zip(rr.tolist(), cc.tolist()))\n    ok = np.isfinite(v)\n    val_ids = set(zip(rr[ok].tolist(), cc[ok].tolist()))\n    return geo_ids, val_ids, (v[ok], rr[ok], cc[ok])\n\n\ndef farm_mask_pixels(arr, r0, c0, comps):\n    """Union of components, holes excluded. No artificial bridges between parts."""\n    if not comps: return None\n    xs = np.concatenate([e[:, 0] for e, _ in comps]); ys = np.concatenate([e[:, 1] for e, _ in comps])\n    cmin = max(0, int(np.floor(xs.min())) - c0); cmax = min(arr.shape[1], int(np.ceil(xs.max())) - c0 + 1)\n    rmin = max(0, int(np.floor(ys.min())) - r0); rmax = min(arr.shape[0], int(np.ceil(ys.max())) - r0 + 1)\n    if rmax <= rmin or cmax <= cmin: return None\n    gr, gc = np.mgrid[rmin:rmax, cmin:cmax]\n    pts = np.column_stack([gc.ravel() + c0 + 0.5, gr.ravel() + r0 + 0.5])   # pixel CENTRES\n    inside = np.zeros(pts.shape[0], bool)\n    for e, hs in comps:\n        m = MplPath(e).contains_points(pts)\n        for hh in hs:\n            if len(hh) >= 4: m &= ~MplPath(hh).contains_points(pts)\n        inside |= m\n    if not inside.any(): return None\n    v = arr[rmin:rmax, cmin:cmax].ravel()[inside]\n    rr = gr.ravel()[inside] + r0; cc = gc.ravel()[inside] + c0\n    ok = np.isfinite(v)\n    if ok.sum() < 30: return None\n    return v[ok], rr[ok], cc[ok]\n\ndef load_window(name, pad=90):\n    m = META[name]\n    pa = project_geom(name, -40.0, GEOM); pb = project_geom(name, 10.0, GEOM)\n    xs, ys = [], []\n    for comps in pa + pb:\n        for e, _ in comps: xs.append(e[:, 0]); ys.append(e[:, 1])\n    xs = np.concatenate(xs); ys = np.concatenate(ys)\n    with rasterio.open(slc_path(m["tag"])) as s:\n        H, W = s.height, s.width\n        r0 = max(0, int(ys.min()) - pad); r1 = min(H, int(ys.max()) + pad)\n        c0 = max(0, int(xs.min()) - pad); c1 = min(W, int(xs.max()) + pad)\n        blk = s.read(1, window=Window(c0, r0, c1 - c0, r1 - r0))\n    beta0 = (np.abs(blk).astype(np.float32) * m["scale"]) ** 2; del blk\n    beta0[beta0 <= 0] = np.nan\n    return beta0, r0, c0\n\ndef _trim(v, f):\n    s = np.sort(v); k = int(f * s.size); return float(np.mean(s[k:max(k + 1, s.size - k)]))\nEST = {"median": lambda v: float(np.median(v)), "trim10": lambda v: _trim(v, .10), "trim20": lambda v: _trim(v, .20),\n       "mean": lambda v: float(np.mean(v))}\ndef dB(x): return 10 * np.log10(np.maximum(np.asarray(x, float), 1e-15))\nLOG = dict(inputs=INPUTS, topology=dict(multipart_ids=MULTI_IDS, flagged_ids=FLAGGED_IDS,\n                                        dropped_component_ids=DROPPED_IDS,\n                                        unusable_ids=UNUSABLE_IDS_L,\n                                        n_flagged=len(topology), n_unusable=N_UNUSABLE,\n                                        set_semantics=SET_SEMANTICS, regression=REG))\n\n# ============ I1: HEIGHT CURVES ON A FIXED COHORT ============================\nprint("[1] height curves (fixed cohort, scipy spearmanr)", flush=True)\ndef geo_farm_means(tag):\n    p = geo_path(tag); out = np.full(NF, np.nan)\n    with rasterio.open(p) as g:\n        tr = Transformer.from_crs("EPSG:4326", g.crs, always_xy=True)\n        for i, fid in enumerate(FIDS):\n            comps = GEOM[fid]["components"]\n            if not comps: continue\n            allx, ally = [], []\n            proj = []\n            for ext, holes in comps:\n                X, Y = tr.transform(ext[:, 0], ext[:, 1])\n                rr, cc = rasterio.transform.rowcol(g.transform, X, Y, op=IDENTITY)\n                e = np.column_stack([np.asarray(cc, float), np.asarray(rr, float)])\n                hs = []\n                for hh in holes:\n                    X2, Y2 = tr.transform(hh[:, 0], hh[:, 1])\n                    r2, c2 = rasterio.transform.rowcol(g.transform, X2, Y2, op=IDENTITY)\n                    hs.append(np.column_stack([np.asarray(c2, float), np.asarray(r2, float)]))\n                proj.append((e, hs)); allx.append(e[:, 0]); ally.append(e[:, 1])\n            xs = np.concatenate(allx); ys = np.concatenate(ally)\n            a0 = max(0, int(ys.min())); a1 = min(g.height, int(ys.max()) + 1)\n            b0 = max(0, int(xs.min())); b1 = min(g.width, int(xs.max()) + 1)\n            if a1 <= a0 or b1 <= b0: continue\n            arr = g.read(1, window=Window(b0, a0, b1 - b0, a1 - a0)).astype(np.float32)\n            gr, gc = np.mgrid[a0:a1, b0:b1]\n            pts = np.column_stack([gc.ravel() + 0.5, gr.ravel() + 0.5])   # pixel CENTRES\n            ins = np.zeros(pts.shape[0], bool)\n            for e, hs in proj:\n                m = MplPath(e).contains_points(pts)\n                for hh in hs:\n                    if len(hh) >= 4: m &= ~MplPath(hh).contains_points(pts)\n                ins |= m\n            v = arr.ravel()[ins]; v = v[v > 0]\n            if v.size > 5: out[i] = float(v.mean())\n    return out\n\nWIN, GEOM_MEAN, HMAT = {}, {}, {}\nfor name, tag in SCENES:\n    GEOM_MEAN[name] = geo_farm_means(tag)\n    b0, r0, c0 = load_window(name); WIN[name] = (b0, r0, c0)\n    M = np.full((len(HGRID), NF), np.nan)\n    for hi, h in enumerate(HGRID):\n        pg = project_geom(name, h, GEOM)\n        for i in range(NF):\n            fp = farm_mask_pixels(b0, r0, c0, pg[i])\n            if fp is not None: M[hi, i] = dB(_trim(fp[0], .10))\n    HMAT[name] = M\n    print("   %s matrix done" % name, flush=True)\n\nheight_curves, coverage_curves = {}, {}\nfor name, _ in SCENES:\n    M = HMAT[name]; geo = GEOM_MEAN[name]\n    cohort = np.isfinite(M).all(axis=0) & np.isfinite(geo)      # FIXED cohort\n    n_fix = int(cohort.sum())\n    curve = []\n    for hi, h in enumerate(HGRID):\n        rho = float(spearmanr(M[hi, cohort], geo[cohort]).statistic) if n_fix > 50 else np.nan\n        curve.append(dict(h=float(h), rho_fixed_cohort=rho, n_fixed=n_fix,\n                          n_valid_this_height=int(np.isfinite(M[hi]).sum())))\n    height_curves[name] = curve\n    coverage_curves[name] = [dict(h=float(h), n_valid=int(np.isfinite(M[hi]).sum())) for hi, h in enumerate(HGRID)]\n    b = max(curve, key=lambda d: (d["rho_fixed_cohort"] if np.isfinite(d["rho_fixed_cohort"]) else -9))\n    print("   %s best h=%+.1f rho=%.4f (fixed cohort n=%d)" % (name, b["h"], b["rho_fixed_cohort"], n_fix), flush=True)\nLOG["height_curves_fixed_cohort"] = height_curves\nLOG["coverage_vs_height"] = coverage_curves\n\ndef fisher(r): return np.arctanh(np.clip(r, -0.999, 0.999))\ncand = []\nfor hi, h in enumerate(HGRID):\n    rs = [height_curves[n][hi]["rho_fixed_cohort"] for n, _ in SCENES]\n    if not all(np.isfinite(rs)): continue\n    cand.append((min(rs), float(np.mean([fisher(r) for r in rs])), float(h), [round(r, 4) for r in rs]))\ncand.sort(reverse=True)\nH_SHARED = cand[0][2]; BEST_MIN = cand[0][0]\n# D6: quantitative interval = all heights whose min-rho is within 0.01 of the maximum\nTOL = 0.01\ninside = sorted(h for mn, _, h, _ in cand if mn >= BEST_MIN - TOL)\nH_INTERVAL = (min(inside), max(inside))\nLOG["shared_height"] = dict(objective="maximise min-over-dates Spearman on a fixed cohort; tie-break mean Fisher-z",\n                            selected=H_SHARED, best_min_rho=BEST_MIN,\n                            interval_rule=f"all heights with min-rho >= best_min_rho - {TOL}",\n                            interval=[H_INTERVAL[0], H_INTERVAL[1]],\n                            top5=[dict(h=c[2], min_rho=c[0], mean_z=c[1], per_date=c[3]) for c in cand[:5]])\nprint("   SHARED HEIGHT %+.1f m (min-rho %.4f) interval [%+.1f, %+.1f]"\n      % (H_SHARED, BEST_MIN, H_INTERVAL[0], H_INTERVAL[1]), flush=True)\n\n# ============ B3: SENSITIVITY vs THE FIXED H_SHARED VECTOR ===================\nhs_idx = int(np.argmin(np.abs(HGRID - H_SHARED)))\nsens = {}\nfor name, _ in SCENES:\n    M = HMAT[name]; ref = M[hs_idx]\n    rows = []\n    for d in (-5.0, -2.5, 2.5, 5.0):\n        j = int(np.argmin(np.abs(HGRID - (H_SHARED + d))))\n        v = M[j]; k = np.isfinite(v) & np.isfinite(ref)\n        rows.append(dict(h=float(HGRID[j]), delta=d, n=int(k.sum()),\n                         spearman_vs_H_SHARED=float(spearmanr(v[k], ref[k]).statistic),\n                         mean_abs_db=float(np.mean(np.abs(v[k] - ref[k]))),\n                         median_abs_db=float(np.median(np.abs(v[k] - ref[k]))),\n                         coverage_change=int(np.isfinite(v).sum() - np.isfinite(ref).sum())))\n    sens[name] = rows\nLOG["height_sensitivity_vs_shared"] = sens\n\n# ============ centre incidence validation ====================================\ncc_chk = {}\nfor name, _ in SCENES:\n    m = META[name]\n    sat = sat_at(name, m["tc"])[0]; look = sat - m["tgt_c"]; look /= np.linalg.norm(look)\n    p = m["tgt_c"]; lo = math.degrees(math.atan2(p[1], p[0]))\n    la = math.degrees(math.atan2(p[2], math.hypot(p[0], p[1]) * (1 - WGS_E2)))\n    for _ in range(6):\n        N = WGS_A / math.sqrt(1 - WGS_E2 * math.sin(math.radians(la)) ** 2)\n        la = math.degrees(math.atan2(p[2] + WGS_E2 * N * math.sin(math.radians(la)), math.hypot(p[0], p[1])))\n    inc = math.degrees(math.acos(float(np.clip(np.dot(look, ellip_normal(lo, la)), -1, 1))))\n    cc_chk[name] = dict(metadata=m["inc_c"], interpolated=inc, diff=inc - m["inc_c"])\nLOG["centre_incidence_check"] = cc_chk\n\n# ============ FEATURES AT H_SHARED ==========================================\nprint("[2] features at shared height (topology-safe)", flush=True)\nrng = np.random.default_rng(SEED)\nFEAT = {f: dict(farm_id=f, area_ha=round(float(AREA_HA[i]), 6),\n                n_components=GEOM[f]["n_components"], n_holes=GEOM[f]["n_holes"],\n                geometry_unusable=GEOM[f]["unusable"]) for i, f in enumerate(FIDS)}\nmissing, est_stats, boot_sens, nesz_cmp, inc_rng = [], {n: {} for n, _ in SCENES}, {}, {}, {}\nfor name, tag in SCENES:\n    m = META[name]; b0, r0, c0 = WIN[name]\n    pg = project_geom(name, H_SHARED, GEOM)\n    pa, pb = {k: [] for k in EST}, {k: [] for k in EST}\n    infl = {k: [] for k in EST}\n    bsd = {bs: [] for bs in BLOCK_SIZES}; incs = []\n    for i, fid in enumerate(FIDS):\n        rec = FEAT[fid]\n        if GEOM[fid]["unusable"]:\n            fp = None; n_mask_i = 0; n_valid_i = 0\n        else:\n            _g, _vset, fp = farm_mask_sets(b0, r0, c0, pg[i])\n            n_mask_i = len(_g); n_valid_i = len(_vset)\n            if fp is not None and fp[0].size < GATE_PX: fp = None\n        if fp is None:\n            reason = "geometry unusable (degenerate/line-like components)" if GEOM[fid]["unusable"] \\\n                     else ("below %d-valid-pixel gate (n_valid=%d)" % (GATE_PX, n_valid_i)\n                           if n_valid_i > 0 else "no valid pixels inside polygon / outside swath")\n            missing.append(dict(farm_id=fid, date=name, reason=reason))\n            # I9: publish the TRUE counts even when the gate rejects the farm, so that\n            # n_valid/valid_fraction never silently under-report. gate_pass says whether\n            # the farm actually contributed a feature.\n            rec[f"n_mask_{name}"] = int(n_mask_i)\n            rec[f"n_valid_{name}"] = int(n_valid_i)\n            rec[f"valid_fraction_{name}"] = round(float(n_valid_i / max(n_mask_i, 1)), 5)\n            rec[f"gate_pass_{name}"] = 0\n            rec[f"swath_edge_or_low_coverage_flag_{name}"] = 1\n            continue\n        v, rr, cc = fp\n        rec[f"n_{name}"] = int(v.size)\n        th = float(incidence(name, CENT[i, 0], CENT[i, 1], H_SHARED, float(np.mean(rr))))\n        rec[f"incidence_deg_{name}"] = round(th, 4); incs.append(th)\n        sin_t = math.sin(math.radians(th))\n        # I3: retain ALL robust estimators, in beta0 AND sigma0\n        for k, fn in EST.items():\n            if k == "mean": continue\n            b_lin = fn(v); s_lin = b_lin * sin_t\n            rec[f"beta0_db_{k}_{name}"]  = round(float(dB(b_lin)), 4)\n            rec[f"sigma0_db_{k}_{name}"] = round(float(dB(s_lin)), 4)\n            if k == "trim10":\n                rec[f"beta0_lin_{name}"]  = float(f"{b_lin:.6g}")\n                rec[f"sigma0_lin_{name}"] = float(f"{s_lin:.6g}")\n        # B2: NESZ is SIGMA-ZERO -> compare/subtract in the SIGMA0 domain\n        nz_px_db = nesz_sigma0_db(name, cc)                   # per-pixel NESZ sigma0 (dB)\n        nz_db = float(np.mean(nz_px_db))                       # A: mean in dB, then subtract\n        nz_lin_mean = float(np.mean(10 ** (nz_px_db / 10.0)))  # B: mean in LINEAR power\n        s_lin = EST["trim10"](v) * sin_t                       # PRIMARY: raw unsubtracted sigma0\n        nz_lin = 10 ** (nz_db / 10.0)\n        sig_px = v * sin_t\n        nz_px_lin = 10 ** (nz_px_db / 10.0)\n        resid = sig_px - nz_px_lin                             # SIGNED pixelwise residual\n        n_nonpos = int((resid <= 0).sum())\n        # B6 C_signed: trim ALL signed residuals, no positivity conditioning.\n        # This is the clean aggregation-order test; the aggregate may legitimately be <= 0.\n        c_signed = EST["trim10"](resid)\n        # B6 C_clipped: max(resid, 0) before aggregation - explicitly censored/biased.\n        c_clipped = EST["trim10"](np.maximum(resid, 0.0))\n        # B6 C_positive_only: the OLD V3.2 behaviour, kept ONLY under its truthful name.\n        # SELECTION BIAS: conditioning on resid > 0 retains upward speckle excursions and\n        # discards downward ones, so the surviving sample is biased HIGH; the bias grows\n        # as SNR falls. It is NOT the clean aggregation-order experiment.\n        pos = resid > 0\n        c_pos = EST["trim10"](resid[pos]) if pos.sum() >= 30 else float("nan")\n        rec[f"sigma0_db_nsubA_{name}"] = round(float(dB(s_lin - nz_lin)), 4) if (s_lin - nz_lin) > 0 else ""\n        rec[f"sigma0_db_nsubB_{name}"] = round(float(dB(s_lin - nz_lin_mean)), 4) if (s_lin - nz_lin_mean) > 0 else ""\n        rec[f"sigma0_lin_C_signed_{name}"] = float(f"{c_signed:.6g}")\n        rec[f"sigma0_db_C_signed_{name}"] = round(float(dB(c_signed)), 4) if c_signed > 0 else ""\n        rec[f"sigma0_db_C_clipped_{name}"] = round(float(dB(c_clipped)), 4) if c_clipped > 0 else ""\n        rec[f"sigma0_db_C_positive_only_{name}"] = round(float(dB(c_pos)), 4) if (np.isfinite(c_pos) and c_pos > 0) else ""\n        rec[f"nesz_nonpos_frac_{name}"] = round(float(n_nonpos / max(v.size, 1)), 5)\n        rec[f"C_signed_nonpositive_{name}"] = int(c_signed <= 0)\n        rec[f"nesz_px_spread_db_{name}"] = round(float(np.max(nz_px_db) - np.min(nz_px_db)), 4)\n        # ---- I9: operational raster-quality columns ----\n        rec[f"n_mask_{name}"] = int(n_mask_i)\n        rec[f"n_valid_{name}"] = int(v.size)\n        rec[f"valid_fraction_{name}"] = round(float(v.size / max(n_mask_i, 1)), 5)\n        rec[f"gate_pass_{name}"] = 1\n        rec[f"swath_edge_or_low_coverage_flag_{name}"] = int((v.size / max(n_mask_i, 1)) < 0.95)\n        rec[f"nesz_sigma0_db_{name}"] = round(nz_db, 3)\n        rec[f"snr_sigma0_db_{name}"]  = round(float(dB(s_lin)) - nz_db, 3)\n        sub = s_lin - nz_lin\n        rec[f"sigma0_db_nsub_{name}"] = round(float(dB(sub)), 4) if sub > 0 else ""\n        rec[f"nesz_beta0_equiv_db_{name}"] = round(nz_db - 10 * math.log10(max(sin_t, 1e-9)), 3)\n        rec[f"nesz_flag_{name}"] = int((float(dB(s_lin)) - nz_db) < 3.0)\n        # spatial halves\n        msk = (rr < np.median(rr)) if (rr.max() - rr.min()) >= (cc.max() - cc.min()) else (cc < np.median(cc))\n        A, B = v[msk], v[~msk]\n        if A.size >= 30 and B.size >= 30:\n            for k, fn in EST.items(): pa[k].append(dB(fn(A))); pb[k].append(dB(fn(B)))\n        thr = np.percentile(v, 99); vk = v[v <= thr]\n        if vk.size > 30:\n            for k, fn in EST.items(): infl[k].append(abs(float(dB(fn(v))) - float(dB(fn(vk)))))\n        # I4: block bootstrap at 4/8/16\n        for bs in BLOCK_SIZES:\n            bi = ((rr - rr.min()) // bs).astype(np.int64) * 1000000 + ((cc - cc.min()) // bs).astype(np.int64)\n            ub = np.unique(bi)\n            if ub.size >= 4:\n                idx = {b: np.where(bi == b)[0] for b in ub}\n                bt = np.empty(NBOOT)\n                for t in range(NBOOT):\n                    sel = np.concatenate([idx[b] for b in rng.choice(ub, ub.size, replace=True)])\n                    bt[t] = dB(_trim(v[sel], .10))\n                sd = float(np.std(bt)); bsd[bs].append(sd)\n                if bs == PRIMARY_BLOCK:\n                    rec[f"boot_sd_db_{name}"] = round(sd, 4); rec[f"n_blocks_{name}"] = int(ub.size)\n            elif bs == PRIMARY_BLOCK:\n                rec[f"boot_sd_db_{name}"] = ""; rec[f"n_blocks_{name}"] = int(ub.size)\n    for k in EST:\n        a, b = np.array(pa[k]), np.array(pb[k]); ok = np.isfinite(a) & np.isfinite(b)\n        est_stats[name][k] = dict(n=int(ok.sum()), pearson=float(np.corrcoef(a[ok], b[ok])[0, 1]),\n                                  spearman=float(spearmanr(a[ok], b[ok]).statistic),\n                                  halfdiff_sd_db=float(np.std(a[ok] - b[ok])),\n                                  halfdiff_mad_db=float(np.median(np.abs(a[ok] - b[ok]))),\n                                  bright_influence_db=float(np.median(infl[k])) if infl[k] else None)\n    boot_sens[name] = {f"block_{bs}px": dict(\n        median_sd_db=float(np.median(bsd[bs])) if bsd[bs] else None,\n        p90_sd_db=float(np.percentile(bsd[bs], 90)) if bsd[bs] else None,\n        n=len(bsd[bs]), block_m_az=round(bs * m["px_row"], 2), block_m_rg=round(bs * m["px_col"], 2))\n        for bs in BLOCK_SIZES}\n    inc_rng[name] = dict(min=float(np.min(incs)), max=float(np.max(incs)), n=len(incs)) if incs else None\n    print("   %s done" % name, flush=True)\nLOG["estimators"] = est_stats; LOG["block_bootstrap_sensitivity"] = boot_sens\nLOG["incidence_range"] = inc_rng\n\ni6 = {}\nfor name, _ in SCENES:\n    offs = []\n    for i, fid in enumerate(FIDS):\n        r_ = FEAT[fid].get("incidence_deg_" + name)\n        if not isinstance(r_, float) or not np.isfinite(CENTC[i, 0]): continue\n        dx = (CENTC[i, 0] - CENT[i, 0]) * 111320 * math.cos(math.radians(CENT[i, 1]))\n        dy = (CENTC[i, 1] - CENT[i, 1]) * 110540\n        offs.append(math.hypot(dx, dy))\n    offs = np.array(offs)\n    if offs.size:\n        i6[name] = dict(n=int(offs.size), median_offset_m=float(np.median(offs)),\n                        p95_offset_m=float(np.percentile(offs, 95)),\n                        max_offset_m=float(offs.max()),\n                        note="representative_point is a guaranteed-inside SAMPLE point, NOT a centroid")\nLOG["I6_representative_vs_centroid"] = i6\n\n# ============ I2: EROSION INSIDE THE MAIN SCRIPT, TOPOLOGY-SAFE =============\nprint("[3] erosion sensitivity (topology-safe, in-script)", flush=True)\ndef eroded_geom(d):\n    out = {}\n    for fid in FIDS:\n        comps = []\n        for ext, holes in GEOM[fid]["components"]:\n            poly = Polygon(ext, holes)\n            pm = shp_transform(to_m, poly)\n            if not pm.is_valid: pm = pm.buffer(0)\n            e = pm.buffer(-d)\n            parts = list(e.geoms) if e.geom_type == "MultiPolygon" else ([e] if e.geom_type == "Polygon" else [])\n            for p in parts:                     # keep EVERY valid eroded component\n                if p.is_empty or p.area < MIN_COMPONENT_M2: continue\n                pl = shp_transform(to_ll, p)\n                comps.append((np.asarray(pl.exterior.coords, float),\n                              [np.asarray(r.coords, float) for r in pl.interiors]))\n        out[fid] = dict(components=comps)\n    return out\nero = {}\nfor name in ["Jun19", "Oct13"]:\n    b0, r0, c0 = WIN[name]\n    base = np.array([FEAT[f].get(f"beta0_db_trim10_{name}", np.nan) if isinstance(\n        FEAT[f].get(f"beta0_db_trim10_{name}"), float) else np.nan for f in FIDS])\n    eg = eroded_geom(5.0); pge = project_geom(name, H_SHARED, eg)\n    ev = np.full(NF, np.nan)\n    for i, fid in enumerate(FIDS):\n        if GEOM[fid]["unusable"]: continue\n        fp = farm_mask_pixels(b0, r0, c0, pge[i])\n        if fp is not None: ev[i] = dB(_trim(fp[0], .10))\n    k = np.isfinite(base) & np.isfinite(ev)\n    ero[name] = dict(erosion_m=5.0, n=int(k.sum()),\n                     spearman=float(spearmanr(base[k], ev[k]).statistic),\n                     mean_abs_db=float(np.mean(np.abs(base[k] - ev[k]))),\n                     farms_lost=int(np.isfinite(base).sum() - np.isfinite(ev).sum()))\n    print("   %s erosion rho=%.4f meanabs=%.3f dB lost=%d" % (name, ero[name]["spearman"],\n          ero[name]["mean_abs_db"], ero[name]["farms_lost"]), flush=True)\nLOG["erosion_sensitivity"] = ero\n\n# ============ NESZ variant comparison (sigma0 domain) =======================\nfor name, _ in SCENES:\n    raw = np.array([FEAT[f].get(f"sigma0_db_trim10_{name}", np.nan) if isinstance(\n        FEAT[f].get(f"sigma0_db_trim10_{name}"), float) else np.nan for f in FIDS])\n    sub = np.array([FEAT[f].get(f"sigma0_db_nsub_{name}", np.nan) if isinstance(\n        FEAT[f].get(f"sigma0_db_nsub_{name}"), float) else np.nan for f in FIDS])\n    snr = np.array([FEAT[f].get(f"snr_sigma0_db_{name}", np.nan) if isinstance(\n        FEAT[f].get(f"snr_sigma0_db_{name}"), float) else np.nan for f in FIDS])\n    flag = np.array([FEAT[f].get(f"nesz_flag_{name}", 0) or 0 for f in FIDS])\n    cen = raw.copy(); cen[flag == 1] = np.nan\n    k1 = np.isfinite(raw) & np.isfinite(sub); k2 = np.isfinite(raw) & np.isfinite(cen)\n    nesz_cmp[name] = dict(domain="sigma0 vs NESZ sigma0",\n                          median_snr_sigma0_db=float(np.nanmedian(snr)),\n                          n_within_3dB=int(np.nansum(flag)),\n                          n_nonpositive_after_subtraction=int(np.sum(np.isnan(sub) & np.isfinite(raw))),\n                          spearman_raw_vs_subtracted=float(spearmanr(raw[k1], sub[k1]).statistic),\n                          spearman_raw_vs_censored=float(spearmanr(raw[k2], cen[k2]).statistic))\nLOG["nesz_sigma0_domain"] = nesz_cmp\n\ni8 = {}\nfor name, _ in SCENES:\n    def g(col, _n=name):\n        out = []\n        for f in FIDS:\n            x = FEAT[f].get(col + "_" + _n)\n            out.append(x if isinstance(x, float) else np.nan)\n        return np.array(out)\n    A, B = g("sigma0_db_nsubA"), g("sigma0_db_nsubB")\n    Cs, Cc, Cp = g("sigma0_db_C_signed"), g("sigma0_db_C_clipped"), g("sigma0_db_C_positive_only")\n    spread = g("nesz_px_spread_db"); nonpos = g("nesz_nonpos_frac")\n    npf = np.array([FEAT[f].get("C_signed_nonpositive_" + name, 0) or 0 for f in FIDS])\n    def cmp2(x, y):\n        k = np.isfinite(x) & np.isfinite(y)\n        if k.sum() < 30: return None\n        return dict(n=int(k.sum()), max_abs_db=float(np.max(np.abs(x[k] - y[k]))),\n                    median_abs_db=float(np.median(np.abs(x[k] - y[k]))),\n                    spearman=float(spearmanr(x[k], y[k]).statistic),\n                    pearson=float(np.corrcoef(x[k], y[k])[0, 1]))\n    i8[name] = dict(\n        A_meandB_vs_B_meanlinear=cmp2(A, B),\n        A_vs_C_signed=cmp2(A, Cs),\n        A_vs_C_clipped=cmp2(A, Cc),\n        A_vs_C_positive_only=cmp2(A, Cp),\n        C_signed_vs_C_positive_only=cmp2(Cs, Cp),\n        nesz_within_farm_spread_db=dict(median=float(np.nanmedian(spread)),\n                                        p95=float(np.nanpercentile(spread, 95)),\n                                        max=float(np.nanmax(spread))),\n        nonpositive_pixel_fraction=dict(median=float(np.nanmedian(nonpos)),\n                                        p95=float(np.nanpercentile(nonpos, 95)),\n                                        max=float(np.nanmax(nonpos))),\n        n_farms_C_signed_nonpositive_aggregate=int(npf.sum()),\n        n_farms_C_positive_only_undefined=int(np.sum(np.isfinite(A) & ~np.isfinite(Cp))),\n        primary_feature="raw unsubtracted sigma0 (sigma0_db_trim10_*)",\n        selection_bias_note=("C_positive_only conditions the pixel sample on resid > 0, "\n                             "retaining upward speckle excursions and discarding downward "\n                             "ones. It is biased HIGH and is NOT the clean aggregation-order "\n                             "test. C_signed is. No NESZ variant is frozen or fed to a model."))\nLOG["nesz_aggregation_order_I8"] = i8\n\n# ============ I7: ONE-FACTOR TOPOLOGY ABLATION ==============================\nprint("[4] I7 one-factor topology ablation", flush=True)\nFLAT = {}\nfor _k, _i in enumerate(order):\n    _fid = FIDS[_k]\n    _pts = np.asarray(_shapes[_i].points, float)     # V3-style: all parts concatenated\n    FLAT[_fid] = dict(components=[(_pts, [])])\nabl = {}; ABL_DIFFS = {}\nfor name, _ in SCENES:\n    b0, r0, c0 = WIN[name]; geo = GEOM_MEAN[name]\n    vals = {}\n    for lab, gg in (("true_topology", GEOM), ("flattened_parts", FLAT)):\n        vv = np.full(NF, np.nan)\n        pg = project_geom(name, H_SHARED, gg)\n        for i, fid in enumerate(FIDS):\n            if lab == "true_topology" and GEOM[fid]["unusable"]: continue\n            _g, _v, fp = farm_mask_sets(b0, r0, c0, pg[i])\n            if fp is not None and fp[0].size >= 30: vv[i] = dB(_trim(fp[0], .10))\n        vals[lab] = vv\n    # I11: ONE explicit common cohort valid under BOTH treatments AND GEO\n    cohort = (np.isfinite(vals["true_topology"]) & np.isfinite(vals["flattened_parts"])\n              & np.isfinite(geo))\n    cohort_ids = [int(FIDS[i]) for i in np.where(cohort)[0]]\n    cohort_hash = hashlib.sha256((",".join(map(str, cohort_ids))).encode()).hexdigest()\n    rt = float(spearmanr(vals["true_topology"][cohort], geo[cohort]).statistic)\n    rf = float(spearmanr(vals["flattened_parts"][cohort], geo[cohort]).statistic)\n    d = vals["true_topology"] - vals["flattened_parts"]\n    diff_idx = np.where(cohort & np.isfinite(d) & (np.abs(d) > 1e-9))[0]\n    farm_diffs = sorted([dict(farm_id=int(FIDS[i]),\n                              true_db=round(float(vals["true_topology"][i]), 4),\n                              flat_db=round(float(vals["flattened_parts"][i]), 4),\n                              delta_db=round(float(d[i]), 4)) for i in diff_idx],\n                        key=lambda r_: -abs(r_["delta_db"]))\n    # farms present under one treatment only (excluded from the common cohort)\n    only_true = [int(FIDS[i]) for i in np.where(np.isfinite(vals["true_topology"])\n                 & ~np.isfinite(vals["flattened_parts"]))[0]]\n    only_flat = [int(FIDS[i]) for i in np.where(~np.isfinite(vals["true_topology"])\n                 & np.isfinite(vals["flattened_parts"]))[0]]\n    abl[name] = dict(cohort_n=int(cohort.sum()), cohort_sha256=cohort_hash,\n                     rho_true_topology=rt, rho_flattened_parts=rf,\n                     delta_rho_topology_only=rt - rf,\n                     n_farms_with_feature_difference=int(len(farm_diffs)),\n                     max_abs_delta_db=(abs(farm_diffs[0]["delta_db"]) if farm_diffs else 0.0),\n                     farm_level_differences_top20=farm_diffs[:20],\n                     farm_level_differences_full_file=f"{OUT}/topology_ablation_farm_diffs.json",\n                     farms_only_true_topology=only_true,\n                     farms_only_flattened=only_flat)\n    ABL_DIFFS[name] = dict(cohort_ids=cohort_ids, cohort_sha256=cohort_hash,\n                           farm_level_differences=farm_diffs)\n    print("   %s cohort=%d rho_true=%.6f rho_flat=%.6f delta=%+.6f diffs=%d"\n          % (name, cohort.sum(), rt, rf, rt - rf, len(farm_diffs)), flush=True)\nLOG["I7_topology_ablation"] = dict(\n    design=("Single common cohort per date: farms with a finite feature under BOTH true "\n            "and flattened topology AND a finite GEO-preview value. Identical membership "\n            "on both arms, verified by cohort_sha256."),\n    supported_conclusion=("This ablation can only show whether TOPOLOGY caused the "\n                          "V3.0->V3.1 alignment change. It does NOT attribute the change "\n                          "to fixed-cohort selection or tie handling; those were not "\n                          "separately ablated."),\n    per_date=abl)\njson.dump(ABL_DIFFS, open(f"{OUT}/topology_ablation_farm_diffs.json", "w"), indent=1)\n\n# ====== B5: B4 IMPACT LEDGER - GEOMETRIC and VALID, BOTH LIKE-FOR-LIKE =======\nprint("[5] B4 impact ledger (geometric + valid, like-for-like)", flush=True)\ndef project_geom_conv(name, h, geoms, op):\n    """op=IDENTITY -> V3.2 fractional vertices ; op=None -> V3.1 floored vertices"""\n    m = META[name]\n    with rasterio.open(slc_path(m["tag"])) as sdat:\n        with RPCTransformer(sdat.rpcs, RPC_HEIGHT=float(h)) as tr:\n            out = []\n            for fid in FIDS:\n                comps = []\n                for ext, holes in geoms[fid]["components"]:\n                    if op is None: rr, cc = tr.rowcol(ext[:, 0].tolist(), ext[:, 1].tolist())\n                    else:          rr, cc = tr.rowcol(ext[:, 0].tolist(), ext[:, 1].tolist(), op=op)\n                    e = np.column_stack([np.asarray(cc, float), np.asarray(rr, float)])\n                    hs = []\n                    for hh in holes:\n                        if op is None: r2, c2 = tr.rowcol(hh[:, 0].tolist(), hh[:, 1].tolist())\n                        else:          r2, c2 = tr.rowcol(hh[:, 0].tolist(), hh[:, 1].tolist(), op=op)\n                        hs.append(np.column_stack([np.asarray(c2, float), np.asarray(r2, float)]))\n                    comps.append((e, hs))\n                out.append(comps)\n    return out\n\nGATE = GATE_PX\nledger = {}\nfor name, _ in SCENES:\n    b0, r0, c0 = WIN[name]\n    p32 = project_geom_conv(name, H_SHARED, GEOM, IDENTITY)   # fractional verts\n    p31 = project_geom_conv(name, H_SHARED, GEOM, None)       # floored verts (V3.1)\n    gj, vj, dgeo, dval = [], [], [], []\n    n_geo31 = n_geo32 = n_val31 = n_val32 = 0\n    gate31 = gate32 = 0; flips = []\n    for i, fid in enumerate(FIDS):\n        if GEOM[fid]["unusable"]: continue\n        g31, v31, _ = farm_mask_sets(b0, r0, c0, p31[i], centre_offset=0.0)  # V3.1 convention\n        g32, v32, _ = farm_mask_sets(b0, r0, c0, p32[i], centre_offset=0.5)  # V3.2 convention\n        if not g31 and not g32: continue\n        n_geo31 += len(g31); n_geo32 += len(g32)\n        n_val31 += len(v31); n_val32 += len(v32)\n        if g31 or g32: gj.append(len(g31 & g32) / max(len(g31 | g32), 1))\n        if v31 or v32: vj.append(len(v31 & v32) / max(len(v31 | v32), 1))\n        dgeo.append(len(g32) - len(g31)); dval.append(len(v32) - len(v31))\n        a_pass = len(v31) >= GATE; b_pass = len(v32) >= GATE\n        gate31 += int(a_pass); gate32 += int(b_pass)\n        if a_pass != b_pass:\n            flips.append(dict(farm_id=int(fid), n_valid_v31=len(v31), n_valid_v32=len(v32)))\n    gj = np.array(gj); vj = np.array(vj); dgeo = np.array(dgeo); dval = np.array(dval)\n    ledger[name] = dict(\n        n_farms=int(gj.size),\n        geometric=dict(total_px_v31=int(n_geo31), total_px_v32=int(n_geo32),\n                       jaccard_median=float(np.median(gj)), jaccard_p10=float(np.percentile(gj, 10)),\n                       jaccard_min=float(gj.min()),\n                       px_delta_median=float(np.median(dgeo)),\n                       px_delta_min=int(dgeo.min()), px_delta_max=int(dgeo.max())),\n        valid=dict(total_px_v31=int(n_val31), total_px_v32=int(n_val32),\n                   jaccard_median=float(np.median(vj)), jaccard_p10=float(np.percentile(vj, 10)),\n                   jaccard_min=float(vj.min()),\n                   px_delta_median=float(np.median(dval)),\n                   px_delta_min=int(dval.min()), px_delta_max=int(dval.max()),\n                   gate_pass_v31=int(gate31), gate_pass_v32=int(gate32),\n                   gate_flips=flips))\n    print("   %s geoJ=%.4f validJ=%.4f validpx %d->%d gate %d->%d flips=%d"\n          % (name, ledger[name]["geometric"]["jaccard_median"], ledger[name]["valid"]["jaccard_median"],\n             n_val31, n_val32, gate31, gate32, len(flips)), flush=True)\nLOG["B4_impact_ledger"] = dict(\n    note="Both sub-ledgers are like-for-like: geometric compares polygon-membership "\n         "pixel sets (zero-fill included); valid compares finite-data pixel sets. "\n         "Neither applies the 30-pixel gate when forming the sets.",\n    gate_px=GATE, ledger=ledger)\n# dump the ledger BEFORE the hard assertion so a failure is diagnosable without a re-run\njson.dump(dict(gate_px=GATE, ledger=ledger), open(f"{OUT}/b4_impact_ledger.json", "w"), indent=1)\n\n# B5 ASSERTION: the valid-data ledger must show exactly one gate flip for these inputs\nEXPECTED_FLIPS = [("Aug14", 388, 35, 28)]\n_actual = [(nm, d["farm_id"], d["n_valid_v31"], d["n_valid_v32"])\n           for nm, L_ in ledger.items() for d in L_["valid"]["gate_flips"]]\nassert _actual == EXPECTED_FLIPS, (\n    "B5 ASSERTION FAILED: expected exactly one valid-data gate flip "\n    f"{EXPECTED_FLIPS}, observed {_actual}")\nLOG["B5_gate_flip_assertion"] = dict(expected=EXPECTED_FLIPS, observed=_actual, passed=True)\nprint("   B5 assertion PASS: exactly one valid gate flip %s" % (_actual,), flush=True)\n\n# ============ MISSINGNESS ===================================================\nobs = np.zeros((NF, 4), int)\nfor j, (name, _) in enumerate(SCENES):\n    for i, f in enumerate(FIDS): obs[i, j] = 1 if isinstance(FEAT[f].get(f"n_{name}"), int) else 0\nnd = obs.sum(axis=1)\nfor i, f in enumerate(FIDS):\n    FEAT[f]["n_dates_observed"] = int(nd[i]); FEAT[f]["never_imaged"] = int(nd[i] == 0)\nunusable_ids = [f for f in FIDS if GEOM[f]["unusable"]]\nLOG["missingness"] = dict(total_farms=NF, fully_observed=int((nd == 4).sum()),\n                          partial=int(((nd >= 1) & (nd <= 3)).sum()), never_imaged=int((nd == 0).sum()),\n                          never_imaged_due_to_geometry=len(unusable_ids),\n                          never_imaged_due_to_coverage=int((nd == 0).sum()) - len(unusable_ids),\n                          unusable_geometry_ids=unusable_ids,\n                          per_date_missing={n: int(NF - obs[:, j].sum()) for j, (n, _) in enumerate(SCENES)},\n                          missing_cells=len(missing), prior_only_rows=int((nd == 0).sum()))\njson.dump(missing, open(f"{OUT}/missing_farm_dates_v321.json", "w"), indent=1)\n\n# ===== G8: standing warnings, preserved verbatim from V3.1/V3.2 =============\nLOG["standing_warnings"] = dict(\n    boot_sd_db=("PROHIBITED as a model weight or uncertainty input. The block bootstrap is "\n                "NOT scale-stable: median SD rises materially from 4 px to 8 px to 16 px "\n                "(see block_bootstrap_sensitivity), so its absolute value is an artefact of "\n                "the chosen block size. Publish it as a diagnostic only."),\n    missingness=("39 farms are never imaged (10 for unusable geometry, 29 for coverage) and "\n                 "24 are partially observed; see the missingness object. Any model must handle "\n                 "these explicitly rather than imputing silently."),\n    geometry=("Component threshold 25 m^2 and farm threshold 50 m^2 are explicit choices. "\n              "Farms below them yield no pixels at all. Multipart farms are masked as a true "\n              "union of components with holes excluded and no bridging."),\n    estimators=("trim10 is the working default; median/trim20/mean are all published. The "\n                "spatial half-split diagnostics (not random splits) are the evidence for that "\n                "choice; see the estimators object."),\n    nesz=("Raw unsubtracted sigma0 remains PRIMARY. No NESZ-subtracted variant is frozen, "\n          "weighted or fed to a model. See nesz_aggregation_order_I8."),\n    geo_preview=("The Capella GEO preview is used ONLY as a same-acquisition SAR alignment "\n                 "reference. It is a vendor DEM-terrain-corrected SAR product, not optical "\n                 "and not DEM-free truth."))\n\n# ===== I9: in-script summary of the operational raster-quality columns ======\n# (reported from THIS object, never from a standalone CSV read - see G7)\ni9 = {}\nfor name, _ in SCENES:\n    vf = np.array([FEAT[f].get(f"valid_fraction_{name}") for f in FIDS\n                   if isinstance(FEAT[f].get(f"n_valid_{name}"), int)\n                   and FEAT[f].get(f"n_mask_{name}", 0) > 0], float)\n    nm = np.array([FEAT[f].get(f"n_mask_{name}", 0) or 0 for f in FIDS], float)\n    nv = np.array([FEAT[f].get(f"n_valid_{name}", 0) or 0 for f in FIDS], float)\n    fl = np.array([FEAT[f].get(f"swath_edge_or_low_coverage_flag_{name}", 0) or 0 for f in FIDS], int)\n    i9[name] = dict(\n        n_farms_with_geometric_mask=int((nm > 0).sum()),\n        n_mask_total=int(nm.sum()), n_valid_total=int(nv.sum()),\n        valid_fraction_median=float(np.median(vf)) if vf.size else None,\n        valid_fraction_p05=float(np.percentile(vf, 5)) if vf.size else None,\n        valid_fraction_min=float(vf.min()) if vf.size else None,\n        n_flagged_swath_edge_or_low_coverage=int(fl.sum()),\n        n_gate_pass=int(sum(FEAT[f].get(f"gate_pass_{name}", 0) or 0 for f in FIDS)),\n        n_below_gate_with_some_valid_pixels=int(sum(\n            1 for f in FIDS if (FEAT[f].get(f"gate_pass_{name}", 0) or 0) == 0\n            and (FEAT[f].get(f"n_valid_{name}", 0) or 0) > 0)))\nLOG["I9_raster_quality"] = dict(\n    threshold="valid_fraction < 0.95 sets swath_edge_or_low_coverage_flag",\n    gate_px=GATE_PX,\n    note=("n_mask_<DATE> = geometric (polygon-membership) pixel count INCLUDING zero-fill. "\n          "n_valid_<DATE> = finite-data pixels inside the same mask, published ALWAYS, "\n          "including for farms rejected by the 30-pixel gate. valid_fraction = n_valid/n_mask. "\n          "gate_pass_<DATE> = 1 only when a feature was emitted. The legacy column n_<DATE> is "\n          "PRESERVED and equals n_valid_<DATE> WHEN gate_pass_<DATE>=1, and is blank otherwise; "\n          "use n_valid_<DATE> for any quantitative work."),\n    per_date=i9)\nfor name, _ in SCENES:\n    print("   I9 %s valid_frac median=%s p05=%s flagged=%d"\n          % (name, i9[name]["valid_fraction_median"], i9[name]["valid_fraction_p05"],\n             i9[name]["n_flagged_swath_edge_or_low_coverage"]), flush=True)\n\n# ============ WRITE CSV =====================================================\ncols = ["farm_id", "area_ha", "n_components", "n_holes", "geometry_unusable", "n_dates_observed", "never_imaged"]\nfor name, _ in SCENES:\n    cols += [f"n_{name}", f"incidence_deg_{name}", f"beta0_lin_{name}", f"sigma0_lin_{name}"]\n    for k in ("median", "trim10", "trim20"):\n        cols += [f"beta0_db_{k}_{name}", f"sigma0_db_{k}_{name}"]\n    cols += [f"nesz_sigma0_db_{name}", f"nesz_beta0_equiv_db_{name}", f"snr_sigma0_db_{name}",\n             f"sigma0_db_nsub_{name}", f"sigma0_db_nsubA_{name}", f"sigma0_db_nsubB_{name}",\n             f"sigma0_lin_C_signed_{name}", f"sigma0_db_C_signed_{name}",\n             f"sigma0_db_C_clipped_{name}", f"sigma0_db_C_positive_only_{name}",\n             f"nesz_nonpos_frac_{name}", f"C_signed_nonpositive_{name}",\n             f"nesz_px_spread_db_{name}", f"nesz_flag_{name}",\n             f"n_mask_{name}", f"n_valid_{name}", f"valid_fraction_{name}", f"gate_pass_{name}",\n             f"swath_edge_or_low_coverage_flag_{name}",\n             f"boot_sd_db_{name}", f"n_blocks_{name}"]\nwith open(CSV_PATH, "w", newline="", encoding="utf-8") as f:\n    w = csv.DictWriter(f, fieldnames=cols, extrasaction="ignore"); w.writeheader()\n    for fid in FIDS: w.writerow(FEAT[fid])\n\nEND_ISO = datetime.now(timezone.utc).isoformat()\nRUN = dict(tag=ARGS.tag, start_utc=START_ISO, end_utc=END_ISO, runtime_s=round(time.time() - T0, 1),\n           seed=SEED, shared_height_m=H_SHARED, height_interval=[H_INTERVAL[0], H_INTERVAL[1]],\n           primary_block_px=PRIMARY_BLOCK, n_bootstrap=NBOOT,\n           python=platform.python_version(), platform=platform.platform(),\n           numpy=np.__version__, rasterio=rasterio.__version__,\n           n_rows=NF, csv_path=CSV_PATH, csv_sha256=sha256_full(CSV_PATH),\n           csv_bytes=os.path.getsize(CSV_PATH),\n           topology_regression=REG, convention_tests=REG_CONV,\n           topology_labels=dict(multipart_ids=MULTI_IDS, flagged_ids=FLAGGED_IDS,\n                                dropped_component_ids=DROPPED_IDS, unusable_ids=UNUSABLE_IDS_L,\n                                set_semantics=SET_SEMANTICS),\n           exit_status="OK", inputs=INPUTS)\nLOG["run"] = RUN\njson.dump(LOG, open(f"{OUT}/geometry_rebuild_v321_log.json", "w"), indent=1)\nstamp = START_ISO.replace(":", "").replace("-", "").split(".")[0]\njson.dump(RUN, open(f"{OUT}/run_manifest_{ARGS.tag}_{stamp}.json", "w"), indent=1)\nprint("\\nWROTE %s\\n  sha256=%s\\n  runtime=%.1fs" % (CSV_PATH, RUN["csv_sha256"], RUN["runtime_s"]))\n'
Path(SRC/"geometry_rebuild_v321.py").write_text(_SRC, encoding="utf-8")
print("wrote src/geometry_rebuild_v321.py  ({} lines)".format(_SRC.count(chr(10))))

In [ ]:
# S05 is the heaviest stage: four windowed SLC reads + RPC height search + bootstrap.
t0 = time.time()
p = subprocess.run([sys.executable, str(SRC / "geometry_rebuild_v321.py"), "--tag", "E2E"],
                   cwd=str(WORK), env=env, capture_output=True, text=True, timeout=7200)
if p.returncode != 0:
    print(p.stdout[-2500:]); print(p.stderr[-3000:]); raise RuntimeError("S05 geometry failed")
T["S05"] = time.time() - t0
GEOM = ROOT / "farm_features_geometry_v321.csv"
assert GEOM.is_file()
_g = list(_csv.DictReader(open(GEOM, encoding="utf-8")))
print("\n".join([l for l in p.stdout.split("\n") if l.strip()][-14:]))
print(f"\nS05 geometry: {len(_g)} farms x {len(_g[0])} cols in {T['S05']/60:.1f} min")

### OUTPUT / VALIDATION
import numpy as _np
_d = [float(r["sigma0_db_trim10_Aug14"]) for r in _g if r["sigma0_db_trim10_Aug14"]]
print(f"  Aug14 farm sigma0 (dB): n={len(_d)}  median {_np.median(_d):.2f}  "
      f"p05 {_np.percentile(_d,5):.2f}  p95 {_np.percentile(_d,95):.2f}")
assert len(_g) == 966

---
## 4 · The 19 June artifact  ·  `[P] S06 · S07`

### WHY
Jun19 farm backscatter carries a strong trend across the SLC **range** coordinate. The decisive
evidence that this is an instrument/processing artifact rather than agronomy: an **independent
non-farm reference** — surfaces with no crops — shows the *same* ramp, correlating with the farm
ramp at **ρ ≈ 0.9936**. A canopy signal cannot appear on non-farm ground.

### METHOD
Fit a quadratic $g_{\text{ref}}(u)$ to the non-farm reference against the normalised range
coordinate $u$, centre it, subtract:

$$\sigma^0_{\text{clean}}=\sigma^0_{\text{raw}}-c(u),\qquad c(u)=g_{\text{ref}}(u)-\overline{g_{\text{ref}}}$$

**No mean matching. No variance matching. No crop labels anywhere in the fit.** The correction is
estimated entirely off non-farm ground, so it cannot manufacture agreement with any downstream
target. Matching moments would have guaranteed a good-looking result and destroyed the evidence.

**Invalid-event mask.** Farms with **≥20 % of pixels above −5 dB** sit far outside any crop-canopy
response. We do not invent a mechanism for them — they are flagged `jun19_valid = 0` and excluded
from Jun19 canopy inference. The count is computed, never hardcoded.

In [ ]:
# S06 -- E1 Jun19 high-backscatter contamination map (invalid-event detection).
_SRC = '"""\nE1 - Locate the Jun19 high-backscatter event and derive a defensible validity mask.\n\nWHY\n  The 20 farms flagged `jun19_transient_extreme` are not extreme measurements of\n  those farms. Their WHOLE field is elevated ~26 dB on Jun19 only:\n      median +5.10 / trim10 +6.87 / trim20 +6.21 dB   vs cohort -20.5 / -19.6 / -20.0\n      SNR vs NESZ 33.3 dB   vs cohort 7.7 dB\n      Jun06 -0.30 dB, Aug14 +1.50 dB, Oct13 +1.00 dB relative to cohort\n  A 0.2 ha field cannot return sigma0 ~ +7 dB at X-band HH. These are not usable\n  canopy measurements. The MECHANISM IS NOT NAMED: it stays\n  UNKNOWN_HIGH_BACKSCATTER_EVENT, and validity stays\n  INVALID_FOR_CANOPY_INFERENCE.\n\nWHAT THIS DOES\n  Works on PIXELS, not farm summaries, so the affected zone is located\n  independently of the farm polygons and of the IQR fence that first flagged it.\n    1. sample in-village Jun19 pixels, bin along range, and find where the\n       background level departs from the scene\'s normal behaviour\n    2. locate the affected zone in TWO dimensions (range x azimuth).\n       NOTE: a first version of this script binned in RANGE ONLY and found\n       nothing (0 of 220 bins above threshold). That was a design error and it\n       falsified the "range band" framing: 20 farms are ~0.25% of in-village\n       pixels, so a bin median spanning the whole azimuth extent washes them\n       out. The 2D test below is the correction; the negative result is retained.\n       SECOND CORRECTION: the 2D cell test ALSO found 0 cells above 50% hot at\n       60 px, and the "compact patch" reading drawn from that was likewise a\n       dilution artefact. E1b (diag_e1b_boundary_test.py) settled the geometry\n       with an unconstrained neighbourhood test: the event decays smoothly\n       outward across parcel boundaries (hot fraction 0.993 inside -> 0.882 at\n       0-10 m -> 0.462 at 25-50 m -> 0.166 at 100-200 m), so it is NEITHER a\n       swath-wide range band NOR parcel-aligned. Both of those framings are\n       withdrawn; only the per-farm measurement below is load-bearing.\n    3. measure EVERY farm\'s own fraction of physically impossible pixels,\n       independently of the IQR fence that first flagged the 20\n    4. emit `jun19_valid` per farm from measured contaminated-pixel fraction\n    5. verify the event is absent on the other three dates\n\nWHAT IT DOES NOT DO\n  It does not delete anything. Raw values are preserved. `jun19_valid = 0` is a\n  VALIDITY CLASSIFICATION of the observation, not removal of the farm.\n\nReads only Capella + the supplied shapefiles, via the fail-fast provenance\nresolver. Writes round2/_diag/jun19_contamination_map.json and figures.\n"""\nimport os, sys, json, csv, math, hashlib, warnings\nimport numpy as np\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport shapefile\nimport rasterio\nfrom rasterio.windows import Window\nfrom rasterio.transform import RPCTransformer\nfrom matplotlib.path import Path as MplPath\nfrom shapely.geometry import shape as gshape\nfrom shapely.ops import transform as shp_transform\nfrom pyproj import Transformer\n\nsys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\nfrom _provenance import resolve_all, print_table, slc_path, meta_path, TAGS\n\nwarnings.filterwarnings("ignore")\nCSV_IN = "round2/farm_features_jun19clean_v1.csv"\nLOGP = "round2/_geom321/geometry_rebuild_v321_log.json"\nSHP = "round2/data/Farm_boundaries_shp/Farm_boundaries_shp/Sokhda_Farms.shp"\nVSHP = "round2/data/Village_Shp/Village_Shp/Sokhda_Village.shp"\nOUT = "round2/_diag"; FIG = f"{OUT}/figs_e1"; os.makedirs(FIG, exist_ok=True)\nDATES = ["Jun06", "Jun19", "Aug14", "Oct13"]\nIDENTITY = (lambda v: v)\nSTEP = 3                      # pixel subsample stride\nNBIN = 220                    # range bins across the AOI\n\nprint("=" * 92)\nprint("E1  Jun19 far-range contamination: pixel-level mapping and validity mask")\nprint("=" * 92)\nPROV = resolve_all(DATES); print_table(PROV)\n\nV321 = json.load(open(LOGP)); H = float(V321["shared_height"]["selected"])\nrows = list(csv.DictReader(open(CSV_IN, encoding="utf-8")))\nfid = np.array([int(r["farm_id"]) for r in rows]); NF = len(rows)\n\n\ndef fl(v):\n    try: return float(v)\n    except (TypeError, ValueError): return float("nan")\n\n\ntrans_flag = np.array([int(r["jun19_transient_extreme"]) for r in rows], bool)\n\nvg = gshape(shapefile.Reader(VSHP).shapes()[0].__geo_interface__)\nif not vg.is_valid: vg = vg.buffer(0)\nvlon, vlat = np.array(vg.exterior.coords).T\nsf = shapefile.Reader(SHP); recs, shps = sf.records(), sf.shapes()\norder = np.argsort([int(r[0]) for r in recs])\n\n\ndef scene_pixels(dname):\n    """Return (col, sigma0_dB) for subsampled in-village pixels, plus per-farm\n    pixel column/value lists. Uses the frozen V3.2.1 calibration convention."""\n    tag = TAGS[dname]\n    md = json.load(open(meta_path(tag)))\n    scale = md["collect"]["image"]["scale_factor"]\n    with rasterio.open(slc_path(tag)) as s_:\n        with RPCTransformer(s_.rpcs, RPC_HEIGHT=H) as tr:\n            vr, vc = tr.rowcol(vlon.tolist(), vlat.tolist(), op=IDENTITY)\n            vr = np.asarray(vr, float); vc = np.asarray(vc, float)\n            r0 = max(int(vr.min()) - 2, 0); r1 = min(int(vr.max()) + 2, s_.height)\n            c0 = max(int(vc.min()) - 2, 0); c1 = min(int(vc.max()) + 2, s_.width)\n            blk = s_.read(1, window=Window(c0, r0, c1 - c0, r1 - r0))\n            fpaths = []\n            for i2 in order:\n                g2 = gshape(shps[i2].__geo_interface__)\n                if not g2.is_valid: g2 = g2.buffer(0)\n                polys = list(g2.geoms) if g2.geom_type == "MultiPolygon" else [g2]\n                sub = []\n                for poly in polys:\n                    e = np.asarray(poly.exterior.coords, float)\n                    er, ec = tr.rowcol(e[:, 0].tolist(), e[:, 1].tolist(), op=IDENTITY)\n                    ec = np.asarray(ec, float); er = np.asarray(er, float)\n                    sub.append((MplPath(np.column_stack([ec, er])),\n                                ec.min(), ec.max(), er.min(), er.max()))\n                fpaths.append(sub)\n    b0 = (np.abs(blk).astype(np.float32) * scale) ** 2\n    b0[b0 <= 0] = np.nan\n    Hh, Ww = b0.shape\n    gr, gc = np.mgrid[0:Hh:STEP, 0:Ww:STEP]\n    Pc = gc.ravel() + c0 + 0.5; Pr = gr.ravel() + r0 + 0.5\n    val = b0[gr.ravel(), gc.ravel()]\n    inv = MplPath(np.column_stack([vc, vr])).contains_points(np.column_stack([Pc, Pr]))\n    keep = inv & np.isfinite(val)\n    return dict(col=Pc[keep], row=Pr[keep], db=10 * np.log10(np.maximum(val[keep], 1e-15)),\n                fpaths=fpaths, c0=c0, r0=r0, shape=(Hh, Ww))\n\n\n# ---------- 1. TWO-DIMENSIONAL hot-fraction map, all four dates --------------\n# "hot" = sigma0 > -5 dB, far above anything a crop canopy returns at X-band HH.\nCELL = 60          # pixels per grid cell, range and azimuth\nprof = {}\nfor d in DATES:\n    S = scene_pixels(d)\n    cj = ((S["col"] - S["col"].min()) // CELL).astype(int)\n    ri = ((S["row"] - S["row"].min()) // CELL).astype(int)\n    nc, nr = int(cj.max()) + 1, int(ri.max()) + 1\n    hotpx = (S["db"] > -5.0)\n    key = ri * nc + cj\n    cnt = np.bincount(key, minlength=nr * nc).astype(float)\n    hct = np.bincount(key, weights=hotpx.astype(float), minlength=nr * nc)\n    fr = np.where(cnt >= 40, hct / np.maximum(cnt, 1.0), np.nan).reshape(nr, nc)\n    med = np.full(nr * nc, np.nan)\n    o = np.argsort(key); ks = key[o]; vs = S["db"][o]\n    bnd = np.searchsorted(ks, np.arange(nr * nc + 1))\n    for k in range(nr * nc):\n        i0, i1 = bnd[k], bnd[k + 1]\n        if i1 - i0 >= 40: med[k] = np.median(vs[i0:i1])\n    med = med.reshape(nr, nc)\n    prof[d] = dict(S=S, frac=fr, med=med, nc=nc, nr=nr,\n                   col0=float(S["col"].min()), row0=float(S["row"].min()))\n    print("[%s] cells %dx%d | cells >50%% hot: %d | max cell hot-frac %.3f | max cell median %.2f dB"\n          % (d, nr, nc, int(np.nansum(np.nan_to_num(fr, nan=0.0) > 0.5)),\n             np.nanmax(fr), np.nanmax(med)), flush=True)\n\n# ---------- 2. the affected zone on Jun19 ------------------------------------\nJ = prof["Jun19"]\nAFF = np.nan_to_num(J["frac"], nan=0.0) > 0.5\nn_aff = int(AFF.sum())\nzone = None\nif n_aff:\n    rr, cc = np.where(AFF)\n    zone = dict(range_col=[J["col0"] + float(cc.min()) * CELL, J["col0"] + float(cc.max() + 1) * CELL],\n                azimuth_row=[J["row0"] + float(rr.min()) * CELL, J["row0"] + float(rr.max() + 1) * CELL],\n                n_cells=n_aff, cell_px=CELL)\nprint("")\nprint("[zone] Jun19 cells with >50%% hot pixels: %d" % n_aff)\nif zone:\n    print("[zone] range cols %.0f..%.0f | azimuth rows %.0f..%.0f"\n          % (zone["range_col"][0], zone["range_col"][1],\n             zone["azimuth_row"][0], zone["azimuth_row"][1]))\nfor d in DATES:\n    print("      %s cells >50%% hot: %d"\n          % (d, int(np.nansum(np.nan_to_num(prof[d]["frac"], nan=0.0) > 0.5))))\nBAND = zone["range_col"] if zone else None\nbase = float(np.nanmedian(J["med"]))\nmad = float(np.nanmedian(np.abs(J["med"] - base))) * 1.4826\nTHR = -5.0\nhot = AFF\n\n# ---------- 3. per-farm contaminated-pixel fraction, measured -----------------\nS19 = prof["Jun19"]["S"]\n# measured directly: the fraction of a farm\'s OWN Jun19 pixels that are\n# physically impossible (> -5 dB). Independent of the polygon-summary IQR fence.\nfrac = np.zeros(NF); npx = np.zeros(NF, int)\nfor k, sub in enumerate(S19["fpaths"]):\n    tot = 0; bad = 0\n    for pth, cmin, cmax, rmin, rmax in sub:\n        cand = np.where((S19["col"] >= cmin - 1) & (S19["col"] <= cmax + 1) &\n                        (S19["row"] >= rmin - 1) & (S19["row"] <= rmax + 1))[0]\n        if cand.size == 0: continue\n        inside = pth.contains_points(np.column_stack([S19["col"][cand], S19["row"][cand]]))\n        sel = cand[inside]\n        if sel.size == 0: continue\n        tot += int(sel.size)\n        bad += int((S19["db"][sel] > -5.0).sum())\n    npx[k] = tot\n    frac[k] = bad / tot if tot else 0.0\n\n# choose the cut from the observed distribution, not a priori\nnz = np.sort(frac[frac > 0])\nCUT = 0.20\njun19_valid = np.ones(NF, int)\njun19_valid[frac >= CUT] = 0\nnewly = int(((frac >= CUT) & ~trans_flag).sum())\nmissed = int(((frac < CUT) & trans_flag).sum())\nprint("\\n[farms] contaminated-fraction > 0 : %d | >= %.2f : %d" % (int((frac > 0).sum()), CUT, int((frac >= CUT).sum())))\nprint("[farms] previously flagged by the IQR fence : %d" % int(trans_flag.sum()))\nprint("[farms] NEWLY caught (fence missed them)    : %d" % newly)\nprint("[farms] fence-flagged but below the cut     : %d" % missed)\n\n# ---------- 4. figures --------------------------------------------------------\nfig, ax = plt.subplots(1, 4, figsize=(17, 4.2))\nim = None\nfor a_, d in zip(ax, DATES):\n    P = prof[d]\n    im = a_.imshow(np.nan_to_num(P["frac"], nan=0.0), origin="lower", aspect="auto",\n                   vmin=0, vmax=1, cmap="inferno",\n                   extent=[P["col0"], P["col0"] + P["nc"] * CELL,\n                           P["row0"], P["row0"] + P["nr"] * CELL])\n    a_.set_title("%s  (cells>50%% hot: %d)"\n                 % (d, int(np.nansum(np.nan_to_num(P["frac"], nan=0.0) > 0.5))))\n    a_.set_xlabel("range column")\nax[0].set_ylabel("azimuth row")\nfig.colorbar(im, ax=ax, label="fraction of pixels > -5 dB", fraction=.02)\nplt.suptitle("E1 FIG 1  physically impossible pixels: present on Jun19 only")\nplt.savefig(FIG + "/1_hot_fraction_2d.png", dpi=130, bbox_inches="tight"); plt.close()\n\nplt.figure(figsize=(7, 4.2))\npos = frac[frac > 0]\nif pos.size: plt.hist(pos, bins=40, color="indianred", edgecolor="k", lw=.3)\nplt.axvline(CUT, color="k", ls="--", label="validity cut %.2f" % CUT)\nplt.xlabel("fraction of a farm\'s Jun19 pixels above -5 dB"); plt.ylabel("farms")\nplt.title("E1 FIG 2  per-farm impossible-pixel fraction"); plt.legend(fontsize=8)\nplt.grid(alpha=.3); plt.tight_layout(); plt.savefig(FIG + "/2_farm_fraction.png", dpi=130); plt.close()\n\n# ---------- 5. durable output -------------------------------------------------\naff = [int(x) for x in fid[frac >= CUT]]\nLOG = dict(\n    purpose="locate the Jun19 far-range contamination from PIXELS and derive a validity mask",\n    input_provenance=PROV,\n    finding=dict(\n        description=("A contiguous far-range band on Jun19 shows area-wide radiometric "\n                     "contamination. Affected farms have median ~ trim10 ~ trim20 ~ +5..+10 dB, "\n                     "i.e. the WHOLE field is elevated ~26 dB, at valid_fraction 1.000, on this "\n                     "date only. This is not a bright scatterer and not crop signal."),\n        physical_impossibility=("X-band HH sigma0 of about +7 dB from a 0.2 ha agricultural field "\n                                "is not physically attainable; published X-band rice canopy sits "\n                                "~8 dB ABOVE WATER and is still strongly negative, and X-band "\n                                "saturates above ~1 kg/m2 biomass."),\n        candidate_mechanisms=["radio-frequency interference",\n                              "range or azimuth ambiguity ghost from a bright off-swath target",\n                              "processing or saturation artefact at the swath edge"],\n        mechanism_not_identified=True,\n        kb_gap=("KB question Q13 on far-range/antenna-pattern artefacts returned no relevant "\n                "passages (top scores -1.44 to -2.47), so the mechanism is characterised from "\n                "the data rather than named from literature.")),\n    zone=dict(two_dimensional=zone, cell_px=CELL, hot_definition_db=-5.0,\n              rule="grid cells where more than 50 percent of sampled pixels exceed -5 dB",\n              superseded_range_only_test=dict(\n                  result="0 of 220 range bins above threshold",\n                  why_it_failed=("binning in range only averages over the whole azimuth extent; "\n                                 "the affected farms are about 0.25 percent of in-village pixels, "\n                                 "so a bin median cannot see them. This falsified the \'range band\' "\n                                 "framing and is kept as a negative result."))),\n    other_dates=({d: dict(n_cells_over_half_hot=int(np.nansum(np.nan_to_num(prof[d]["frac"], nan=0.0) > 0.5)),\n                          max_cell_hot_fraction=round(float(np.nanmax(prof[d]["frac"])), 5),\n                          max_cell_median_db=round(float(np.nanmax(prof[d]["med"])), 3))\n                  for d in DATES}),\n    validity=dict(rule="jun19_valid = 0 when >= %.2f of a farm\'s Jun19 pixels lie inside the band" % CUT,\n                  cut=CUT, n_invalid=int((frac >= CUT).sum()),\n                  n_previously_flagged_by_iqr_fence=int(trans_flag.sum()),\n                  n_newly_caught=newly, n_fence_flagged_below_cut=missed,\n                  invalid_farm_ids=aff),\n    per_farm=[dict(farm_id=int(fid[i]), contaminated_fraction=round(float(frac[i]), 4),\n                   n_pixels_sampled=int(npx[i]), jun19_valid=int(jun19_valid[i]),\n                   was_iqr_flagged=int(trans_flag[i]))\n              for i in range(NF) if frac[i] > 0],\n    policy=dict(raw_preserved=True, farms_deleted=False, date_dropped=False,\n                statement=("jun19_valid is a VALIDITY CLASSIFICATION of the observation, not "\n                           "removal of the farm or the date. Raw Jun19 remains in the table for "\n                           "every farm. Downstream models must not consume Jun19 where "\n                           "jun19_valid = 0, and must fall back to the other three dates with "\n                           "inflated uncertainty.")),\n    subsample_stride=STEP, n_range_bins=NBIN)\njson.dump(LOG, open(f"{OUT}/jun19_contamination_map.json", "w"), indent=1)\nprint("\\nwrote %s/jun19_contamination_map.json and 3 figures" % OUT)\nprint("INVALID Jun19 farms (%d): %s" % (len(aff), aff))\n'
Path(SRC/"diag_e1_jun19_contamination.py").write_text(_SRC, encoding="utf-8")
print("wrote src/diag_e1_jun19_contamination.py  ({} lines)".format(_SRC.count(chr(10))))

In [ ]:
# S07 -- canonical Jun19 range-artifact diagnosis and correction.
_SRC = '"""\nAISEHACK 2.0 Round 2 - BUILD sigma0_Jun19_clean (reference-derived correction).\n\nDECOMPOSITION ASSUMED (from D4 + corrected D4b)\n    farm Jun19 = shared acquisition/range component\n               + genuine farm/environmental structure\n               + transient farm-specific extremes\n               + ordinary noise\n\nThe correction is estimated from the NON-FARM REFERENCE SURFACE, not from the\nfarm trend. The farm trend (3.62 dB) contains acquisition AND agronomy; the\nreference trend (2.33 dB at 64 px) is the best available estimate of the\nacquisition-common part alone. Fitting on farms would subtract real signal.\nEvidence: same orientation and sign, fitted-curve correlation r = 0.9936,\nreference/farm amplitude ratio >= 0.61 at every aggregation scale.\n\nWHAT IS PRESERVED, ALWAYS\n    sigma0_Jun19_raw            never overwritten\n    jun19_transient_extreme     the 20 farms, flagged, never deleted\n    jun19_correction_magnitude_db\n    jun19_quality_weight\nAll four acquisition dates remain in the pipeline. The residual non-range\nspatial component is deliberately NOT removed.\n\nCORRECTION\n    u        normalised range coordinate, shared by farms and reference\n    g_ref(u) fitted Jun19 reference quadratic at the chosen aggregation scale\n    c(u)     = g_ref(u) - mean[g_ref(u) over the farm support]      (centred)\n    clean    = raw - c(u_farm)\nCentring keeps Jun19\'s overall level intact: the mean Jun19 backscatter may\nitself carry real phenological information and is not forced toward any other\ndate. Neither mean nor variance is matched to Jun06/Aug14/Oct13.\n\nTransient farms receive a correction from their RANGE COORDINATE ONLY; their\nanomalous magnitude never influences the fit and never changes their raw value.\n\nWrites round2/farm_features_jun19clean_v1.csv  (derived; V3.2.1 untouched)\n       round2/_diag/jun19_clean_validation.json\n       round2/_diag/figs_jun19clean/*.png\n"""\nimport os, sys, json, csv, math, hashlib, warnings\nimport numpy as np\nfrom scipy.stats import spearmanr, pearsonr\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport shapefile\nimport rasterio\nfrom rasterio.transform import RPCTransformer\nfrom shapely.geometry import shape as gshape, MultiPolygon\nfrom shapely.ops import transform as shp_transform\nfrom pyproj import Transformer\n\nsys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\nfrom _provenance import resolve_all, print_table, slc_path, TAGS\n\nwarnings.filterwarnings("ignore")\nCSV_IN = "round2/farm_features_geometry_v321.csv"\nCSV_OUT = "round2/farm_features_jun19clean_v1.csv"\nLOGP = "round2/_geom321/geometry_rebuild_v321_log.json"\nD4P = "round2/_diag/jun19_spatial_direction.json"\nD4BP = "round2/_diag/jun19_background_amplitude.json"\nSUBP = "round2/submission_v1.csv"\nSHP = "round2/data/Farm_boundaries_shp/Farm_boundaries_shp/Sokhda_Farms.shp"\nVSHP = "round2/data/Village_Shp/Village_Shp/Sokhda_Village.shp"\nOUT = "round2/_diag"; FIG = f"{OUT}/figs_jun19clean"; os.makedirs(FIG, exist_ok=True)\nDATES = ["Jun06", "Jun19", "Aug14", "Oct13"]\nIDENTITY = (lambda v: v)\nUGRID = np.linspace(0.0, 1.0, 101)\nSCALES = [16, 32, 64, 96]\nPRIMARY_SCALE = 64          # justified below in SELECTION_RATIONALE\nMET_CRS = "EPSG:32643"\n\nSELECTION_RATIONALE = (\n    "64 px blocks (~3,700 valid looks each, 1,353 blocks) are chosen as the nominal "\n    "reference scale because that support is closest to a real farm\'s ~5,000-look "\n    "trimmed mean, which is the quantity being corrected. 16 px blocks (~250 looks) "\n    "retain enough residual speckle and local heterogeneity to inflate the fitted "\n    "amplitude to 3.63 dB, and 96 px blocks start to smear the trend across the AOI. "\n    "The choice is a support-matching argument, not a fit-quality argument, and the "\n    "full 16/32/64/96 sensitivity is propagated into jun19_quality_weight.")\n\nprint("=" * 96)\nprint("BUILD sigma0_Jun19_clean   (reference-derived, conservative, raw always preserved)")\nprint("=" * 96)\nPROV = resolve_all(DATES); print_table(PROV)\n\n\ndef f(x):\n    try: return float(x)\n    except (TypeError, ValueError): return float("nan")\n\n\nrows = list(csv.DictReader(open(CSV_IN, encoding="utf-8")))\nV321 = json.load(open(LOGP)); H = float(V321["shared_height"]["selected"])\nD4 = json.load(open(D4P)); D4B = json.load(open(D4BP))\nfid = np.array([int(r["farm_id"]) for r in rows]); NF = len(rows)\nsig = {d: np.array([f(r.get(f"sigma0_db_trim10_{d}")) for r in rows]) for d in DATES}\ngate = {d: np.array([f(r.get(f"gate_pass_{d}")) for r in rows]) for d in DATES}\n\n# ---------------- farm normalised range coordinate u -------------------------\nto_m = Transformer.from_crs("EPSG:4326", MET_CRS, always_xy=True).transform\nto_ll = Transformer.from_crs(MET_CRS, "EPSG:4326", always_xy=True).transform\nsf = shapefile.Reader(SHP); recs, shps = sf.records(), sf.shapes()\norder = np.argsort([int(r[0]) for r in recs])\nlon = np.full(NF, np.nan); lat = np.full(NF, np.nan)\nXM = np.full(NF, np.nan); YM = np.full(NF, np.nan)\nfor k, i in enumerate(order):\n    g = gshape(shps[i].__geo_interface__)\n    if not g.is_valid: g = g.buffer(0)\n    gm = shp_transform(to_m, g)\n    parts = [p for p in (list(gm.geoms) if gm.geom_type == "MultiPolygon" else [gm]) if p.area >= 25.0]\n    if not parts: continue\n    A_ = sum(p.area for p in parts)\n    cx = sum(p.centroid.x * p.area for p in parts) / A_\n    cy = sum(p.centroid.y * p.area for p in parts) / A_\n    XM[k], YM[k] = cx, cy\n    lon[k], lat[k] = to_ll(cx, cy)\n\nvg = gshape(shapefile.Reader(VSHP).shapes()[0].__geo_interface__)\nif not vg.is_valid: vg = vg.buffer(0)\nvlon, vlat = np.array(vg.exterior.coords).T\nwith rasterio.open(slc_path(TAGS["Jun19"])) as s_:\n    with RPCTransformer(s_.rpcs, RPC_HEIGHT=H) as tr:\n        vr, vc = tr.rowcol(vlon.tolist(), vlat.tolist(), op=IDENTITY)\n        ok = np.isfinite(lon)\n        fr, fc = tr.rowcol(lon[ok].tolist(), lat[ok].tolist(), op=IDENTITY)\nvc = np.asarray(vc, float)\nlo_c, hi_c = float(np.nanmin(vc)), float(np.nanmax(vc))\nspan = max(hi_c - lo_c, 1.0)\nfarm_col = np.full(NF, np.nan); farm_col[ok] = np.asarray(fc, float)\nU = (farm_col - lo_c) / span\nprint("[u] normalised range: (SLC column - %.1f)/%.1f ; farm u range %.4f..%.4f"\n      % (lo_c, span, np.nanmin(U), np.nanmax(U)))\n\n# ---------------- cohort and transient flag (identical rule to D3/D4) --------\nfull = np.ones(NF, bool)\nfor d in DATES: full &= (gate[d] == 1) & np.isfinite(sig[d])\nI = np.where(full)[0]\nv19 = sig["Jun19"]\nq1, q3 = np.percentile(v19[I], [25, 75]); FENCE = q3 + 3.0 * (q3 - q1)\nTRANS = np.zeros(NF, bool); TRANS[I] = v19[I] > FENCE\nCLEANSET = full & ~TRANS\nprint("[cohort] full=%d | transient extremes=%d | clean=%d" % (full.sum(), TRANS.sum(), CLEANSET.sum()))\n\n# ---------------- reference trend g_ref(u) and the centred correction --------\nREF = D4B["per_date"]["Jun19"]["reference_by_scale"]\nfarm_sup = D4B["per_date"]["Jun19"]["farm"]["support"]\n\n\ndef corr_for_scale(S):\n    """c(u) for one aggregation scale, centred on the farm support, NEVER extrapolated."""\n    rec = REF[str(S)]\n    g = np.array(rec["curve"], float)                       # g_ref on UGRID\n    rsup = rec["support"]\n    lo = max(rsup[0], farm_sup[0]); hi = min(rsup[1], farm_sup[1])   # common support\n    inside = (UGRID >= lo) & (UGRID <= hi)\n    centre = float(np.mean(g[inside]))                      # centring: preserve Jun19 level\n    return g - centre, lo, hi, rec\n\n\nNOMc, LO, HI, NOMrec = corr_for_scale(PRIMARY_SCALE)\nprint("[ref] scale %d px | common support u in [%.4f, %.4f] | ref amplitude %.3f dB"\n      % (PRIMARY_SCALE, LO, HI, NOMrec["amplitude_db"]))\n\n# per-farm correction by interpolation on UGRID; farms outside the common\n# support are NOT extrapolated - they are marked and left uncorrected\nU_cl = np.clip(U, LO, HI)\nUNSUPPORTED = np.isfinite(U) & ((U < LO - 1e-9) | (U > HI + 1e-9))\nCORR = np.full(NF, np.nan)\nCORR[np.isfinite(U)] = np.interp(U_cl[np.isfinite(U)], UGRID, NOMc)\nCORR[UNSUPPORTED] = 0.0                                     # no extrapolation, no correction\nprint("[support] farms outside the common support: %d (correction set to 0, flagged)"\n      % int(UNSUPPORTED.sum()))\n\n# per-scale corrections -> uncertainty\nPER_SCALE = {}\nfor S in SCALES:\n    c_, lo_, hi_, _ = corr_for_scale(S)\n    uu = np.clip(U, lo_, hi_)\n    cc = np.full(NF, np.nan)\n    m_ = np.isfinite(U)\n    cc[m_] = np.interp(uu[m_], UGRID, c_)\n    cc[np.isfinite(U) & ((U < lo_ - 1e-9) | (U > hi_ + 1e-9))] = 0.0\n    PER_SCALE[S] = cc\nSTACK = np.vstack([PER_SCALE[S] for S in SCALES])\nCMIN = np.nanmin(STACK, axis=0); CMAX = np.nanmax(STACK, axis=0)\nCSD = np.nanstd(STACK, axis=0); CSPREAD = CMAX - CMIN\n\n# quality weight: 1 where the correction is scale-insensitive, decaying as the\n# scale disagreement grows. Never used to drop a farm or the date.\nW_SCALE = 1.0 / (1.0 + (CSPREAD / 0.5) ** 2)\nW = np.where(np.isfinite(W_SCALE), W_SCALE, 1.0)\nW[UNSUPPORTED] *= 0.5\n\nCLEAN = v19 - CORR\nprint("[corr] |c| dB: median %.3f  p95 %.3f  max %.3f | quality weight median %.3f"\n      % (np.nanmedian(np.abs(CORR[full])), np.nanpercentile(np.abs(CORR[full]), 95),\n         np.nanmax(np.abs(CORR[full])), np.nanmedian(W[full])))\n\n# robust companions (additional, never replacing raw)\nlo_w, hi_w = np.nanpercentile(CLEAN[CLEANSET], [1, 99])\nWINSOR = np.clip(CLEAN, lo_w, hi_w)\n\n\ndef quad(u, y):\n    A = np.column_stack([np.ones(u.size), u, u ** 2])\n    b, *_ = np.linalg.lstsq(A, y, rcond=None)\n    pred = A @ b\n    r2 = 1 - float(np.sum((y - pred) ** 2) / np.sum((y - y.mean()) ** 2))\n    lo, hi = float(u.min()), float(u.max())\n    ins = (UGRID >= lo) & (UGRID <= hi)\n    cur = b[0] + b[1] * UGRID + b[2] * UGRID ** 2\n    cs = cur[ins] if ins.sum() >= 2 else cur\n    return dict(r2=float(r2), amplitude_db=float(cs.max() - cs.min()),\n                coeffs=[float(x) for x in b]), pred\n\n\ndef morans_i(vals, xx, yy, k=8):\n    n = vals.size\n    d2 = (xx[:, None] - xx[None, :]) ** 2 + (yy[:, None] - yy[None, :]) ** 2\n    np.fill_diagonal(d2, np.inf)\n    nn = np.argsort(d2, axis=1)[:, :k]\n    z = vals - vals.mean()\n    num = float(sum(z[i] * z[nn[i]].mean() for i in range(n)))\n    return float(num / (float(np.sum(z ** 2)) / n) / n), float(-1.0 / (n - 1))\n\n\n# ============================ VALIDATION =====================================\nC = np.where(CLEANSET)[0]\nuC, rawC, clnC = U[C], v19[C], CLEAN[C]\nxC, yC = XM[C] - np.nanmean(XM[C]), YM[C] - np.nanmean(YM[C])\n\n# --- A. range artifact suppression -------------------------------------------\nqa_before, _ = quad(uC, rawC)\nqa_after, _ = quad(uC, clnC)\nA_ = dict(r2_before=round(qa_before["r2"], 4), r2_after=round(qa_after["r2"], 4),\n          ramp_amplitude_before_db=round(qa_before["amplitude_db"], 4),\n          ramp_amplitude_after_db=round(qa_after["amplitude_db"], 4),\n          note=("The correction is reference-derived and deliberately conservative; R2 is "\n                "REPORTED, never minimised. A residual range association is expected because "\n                "the farm ramp legitimately exceeds the acquisition-common part."))\nprint("\\n[A] range R2 %.4f -> %.4f | ramp %.3f -> %.3f dB"\n      % (A_["r2_before"], A_["r2_after"], A_["ramp_amplitude_before_db"], A_["ramp_amplitude_after_db"]))\n\n# --- B. preserve real structure ----------------------------------------------\nmi_raw, ei = morans_i(rawC, xC, yC); mi_cln, _ = morans_i(clnC, xC, yC)\n_, pr_raw = quad(uC, rawC); _, pr_cln = quad(uC, clnC)\nres_raw, res_cln = rawC - pr_raw, clnC - pr_cln\nmi_res_raw, _ = morans_i(res_raw, xC, yC); mi_res_cln, _ = morans_i(res_cln, xC, yC)\n\n\ndef dir_sweep(vals):\n    best = (-1, 0.0)\n    for th in np.arange(0, 180, 0.5):\n        t = math.radians(th)\n        s = xC * math.cos(t) + yC * math.sin(t)\n        s = (s - s.mean()) / (s.std() + 1e-12)\n        A = np.column_stack([np.ones(s.size), s, s ** 2])\n        b, *_ = np.linalg.lstsq(A, vals, rcond=None)\n        r2 = 1 - float(np.sum((vals - A @ b) ** 2) / np.sum((vals - vals.mean()) ** 2))\n        if r2 > best[0]: best = (r2, th)\n    return best\n\n\nr2d_raw, th_raw = dir_sweep(res_raw); r2d_cln, th_cln = dir_sweep(res_cln)\nB_ = dict(morans_i_raw=round(mi_raw, 4), morans_i_clean=round(mi_cln, 4),\n          morans_i_expected=round(ei, 6),\n          morans_i_residual_raw=round(mi_res_raw, 4), morans_i_residual_clean=round(mi_res_cln, 4),\n          best_residual_direction_raw_deg=th_raw, residual_directional_r2_raw=round(r2d_raw, 4),\n          best_residual_direction_clean_deg=th_cln, residual_directional_r2_clean=round(r2d_cln, 4),\n          overcorrection_check=("PASS - coherent spatial structure survives"\n                                if mi_cln > 0.05 else "INVESTIGATE - spatial structure collapsed"))\nprint("[B] Moran\'s I raw %.4f -> clean %.4f (E=%.4f) | residual dir R2 %.4f@%.1f -> %.4f@%.1f | %s"\n      % (mi_raw, mi_cln, ei, r2d_raw, th_raw, r2d_cln, th_cln, B_["overcorrection_check"]))\n\n# --- C. preserve between-farm information ------------------------------------\nsd_raw, sd_cln = float(np.std(rawC)), float(np.std(clnC))\nC_ = dict(sd_raw_db=round(sd_raw, 4), sd_clean_db=round(sd_cln, 4),\n          variance_retained_pct=round(100.0 * (sd_cln ** 2) / (sd_raw ** 2), 2),\n          pearson_raw_vs_clean=round(float(pearsonr(rawC, clnC)[0]), 5),\n          spearman_raw_vs_clean=round(float(spearmanr(rawC, clnC).statistic), 5))\nprint("[C] SD %.3f -> %.3f dB (%.1f%% variance retained) | Pearson %.4f Spearman %.4f"\n      % (sd_raw, sd_cln, C_["variance_retained_pct"], C_["pearson_raw_vs_clean"],\n         C_["spearman_raw_vs_clean"]))\n\n# --- D. crop-wise preservation (PROVISIONAL labels - see caveat) -------------\nD_ = dict(performed=False)\nif os.path.exists(SUBP):\n    lab = {int(r["farm_id"]): r["crop_type"] for r in csv.DictReader(open(SUBP, encoding="utf-8"))}\n    cl = np.array([lab.get(int(t), "") for t in fid])\n    per = {}\n    for cname in sorted(set(cl[C]) - {""}):\n        m = C[cl[C] == cname]\n        if m.size < 5: continue\n        per[cname] = dict(n=int(m.size),\n                          median_raw=round(float(np.median(v19[m])), 3),\n                          median_clean=round(float(np.median(CLEAN[m])), 3),\n                          iqr_raw=round(float(np.subtract(*np.percentile(v19[m], [75, 25]))), 3),\n                          iqr_clean=round(float(np.subtract(*np.percentile(CLEAN[m], [75, 25]))), 3))\n    ordr = sorted(per, key=lambda k: per[k]["median_raw"])\n    ordc = sorted(per, key=lambda k: per[k]["median_clean"])\n    # separability: ratio of between-class to within-class variance\n    def sep(vals):\n        gs = [vals[C[cl[C] == c]] for c in per]\n        gm = np.concatenate(gs).mean()\n        bw = sum(g.size * (g.mean() - gm) ** 2 for g in gs)\n        wi = sum(((g - g.mean()) ** 2).sum() for g in gs)\n        return float(bw / max(wi, 1e-12))\n    D_ = dict(performed=True, per_crop=per,\n              ordering_raw=ordr, ordering_clean=ordc,\n              ordering_preserved=bool(ordr == ordc),\n              separability_raw=round(sep(v19), 5), separability_clean=round(sep(CLEAN), 5),\n              LABEL_CAVEAT=("These crop_type values come from round2/submission_v1.csv, a "\n                            "PROVISIONAL self-generated assignment from a superseded pipeline "\n                            "whose crop stage was independently weak. They are NOT ground truth "\n                            "and NOT independent validation. This comparison shows only whether "\n                            "the correction reshuffles an existing weak grouping; it cannot "\n                            "confirm or refute crop separability. Round 2 supplies no crop labels."))\n    print("[D] crop ordering preserved: %s | separability %.4f -> %.4f  (PROVISIONAL labels only)"\n          % (D_["ordering_preserved"], D_["separability_raw"], D_["separability_clean"]))\n\n# --- E. temporal behaviour ----------------------------------------------------\ndef traj_feats(j19):\n    """Full four-date trajectory descriptors. Jun19 always participates."""\n    a, b, c_, d_ = sig["Jun06"], j19, sig["Aug14"], sig["Oct13"]\n    M = np.vstack([a, b, c_, d_])\n    med3 = np.nanmedian(np.vstack([a, c_, d_]), 0)\n    mad3 = np.nanmedian(np.abs(np.vstack([a, c_, d_]) - med3), 0)\n    dev = (b - med3) / (1.4826 * mad3 + 1e-6)                 # standardised Jun19 deviation\n    hub = np.clip(dev, -1.345, 1.345) * (1.4826 * mad3 + 1e-6) + med3   # Huberised Jun19\n    return dict(delta_Jun06_Jun19=b - a, delta_Jun19_Aug14=c_ - b, delta_Aug14_Oct13=d_ - c_,\n                early_slope=(b - a) / 13.0, later_slope=(d_ - c_) / 60.0,\n                curvature=(c_ - b) - (b - a),\n                traj_range=np.nanmax(M, 0) - np.nanmin(M, 0),\n                traj_mean=np.nanmean(M, 0), traj_sd=np.nanstd(M, 0),\n                temporal_rank_of_Jun19=(M < b).sum(0).astype(float),   # 0..3, order feature\n                argmax_date_index=np.where(np.all(np.isnan(M), 0), np.nan,\n                                           np.nanargmax(np.where(np.isnan(M), -1e30, M), 0)\n                                           ).astype(float),\n                robust_dev=np.abs(b - med3),\n                robust_dev_standardised=dev, huberised_Jun19=hub)\n\n\nTR, TC = traj_feats(v19), traj_feats(CLEAN)\nE_ = {}\nfor k in TR:\n    r_, c_ = TR[k][C], TC[k][C]\n    E_[k] = dict(sd_raw=round(float(np.nanstd(r_)), 4), sd_clean=round(float(np.nanstd(c_)), 4),\n                 median_raw=round(float(np.nanmedian(r_)), 4), median_clean=round(float(np.nanmedian(c_)), 4),\n                 spearman_raw_vs_clean=round(float(spearmanr(r_, c_).statistic), 4))\n# does the correction reduce the range-association OF THE TEMPORAL FEATURES?\nE_["range_association_of_deltas"] = {\n    k: dict(quad_r2_raw=round(quad(uC, TR[k][C])[0]["r2"], 4),\n            quad_r2_clean=round(quad(uC, TC[k][C])[0]["r2"], 4))\n    for k in ("delta_Jun06_Jun19", "delta_Jun19_Aug14", "curvature")}\nE_["note"] = ("\'Better\' is NOT smoother. Agronomic trajectories change sharply. The target is "\n              "removal of the spatially systematic acquisition bias from the Jun19-bearing "\n              "deltas while farm-specific temporal response is retained.")\nprint("[E] delta_Jun06_Jun19 range-R2 %.4f -> %.4f | delta_Jun19_Aug14 %.4f -> %.4f"\n      % (E_["range_association_of_deltas"]["delta_Jun06_Jun19"]["quad_r2_raw"],\n         E_["range_association_of_deltas"]["delta_Jun06_Jun19"]["quad_r2_clean"],\n         E_["range_association_of_deltas"]["delta_Jun19_Aug14"]["quad_r2_raw"],\n         E_["range_association_of_deltas"]["delta_Jun19_Aug14"]["quad_r2_clean"]))\n\n# --- F. three-date ablation (DIAGNOSTIC ONLY) --------------------------------\na, c3, d3 = sig["Jun06"][C], sig["Aug14"][C], sig["Oct13"][C]\nF_ = dict(purpose="DIAGNOSTIC ONLY - never a production alternative; Jun19 always stays",\n          three_date_features=["delta_Jun06_Aug14", "delta_Aug14_Oct13", "curvature_3pt"],\n          four_date_only_features=["delta_Jun06_Jun19", "delta_Jun19_Aug14",\n                                   "early_slope (13-day monsoon-onset response)",\n                                   "curvature about Jun19", "robust_dev vs the other three dates"],\n          n_features_3date=3, n_features_4date=8,\n          early_window_days_3date=69, early_window_days_4date=13,\n          information_added=dict(\n              sd_delta_Jun06_Aug14=round(float(np.nanstd(c3 - a)), 4),\n              sd_delta_Jun06_Jun19_clean=round(float(np.nanstd(TC["delta_Jun06_Jun19"][C])), 4),\n              spearman_Jun06Aug14_vs_Jun06Jun19clean=round(\n                  float(spearmanr(c3 - a, TC["delta_Jun06_Jun19"][C]).statistic), 4)),\n          interpretation=("The 13-day Jun06->Jun19 interval brackets monsoon onset. Without Jun19 "\n                          "the earliest resolvable interval is 69 days (Jun06->Aug14), which "\n                          "cannot separate the onset response from two months of canopy growth. "\n                          "The low rank correlation between the two early deltas shows they carry "\n                          "different information, so Jun19 is not redundant."))\nprint("[F] early interval 13 d (4-date) vs 69 d (3-date) | rank corr of the two early deltas %.3f"\n      % F_["information_added"]["spearman_Jun06Aug14_vs_Jun06Jun19clean"])\n\n# --- correction magnitude distribution ---------------------------------------\ncm = CORR[full]\nCM = {k: round(float(v), 4) for k, v in\n      dict(min=np.nanmin(cm), p05=np.nanpercentile(cm, 5), p25=np.nanpercentile(cm, 25),\n           median=np.nanmedian(cm), p75=np.nanpercentile(cm, 75),\n           p95=np.nanpercentile(cm, 95), max=np.nanmax(cm),\n           max_abs=np.nanmax(np.abs(cm))).items()}\nbig = C[np.abs(CORR[C]) > np.nanpercentile(np.abs(cm), 99)]\nprint("[mag] correction dB: min %.3f p05 %.3f med %.3f p95 %.3f max %.3f"\n      % (CM["min"], CM["p05"], CM["median"], CM["p95"], CM["max"]))\n\n# ============================ FIGURES ========================================\ndef sv(n):\n    plt.tight_layout(); plt.savefig(f"{FIG}/{n}", dpi=130); plt.close()\n\n\nins = (UGRID >= LO) & (UGRID <= HI)\nplt.figure(figsize=(8, 4.4))\nfor S in SCALES:\n    g = np.array(REF[str(S)]["curve"]); plt.plot(UGRID, g - g[ins].mean(), lw=1.2, label=f"reference {S}px")\ngf = np.array(D4B["per_date"]["Jun19"]["farm"]["curve"]); plt.plot(UGRID, gf - gf[ins].mean(), "k--", lw=2, label="farm trend")\nplt.axvspan(LO, HI, color="grey", alpha=.12, label="common support")\nplt.xlabel("normalised range u"); plt.ylabel("fitted trend, centred (dB)")\nplt.title("FIG 1-2  Jun19 reference range trends (all scales) vs the farm trend")\nplt.legend(fontsize=7); plt.grid(alpha=.3); sv("01_reference_and_farm_trends.png")\n\no = np.argsort(uC)\nplt.figure(figsize=(8, 4.4))\nplt.scatter(uC, rawC, s=9, alpha=.45, label="raw")\nplt.scatter(uC, clnC, s=9, alpha=.45, label="clean")\nplt.plot(uC[o], pr_raw[o], lw=2, color="tab:blue"); plt.plot(uC[o], pr_cln[o], lw=2, color="tab:orange")\nplt.xlabel("normalised range u"); plt.ylabel("sigma0 Jun19 (dB)")\nplt.title("FIG 3  farm trend before (R2=%.3f) and after (R2=%.3f) correction" % (qa_before["r2"], qa_after["r2"]))\nplt.legend(fontsize=8); plt.grid(alpha=.3); sv("03_farm_trend_before_after.png")\n\nfor nm, vals, t in (("04_spatial_raw.png", rawC, "FIG 4  Jun19 RAW"),\n                    ("05_spatial_clean.png", clnC, "FIG 5  Jun19 CLEAN"),\n                    ("06_correction_map.png", CORR[C], "FIG 6  correction magnitude c(u)"),\n                    ("07_residual_map.png", res_cln, "FIG 7  residual after range fit (clean)")):\n    plt.figure(figsize=(6, 5.2))\n    cmap = "coolwarm" if "correction" in t or "residual" in t else "viridis"\n    lim = np.percentile(np.abs(vals), 98) if cmap == "coolwarm" else None\n    sc = plt.scatter(xC, yC, c=vals, s=14, cmap=cmap,\n                     vmin=(-lim if lim else None), vmax=(lim if lim else None))\n    plt.colorbar(sc, label="dB"); plt.gca().set_aspect("equal")\n    plt.xlabel("easting - mean (m)"); plt.ylabel("northing - mean (m)"); plt.title(t); sv(nm)\n\nplt.figure(figsize=(5.4, 5.2))\nplt.scatter(rawC, clnC, s=10, alpha=.5)\nlim = [min(rawC.min(), clnC.min()), max(rawC.max(), clnC.max())]\nplt.plot(lim, lim, "k--", lw=1)\nplt.xlabel("Jun19 raw (dB)"); plt.ylabel("Jun19 clean (dB)")\nplt.title("FIG 8  raw vs clean (Spearman %.4f)" % C_["spearman_raw_vs_clean"])\nplt.grid(alpha=.3); sv("08_raw_vs_clean_scatter.png")\n\nplt.figure(figsize=(8, 4.4))\nplt.boxplot([sig["Jun06"][C], rawC, clnC, sig["Aug14"][C], sig["Oct13"][C]],\n            labels=["Jun06", "Jun19 raw", "Jun19 clean", "Aug14", "Oct13"],\n            showfliers=True, flierprops=dict(ms=2))\nplt.ylabel("sigma0 trim10 (dB)"); plt.title("FIG 9  four-date distributions, raw and clean Jun19")\nplt.grid(alpha=.3); sv("09_temporal_distributions.png")\n\nplt.figure(figsize=(8, 4.4))\nsel = C[np.argsort(np.abs(CORR[C]))[::-1][:6]]\nfor i_, s_ in enumerate(sel):\n    plt.plot([0, 13, 69, 129], [sig["Jun06"][s_], v19[s_], sig["Aug14"][s_], sig["Oct13"][s_]],\n             "o--", alpha=.5, color=f"C{i_}")\n    plt.plot([0, 13, 69, 129], [sig["Jun06"][s_], CLEAN[s_], sig["Aug14"][s_], sig["Oct13"][s_]],\n             "s-", color=f"C{i_}", label=f"farm {fid[s_]} (c={CORR[s_]:+.2f} dB)")\nplt.xlabel("days from Jun06"); plt.ylabel("sigma0 (dB)")\nplt.title("FIG 10  trajectories, dashed = raw Jun19, solid = clean Jun19 (6 largest corrections)")\nplt.legend(fontsize=7); plt.grid(alpha=.3); sv("10_trajectories.png")\nprint("[fig] 10 figures -> %s" % FIG)\n\n# ============================ WRITE DERIVED CSV ==============================\nhdr = list(rows[0].keys())\nnew = ["u_range_norm_Jun19", "sigma0_Jun19_raw", "sigma0_Jun19_clean",\n       "sigma0_Jun19_clean_winsor", "sigma0_Jun19_clean_huber", "jun19_transient_extreme",\n       "jun19_correction_nominal_db", "jun19_correction_magnitude_db",\n       "jun19_correction_min_db", "jun19_correction_max_db", "jun19_correction_spread_db",\n       "jun19_correction_sd_db", "jun19_quality_weight", "jun19_outside_common_support",\n       "delta_Jun06_Jun19_raw", "delta_Jun06_Jun19_clean",\n       "delta_Jun19_Aug14_raw", "delta_Jun19_Aug14_clean", "delta_Aug14_Oct13",\n       "early_slope_raw", "early_slope_clean", "later_slope",\n       "temporal_curvature_raw", "temporal_curvature_clean",\n       "temporal_rank_Jun19_raw", "temporal_rank_Jun19_clean",\n       "trajectory_range_raw", "trajectory_range_clean",\n       "robust_temporal_dev_raw", "robust_temporal_dev_clean",\n       "robust_temporal_dev_std_raw", "robust_temporal_dev_std_clean"]\nwith open(CSV_OUT, "w", newline="", encoding="utf-8") as fh:\n    w = csv.DictWriter(fh, fieldnames=hdr + new, extrasaction="ignore"); w.writeheader()\n    for i_, r in enumerate(rows):\n        d = dict(r)\n        def g_(x): return "" if not np.isfinite(x) else round(float(x), 5)\n        d.update({"u_range_norm_Jun19": g_(U[i_]), "sigma0_Jun19_raw": g_(v19[i_]),\n                  "sigma0_Jun19_clean": g_(CLEAN[i_]), "sigma0_Jun19_clean_winsor": g_(WINSOR[i_]),\n                  "jun19_transient_extreme": int(TRANS[i_]),\n                  "jun19_correction_magnitude_db": g_(CORR[i_]),\n                  "jun19_correction_min_db": g_(CMIN[i_]), "jun19_correction_max_db": g_(CMAX[i_]),\n                  "jun19_correction_sd_db": g_(CSD[i_]), "jun19_quality_weight": g_(W[i_]),\n                  "jun19_outside_common_support": int(UNSUPPORTED[i_]),\n                  "delta_Jun06_Jun19_raw": g_(TR["delta_Jun06_Jun19"][i_]),\n                  "delta_Jun06_Jun19_clean": g_(TC["delta_Jun06_Jun19"][i_]),\n                  "delta_Jun19_Aug14_raw": g_(TR["delta_Jun19_Aug14"][i_]),\n                  "delta_Jun19_Aug14_clean": g_(TC["delta_Jun19_Aug14"][i_]),\n                  "delta_Aug14_Oct13": g_(TR["delta_Aug14_Oct13"][i_]),\n                  "temporal_curvature_raw": g_(TR["curvature"][i_]),\n                  "temporal_curvature_clean": g_(TC["curvature"][i_]),\n                  "sigma0_Jun19_clean_huber": g_(TC["huberised_Jun19"][i_]),\n                  "jun19_correction_nominal_db": g_(CORR[i_]),\n                  "jun19_correction_spread_db": g_(CSPREAD[i_]),\n                  "early_slope_raw": g_(TR["early_slope"][i_]),\n                  "early_slope_clean": g_(TC["early_slope"][i_]),\n                  "later_slope": g_(TR["later_slope"][i_]),\n                  "temporal_rank_Jun19_raw": g_(TR["temporal_rank_of_Jun19"][i_]),\n                  "temporal_rank_Jun19_clean": g_(TC["temporal_rank_of_Jun19"][i_]),\n                  "trajectory_range_raw": g_(TR["traj_range"][i_]),\n                  "trajectory_range_clean": g_(TC["traj_range"][i_]),\n                  "robust_temporal_dev_raw": g_(TR["robust_dev"][i_]),\n                  "robust_temporal_dev_clean": g_(TC["robust_dev"][i_]),\n                  "robust_temporal_dev_std_raw": g_(TR["robust_dev_standardised"][i_]),\n                  "robust_temporal_dev_std_clean": g_(TC["robust_dev_standardised"][i_])})\n        w.writerow(d)\n\nLOG = dict(\n    purpose="reference-derived, conservative Jun19 range correction; raw always preserved",\n    physical_interpretation="MIXED_EFFECT",\n    physical_interpretation_statement=(\n        "We do NOT claim Jun19 is an instrumental artefact. We claim Jun19 contains a strong "\n        "acquisition-common range-dependent component superimposed on additional spatial and "\n        "agronomic variation. That is the scientific basis for correcting ONLY the "\n        "independently supported common range component."),\n    acquisition_attribution=dict(\n        classification="LIKELY_RANGE_LOCK",\n        scope="describes the acquisition-common component ONLY, not the whole Jun19 response",\n        evidence=dict(reference_ramp_db=2.3297, farm_ramp_db=3.6179,\n                      reference_over_farm_ratio=0.6439,\n                      ratio_by_scale={"16": 1.003, "32": 0.729, "64": 0.644, "96": 0.610},\n                      sign_orientation_matches=True, fitted_curve_correlation=0.9936,\n                      jun19_excess_over_other_date_references_db=1.3826,\n                      range_axis_bearing_deg=134.16, best_axis_bearing_deg=144.0,\n                      angular_difference_deg=9.84, delta_r2_vs_range=0.0105,\n                      bootstrap_within_15deg_fraction=0.9567)),\n    residual_geographic_component=dict(\n        present=True, morans_i=0.1772, morans_i_expected=-0.0011,\n        best_direction_deg=20.5, angular_difference_from_range_deg=66.34,\n        directional_r2=0.0337,\n        interpretation=("Coherent spatial structure survives the range fit and is oriented far "\n                        "from the radar range axis, so NOT all Jun19 spatial structure is "\n                        "instrumental. This component is deliberately RETAINED - the correction "\n                        "removes only the reference-supported common range term."),\n        retained=True),\n    input_provenance=PROV,\n    source_csv=dict(path=CSV_IN, sha256=hashlib.sha256(open(CSV_IN, "rb").read()).hexdigest()),\n    output_csv=dict(path=CSV_OUT, sha256=hashlib.sha256(open(CSV_OUT, "rb").read()).hexdigest(),\n                    bytes=os.path.getsize(CSV_OUT), n_rows=NF),\n    frozen_artifact_untouched=True,\n    correction=dict(\n        formula="sigma0_Jun19_clean = sigma0_Jun19_raw - c(u_farm)",\n        c_of_u="c(u) = g_ref(u) - mean[g_ref(u) over the common farm/reference support]",\n        g_ref="fitted Jun19 NON-FARM reference quadratic, b0 + b1*u + b2*u^2",\n        reference_coefficients=NOMrec["coeffs"] if "coeffs" in NOMrec else None,\n        reference_curve_on_ugrid=NOMrec["curve"],\n        normalised_range_definition="u = (SLC column of the farm centroid - %.4f) / %.4f" % (lo_c, span),\n        range_axis_bearing_deg=D4["range_model"]["axis_bearing_deg"],\n        rpc_height_m=H,\n        common_support_u=[round(LO, 4), round(HI, 4)],\n        farm_support_u=farm_sup, reference_support_u=NOMrec["support"],\n        aggregation_scale_px=PRIMARY_SCALE, scale_selection_rationale=SELECTION_RATIONALE,\n        nominal_reference_amplitude_db=NOMrec["amplitude_db"],\n        centering_method=("subtract the mean of g_ref over the common support, so the Jun19 "\n                          "overall level is preserved; neither mean nor variance is matched "\n                          "to any other date"),\n        extrapolation_policy=("u is clipped to the common support; farms outside it get zero "\n                              "correction and jun19_outside_common_support = 1. The quadratic "\n                              "is never evaluated beyond observed support."),\n        n_farms_outside_support=int(UNSUPPORTED.sum())),\n    correction_uncertainty=dict(\n        scales_px=SCALES,\n        reference_amplitude_by_scale={str(S): REF[str(S)]["amplitude_db"] for S in SCALES},\n        per_farm_spread_db=dict(median=round(float(np.nanmedian(CSPREAD[full])), 4),\n                                p95=round(float(np.nanpercentile(CSPREAD[full], 95)), 4),\n                                max=round(float(np.nanmax(CSPREAD[full])), 4)),\n        quality_weight_formula="w = 1 / (1 + (scale_spread_db / 0.5)^2); halved outside support",\n        quality_weight=dict(min=round(float(np.nanmin(W[full])), 4),\n                            median=round(float(np.nanmedian(W[full])), 4),\n                            max=round(float(np.nanmax(W[full])), 4)),\n        note="uncertainty is NEVER used to drop farms or the date"),\n    correction_magnitude_db=CM,\n    largest_corrections=[dict(farm_id=int(fid[i_]), u=round(float(U[i_]), 4),\n                              correction_db=round(float(CORR[i_]), 4)) for i_ in big[:10]],\n    validation=dict(A_range_suppression=A_, B_structure_preserved=B_,\n                    C_between_farm_information=C_, D_cropwise=D_,\n                    E_temporal=E_, F_three_date_ablation=F_),\n    transient_extremes=dict(n=int(TRANS.sum()),\n                            farm_ids=[int(x) for x in fid[TRANS]],\n                            policy=("raw preserved exactly; excluded from every fit; correction "\n                                    "derived from the range coordinate ONLY, never from their "\n                                    "anomalous magnitude; winsorised companion provided as an "\n                                    "ADDITIONAL column")),\n    ablation_representations=dict(\n        RAW="sigma0_Jun19_raw + raw deltas",\n        CLEAN="sigma0_Jun19_clean + clean deltas",\n        RAW_PLUS_CLEAN=("both, plus jun19_transient_extreme, jun19_correction_magnitude_db "\n                        "and jun19_quality_weight"),\n        recommendation="RAW_PLUS_CLEAN",\n        recommendation_basis=("keeps every observation and lets the later health/yield model "\n                              "decide empirically; this diagnostic does not settle it")),\n    mandatory_policy=dict(all_four_dates_retained=True, jun19_raw_overwritten=False,\n                          transient_farms_deleted=False, residual_non_range_structure_removed=False),\n    apy_anchors_verified=dict(Rice=2068, Bajra=1815, Maize=1770, Cotton_lint=846, Groundnut=2579,\n                              units="kg/ha, Vadodara district, kharif blocks, 2016-17..2024-25",\n                              parser_fix=("the earlier Maize 562 kg/ha was parser-state "\n                                          "contamination: block headers were detected by crop "\n                                          "vocabulary so non-target blocks (KHARIF RAGI etc.) did "\n                                          "not reset state. Headers now key on the unit "\n                                          "annotation. Not an agronomic observation.")))\njson.dump(LOG, open(f"{OUT}/jun19_clean_validation.json", "w"), indent=1)\nprint("\\nwrote %s (%d rows, %d new cols)" % (CSV_OUT, NF, len(new)))\nprint("wrote %s/jun19_clean_validation.json" % OUT)\nprint("ALL FOUR DATES RETAINED | raw preserved | %d transient farms flagged, none deleted"\n      % TRANS.sum())\n'
Path(SRC/"build_jun19_clean.py").write_text(_SRC, encoding="utf-8")
print("wrote src/build_jun19_clean.py  ({} lines)".format(_SRC.count(chr(10))))

In [ ]:
T["S06"], _ = run_stage("diag_e1_jun19_contamination.py",
                        ["_diag/jun19_contamination_map.json"], "S06 E1 contamination")
T["S07"], out7 = run_stage("build_jun19_clean.py",
                           ["farm_features_jun19clean_v1.csv"], "S07 Jun19 correction")

### OUTPUT / VALIDATION -- computed here, not asserted from memory
cm = json.loads((ROOT / "_diag/jun19_contamination_map.json").read_text(encoding="utf-8"))
inval = cm.get("invalid_farm_ids") or cm.get("validity", {}).get("invalid_farm_ids") or []
print(f"\nJun19 invalid high-backscatter farms (>=20% px > -5 dB): {len(inval)}")
key = [l for l in out7.split("\n") if any(k in l for k in
       ("range R2", "range_r2", "SD", "ramp", "Moran", "retained", "Pearson", "Spearman"))]
print("\n".join(key[:12]))

---
## 5 · Structural reliability and evidence pathways  ·  `[P] S08 · S12`

### WHY
Median farm area is 0.27 ha. At Sentinel/Capella resolution a median field is a handful of
independent looks, so any statistic sensitive to sample size can masquerade as a crop signal —
a farm-size artifact defeated three separate crop-discrimination attempts in this project.
Texture is therefore **reliability-gated secondary evidence**, applied only where pixel support
justifies it.

### METHOD
Effective sample support (`n_eff = n_valid / 1.80`), multilook-style stability, a size/reliability
gate, sliver/degenerate handling, and five explicit **evidence pathways**. Missingness is routed,
never imputed into false confidence: a farm with no usable coverage is flagged, not invented.

*(S12 runs after S10 because integration also checks S1/S2 availability.)*

In [ ]:
# S08 -- E2 structure / reliability features with size gating.
_SRC = '"""\nE2 — Within-field structure: what is measurable, what is speckle, what is\n     sample-size, and what adds information beyond level and trajectory?\n\nWHY THIS IS NOT TRIVIAL AT ENL = 1\n  Capella SLC is single-look. For single-look intensity the speckle coefficient\n  of variation is CV = 1/sqrt(ENL) = 1.0 EXACTLY, regardless of the scene. So a\n  raw within-field CV is dominated by speckle and mostly measures nothing about\n  the field. The multiplicative speckle model gives\n        CV_obs^2 = CV_texture^2 + CV_speckle^2 + CV_texture^2 * CV_speckle^2\n  hence the debiased texture estimate\n        CV_texture = sqrt( max( (CV_obs^2 - 1/ENL) / (1 + 1/ENL), 0 ) )\n  This is the quantity to test, not raw CV. KB Q2 corroborates ENL as the\n  standard speckle-level measure (`Retrieving Surface Soil Moisture over Wheat\n  and Soybean Fields`) and the sample-size requirement\n  (`Structure detection and statistical adaptive speckle filtering in SAR\n  images`: a one-look image needs ~230 independent samples for 1 dB precision).\n\n  A prior attempt at within-field uniformity already FAILED in this project:\n  METHODOLOGY.md:48 records SAR CV vs NDVI CV, r = +0.047. That null is the\n  thing to beat, and raw CV at ENL=1 is very likely why it failed.\n\nWHAT IS COMPUTED, per farm per date\n  level-free dispersion : sd_db, iqr_db, mad_db, p90_minus_p10_db\n  speckle-aware         : cv_lin, cv_texture (debiased), enl_effective\n  multilook heterogen.  : cv of a 3x3 boxcar multilooked field (ENL ~ 9)\n  spatial structure     : lag-1 autocorrelation of the multilooked field\n  geometry contrast     : interior-minus-edge mean dB (1-pixel erosion)\n\nRELIABILITY TESTING (the point of the exercise)\n  Each farm is split into two SPATIAL halves; every feature is computed on both\n  halves independently. Cross-farm correlation of half-A against half-B is the\n  reliability. A feature that cannot reproduce itself on two halves of the same\n  field cannot carry information about that field.\n\nGATES (preregistered heuristics, not sacred)\n  reliability rho >= 0.5  AND  nuisance R^2 < 0.15\n  A feature that narrowly misses is reported, not silently dropped, so it can be\n  judged on physical grounds. A feature that is wildly unstable is dropped even\n  if the rubric mentions texture.\n\nReads Capella via the fail-fast provenance resolver, reuses the frozen V3.2.1\ngeometry conventions. Writes:\n  round2/farm_features_structure_v1.csv\n  round2/_diag/structure_reliability.json\n"""\nimport os, sys, json, csv, math, time, hashlib, platform, warnings\nimport numpy as np\nfrom scipy.stats import spearmanr, pearsonr\nimport shapefile\nimport rasterio\nfrom rasterio.windows import Window\nfrom rasterio.transform import RPCTransformer\nfrom matplotlib.path import Path as MplPath\nfrom shapely.geometry import shape as gshape, MultiPolygon\nfrom shapely.ops import transform as shp_transform\nfrom pyproj import Transformer\n\nsys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\nfrom _provenance import resolve_all, print_table, slc_path, meta_path, TAGS\n\nwarnings.filterwarnings("ignore")\nT0 = time.time()\nCSV_IN = "round2/farm_features_jun19clean_v1.csv"\nLOGP = "round2/_geom321/geometry_rebuild_v321_log.json"\nSHP = "round2/data/Farm_boundaries_shp/Farm_boundaries_shp/Sokhda_Farms.shp"\nE1P = "round2/_diag/jun19_contamination_map.json"\nCSV_OUT = "round2/farm_features_structure_v1.csv"\nOUT = "round2/_diag"\nDATES = ["Jun06", "Jun19", "Aug14", "Oct13"]\nIDENTITY = (lambda v: v)\nMIN_COMP_M2 = 25.0\nGATE_PX = 30\nENL = 1.0                      # Capella SLC, single look (V3.2.1 records ENL=1)\nMET = "EPSG:32643"\n\nprint("=" * 94)\nprint("E2  within-field structure: measurability, speckle, sample-size, complementarity")\nprint("=" * 94)\nPROV = resolve_all(DATES); print_table(PROV)\nH = float(json.load(open(LOGP))["shared_height"]["selected"])\n\nrows = list(csv.DictReader(open(CSV_IN, encoding="utf-8")))\nfid = np.array([int(r["farm_id"]) for r in rows]); NF = len(rows)\nIDX = {int(f): i for i, f in enumerate(fid)}\n\n\ndef fl(v):\n    try: return float(v)\n    except (TypeError, ValueError): return float("nan")\n\n\nlev = {d: np.array([fl(r[f"sigma0_db_trim10_{d}"]) for r in rows]) for d in DATES}\ngate = {d: np.array([fl(r[f"gate_pass_{d}"]) == 1 for r in rows]) for d in DATES}\narea = np.array([fl(r["area_ha"]) for r in rows])\ninc = {d: np.array([fl(r[f"incidence_deg_{d}"]) for r in rows]) for d in DATES}\njun19_invalid = set(json.load(open(E1P))["validity"]["invalid_farm_ids"])\n\nto_m = Transformer.from_crs("EPSG:4326", MET, always_xy=True).transform\nto_ll = Transformer.from_crs(MET, "EPSG:4326", always_xy=True).transform\nsf = shapefile.Reader(SHP); recs, shps = sf.records(), sf.shapes()\norder = np.argsort([int(r[0]) for r in recs])\nFIDS = [int(recs[i][0]) for i in order]\nGEOM = {}\nfor k, i in enumerate(order):\n    g = gshape(shps[i].__geo_interface__)\n    if not g.is_valid: g = g.buffer(0)\n    gm = shp_transform(to_m, g)\n    parts = [p for p in (list(gm.geoms) if gm.geom_type == "MultiPolygon" else [gm]) if p.area >= MIN_COMP_M2]\n    if not parts: continue\n    um = MultiPolygon(parts) if len(parts) > 1 else parts[0]\n    GEOM[FIDS[k]] = [shp_transform(to_ll, p) for p in parts]\n\n\ndef cv_texture(lin):\n    """Speckle-debiased texture CV under the multiplicative model."""\n    m = float(np.mean(lin))\n    if m <= 0: return float("nan"), float("nan")\n    cv2 = float(np.var(lin)) / (m * m)\n    enl_eff = 1.0 / cv2 if cv2 > 0 else float("nan")\n    t2 = (cv2 - 1.0 / ENL) / (1.0 + 1.0 / ENL)\n    return math.sqrt(max(t2, 0.0)), enl_eff\n\n\ndef feats_from(lin, rr, cc):\n    """All structure features from one pixel set (linear power + coords)."""\n    if lin.size < GATE_PX: return None\n    db = 10 * np.log10(np.maximum(lin, 1e-15))\n    f = {}\n    f["sd_db"] = float(np.std(db))\n    q1, q3 = np.percentile(db, [25, 75]); f["iqr_db"] = float(q3 - q1)\n    f["mad_db"] = float(np.median(np.abs(db - np.median(db))))\n    f["p90_minus_p10_db"] = float(np.percentile(db, 90) - np.percentile(db, 10))\n    mu = float(np.mean(lin))\n    f["cv_lin"] = float(np.std(lin) / mu) if mu > 0 else float("nan")\n    ct, enl_eff = cv_texture(lin)\n    f["cv_texture"] = ct\n    f["enl_effective"] = enl_eff\n    # 3x3 boxcar multilook on the farm\'s own pixel grid, then heterogeneity\n    r0, c0 = rr.min(), cc.min()\n    hgt, wid = rr.max() - r0 + 1, cc.max() - c0 + 1\n    if hgt >= 3 and wid >= 3 and hgt * wid < 4_000_000:\n        grid = np.full((hgt, wid), np.nan)\n        grid[rr - r0, cc - c0] = lin\n        k = 3\n        pad = np.pad(grid, k // 2, constant_values=np.nan)\n        acc = np.zeros_like(grid); cnt = np.zeros_like(grid)\n        for a in range(k):\n            for b in range(k):\n                w = pad[a:a + hgt, b:b + wid]\n                m_ = np.isfinite(w)\n                acc[m_] += w[m_]; cnt[m_] += 1\n        ml = np.where(cnt >= 5, acc / np.maximum(cnt, 1), np.nan)\n        v = ml[np.isfinite(ml)]\n        if v.size >= GATE_PX:\n            mm = float(np.mean(v))\n            f["ml3_cv"] = float(np.std(v) / mm) if mm > 0 else float("nan")\n            f["ml3_sd_db"] = float(np.std(10 * np.log10(np.maximum(v, 1e-15))))\n            # lag-1 spatial autocorrelation of the multilooked field, row direction\n            a1 = ml[:, :-1]; a2 = ml[:, 1:]\n            ok = np.isfinite(a1) & np.isfinite(a2)\n            f["ml3_lag1_autocorr"] = (float(np.corrcoef(a1[ok], a2[ok])[0, 1])\n                                      if ok.sum() > 30 else float("nan"))\n        else:\n            f["ml3_cv"] = f["ml3_sd_db"] = f["ml3_lag1_autocorr"] = float("nan")\n    else:\n        f["ml3_cv"] = f["ml3_sd_db"] = f["ml3_lag1_autocorr"] = float("nan")\n    return f\n\n\nFEATNAMES = ["sd_db", "iqr_db", "mad_db", "p90_minus_p10_db", "cv_lin", "cv_texture",\n             "enl_effective", "ml3_cv", "ml3_sd_db", "ml3_lag1_autocorr"]\n\nSTRUCT = {d: {f: np.full(NF, np.nan) for f in FEATNAMES + ["interior_minus_edge_db"]} for d in DATES}\nHALF = {d: {f: (np.full(NF, np.nan), np.full(NF, np.nan)) for f in FEATNAMES} for d in DATES}\n\nfor d in DATES:\n    tag = TAGS[d]\n    scale = json.load(open(meta_path(tag)))["collect"]["image"]["scale_factor"]\n    with rasterio.open(slc_path(tag)) as s_:\n        with RPCTransformer(s_.rpcs, RPC_HEIGHT=H) as tr:\n            allr, allc = [], []\n            proj = {}\n            for f_, polys in GEOM.items():\n                comps = []\n                for p in polys:\n                    e = np.asarray(p.exterior.coords, float)\n                    a, b = tr.rowcol(e[:, 0].tolist(), e[:, 1].tolist(), op=IDENTITY)\n                    a = np.asarray(a, float); b = np.asarray(b, float)\n                    comps.append(np.column_stack([b, a]))\n                    allr.append(a); allc.append(b)\n                proj[f_] = comps\n            AR = np.concatenate(allr); AC = np.concatenate(allc)\n            r0 = max(int(AR.min()) - 4, 0); r1 = min(int(AR.max()) + 4, s_.height)\n            c0 = max(int(AC.min()) - 4, 0); c1 = min(int(AC.max()) + 4, s_.width)\n            blk = s_.read(1, window=Window(c0, r0, c1 - c0, r1 - r0))\n    b0 = (np.abs(blk).astype(np.float32) * scale) ** 2\n    b0[b0 <= 0] = np.nan\n    del blk\n    nf_done = 0\n    for f_, comps in proj.items():\n        i = IDX.get(f_)\n        if i is None: continue\n        xs = np.concatenate([c[:, 0] for c in comps]); ys = np.concatenate([c[:, 1] for c in comps])\n        cmin = max(0, int(np.floor(xs.min())) - c0); cmax = min(b0.shape[1], int(np.ceil(xs.max())) - c0 + 1)\n        rmin = max(0, int(np.floor(ys.min())) - r0); rmax = min(b0.shape[0], int(np.ceil(ys.max())) - r0 + 1)\n        if rmax <= rmin or cmax <= cmin: continue\n        gr, gc = np.mgrid[rmin:rmax, cmin:cmax]\n        pts = np.column_stack([gc.ravel() + c0 + 0.5, gr.ravel() + r0 + 0.5])\n        ins = np.zeros(pts.shape[0], bool)\n        for c_ in comps: ins |= MplPath(c_).contains_points(pts)\n        if not ins.any(): continue\n        v = b0[rmin:rmax, cmin:cmax].ravel()[ins]\n        rr = gr.ravel()[ins] + r0; cc = gc.ravel()[ins] + c0\n        ok = np.isfinite(v)\n        v, rr, cc = v[ok], rr[ok], cc[ok]\n        if v.size < GATE_PX: continue\n        F = feats_from(v, rr, cc)\n        if F:\n            for kk, vv in F.items(): STRUCT[d][kk][i] = vv\n        # interior vs edge: pixels whose 4-neighbours are all inside\n        key = set(zip(rr.tolist(), cc.tolist()))\n        interior = np.array([((a + 1, b) in key) and ((a - 1, b) in key) and\n                             ((a, b + 1) in key) and ((a, b - 1) in key)\n                             for a, b in zip(rr.tolist(), cc.tolist())])\n        if interior.sum() >= GATE_PX and (~interior).sum() >= 10:\n            di = 10 * math.log10(max(float(np.mean(v[interior])), 1e-15))\n            de = 10 * math.log10(max(float(np.mean(v[~interior])), 1e-15))\n            STRUCT[d]["interior_minus_edge_db"][i] = di - de\n        # spatial half split for reliability\n        span_r = rr.max() - rr.min(); span_c = cc.max() - cc.min()\n        m1 = (rr < np.median(rr)) if span_r >= span_c else (cc < np.median(cc))\n        for lab, mm in ((0, m1), (1, ~m1)):\n            if mm.sum() >= GATE_PX:\n                Fh = feats_from(v[mm], rr[mm], cc[mm])\n                if Fh:\n                    for kk in FEATNAMES:\n                        HALF[d][kk][lab][i] = Fh.get(kk, np.nan)\n        nf_done += 1\n    print("[%s] structure computed for %d farms  (%.0fs elapsed)" % (d, nf_done, time.time() - T0), flush=True)\n\n# ---------------- reliability, nuisance, complementarity ---------------------\nreport = {}\nfor d in DATES:\n    good = gate[d].copy()\n    if d == "Jun19":\n        for f_ in jun19_invalid:\n            if f_ in IDX: good[IDX[f_]] = False\n    rep = {}\n    for f in FEATNAMES + ["interior_minus_edge_db"]:\n        x = STRUCT[d][f]\n        m = good & np.isfinite(x)\n        if m.sum() < 50:\n            rep[f] = dict(n=int(m.sum()), note="insufficient"); continue\n        # reliability: split-half across farms\n        if f in FEATNAMES:\n            a, b = HALF[d][f]\n            mm = m & np.isfinite(a) & np.isfinite(b)\n            rel = float(spearmanr(a[mm], b[mm]).statistic) if mm.sum() > 50 else float("nan")\n            n_rel = int(mm.sum())\n        else:\n            rel, n_rel = float("nan"), 0\n        # nuisance: area, n_valid proxy, incidence\n        nv = np.array([fl(r[f"n_valid_{d}"]) for r in rows], float)\n        X = np.column_stack([np.ones(m.sum()), np.log10(np.maximum(area[m], 1e-6)),\n                             np.log10(np.maximum(nv[m], 1)), inc[d][m]])\n        beta, *_ = np.linalg.lstsq(X, x[m], rcond=None)\n        pred = X @ beta\n        r2 = 1 - float(np.sum((x[m] - pred) ** 2) / np.sum((x[m] - x[m].mean()) ** 2))\n        # complementarity: variance NOT explained by level + trajectory\n        L = np.column_stack([np.ones(m.sum()), lev[d][m],\n                             lev["Oct13"][m] - lev["Jun06"][m]])\n        okL = np.all(np.isfinite(L), axis=1)\n        if okL.sum() > 50:\n            bl, *_ = np.linalg.lstsq(L[okL], x[m][okL], rcond=None)\n            pl = L[okL] @ bl\n            r2_lev = 1 - float(np.sum((x[m][okL] - pl) ** 2) / np.sum((x[m][okL] - x[m][okL].mean()) ** 2))\n        else:\n            r2_lev = float("nan")\n        passes = bool(np.isfinite(rel) and rel >= 0.5 and r2 < 0.15)\n        rep[f] = dict(n=int(m.sum()), reliability_split_half_rho=(round(rel, 4) if np.isfinite(rel) else None),\n                      n_reliability=n_rel, nuisance_r2_area_npx_incidence=round(r2, 4),\n                      r2_explained_by_level_and_trajectory=(round(r2_lev, 4) if np.isfinite(r2_lev) else None),\n                      median=round(float(np.nanmedian(x[m])), 4),\n                      passes_preregistered_gates=passes)\n    report[d] = rep\n    print("\\n[%s] %-24s %8s %10s %10s %8s" % (d, "feature", "rel_rho", "nuis_R2", "levR2", "gate"))\n    for f, v in rep.items():\n        if "note" in v: continue\n        print("      %-24s %8s %10.3f %10s %8s"\n              % (f, v["reliability_split_half_rho"], v["nuisance_r2_area_npx_incidence"],\n                 v["r2_explained_by_level_and_trajectory"], "PASS" if v["passes_preregistered_gates"] else "-"))\n\n# accepted set: passes on a majority of the dates where it is defined\nacc = []\nfor f in FEATNAMES + ["interior_minus_edge_db"]:\n    p = [report[d][f].get("passes_preregistered_gates", False) for d in DATES if f in report[d]]\n    if sum(bool(x) for x in p) >= 3: acc.append(f)\nprint("\\nACCEPTED (passes on >=3 of 4 dates): %s" % (acc or "NONE"))\n\n# ---------------- is the null a SAMPLE-SIZE effect? ---------------------------\n# If reliability is limited by how many looks a field provides, it must improve\n# with field size. If it does not, the limitation is that there is no stable\n# field-level texture to measure, which is a different and stronger conclusion.\nprint("\\n[size-stratified reliability] does more area buy reliability?")\nstrat = {}\nfor d in DATES:\n    good = gate[d].copy()\n    if d == "Jun19":\n        for f_ in jun19_invalid:\n            if f_ in IDX: good[IDX[f_]] = False\n    nv = np.array([fl(r[f"n_valid_{d}"]) for r in rows], float)\n    qs = np.nanpercentile(area[good], [25, 50, 75])\n    bins = [(-np.inf, qs[0]), (qs[0], qs[1]), (qs[1], qs[2]), (qs[2], np.inf)]\n    strat[d] = {}\n    for f in ["cv_texture", "ml3_cv", "ml3_sd_db", "sd_db"]:\n        a, b = HALF[d][f]\n        row = []\n        for lo, hi in bins:\n            m = good & np.isfinite(a) & np.isfinite(b) & (area > lo) & (area <= hi)\n            if m.sum() < 40: row.append(None); continue\n            row.append(round(float(spearmanr(a[m], b[m]).statistic), 3))\n        strat[d][f] = dict(quartile_rho=row,\n                           median_area_ha=[round(float(np.nanmedian(area[good & (area > lo) & (area <= hi)])), 3)\n                                           for lo, hi in bins],\n                           median_n_valid=[int(np.nanmedian(nv[good & (area > lo) & (area <= hi)]))\n                                           for lo, hi in bins])\n    print("   %s  %-12s Q1..Q4 rho: %s" % (d, "cv_texture", strat[d]["cv_texture"]["quartile_rho"]))\n    print("   %s  %-12s Q1..Q4 rho: %s" % (d, "ml3_sd_db", strat[d]["ml3_sd_db"]["quartile_rho"]))\n\n# largest-farm reliability, pooled across dates, for the best two candidates\nbig = area >= np.nanpercentile(area, 90)\npooled = {}\nfor f in ["cv_texture", "ml3_cv", "ml3_sd_db", "sd_db"]:\n    aa, bb = [], []\n    for d in DATES:\n        good = gate[d].copy()\n        if d == "Jun19":\n            for f_ in jun19_invalid:\n                if f_ in IDX: good[IDX[f_]] = False\n        a, b = HALF[d][f]\n        m = good & big & np.isfinite(a) & np.isfinite(b)\n        aa.append(a[m]); bb.append(b[m])\n    aa = np.concatenate(aa); bb = np.concatenate(bb)\n    pooled[f] = dict(n=int(aa.size),\n                     rho=(round(float(spearmanr(aa, bb).statistic), 4) if aa.size > 40 else None))\n    print("   top-decile-area pooled  %-12s n=%4d rho=%s" % (f, pooled[f]["n"], pooled[f]["rho"]))\n\n# theoretical speckle reference: 1-look intensity in dB has SD = 5.57 dB\nSPECKLE_SD_DB = 5.5730\nobs_sd = {d: round(float(np.nanmedian(STRUCT[d]["sd_db"][gate[d]])), 4) for d in DATES}\nprint("\\n[speckle check] theoretical 1-look intensity SD in dB = %.3f" % SPECKLE_SD_DB)\nprint("   observed median sd_db per date: %s" % obs_sd)\n\n\n# ---------------- write derived CSV -------------------------------------------\nhdr = list(rows[0].keys())\nnew = [f"{f}_{d}" for d in DATES for f in FEATNAMES + ["interior_minus_edge_db"]]\nwith open(CSV_OUT, "w", newline="", encoding="utf-8") as fh:\n    w = csv.DictWriter(fh, fieldnames=hdr + new, extrasaction="ignore"); w.writeheader()\n    for i, r in enumerate(rows):\n        dd = dict(r)\n        for d in DATES:\n            for f in FEATNAMES + ["interior_minus_edge_db"]:\n                v = STRUCT[d][f][i]\n                dd[f"{f}_{d}"] = "" if not np.isfinite(v) else round(float(v), 6)\n        w.writerow(dd)\n\nLOG = dict(\n    purpose="determine which within-field structural quantities are measurable at ENL=1",\n    input_provenance=PROV, enl_assumed=ENL,\n    speckle_model=("CV_obs^2 = CV_t^2 + CV_s^2 + CV_t^2*CV_s^2 with CV_s = 1/sqrt(ENL); "\n                   "CV_texture = sqrt(max((CV_obs^2 - 1/ENL)/(1 + 1/ENL), 0))"),\n    prior_null=("METHODOLOGY.md:48 records a failed within-field uniformity test, SAR CV vs "\n                "NDVI CV r=+0.047. Raw CV at ENL=1 is speckle-dominated, which is the likely "\n                "cause; cv_texture is the debiased retest."),\n    jun19_invalid_excluded=sorted(jun19_invalid),\n    gates=dict(reliability_rho=0.5, nuisance_r2=0.15,\n               status="preregistered heuristics; near-misses are reported, not silently dropped"),\n    per_date=report, accepted_features=acc,\n    size_stratified_reliability=strat,\n    top_decile_area_pooled_reliability=pooled,\n    speckle_reference=dict(\n        theoretical_1look_intensity_sd_db=5.5730,\n        observed_median_sd_db=obs_sd,\n        reading=("sd_db sitting at the theoretical single-look speckle value, with a p05-p95 "\n                 "spread of only about 0.5 dB across 900 farms, means dB dispersion at ENL=1 "\n                 "is speckle and carries almost no field information.")),\n    output_csv=dict(path=CSV_OUT, sha256=hashlib.sha256(open(CSV_OUT, "rb").read()).hexdigest(),\n                    n_rows=NF, n_new_columns=len(new)),\n    runtime_s=round(time.time() - T0, 1),\n    environment=dict(python=platform.python_version(), numpy=np.__version__,\n                     rasterio=rasterio.__version__))\njson.dump(LOG, open(f"{OUT}/structure_reliability.json", "w"), indent=1)\nprint("\\nwrote %s and %s/structure_reliability.json  (%.0fs)" % (CSV_OUT, OUT, time.time() - T0))\n'
Path(SRC/"build_e2_structure_features.py").write_text(_SRC, encoding="utf-8")
print("wrote src/build_e2_structure_features.py  ({} lines)".format(_SRC.count(chr(10))))

In [ ]:
T["S08"], _ = run_stage("build_e2_structure_features.py",
                        ["farm_features_structure_v1.csv", "_diag/structure_reliability.json"],
                        "S08 structure/reliability")
_s = list(_csv.DictReader(open(ROOT / "farm_features_structure_v1.csv", encoding="utf-8")))
print(f"  structure table: {len(_s)} farms x {len(_s[0])} cols")
assert len(_s) == 966

---
## 6 · Public auxiliary Sentinel source data  ·  `[P] S09 · S10`

### WHY
Capella is single-polarisation HH. Four HH numbers per farm give a continuous phenological
gradient but cannot separate five species — the literature is consistent that the short-duration
cereals need polarimetric or optical phenology. Sentinel-1 adds **VV+VH**; Sentinel-2 adds
red-edge and SWIR, though the monsoon removes most of Jun–Sep.

### INPUT — source imagery only
Rasters come exclusively from the pinned public Hugging Face source-only dataset
`aisehack-r2-public-auxiliary-source-data` (AOI clips at native value / resolution / CRS).
Microsoft Planetary Computer STAC discovery is retained only as development/forensic evidence;
it is not performed by this production notebook.

**No farm-level Sentinel feature is ever loaded.** Per-polygon masking, dB conversion, SCL cloud
masking and the 20 m → 10 m resampling all execute in this notebook.

### METHOD
- **S09** optical reference: cloud-screened S2 scenes → NDVI/NDMI/NDRE/NDWI per farm, then the
  **BOA_ADD_OFFSET = −1000** correction (Sentinel-2 processing baseline ≥ 04.00).
- **S10** full season: S1 RTC γ⁰ → dB → per-farm means; S2 → per-pixel SCL mask (keep classes
  4/5/6/7) → five indices per farm per date.

In [ ]:
# S09a -- canonical Sentinel-2 multi-index optical reference from pinned source clips.
_SRC = '"""Build the canonical S2 optical reference from frozen public source clips.\n\nThis is the source-transport equivalent of build_reference_v2.py.  It preserves the\ncanonical float-window read, default nearest-neighbour resampling, formulae, output order and\nprecision.  It replaces only the live STAC lookup with the pinned auxiliary manifest.\n"""\nimport csv, json, os\nimport numpy as np\nimport rasterio, shapefile\nfrom rasterio.features import geometry_mask\nfrom rasterio.windows import Window\nfrom pyproj import Transformer\n\nR = os.environ.get("E2E_ROOT", "round2") + "/"\nCLIPS = os.environ["SENTINEL_CLIPS"]\nMAN = json.load(open(os.path.join(CLIPS, "AUXILIARY_MANIFEST.json"), encoding="utf-8"))\nentries = {e["local_file"]: e for e in MAN["files"] if e.get("collection") == "sentinel-2-l2a"}\ndates = sorted({e["date"] for e in entries.values() if float(e.get("scene_cloud_pct", 100)) < 20})\nbands = ("B04", "B08", "B11", "B05", "B03")  # B8A was read historically but never used\n\nreader = shapefile.Reader(R + "data/Farm_boundaries_shp/Farm_boundaries_shp/Sokhda_Farms.shp")\nshapes, recs = reader.shapes(), reader.records()\nfids = [str(r[0]) for r in recs]\n\ndef read_date(d):\n    out = {}; wt = crs = None\n    for b in bands:\n        key = f"s2/S2_{d}_{b}.tif"; e = entries[key]\n        with rasterio.open(os.path.join(CLIPS, key)) as src:\n            wf = e["window_float"]; c0, r0 = e["clip_origin"]\n            win = Window(wf[0]-c0, wf[1]-r0, wf[2], wf[3])\n            res = abs(src.transform.a)\n            osh = (int(wf[3]*res/10), int(wf[2]*res/10)) if res != 10 else None\n            a = src.read(1, window=win, out_shape=osh).astype("float32")\n            if res == 10:\n                wt, crs = src.window_transform(win), src.crs\n        out[b] = a\n    h = min(v.shape[0] for v in out.values()); w = min(v.shape[1] for v in out.values())\n    return {k:v[:h,:w] for k,v in out.items()}, wt, crs\n\ndef indices(B):\n    e=1e-6; red,nir=B["B04"],B["B08"]\n    return {\n        "NDVI":(nir-red)/np.clip(nir+red,e,None),\n        "NDMI":(nir-B["B11"])/np.clip(nir+B["B11"],e,None),\n        "NDWI":(B["B03"]-nir)/np.clip(B["B03"]+nir,e,None),\n        "NDRE":(nir-B["B05"])/np.clip(nir+B["B05"],e,None),\n        "EVI":2.5*(nir-red)/np.clip(nir+6*red-7.5*B["B03"]+1e4,e,None),\n        "SAVI":1.5*(nir-red)/np.clip(nir+red+0.5e4,e,None),\n    }\n\nresult = {}\nfor d in dates:\n    B, wt, crs = read_date(d); idx = indices(B)\n    tr = Transformer.from_crs("EPSG:4326", crs, always_xy=True)\n    per = {k:{} for k in idx}\n    for rec, shp in zip(recs, shapes):\n        X,Y=tr.transform([p[0] for p in shp.points],[p[1] for p in shp.points])\n        try:\n            m=geometry_mask([{"type":"Polygon","coordinates":[list(zip(X,Y))]}],\n                            out_shape=next(iter(idx.values())).shape,transform=wt,invert=True)\n        except Exception:\n            continue\n        if m.sum()<3: continue\n        for k,a in idx.items():\n            v=a[m & np.isfinite(a)]\n            if v.size: per[k][str(rec[0])] = float(np.nanmean(v))\n    result[d]=per\n    print(f"  {d} {list(idx)} farms={len(per[\'NDVI\'])}", flush=True)\n\nkeys=sorted({k for d in result for k in result[d]})\nwith open(R+"farm_reference_v2.csv","w",newline="",encoding="utf-8") as fh:\n    w=csv.writer(fh); w.writerow(["fid"]+[f"{k}_{d}" for d in dates for k in keys])\n    for fid in fids:\n        w.writerow([fid]+[result[d].get(k,{}).get(fid,"") for d in dates for k in keys])\nprint(f"wrote {R}farm_reference_v2.csv | dates={len(dates)} indices={keys}")\n'
Path(SRC/"build_reference_from_source.py").write_text(_SRC, encoding="utf-8")
print("wrote src/build_reference_from_source.py  ({} lines)".format(_SRC.count(chr(10))))

In [ ]:
# S09b -- Sentinel-2 BOA_ADD_OFFSET correction.
_SRC = '"""\nCorrect the Sentinel-2 reference for the missing BOA_ADD_OFFSET, then re-test the\nancillary crop evidence and re-extract the crop-AREA prior.\n\nTHE DEFECT\n  round2/build_reference_v2.py reads Sentinel-2 L2A assets from Planetary\n  Computer as raw integer DN and computes normalised-difference indices directly.\n  Since ESA processing baseline 04.00 (mandatory from 2022-01-25) L2A carries\n      BOA_ADD_OFFSET = -1000,  so  reflectance = (DN - 1000) / 10000\n  The script never applies it. For a normalised difference this does NOT cancel:\n      NDVI_computed = (NIR-RED) / (NIR+RED)            [DN]\n      NDVI_true     = (NIR-RED) / (NIR+RED-2000)       [offset-corrected DN]\n  The numerator is offset-free; the denominator is inflated by 2000 DN. Every\n  index is therefore compressed toward zero, and the compression is strongest\n  for vigorous vegetation.\n\nWHY IT CAN BE FIXED WITHOUT RE-DOWNLOADING\n  The script stored BOTH NDVI and SAVI, and SAVI was written with the DN-scaled\n  soil term:\n      SAVI = 1.5 (NIR-RED) / (NIR+RED+5000)\n      NDVI = (NIR-RED) / (NIR+RED)\n      => SAVI/NDVI = 1.5 S / (S + 5000),   S = NIR+RED in DN\n  So S is recoverable per farm per date, and with it A = NIR-RED = NDVI * S.\n  Every normalised-difference index can then be re-expressed offset-corrected.\n  NDMI, NDRE and NDWI each involve one further band, which is recoverable from\n  its own index and the already-known NIR.\n\nVERIFICATION OF THE DIAGNOSIS\n  Uncorrected, the median field needs rho_RED = 0.180 - bright bare soil - for\n  all 966 farms in November. Corrected, rho_RED = 0.080 and rho_NIR = 0.284,\n  a normal mixed crop/soil signature. Corrected NDVI: median 0.558, p90 0.699,\n  max 0.836, with 36.7% of fields above 0.6.\n\nWrites round2/farm_reference_v2_offsetcorrected.csv and\n       round2/_diag/s2_offset_correction.json\n"""\nimport os, csv, re, json, warnings\nimport numpy as np\nfrom scipy.stats import spearmanr\n\nwarnings.filterwarnings("ignore")\nSRC = "round2/farm_reference_v2.csv"\nDST = "round2/farm_reference_v2_offsetcorrected.csv"\nOUT = "round2/_diag"\nOFFSET_DN = 1000.0          # ESA baseline 04.00 BOA_ADD_OFFSET\nSOIL_DN = 5000.0            # the 0.5e4 soil term the original script used in SAVI\n\n\ndef fl(v):\n    try: return float(v)\n    except (TypeError, ValueError): return float("nan")\n\n\nrows = list(csv.DictReader(open(SRC, encoding="utf-8")))\ncols = [c for c in rows[0] if c != "fid"]\nDATES = sorted(set(re.findall(r"\\d{4}-\\d{2}-\\d{2}", " ".join(cols))))\nprint("=" * 88)\nprint("Sentinel-2 BOA_ADD_OFFSET correction  (%d farms, %d dates)" % (len(rows), len(DATES)))\nprint("=" * 88)\n\nout_rows = [dict(fid=r["fid"]) for r in rows]\nreport = {}\nfor d in DATES:\n    ndvi = np.array([fl(r.get(f"NDVI_{d}", "")) for r in rows])\n    savi = np.array([fl(r.get(f"SAVI_{d}", "")) for r in rows])\n    ndmi = np.array([fl(r.get(f"NDMI_{d}", "")) for r in rows])\n    ndre = np.array([fl(r.get(f"NDRE_{d}", "")) for r in rows])\n    ndwi = np.array([fl(r.get(f"NDWI_{d}", "")) for r in rows])\n    ok = np.isfinite(ndvi) & np.isfinite(savi) & (np.abs(ndvi) > 1e-6)\n    ratio = np.full(len(rows), np.nan); ratio[ok] = savi[ok] / ndvi[ok]\n    # S = NIR + RED in raw DN\n    S = np.full(len(rows), np.nan)\n    good = ok & np.isfinite(ratio) & (ratio < 1.5 - 1e-6) & (ratio > 0)\n    S[good] = SOIL_DN * ratio[good] / (1.5 - ratio[good])\n    A = ndvi * S                                   # NIR - RED, offset-free\n    NIR = (S + A) / 2.0\n    Sc = S - 2 * OFFSET_DN                         # offset-corrected NIR+RED\n    valid = np.isfinite(Sc) & (Sc > 500)\n    ndvi_c = np.where(valid, A / Sc, np.nan)\n\n    def corr_nd(idx_obs, partner_sign):\n        """Recover the partner band from its own index and NIR, then recompute\n        the index with both bands offset-corrected."""\n        with np.errstate(invalid="ignore", divide="ignore"):\n            if partner_sign > 0:      # index = (NIR - X)/(NIR + X)\n                X = NIR * (1 - idx_obs) / (1 + idx_obs)\n                num = (NIR - OFFSET_DN) - (X - OFFSET_DN)\n                den = (NIR - OFFSET_DN) + (X - OFFSET_DN)\n            else:                     # index = (X - NIR)/(X + NIR)\n                X = NIR * (1 + idx_obs) / (1 - idx_obs)\n                num = (X - OFFSET_DN) - (NIR - OFFSET_DN)\n                den = (X - OFFSET_DN) + (NIR - OFFSET_DN)\n            r = np.where(np.isfinite(den) & (np.abs(den) > 100), num / den, np.nan)\n        return np.where(valid, r, np.nan)\n\n    ndmi_c = corr_nd(ndmi, +1)\n    ndre_c = corr_nd(ndre, +1)\n    ndwi_c = corr_nd(ndwi, -1)\n    for i, r in enumerate(out_rows):\n        r[f"NDVI_{d}"] = "" if not np.isfinite(ndvi_c[i]) else round(float(ndvi_c[i]), 5)\n        r[f"NDMI_{d}"] = "" if not np.isfinite(ndmi_c[i]) else round(float(ndmi_c[i]), 5)\n        r[f"NDRE_{d}"] = "" if not np.isfinite(ndre_c[i]) else round(float(ndre_c[i]), 5)\n        r[f"NDWI_{d}"] = "" if not np.isfinite(ndwi_c[i]) else round(float(ndwi_c[i]), 5)\n        r[f"NDVI_raw_{d}"] = "" if not np.isfinite(ndvi[i]) else round(float(ndvi[i]), 5)\n    v = ndvi_c[np.isfinite(ndvi_c)]\n    o = ndvi[np.isfinite(ndvi)]\n    report[d] = dict(n=int(v.size),\n                     ndvi_uncorrected=dict(median=round(float(np.median(o)), 4),\n                                           p90=round(float(np.percentile(o, 90)), 4),\n                                           max=round(float(o.max()), 4)),\n                     ndvi_corrected=dict(median=round(float(np.median(v)), 4),\n                                         p90=round(float(np.percentile(v, 90)), 4),\n                                         p99=round(float(np.percentile(v, 99)), 4),\n                                         max=round(float(v.max()), 4)),\n                     frac_above_0p6=round(float((v > 0.6).mean()), 4),\n                     frac_above_0p7=round(float((v > 0.7).mean()), 4),\n                     implied_rho_red_uncorrected=round(float(np.nanmedian((S - A) / 2 / 1e4)), 4),\n                     implied_rho_red_corrected=round(float(np.nanmedian((Sc - A) / 2 / 1e4)), 4))\n    print("  %s  NDVI med %.3f -> %.3f | max %.3f -> %.3f | >0.6 %.1f%% | rho_RED %.3f -> %.3f"\n          % (d, report[d]["ndvi_uncorrected"]["median"], report[d]["ndvi_corrected"]["median"],\n             report[d]["ndvi_uncorrected"]["max"], report[d]["ndvi_corrected"]["max"],\n             100 * report[d]["frac_above_0p6"], report[d]["implied_rho_red_uncorrected"],\n             report[d]["implied_rho_red_corrected"]))\n\nhdr = ["fid"] + [f"{ix}_{d}" for d in DATES for ix in ("NDVI", "NDMI", "NDRE", "NDWI")] \\\n      + [f"NDVI_raw_{d}" for d in DATES]\nwith open(DST, "w", newline="", encoding="utf-8") as fh:\n    w = csv.DictWriter(fh, fieldnames=hdr, extrasaction="ignore"); w.writeheader()\n    for r in out_rows: w.writerow(r)\n\nLOG = dict(\n    defect=dict(\n        file="round2/build_reference_v2.py",\n        description=("Sentinel-2 L2A assets are read as raw integer DN and normalised-difference "\n                     "indices are computed directly, without applying BOA_ADD_OFFSET = -1000 "\n                     "(ESA processing baseline 04.00, mandatory since 2022-01-25)."),\n        why_it_matters=("the offset does NOT cancel in a normalised difference: the numerator is "\n                        "offset-free but the denominator is inflated by 2000 DN, so every index is "\n                        "compressed toward zero, most strongly for vigorous vegetation"),\n        note_on_EVI_SAVI=("EVI and SAVI already carried DN-scaled constants (1e4, 0.5e4), so the "\n                          "original author was aware of DN scaling but not of the baseline offset")),\n    correction=dict(\n        method=("S = NIR+RED recovered per farm per date from the stored SAVI/NDVI ratio "\n                "(SAVI/NDVI = 1.5 S/(S+5000)); A = NIR-RED = NDVI*S; partner bands recovered from "\n                "their own indices and NIR; all indices recomputed with both bands offset-corrected"),\n        no_download_required=True,\n        indices_corrected=["NDVI", "NDMI", "NDRE", "NDWI"],\n        indices_not_corrected=["EVI", "SAVI"],\n        indices_not_corrected_reason="their DN-scaled constants make analytic inversion ambiguous"),\n    evidence_the_diagnosis_is_right=dict(\n        uncorrected_implied_rho_red_median=report[DATES[0]]["implied_rho_red_uncorrected"],\n        corrected_implied_rho_red_median=report[DATES[0]]["implied_rho_red_corrected"],\n        argument=("uncorrected requires rho_RED near 0.18 - bright bare soil - for essentially "\n                  "every one of 966 farms in November, which is not credible. Corrected gives "\n                  "rho_RED near 0.08 and rho_NIR near 0.28, a normal mixed crop/soil signature.")),\n    per_date=report,\n    consequences=dict(\n        withdrawn=("the E4 conclusion \'late-season cotton persistence NOT SUPPORTED\'. It measured "\n                   "the preprocessing defect, not the vegetation. With the offset applied, 36.7% "\n                   "of fields exceed NDVI 0.6 and 9.9% exceed 0.7 in November - fully consistent "\n                   "with substantial standing cotton and with the district prior."),\n        dissolved=("the contradiction between the district cotton share and the optical evidence"),\n        needs_recheck=("E4b joint separability. The correction is NOT a pure monotone transform - "\n                       "the denominator adjustment depends on each farm\'s own S - so dip tests, "\n                       "silhouettes and cluster assignments must be recomputed, not assumed to "\n                       "carry over."),\n        unaffected=("the Capella-only results in E3, which never used optical data")),\n    verdict="S2_REFERENCE = UNTRUSTED as published, CORRECTED here, and now usable")\njson.dump(LOG, open(f"{OUT}/s2_offset_correction.json", "w"), indent=1)\nprint("\\nwrote %s" % DST)\nprint("wrote %s/s2_offset_correction.json" % OUT)\nprint("VERDICT: %s" % LOG["verdict"])\n'
Path(SRC/"fix_s2_reference_offset.py").write_text(_SRC, encoding="utf-8")
print("wrote src/fix_s2_reference_offset.py  ({} lines)".format(_SRC.count(chr(10))))

In [ ]:
# ---- deterministic public Hugging Face source snapshot; no live STAC in production ----
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], check=True)
from huggingface_hub import snapshot_download
HF_REPO = "SharadhNaiduTrains/aisehack-r2-public-auxiliary-source-data"  # replaced from the publication receipt by the notebook builder
HF_REVISION = "9b9b3ec711250df2c1111bdd241c8691d37ecbe9"  # replaced from the publication receipt by the notebook builder
assert HF_REPO != "SharadhNaiduTrains/aisehack-r2-public-auxiliary-source-data" and HF_REVISION != "9b9b3ec711250df2c1111bdd241c8691d37ecbe9", \
    "HF dataset repository/revision was not pinned at build time"
CLIPDIR = snapshot_download(repo_id=HF_REPO, repo_type="dataset", revision=HF_REVISION,
                            local_dir=str(WORK / "hf_aux"), token=False)
SENTINEL_MODE = "clips"
print(f"SENTINEL SOURCE: public Hugging Face dataset {HF_REPO}@{HF_REVISION}")

# verify every manifest hash before using any raster or agronomic table
man_path = Path(CLIPDIR) / "AUXILIARY_MANIFEST.json"
man = json.loads(man_path.read_text(encoding="utf-8"))
assert man.get("contains_farm_level_model_outputs") is False, \
    "auxiliary dataset declares farm-level model outputs -- refusing to use it"
bad_hash = []
for ent in man["files"]:
    p = Path(CLIPDIR) / ent["local_file"]
    if not p.is_file() or sha256(p) != ent["sha256"]:
        bad_hash.append(ent["local_file"])
assert not bad_hash, f"HF auxiliary hash mismatch/missing files: {bad_hash[:5]}"
print(f"  hash audit PASS: {len(man['files'])}/{len(man['files'])} manifest files")
print(f"  farm-level model outputs declared: {man['contains_farm_level_model_outputs']}")
for nm in ("crop_area_prior_verified.json", "vadodara_apy_series.json"):
    shutil.copyfile(Path(CLIPDIR) / "agronomic" / nm, ROOT / "_diag" / nm)

In [ ]:
env2 = dict(env, SENTINEL_MODE=SENTINEL_MODE, SENTINEL_CLIPS=CLIPDIR)

def run_sentinel(mod, produces, label, timeout=5400):
    t0 = time.time()
    p = subprocess.run([sys.executable, str(SRC / mod)], cwd=str(WORK), env=env2,
                       capture_output=True, text=True, timeout=timeout)
    if p.returncode != 0:
        print(p.stdout[-2000:]); print(p.stderr[-3000:])
        raise RuntimeError(f"{label} failed")
    for out in produces:
        assert (ROOT / out).is_file(), f"{label}: {out} missing"
    dt = time.time() - t0
    print(f"  {label:<34} {dt:7.1f}s")
    return dt, p.stdout

T["S09a"], _ = run_sentinel("build_reference_from_source.py", ["farm_reference_v2.csv"],
                            "S09a S2 optical reference")
T["S09b"], _ = run_sentinel("fix_s2_reference_offset.py",
                            ["farm_reference_v2_offsetcorrected.csv"], "S09b BOA offset fix")

---
## 7 · Full-season Sentinel farm features  ·  `[P] S10`

### WHY → INPUT → METHOD
The pinned Hugging Face snapshot is source-data transport only. This stage performs every farm
operation live: S1 RTC linear γ⁰ → dB → polygon means, and S2 canonical fractional-window reads
→ SCL classes 4/5/6/7 → the unchanged `keep.mean() >= 0.25` gate → `EVI, NBR2, NDMI, NDRE,
NDVI` farm means. `OFF = -1000`; masking precedes index calculation; EVI retains its `+1e4`
denominator term. Rasterio's default/nearest resampling is used exactly as in development.

The historical 18-date source was recovered by reconstructing the original selection semantics:
sorted same-date candidates were tried sequentially, and a failed candidate did not consume the
calendar date. Four accepted observations therefore come from overlapping tile T43QBE after
T43QCE failed the original filters. No date, threshold, crop count or label is forced.

In [ ]:
# S10 -- canonical S1/S2 source-to-farm processing with fractional source-window preservation.
_SRC = '"""\nS10 -- Sentinel-1 / Sentinel-2 farm features from PUBLIC SOURCE RASTERS.\n\nThis is a minimal patch of the canonical fetch_s1_s2_fullseason.py. Every index formula,\nconstant, mask class, clip epsilon, ordering and output precision is preserved verbatim; the\nONLY change is where the pixels come from:\n\n    canonical : rasterio.open(<signed STAC href>)  -> s.read(1, window=win, out_shape=osh)\n    here      : rasterio.open(<AOI clip GeoTIFF>)  -> s.read(1,             out_shape=osh)\n\nThe clips were produced with exactly `s.read(1, window=win)` at native resolution, so the clip\nIS the window. `osh` is recomputed from the ORIGINAL float window recorded in the auxiliary\nmanifest (`window_float`), which is what preserves the 20 m -> 10 m half-pixel alignment.\n\nThere is deliberately no live-discovery mode in this production module. STAC was used only to\nbuild and audit the public source package; production consumes the pinned, hash-locked clips.\n"""\nimport csv, json, math, os, sys\nimport numpy as np\nimport rasterio, shapefile\nfrom rasterio.windows import from_bounds, Window\nfrom rasterio.features import geometry_mask\nfrom pyproj import Transformer\n\nsys.stdout.reconfigure(encoding="utf-8", errors="replace")\nBBOX = [73.133, 22.408, 73.181, 22.443]\nBANDS = {"B03": 1, "B04": 1, "B05": 2, "B08": 1, "B11": 2, "B12": 2, "SCL": 2}\nR = os.environ.get("E2E_ROOT", "round2") + "/"\nCLIPS = os.environ.get("SENTINEL_CLIPS", "")\n\nr = shapefile.Reader(R + "data/Farm_boundaries_shp/Farm_boundaries_shp/Sokhda_Farms.shp")\nshapes, recs = r.shapes(), r.records()\nFIDS = [str(rec[0]) for rec in recs]\n\n\ndef per_farm(arr, wt, crs):                      # verbatim from the canonical script\n    tr = Transformer.from_crs("EPSG:4326", crs, always_xy=True)\n    out = {}\n    for rec, s in zip(recs, shapes):\n        pts = s.points\n        X, Y = tr.transform([p[0] for p in pts], [p[1] for p in pts])\n        try:\n            m = geometry_mask([{"type": "Polygon", "coordinates": [list(zip(X, Y))]}],\n                              out_shape=arr.shape, transform=wt, invert=True)\n        except Exception:\n            continue\n        if m.sum() < 3:\n            continue\n        v = arr[m & np.isfinite(arr)]\n        if v.size:\n            out[str(rec[0])] = float(np.nanmean(v))\n    return out\n\n\n# ---------------- source index ----------------\nMAN = json.loads(open(os.path.join(CLIPS, "AUXILIARY_MANIFEST.json"),\n                      encoding="utf-8").read())\nWIN = {}\nfor e in MAN["files"]:\n    lf = e.get("local_file", "")\n    if lf.startswith(("s1/", "s2/")):\n        WIN[lf] = e                              # full manifest entry (float window + origin)\nS1_DATES = sorted({e["date"] for e in MAN["files"] if e.get("collection") == "sentinel-1-rtc"})\nS2_DATES = sorted({e["date"] for e in MAN["files"] if e.get("collection") == "sentinel-2-l2a"})\nprint(f"[src] public source clips: {len(WIN)} rasters | "\n      f"S1 {len(S1_DATES)} dates | S2 {len(S2_DATES)} dates")\n\n\n# ===================== 1. SENTINEL-1 =====================\ns1, meta = {}, {}\nfor d in S1_DATES:\n    try:\n        rec = {}\n        for pol in ("vv", "vh"):\n            key = f"s1/S1_{d}_{pol.upper()}.tif"\n            if key not in WIN:\n                continue\n            with rasterio.open(os.path.join(CLIPS, key)) as s:\n                a = s.read(1).astype("float32")\n                wt, crs = s.transform, s.crs\n            a[a <= 0] = np.nan\n            rec[pol] = per_farm(10.0 * np.log10(a), wt, crs)\n        if "vv" in rec and rec["vv"]:\n            s1[d] = rec\n            print(f"  s1 {d}  vv={len(rec.get(\'vv\',{}))} vh={len(rec.get(\'vh\',{}))}", flush=True)\n    except Exception as e:\n        print(f"  s1 {d} FAILED {str(e)[:70]}", flush=True)\n\nif s1:\n    dates = sorted(s1)\n    hdr = ["fid"] + [f"{p.upper()}_{d}" for d in dates for p in ("vv", "vh")]\n    with open(R + "farm_s1_timeseries.csv", "w", newline="", encoding="utf-8") as fh:\n        w = csv.writer(fh); w.writerow(hdr)\n        for fid in FIDS:\n            row = [fid]\n            for d in dates:\n                for p in ("vv", "vh"):\n                    row.append(s1[d].get(p, {}).get(fid, ""))\n            w.writerow(row)\n    print(f"[s1] wrote farm_s1_timeseries.csv  dates={len(dates)}")\n\n# ===================== 2. SENTINEL-2 =====================\ns2 = {}\nfor d in S2_DATES:\n    try:\n        B, wt, crs = {}, None, None\n        for b in BANDS:\n            key = f"s2/S2_{d}_{b}.tif"\n            if key not in WIN:\n                continue\n            ent = WIN[key]\n            with rasterio.open(os.path.join(CLIPS, key)) as s:\n                res = abs(s.transform.a)\n                wf = ent["window_float"]           # canonical FLOAT window in SOURCE pixels\n                c0, r0 = ent["clip_origin"]        # integer superset origin\n                # Rebuild the canonical float window RELATIVE to the clip, then perform the\n                # identical single fractional-window resampling read. Verified pixel-exact\n                # against a direct source read for SCL and B11 (0 differing pixels).\n                rel = Window(wf[0] - c0, wf[1] - r0, wf[2], wf[3])\n                osh = (int(wf[3] * res / 10), int(wf[2] * res / 10)) if res != 10 else None\n                a = s.read(1, window=rel, out_shape=osh).astype("float32")\n                if res == 10:\n                    wt, crs = s.window_transform(rel), s.crs\n            B[b] = a\n        if wt is None or "B08" not in B:\n            raise RuntimeError("no 10m grid")\n        h = min(v.shape[0] for v in B.values()); wd = min(v.shape[1] for v in B.values())\n        B = {k: v[:h, :wd] for k, v in B.items()}\n        if "SCL" in B:\n            keep = np.isin(np.rint(B["SCL"]), [4, 5, 6, 7])\n        else:\n            keep = np.ones(B["B08"].shape, bool)\n        if keep.mean() < 0.25:\n            raise RuntimeError("clear fraction %.2f too low" % keep.mean())\n        OFF = -1000.0                      # ESA baseline 04.00 BOA_ADD_OFFSET\n        g = {k: np.where(keep, B[k] + OFF, np.nan) for k in B if k != "SCL"}\n        e = 1e-6\n        nir, red = g["B08"], g["B04"]\n        idx = {"NDVI": (nir - red) / np.clip(nir + red, e, None)}\n        if "B11" in g:\n            idx["NDMI"] = (nir - g["B11"]) / np.clip(nir + g["B11"], e, None)\n        if "B12" in g:\n            idx["NBR2"] = (g["B11"] - g["B12"]) / np.clip(g["B11"] + g["B12"], e, None)\n        if "B05" in g:\n            idx["NDRE"] = (nir - g["B05"]) / np.clip(nir + g["B05"], e, None)\n        if "B03" in g:\n            idx["EVI"] = 2.5 * (nir - red) / np.clip(nir + 6 * red - 7.5 * g["B03"] + 1e4, e, None)\n        s2[d] = {k: per_farm(v, wt, crs) for k, v in idx.items()}\n        print("  s2 %s  clear=%.2f farms=%d" % (d, keep.mean(), len(s2[d]["NDVI"])), flush=True)\n    except Exception as ex:\n        print("  s2 %s skipped: %s" % (d, str(ex)[:70]), flush=True)\n\nif s2:\n    dates = sorted(s2)\n    keys = sorted({k for d in s2 for k in s2[d]})\n    with open(R + "farm_s2_fullseason.csv", "w", newline="", encoding="utf-8") as fh:\n        w = csv.writer(fh)\n        w.writerow(["fid"] + [f"{k}_{d}" for d in dates for k in keys])\n        for fid in FIDS:\n            row = [fid]\n            for d in dates:\n                for k in keys:\n                    row.append(s2[d].get(k, {}).get(fid, ""))\n            w.writerow(row)\n    print("[s2] wrote farm_s2_fullseason.csv  dates=%d" % len(dates))\njson.dump(dict(mode="pinned_public_source_clips", s1_dates=sorted(s1), s2_dates=sorted(s2)),\n          open(R + "_diag/sentinel_source_provenance.json", "w"), indent=1)\n'
Path(SRC/"build_sentinel_from_source.py").write_text(_SRC, encoding="utf-8")
print("wrote src/build_sentinel_from_source.py  ({} lines)".format(_SRC.count(chr(10))))

In [ ]:
T["S10"], out10 = run_sentinel("build_sentinel_from_source.py",
    ["farm_s1_timeseries.csv", "farm_s2_fullseason.csv", "_diag/sentinel_source_provenance.json"],
    "S10 Sentinel full season")
sp = json.loads((ROOT/"_diag/sentinel_source_provenance.json").read_text())
assert len(sp["s1_dates"]) == 13 and len(sp["s2_dates"]) == 18
print(f"S10 validation: S1={len(sp['s1_dates'])} dates; S2={len(sp['s2_dates'])} accepted dates")
print("S2 accepted:", sp["s2_dates"])

---
## 8 · Crop identity  ·  `[P] S11 · S12 · S13`

### WHY
The four-date Capella trajectory is a physically meaningful continuous phenological gradient,
but it does **not** contain five natural clusters. CROP_ID is therefore a **hybrid multi-sensor
crop-identity refinement using Capella X-band trajectory + Sentinel-1/2 phenology**.

### METHOD
S11 fits the v3 component shapes on the current 966 farms—those fitted crop-conditioned shapes
cannot be shipped as a cached model input. S12 constructs the invalid mask, coverage and five
evidence pathways. S13 fuses Capella, S1 and S2 with a weak district-area prior. Reported scores
are **normalized relative crop support**, never calibrated probabilities. MAP labels emerge from
the support surface; no quota forcing is used.

In [ ]:
# S11 -- fit crop_id_v3 component shapes.
_SRC = '"""\nCROP_ID v3 - literature-informed, weakly-supervised, multi-temporal probabilistic\ninference over five kharif crops for all 966 Sokhda farms.\n\nARCHITECTURE (Part A)\n  The templates supply ORDINAL constraints (sign of each inter-date change, plus a\n  cross-crop ordering on the Aug->Oct change). The DATA supplies the magnitudes.\n  No dB value from any paper is imported. This is the transferability policy\n  (calendar HIGH, shape MODERATE, magnitude LOW, thresholds NEVER) made operational.\n\n  THREE MODEL FAMILIES, each encoding a different scientific hypothesis:\n    M-A  constrained temporal mixture with latent sowing offset\n         H: crops are generative components whose mean SHAPE obeys the literature\n            ordering; sowing date is a discrete latent sub-template.\n    M-B  hierarchical duration-first classifier\n         H: the single most reliable axis is "still standing in Nov vs not"\n            (the only literature prediction our own data confirmed, rho=+0.313);\n            crop identity within a duration group is then a timing question.\n    M-C  flat Bayesian evidence fusion (the v2 incumbent, re-run as control).\n\n  Prototype-distance and unconstrained GMM variants are NOT built: on centred\n  shape vectors with a shared covariance they are algebraically near-identical to\n  M-A and would pad the comparison table without adding information.\n\nNUISANCE HANDLING\n  Every farm trajectory is CENTRED before matching. Absolute X-HH level is driven\n  by soil moisture, incidence and roughness; the crop signal is in the SHAPE.\n  Centring marginalises the per-farm level offset analytically.\n\nRARE-CROP POLICY (A7)\n  The concentration test runs with NO PRIOR AT ALL, on likelihood ratios only.\n  Whether the prior may erase a rare crop is decided by that test, not a threshold.\n\nOutputs\n  round2/farm_crop_probabilities_v3.csv\n  round2/_diag/crop_id_v3.json\n  round2/_diag/crop_temporal_templates_v2.json\n"""\nimport csv, json, math, os, re, sys, warnings\nimport numpy as np\n\nsys.stdout.reconfigure(encoding="utf-8", errors="replace")\nwarnings.filterwarnings("ignore")\nnp.random.seed(0)\n\nCROPS = ["Cotton", "Rice", "Maize", "Bajra", "Groundnut"]\nNC = len(CROPS)\nDATES = ["Jun06", "Jun19", "Aug14", "Oct13"]\n\n# ============================================================================\n# 1. TEMPLATE LIBRARY v2  - crop x sowing/maturity subtemplate\n# ============================================================================\n# sign: required sign of (d1=Jun19-Jun06, d2=Aug14-Jun19, d3=Oct13-Aug14)\n#       +1 must increase, -1 must decrease, 0 unconstrained\n# stand: P(canopy still green in mid-November)  [drives the S2 factor]\n# All entries carry an evidence tag; NOT SOURCED items are absent, never invented.\nTEMPLATES = {\n "Cotton_early": dict(crop="Cotton", sign=(+1, +1, -1), stand=0.90, w=0.25,\n   basis="P1 RISAT-1 Gujarat incl. Vadodara: sowing end-May, >60% by end-June; "\n         "Aug-Sep vegetative peak; first picking mid-Oct, crop stays standing.",\n   note="early-sown: canopy already developing by Jun19, so d1 must be positive"),\n "Cotton_timely": dict(crop="Cotton", sign=(0, +1, -1), stand=0.90, w=0.50,\n   basis="P1: modal Gujarat cotton sowing mid-to-late June.",\n   note="sown around Jun19 itself: d1 unconstrained, strong rise into Aug"),\n "Cotton_late": dict(crop="Cotton", sign=(0, +1, 0), stand=0.90, w=0.25,\n   basis="P2 Sirsa: late-sown cotton continues rising into Aug-Sep.",\n   note="late-sown: still gaining biomass at Oct13, d3 may be >=0"),\n "Rice": dict(crop="Rice", sign=(0, +1, -1), stand=0.55, w=1.00,\n   basis="KB L1/L3: transplant mid-Jul to early-Aug, 125-150 d; lowest backscatter "\n         "at transplant (standing-water specularity), sharp Jul-Aug rise with LAI, "\n         "senescent decline Sep-Oct, mid-Oct maturity minimum.",\n   note="Jun19 precedes transplanting, so d1 carries no rice signal"),\n "Maize": dict(crop="Maize", sign=(+1, +1, -1), stand=0.18, w=1.00,\n   basis="KB (Sentinel-1 Tamil Nadu, MAPscape): minimum at sowing, STEEP increase "\n         "seedling->vegetative, marginal increase to tasseling, decline at maturity. "\n         "KB (Netherlands S1): abrupt harvest drop mid-Sep to mid-Oct. "\n         "KB (corn coherence): post-maturity decline is muted because fields are left "\n         "standing for grain dry-down.",\n   note="SOURCED in v3 - was empty in v1. Steepest d2 of the five crops."),\n "Bajra": dict(crop="Bajra", sign=(+1, +1, -1), stand=0.05, w=1.00,\n   basis="KB (Optimal Datasets Suitability for Pearl Millet (Bajra) Discrimination, "\n         "multiparametric SAR): temporal backscatter profile separates bajra from "\n         "paddy/cotton; HH is ADVANTAGEOUS over VV at advanced growth stages because "\n         "VV suffers higher attenuation. KB: late-June sowing, short duration.",\n   note="earliest responder (d1>0), rising to an early-August peak (d2>0), harvested "\n        "well before Oct13 (d3<0, the steepest decline group). The HH finding is "\n        "directly relevant to our X-HH sensor and is the only pol-specific transfer "\n        "we accept. NOTE: this ordinal signature is IDENTICAL to Maize and Groundnut - "\n        "the same KB source states that separating bajra from maize/jowar required "\n        "POLARIMETRIC parameters, which a single-pol sensor cannot supply. The "\n        "within-group split is therefore reported as unresolved, not asserted."),\n "Groundnut": dict(crop="Groundnut", sign=(+1, +1, -1), stand=0.05, w=1.00,\n   basis="P1 RISAT-1 Gujarat: June sowing on monsoon onset, 110-120 d, peak at pod "\n         "development, declining backscatter toward the October harvest.",\n   note="same ordinal signature as Maize; separated only by d2 steepness and by the "\n        "prior. This confusion is REPORTED, not hidden."),\n}\nSUBS = list(TEMPLATES)\nNS = len(SUBS)\nSUB_CROP = np.array([CROPS.index(TEMPLATES[s]["crop"]) for s in SUBS])\nSUB_W = np.array([TEMPLATES[s]["w"] for s in SUBS], float)\nSIGN = np.array([TEMPLATES[s]["sign"] for s in SUBS], float)\nSTAND = np.array([TEMPLATES[s]["stand"] for s in SUBS], float)\n\n# Cross-crop ordering on d3 (Aug->Oct change). This is the ONE literature prediction\n# our own Capella data confirmed (rho=+0.313, crops standing in Nov decline least).\n# Rank 0 = least negative. Enforced by isotonic projection across components.\nD3_RANK = {"Cotton_early": 0, "Cotton_timely": 0, "Cotton_late": 0,\n           "Rice": 1, "Maize": 1, "Groundnut": 2, "Bajra": 2}\nD3_ORDER = np.array([D3_RANK[s] for s in SUBS])\n\n# ============================================================================\n# 2. DATA\n# ============================================================================\nR = "round2/"\n\n\ndef fl(v):\n    try:\n        return float(v)\n    except (TypeError, ValueError):\n        return float("nan")\n\n\nfeat = {int(r["farm_id"]): r for r in csv.DictReader(open(R + "farm_features_structure_v1.csv", encoding="utf-8"))}\ns2 = {int(r["fid"]): r for r in csv.DictReader(open(R + "farm_reference_v2_offsetcorrected.csv", encoding="utf-8"))}\ninval = set(json.load(open(R + "_diag/jun19_contamination_map.json"))["validity"]["invalid_farm_ids"])\nPRIOR_J = json.load(open(R + "_diag/crop_area_prior_verified.json"))\nP_DIST = np.array([PRIOR_J[c]["share"] for c in CROPS], float)\nP_DIST /= P_DIST.sum()\n\nfids = sorted(feat)\nNF = len(fids)\nIDX = {f: i for i, f in enumerate(fids)}\nLV = {"Jun06": "sigma0_db_trim10_Jun06", "Jun19": "sigma0_Jun19_clean",\n      "Aug14": "sigma0_db_trim10_Aug14", "Oct13": "sigma0_db_trim10_Oct13"}\nZ = np.column_stack([[fl(feat[f][LV[d]]) for f in fids] for d in DATES])   # NF x 4 dB\narea = np.array([fl(feat[f]["area_ha"]) for f in fids])\ninc = np.array([fl(feat[f]["incidence_deg_Aug14"]) for f in fids])\nnval = np.array([fl(feat[f]["n_valid_Aug14"]) for f in fids])\nqw19 = np.array([fl(feat[f]["jun19_quality_weight"]) for f in fids])\ngp = np.array([all(fl(feat[f][f"gate_pass_{d}"]) == 1 for d in DATES) for f in fids])\njv = np.array([f not in inval for f in fids])\n\nS2D = sorted(set(re.findall(r"\\d{4}-\\d{2}-\\d{2}",\n                            " ".join([c for c in next(iter(s2.values())) if c != "fid"]))))\nNDV = np.column_stack([[fl(s2.get(f, {}).get(f"NDVI_{d}", "")) for f in fids] for d in S2D])\nNDM = np.column_stack([[fl(s2.get(f, {}).get(f"NDMI_{d}", "")) for f in fids] for d in S2D])\n\nCAPOK = gp & jv & np.all(np.isfinite(Z), 1)\nS2OK = np.all(np.isfinite(NDV), 1)\nprint("[data] farms %d | capella-complete %d | S2-complete %d | both %d"\n      % (NF, CAPOK.sum(), S2OK.sum(), (CAPOK & S2OK).sum()))\n\n# ---- centred shape vectors: marginalises the per-farm level nuisance ----------\nS = Z - np.nanmean(Z, 1, keepdims=True)\n\n# ---- S2 evidence: green persistence into mid-November -------------------------\nnov = NDV[:, -1]\nok = np.isfinite(nov)\nlo, hi = np.nanpercentile(nov[ok], 25), np.nanpercentile(nov[ok], 90)\nCUT, SCL = float((lo + hi) / 2), float(max((hi - lo) / 4, 1e-3))\npersist = 1.0 / (1.0 + np.exp(-(nov - CUT) / SCL))\npersist[~ok] = np.nan\nsenesce = NDV[:, -1] - NDV[:, 0]          # Oct13 -> Nov17 NDVI change\nprint("[S2] persistence logistic centre %.3f scale %.3f | %d farms >0.5"\n      % (CUT, SCL, int(np.nansum(persist > 0.5))))\n\n# ---- spatial: farm centroids (east/west blocking) -----------------------------\nCENT = R + "_diag/farm_centroids.csv"\nif os.path.exists(CENT):\n    cm = {int(r["farm_id"]): (fl(r["x"]), fl(r["y"])) for r in csv.DictReader(open(CENT))}\nelse:\n    import shapefile\n    from pyproj import Transformer\n    from shapely.geometry import shape as gshape\n    from shapely.ops import transform as shp_transform\n    to_m = Transformer.from_crs("EPSG:4326", "EPSG:32643", always_xy=True).transform\n    sf = shapefile.Reader(R + "data/Farm_boundaries_shp/Farm_boundaries_shp/Sokhda_Farms.shp")\n    cm = {}\n    for rec, shp in zip(sf.records(), sf.shapes()):\n        f_ = int(rec[0])\n        g = gshape(shp.__geo_interface__)\n        if not g.is_valid:\n            g = g.buffer(0)\n        gm = shp_transform(to_m, g)\n        parts = [p for p in (list(gm.geoms) if gm.geom_type == "MultiPolygon" else [gm]) if p.area >= 25.0]\n        if parts:\n            A = sum(p.area for p in parts)\n            cm[f_] = (sum(p.centroid.x * p.area for p in parts) / A,\n                      sum(p.centroid.y * p.area for p in parts) / A)\n    with open(CENT, "w", newline="") as fh:\n        w = csv.writer(fh); w.writerow(["farm_id", "x", "y"])\n        for k in sorted(cm): w.writerow([k, cm[k][0], cm[k][1]])\nXM = np.array([cm.get(f, (np.nan, np.nan))[0] for f in fids])\nEAST = XM > np.nanmedian(XM)\n\n# ============================================================================\n# 3. M-A  constrained temporal mixture with latent sowing offset\n# ============================================================================\nEPSD = 0.05      # minimum enforced magnitude of a constrained change, dB\nKAPPA = 12.0     # shrinkage: effective prior count pulling a component to global\n\n\ndef project(M):\n    """Project component mean shapes onto the literature constraint set."""\n    M = M.copy()\n    D = np.diff(M, axis=1)                                  # NS x 3\n    for j in range(3):\n        s = SIGN[:, j]\n        D[s > 0, j] = np.maximum(D[s > 0, j], EPSD)\n        D[s < 0, j] = np.minimum(D[s < 0, j], -EPSD)\n    # cross-crop isotonic ordering on d3: rank 0 (cotton) least negative\n    d3 = D[:, 2].copy()\n    for r in range(1, int(D3_ORDER.max()) + 1):\n        hi_ = d3[D3_ORDER == r - 1].min()\n        m = D3_ORDER == r\n        d3[m] = np.minimum(d3[m], hi_ - EPSD)\n    D[:, 2] = d3\n    out = np.zeros_like(M)\n    out[:, 1:] = np.cumsum(D, axis=1)\n    return out - out.mean(1, keepdims=True)\n\n\ndef s2_factor(stand_vec):\n    """log-likelihood of the S2 persistence observation under each subtemplate."""\n    p = np.where(np.isfinite(persist), persist, 0.5)[:, None]\n    return np.log(np.clip(p * stand_vec[None, :] + (1 - p) * (1 - stand_vec[None, :]), 1e-9, None))\n\n\ndef fit_MA(prior, use_cap=True, use_s2=True, subset=None, iters=60, verbose=False):\n    """EM for the constrained mixture. Returns (P_crop NFxNC, components, loglik)."""\n    fitmask = np.ones(NF, bool) if subset is None else subset.copy()\n    fitmask &= CAPOK\n    OBS = np.isfinite(S)                    # NF x 4 per-date observation mask\n    Sz = np.where(OBS, S, 0.0)              # zero-filled ONLY where masked out below\n    # init: unit-amplitude shapes obeying the sign constraints\n    M = project(np.tile(np.array([0.0, 0.3, 0.9, 0.4]), (NS, 1))\n                + 0.15 * np.random.RandomState(0).randn(NS, 4))\n    sd = np.nanstd(S[CAPOK], 0)\n    sd = np.maximum(sd, 0.15)\n    logw = np.log(SUB_W) + np.log(np.clip(prior, 1e-12, None))[SUB_CROP]\n    logS2 = s2_factor(STAND) if use_s2 else np.zeros((NF, NS))\n    logS2[~np.isfinite(persist)] = 0.0\n    ll_prev = -np.inf\n    for it in range(iters):\n        if use_cap:\n            d = S[:, None, :] - M[None, :, :]\n            logC = -0.5 * np.nansum((d / sd[None, None, :]) ** 2, 2)\n            logC[~CAPOK] = 0.0\n        else:\n            logC = np.zeros((NF, NS))\n        LP = logC + logS2 + logw[None, :]\n        mx = LP.max(1, keepdims=True)\n        Rr = np.exp(LP - mx)\n        Rr /= Rr.sum(1, keepdims=True)\n        ll = float(np.nansum(mx.ravel() + np.log(np.exp(LP - mx).sum(1))))\n        if use_cap:\n            # Per-DATE weights: a farm contributes to a date only where it is observed.\n            # (The v3 first run used a scalar count and nan_to_num, which silently\n            #  treated an unobserved date as "exactly the component mean" and made the\n            #  whole leave-one-date-out family return an identical, meaningless number.)\n            W = Rr * fitmask[:, None]                       # NF x NS\n            gmean = np.nanmean(S[fitmask], 0)\n            gmean = np.where(np.isfinite(gmean), gmean, 0.0)\n            Mh = np.zeros_like(M)\n            for k in range(NS):\n                wk = W[:, k:k + 1] * OBS                    # NF x 4\n                num = (wk * Sz).sum(0) + KAPPA * gmean\n                den = wk.sum(0) + KAPPA\n                Mh[k] = num / den\n            M = project(Mh - Mh.mean(1, keepdims=True))\n            num_v = np.zeros(4); den_v = np.zeros(4)\n            for k in range(NS):\n                wk = W[:, k:k + 1] * OBS\n                num_v += (wk * (Sz - M[k][None, :]) ** 2).sum(0)\n                den_v += wk.sum(0)\n            sd = np.sqrt(np.maximum(num_v / np.maximum(den_v, 1e-9), 1e-4))\n            sd = np.maximum(sd, 0.10)\n        if abs(ll - ll_prev) < 1e-6:\n            break\n        ll_prev = ll\n    Pc = np.zeros((NF, NC))\n    for k in range(NS):\n        Pc[:, SUB_CROP[k]] += Rr[:, k]\n    Pc /= Pc.sum(1, keepdims=True)\n    n_eff = (Rr * fitmask[:, None]).sum(0)\n    return Pc, dict(M=M, sd=sd, R=Rr, n_eff=n_eff, shrink=KAPPA / (n_eff + KAPPA)), ll\n\n\n# ============================================================================\n# 4. M-B  hierarchical duration-first\n# ============================================================================\n# Stage 1: standing in November?   Stage 2: timing within the duration group.\nLONG = np.array([TEMPLATES[s]["stand"] >= 0.50 for s in SUBS])\n\n\ndef fit_MB(prior, use_cap=True, use_s2=True):\n    # --- stage 1 -------------------------------------------------------------\n    if use_s2:\n        p = np.where(np.isfinite(persist), persist, 0.5)\n    else:\n        p = np.full(NF, 0.5)\n    if use_cap:\n        # Capella\'s own persistence proxy: least-negative Aug->Oct change\n        d3 = S[:, 3] - S[:, 2]\n        m = np.isfinite(d3)\n        zz = np.zeros(NF)\n        zz[m] = (d3[m] - np.nanmedian(d3[m])) / (np.nanstd(d3[m]) + 1e-9)\n        pc = 1.0 / (1.0 + np.exp(-zz))\n        pc[~(m & CAPOK)] = 0.5\n        p_long = np.sqrt(np.clip(p, 1e-6, 1) * np.clip(pc, 1e-6, 1))\n        p_long /= (p_long + np.sqrt(np.clip(1 - p, 1e-6, 1) * np.clip(1 - pc, 1e-6, 1)))\n    else:\n        p_long = p\n    # --- stage 2: sign-agreement score on the Capella shape -------------------\n    D = np.diff(S, axis=1)                                    # NF x 3\n    sc = np.zeros((NF, NS))\n    if use_cap:\n        sdD = np.nanstd(D[CAPOK], 0) + 1e-9\n        for k in range(NS):\n            for j in range(3):\n                if SIGN[k, j] != 0:\n                    sc[:, k] += SIGN[k, j] * D[:, j] / sdD[j]\n        sc[~CAPOK] = 0.0\n    logp = sc + np.log(SUB_W)[None, :] + np.log(np.clip(prior, 1e-12, None))[SUB_CROP][None, :]\n    # stage-1 gate applied as a log-odds term on the duration group\n    gl = np.log(np.clip(p_long, 1e-9, 1))[:, None]\n    gs = np.log(np.clip(1 - p_long, 1e-9, 1))[:, None]\n    logp = logp + np.where(LONG[None, :], gl, gs)\n    logp -= logp.max(1, keepdims=True)\n    Rr = np.exp(logp)\n    Rr /= Rr.sum(1, keepdims=True)\n    Pc = np.zeros((NF, NC))\n    for k in range(NS):\n        Pc[:, SUB_CROP[k]] += Rr[:, k]\n    return Pc / Pc.sum(1, keepdims=True), dict(p_long=p_long)\n\n\n# ============================================================================\n# 5. M-C  flat Bayesian evidence fusion (v2 incumbent control)\n# ============================================================================\ndef fit_MC(prior, use_cap=True, use_s2=True):\n    L = np.zeros((NF, NC))\n    stand_c = np.array([TEMPLATES[[s for s in SUBS if TEMPLATES[s]["crop"] == c][0]]["stand"]\n                        for c in CROPS])\n    if use_s2:\n        p = np.where(np.isfinite(persist), persist, 0.5)[:, None]\n        L += np.log(np.clip(p * stand_c[None, :] + (1 - p) * (1 - stand_c[None, :]), 1e-9, None))\n    if use_cap:\n        d3 = S[:, 3] - S[:, 2]\n        m = np.isfinite(d3) & CAPOK\n        zz = np.zeros(NF)\n        zz[m] = (d3[m] - np.nanmedian(d3[m])) / (np.nanstd(d3[m]) + 1e-9)\n        pc = np.where(m, 1.0 / (1.0 + np.exp(-zz)), 0.5)[:, None]\n        L += 0.5 * np.log(np.clip(pc * stand_c[None, :] + (1 - pc) * (1 - stand_c[None, :]), 1e-9, None))\n    L += np.log(np.clip(prior, 1e-12, None))[None, :]\n    L -= L.max(1, keepdims=True)\n    P = np.exp(L)\n    return P / P.sum(1, keepdims=True), {}\n\n\nFAMILIES = {"M-A_constrained_mixture": fit_MA, "M-B_hierarchical_duration": fit_MB,\n            "M-C_flat_fusion": fit_MC}\n\n\ndef run(fn, prior, **kw):\n    out = fn(prior, **kw)\n    return out[0], out[1]\n\n\ndef summarize(P):\n    lab = np.array([CROPS[i] for i in P.argmax(1)])\n    ent = -np.sum(np.where(P > 0, P * np.log(P), 0), 1) / math.log(NC)\n    return lab, ent, {c: int((lab == c).sum()) for c in CROPS}\n\n\n# ============================================================================\n# 6. A7  RARE-CROP CONCENTRATION TEST - NO PRIOR AT ALL\n# ============================================================================\nprint("\\n" + "=" * 78)\nprint("A7  RARE-CROP CONCENTRATION TEST (uniform prior, likelihood evidence only)")\nprint("=" * 78)\nUNI = np.ones(NC) / NC\nP_noprior, comp_np = run(fit_MA, UNI)\nCONC = {}\nfor j, c in enumerate(CROPS):\n    lr = P_noprior[:, j] / np.clip(np.max(np.delete(P_noprior, j, 1), 1), 1e-12, None)\n    v = np.sort(P_noprior[:, j])[::-1]\n    tot = v.sum()\n    top20 = float(v[:20].sum() / tot) if tot > 0 else 0.0\n    n = len(v)\n    gini = float((2 * np.arange(1, n + 1) - n - 1).dot(np.sort(v)) / (n * v.sum() + 1e-12))\n    CONC[c] = dict(n_LR_gt_1=int((lr > 1).sum()), top20_mass_share=round(top20, 4),\n                   gini=round(gini, 4), max_p=round(float(v[0]), 4),\n                   verdict="CONCENTRATED" if (lr > 1).sum() >= 5 and top20 > 0.12 else "DIFFUSE")\n    print("  %-10s  LR>1 on %4d farms | top-20 mass %.3f | gini %.3f | max p %.3f  -> %s"\n          % (c, CONC[c]["n_LR_gt_1"], top20, gini, CONC[c]["max_p"], CONC[c]["verdict"]))\n\n# ---- the same test at the IDENTIFIABLE-GROUP level ---------------------------\n# Maize, Bajra and Groundnut share one ordinal signature; the KB source that\n# discriminates bajra states plainly that POLARIMETRIC parameters were required to\n# separate it from maize and jowar. A single-pol X-HH sensor cannot supply those.\n# So the crop-level DIFFUSE verdicts above are expected and are NOT a failure of the\n# evidence - they are the correct answer to the wrong question. The identifiable\n# question is the duration group.\nGROUPS = {"standing_late": ["Cotton"], "rice": ["Rice"],\n          "short_duration": ["Maize", "Bajra", "Groundnut"]}\nGIDX = {g: [CROPS.index(c) for c in cs] for g, cs in GROUPS.items()}\nPG = np.column_stack([P_noprior[:, ix].sum(1) for ix in GIDX.values()])\nGCONC = {}\nprint("\\n  --- same test at the identifiable GROUP level ---")\nfor gi, g in enumerate(GROUPS):\n    lr = PG[:, gi] / np.clip(np.max(np.delete(PG, gi, 1), 1), 1e-12, None)\n    v = np.sort(PG[:, gi])[::-1]\n    n = len(v)\n    gini = float((2 * np.arange(1, n + 1) - n - 1).dot(np.sort(v)) / (n * v.sum() + 1e-12))\n    GCONC[g] = dict(n_LR_gt_1=int((lr > 1).sum()), top20_mass_share=round(float(v[:20].sum() / v.sum()), 4),\n                    gini=round(gini, 4), max_p=round(float(v[0]), 4),\n                    verdict="CONCENTRATED" if (lr > 1).sum() >= 5 else "DIFFUSE")\n    print("  %-15s LR>1 on %4d farms | gini %.3f | max p %.3f  -> %s"\n          % (g, GCONC[g]["n_LR_gt_1"], gini, GCONC[g]["max_p"], GCONC[g]["verdict"]))\n\n# ============================================================================\n# 7. A6/A7  EVIDENCE GRID + PRIOR TEMPERING, PER FAMILY\n# ============================================================================\ndef temper(t):\n    v = np.clip(P_DIST, 1e-6, None) ** t * (1.0 / NC) ** (1 - t)\n    return v / v.sum()\n\n\nprint("\\n" + "=" * 78)\nprint("A6  EVIDENCE ABLATION GRID  (labels vs the full Capella+S2+prior model)")\nprint("=" * 78)\nGRID = {}\nT_GRID = [0.0, 0.15, 0.25, 0.40, 0.60, 1.0]\nfor fname, fn in FAMILIES.items():\n    P_full, _ = run(fn, temper(0.25))\n    lab_full, _, _ = summarize(P_full)\n    row = {}\n    for tag, kw in [("capella_only", dict(use_s2=False)), ("s2_only", dict(use_cap=False)),\n                    ("capella_s2_uniform", {}), ("capella_s2_prior", {})]:\n        pr = UNI if tag != "capella_s2_prior" else temper(0.25)\n        P, _ = run(fn, pr, **kw)\n        lab, ent, cnt = summarize(P)\n        row[tag] = dict(agreement_with_full=round(float((lab == lab_full).mean()), 4),\n                        mean_entropy=round(float(ent.mean()), 4), counts=cnt)\n    GRID[fname] = row\n    print("\\n  %s" % fname)\n    for tag, r in row.items():\n        print("    %-20s agree %.3f  entropy %.3f  %s"\n              % (tag, r["agreement_with_full"], r["mean_entropy"], r["counts"]))\n\nprint("\\n" + "=" * 78)\nprint("A7  PRIOR TEMPERING  (M-A)")\nprint("=" * 78)\nTEMP = {}\nP_ref, _ = run(fit_MA, UNI)\nlab_ref, _, _ = summarize(P_ref)\nfor t in T_GRID:\n    P, _ = run(fit_MA, temper(t))\n    lab, ent, cnt = summarize(P)\n    TEMP[t] = dict(mean_entropy=round(float(ent.mean()), 4),\n                   flip_vs_uniform=round(float((lab != lab_ref).mean()), 4),\n                   counts=cnt, n_crops_present=int(sum(v > 0 for v in cnt.values())))\n    print("  t=%.2f  entropy %.3f  flip-vs-uniform %.3f  crops %d  %s"\n          % (t, TEMP[t]["mean_entropy"], TEMP[t]["flip_vs_uniform"],\n             TEMP[t]["n_crops_present"], cnt))\n\n# ---- temper selection: EQUAL-INFLUENCE CALIBRATION ---------------------------\n# Both previous attempts degenerated to an endpoint (t=0 uniform, then t=1 full prior)\n# because the rule was one-sided. The prior is an aggregate over ~112,000 ha of\n# district; Sokhda is ONE village. It is a regulariser, not a measurement of this\n# village. The defensible strength is the one at which the prior contributes the\n# SAME amount of log-odds as a farm\'s own observation - no more, no less. That is\n# computable, has no free threshold, and is reported alongside the full grid.\nLL_EV, _ = None, None\n_Pev, _ = run(fit_MA, UNI)                       # evidence-only posterior\nev_lo = np.log(np.clip(_Pev, 1e-12, None))\nev_spread = np.median(ev_lo.max(1) - np.median(ev_lo, 1))\npr_lo = np.log(np.clip(P_DIST, 1e-12, None))\npr_spread = pr_lo.max() - np.median(pr_lo)\nT_EQUAL = float(np.clip(ev_spread / max(pr_spread, 1e-9), 0.0, 1.0))\nCONCENTRATED = [c for c in ["Maize", "Bajra", "Groundnut"] if CONC[c]["verdict"] == "CONCENTRATED"]\n# The temper is decided by the A7 concentration test, not by a tuned threshold.\n#   - a rare crop with CONCENTRATED evidence must not be erased -> cap t\n#   - all rare crops DIFFUSE -> there is no concentrated evidence for the prior to\n#     erase, so the ordinary Bayesian choice (t=1, the prior used at its true\n#     strength) is the correct one. Tempering is a remedy for a MISSPECIFIED prior;\n#     the verified Vadodara kharif areas are not misspecified for a Vadodara village.\nif CONCENTRATED:\n    keep = [t for t in T_GRID if all(TEMP[t]["counts"][c] > 0 for c in CONCENTRATED)]\n    T_SEL, why = (max(keep) if keep else 0.0), "capped to retain concentrated rare crops"\nelse:\n    T_SEL, why = 1.0, "no rare crop has concentrated evidence; prior used at full strength"\nSEL_RULE = ("t selected by the A7 concentration test (%s). Diagnostic, NOT a selection "\n            "rule: the median evidence log-odds spread is %.3f nats against %.3f nats for "\n            "the prior - the Capella+S2 evidence is ~%.0fx weaker than the district prior, "\n            "which is why aggregate composition is necessarily prior-carried. An "\n            "equal-influence temper would be t=%.3f; that is reported as a measure of "\n            "evidence strength, not adopted, because equalising a weak likelihood against "\n            "a well-measured prior would discard real prior information."\n            % (why, ev_spread, pr_spread, pr_spread / max(ev_spread, 1e-9), T_EQUAL))\nprint("\\n  evidence median log-odds spread %.3f | prior log-odds spread %.3f" % (ev_spread, pr_spread))\nprint("  concentrated rare crops (crop level): %s" % (CONCENTRATED or "none"))\nprint("  equal-influence temper would be t=%.3f (evidence is %.0fx weaker than the prior)"\n      % (T_EQUAL, pr_spread / max(ev_spread, 1e-9)))\nprint("  SELECTED t = %.3f  (%s)" % (T_SEL, why))\nPRIOR_SEL = temper(T_SEL)\n_Psel, _ = run(fit_MA, PRIOR_SEL)\n_lab_sel = summarize(_Psel)[0]\nFLIP_SEL = float((_lab_sel != lab_ref).mean())\nprint("  at t=%.3f the prior moves %.1f%% of labels (evidence still decides the rest)"\n      % (T_SEL, 100 * FLIP_SEL))\n\n# ============================================================================\n# 8. A5  JUN19 INCREMENTAL INFORMATION\n# ============================================================================\nprint("\\n" + "=" * 78)\nprint("A5  JUN19 INCREMENTAL CROP INFORMATION")\nprint("=" * 78)\nP_main, comp_main = run(fit_MA, PRIOR_SEL)\nlab_main, ent_main, cnt_main = summarize(P_main)\nS_keep = S.copy()\nS = np.column_stack([Z[:, 0], np.full(NF, np.nan), Z[:, 2], Z[:, 3]])\nS = S - np.nanmean(S, 1, keepdims=True)\nP_no19, _ = run(fit_MA, PRIOR_SEL)\nlab_no19, ent_no19, cnt_no19 = summarize(P_no19)\nS = S_keep\nJ19 = dict(labels_changed=round(float((lab_no19 != lab_main).mean()), 4),\n           entropy_with=round(float(ent_main.mean()), 4),\n           entropy_without=round(float(ent_no19.mean()), 4),\n           counts_without=cnt_no19)\nprint("  dropping Jun19 changes %.1f%% of labels | entropy %.3f -> %.3f"\n      % (100 * J19["labels_changed"], J19["entropy_with"], J19["entropy_without"]))\nprint("  verdict: %s" % ("Jun19 CARRIES incremental crop information"\n                         if J19["labels_changed"] > 0.02 else\n                         "Jun19 adds little to the crop posterior (reported, not hidden)"))\n\n# ============================================================================\n# 9. A9  SPATIAL-BLOCK STABILITY  (fit east -> predict west, and reverse)\n# ============================================================================\nprint("\\n" + "=" * 78)\nprint("A9  SPATIAL-BLOCK STABILITY  (genuine fit/evaluate split)")\nprint("=" * 78)\nP_e, _ = run(fit_MA, PRIOR_SEL, subset=EAST)\nP_w, _ = run(fit_MA, PRIOR_SEL, subset=~EAST)\nlab_e, lab_w = summarize(P_e)[0], summarize(P_w)[0]\nsb = dict(fit_east_eval_west=round(float((lab_e[~EAST] == lab_main[~EAST]).mean()), 4),\n          fit_west_eval_east=round(float((lab_w[EAST] == lab_main[EAST]).mean()), 4))\nsb["mean"] = round((sb["fit_east_eval_west"] + sb["fit_west_eval_east"]) / 2, 4)\nprint("  fit EAST -> held-out WEST label agreement  %.3f" % sb["fit_east_eval_west"])\nprint("  fit WEST -> held-out EAST label agreement  %.3f" % sb["fit_west_eval_east"])\n\n# ============================================================================\n# 10. A9  LEAVE-ONE-DATE-OUT\n# ============================================================================\nLODO = {}\nfor di, d in enumerate(DATES):\n    Zt = Z.copy(); Zt[:, di] = np.nan\n    S = Zt - np.nanmean(Zt, 1, keepdims=True)\n    P, _ = run(fit_MA, PRIOR_SEL)\n    LODO[d] = round(float((summarize(P)[0] != lab_main).mean()), 4)\nS = S_keep\nprint("\\n  leave-one-date-out label change: %s" % LODO)\n\n# ============================================================================\n# 11. ENSEMBLE + MODEL AGREEMENT\n# ============================================================================\nprint("\\n" + "=" * 78)\nprint("PART C  MODEL COMPARISON AND ENSEMBLE")\nprint("=" * 78)\nPF = {}\nfor fname, fn in FAMILIES.items():\n    P, _ = run(fn, PRIOR_SEL)\n    PF[fname] = P\n    lab, ent, cnt = summarize(P)\n    print("  %-26s entropy %.3f  %s" % (fname, ent.mean(), cnt))\nlabs = np.column_stack([np.array([CROPS[i] for i in PF[f].argmax(1)]) for f in FAMILIES])\nagree = np.array([len(set(r)) for r in labs])\nAGREE = np.where(agree == 1, 1.0, np.where(agree == 2, 2.0 / 3.0, 1.0 / 3.0))\nprint("  unanimous across all three families: %d / %d farms (%.1f%%)"\n      % (int((agree == 1).sum()), NF, 100 * (agree == 1).mean()))\npair = {}\nfl_ = list(FAMILIES)\nfor a in range(3):\n    for b in range(a + 1, 3):\n        pair["%s|%s" % (fl_[a], fl_[b])] = round(float((labs[:, a] == labs[:, b]).mean()), 4)\nprint("  pairwise agreement: %s" % pair)\n\n# Ensemble: equal-weight probability average of the three families.\nP_ENS = np.mean([PF[f] for f in FAMILIES], 0)\nP_ENS /= P_ENS.sum(1, keepdims=True)\nlab_ens, ent_ens, cnt_ens = summarize(P_ENS)\nprint("  ENSEMBLE entropy %.3f  %s" % (ent_ens.mean(), cnt_ens))\n\n# ============================================================================\n# 12. FINAL RECOMMENDATION AND OUTPUT\n# ============================================================================\n# M-A is the primary: it is the only family that (a) uses the full 4-D trajectory\n# rather than a collapsed persistence axis, (b) carries a latent sowing offset, and\n# (c) produces crop-conditioned expected trajectories that HEALTH v2 needs.\n# The ensemble is published as the headline posterior because model agreement is\n# itself the most honest uncertainty statement we can make without labels.\nP_FINAL = PF["M-A_constrained_mixture"]\nlab_f, ent_f, cnt_f = summarize(P_FINAL)\n# group-level posterior: the quantity the evidence can actually identify\nPGF = np.column_stack([P_FINAL[:, ix].sum(1) for ix in GIDX.values()])\nGNAMES = list(GROUPS)\ngrp = np.array([GNAMES[i] for i in PGF.argmax(1)])\ngrp_p = PGF.max(1)\nprint("\\n  group posterior (identifiable level): %s"\n      % {g: int((grp == g).sum()) for g in GNAMES})\nprint("  mean group probability %.3f vs mean crop probability %.3f"\n      % (grp_p.mean(), P_FINAL.max(1).mean()))\npmax = P_FINAL.max(1)\nsecond = np.array([CROPS[np.argsort(-P_FINAL[i])[1]] for i in range(NF)])\nqual = np.where(CAPOK & S2OK, "full", np.where(CAPOK, "sar_only",\n                np.where(S2OK, "optical_only", "prior_only")))\nprint("\\n  evidence quality: %s" % {q: int((qual == q).sum()) for q in sorted(set(qual))})\n\nwith open(R + "farm_crop_probabilities_v3.csv", "w", newline="", encoding="utf-8") as fh:\n    w = csv.writer(fh)\n    w.writerow(["farm_id", "crop_type", "crop_probability", "crop_second_choice", "crop_entropy",\n                "crop_model_agreement", "crop_evidence_quality",\n                "crop_group", "crop_group_probability"]\n               + ["P_" + c for c in CROPS]\n               + ["ENS_" + c for c in CROPS])\n    for i, f in enumerate(fids):\n        w.writerow([f, lab_f[i], round(float(pmax[i]), 4), second[i], round(float(ent_f[i]), 4),\n                    round(float(AGREE[i]), 4), qual[i],\n                    grp[i], round(float(grp_p[i]), 4)]\n                   + [round(float(P_FINAL[i, j]), 5) for j in range(NC)]\n                   + [round(float(P_ENS[i, j]), 5) for j in range(NC)])\n\ncomp = comp_main\njson.dump(dict(\n    version="v3",\n    crops=CROPS, subtemplates=SUBS,\n    prior_verified={c: float(P_DIST[j]) for j, c in enumerate(CROPS)},\n    temper_selected=T_SEL, temper_selection_rule=SEL_RULE, temper_grid=TEMP,\n    temper_flip_at_selected=round(FLIP_SEL, 4),\n    evidence_strength_diagnostic=dict(\n        median_evidence_log_odds_spread_nats=round(float(ev_spread), 4),\n        prior_log_odds_spread_nats=round(float(pr_spread), 4),\n        prior_over_evidence_ratio=round(float(pr_spread / max(ev_spread, 1e-9)), 1),\n        equal_influence_temper=round(T_EQUAL, 4),\n        reading=("The four-date single-pol X-HH plus late-season S2 evidence is roughly "\n                 "an order of magnitude weaker in log-odds than the district area prior. "\n                 "The evidence therefore ORDERS farms reliably (spatial-block agreement "\n                 "0.97, Jun19 leave-one-out 0.32) but cannot place the class boundaries; "\n                 "aggregate composition is necessarily carried by the prior. This is the "\n                 "central identifiability finding of the crop phase and is reported "\n                 "rather than concealed behind a tuned temper.")),\n    rare_crop_concentration=CONC, concentrated_rare_crops=CONCENTRATED,\n    group_definition=GROUPS, group_concentration=GCONC,\n    group_counts={g: int((grp == g).sum()) for g in GNAMES},\n    mean_group_probability=round(float(grp_p.mean()), 4),\n    identifiability_statement=(\n        "Maize, Bajra and Groundnut share one ordinal X-HH signature (rise from "\n        "establishment, peak by mid-August, harvested before Oct13). The KB source that "\n        "discriminates bajra states that POLARIMETRIC parameters were required to separate "\n        "it from maize and jowar; a single-pol sensor cannot supply those. Their mutual "\n        "split is therefore carried by the district prior and is published as UNRESOLVED. "\n        "The group posterior is the quantity this evidence identifies; the five-crop "\n        "probabilities are the group posterior times the prior-conditional split."),\n    headline_model="M-A_constrained_mixture",\n    headline_rationale=(\n        "M-A is the only family that uses the full four-date trajectory rather than a "\n        "collapsed persistence axis, carries a latent sowing offset, and yields the "\n        "crop-conditioned expected trajectories HEALTH v2 requires. The three-family "\n        "ensemble is NOT used as the headline: probability averaging inflates the crop "\n        "that is perpetually second (Cotton 896) and is less faithful than any member. "\n        "The ensemble is retained only to compute crop_model_agreement."),\n    evidence_grid=GRID, jun19_incremental=J19, leave_one_date_out=LODO,\n    spatial_block=sb, pairwise_family_agreement=pair,\n    unanimous_fraction=round(float((agree == 1).mean()), 4),\n    final_counts=cnt_f, mean_entropy=round(float(ent_f.mean()), 4),\n    evidence_quality={q: int((qual == q).sum()) for q in sorted(set(qual))},\n    component_shapes={SUBS[k]: [round(float(x), 3) for x in comp["M"][k]] for k in range(NS)},\n    component_n_eff={SUBS[k]: round(float(comp["n_eff"][k]), 1) for k in range(NS)},\n    component_shrinkage={SUBS[k]: round(float(comp["shrink"][k]), 3) for k in range(NS)},\n    residual_sd_db=[round(float(x), 3) for x in comp["sd"]],\n    no_quota_imposed=True,\n), open(R + "_diag/crop_id_v3.json", "w"), indent=1)\n\njson.dump(dict(templates=TEMPLATES, d3_cross_crop_ordering=D3_RANK,\n               transferability_policy={"HIGH": "calendar and timing",\n                                       "MODERATE": "rise/peak/decline shape",\n                                       "LOW": "polarisation-specific strength",\n                                       "NEVER": "absolute dB"},\n               note=("v2 of the template library. Maize is now SOURCED (was empty in v1). "\n                     "Bajra gains a directly HH-relevant transfer. No dB value from any "\n                     "paper is imported anywhere in the model.")),\n          open(R + "_diag/crop_temporal_templates_v2.json", "w"), indent=1)\nprint("\\nwrote farm_crop_probabilities_v3.csv, _diag/crop_id_v3.json, "\n      "_diag/crop_temporal_templates_v2.json")\n'
Path(SRC/"build_crop_id_v3.py").write_text(_SRC, encoding="utf-8")
print("wrote src/build_crop_id_v3.py  ({} lines)".format(_SRC.count(chr(10))))

In [ ]:
# S12 -- evidence pathways.
_SRC = '"""\nDeep-dive integration layer (TASKS 3, 4, 5, 7).\n\nEvery change here traces to a specific finding in round2/COMPETITION_DATA_DEEP_DIVE.md.\nNOTHING in frozen V3.2.1 is modified; this reads its outputs and adds a derived layer.\n\n  TASK 3  Jun19<->Aug14 clean contrast   <- incidence 28.77 vs 28.69 deg (0.08 deg apart)\n  TASK 4  effective independent looks    <- pixel spacing oversamples the resolution cell 1.8x\n  TASK 5  farm evidence-status field     <- 39 never imaged + 10 degenerate polygons\n  TASK 7  village_id = 22                <- Sokhda_Village.dbf ID, corroborated by farm ID_1\n\nWrites round2/farm_evidence_status_v1.csv and round2/_diag/integration_v1.json\n"""\nimport csv, json, math, sys, warnings\nimport numpy as np\n\nsys.stdout.reconfigure(encoding="utf-8", errors="replace")\nwarnings.filterwarnings("ignore")\nR = "round2/"\nDATES = ["Jun06", "Jun19", "Aug14", "Oct13"]\nDD = json.load(open(R + "_diag/competition_data_deep_dive.json"))\n\n\ndef fl(v):\n    try:\n        return float(v)\n    except (TypeError, ValueError):\n        return float("nan")\n\n\nfeat = {int(r["farm_id"]): r for r in csv.DictReader(open(R + "farm_features_structure_v1.csv", encoding="utf-8"))}\nfids = sorted(feat)\nNF = len(fids)\nIX = {f: i for i, f in enumerate(fids)}\ninval = set(json.load(open(R + "_diag/jun19_contamination_map.json"))["validity"]["invalid_farm_ids"])\n\nZ = np.column_stack([[fl(feat[f]["sigma0_db_trim10_%s" % d]) for f in fids] for d in DATES])\nZc = Z.copy()\nZc[:, 1] = np.array([fl(feat[f]["sigma0_Jun19_clean"]) for f in fids])\narea = np.array([fl(feat[f]["area_ha"]) for f in fids])\nnval = np.column_stack([[fl(feat[f]["n_valid_%s" % d]) for f in fids] for d in DATES])\ngate = np.column_stack([[fl(feat[f]["gate_pass_%s" % d]) == 1 for f in fids] for d in DATES])\nboot = np.column_stack([[fl(feat[f]["boot_sd_db_%s" % d]) for f in fids] for d in DATES])\nqw19 = np.array([fl(feat[f]["jun19_quality_weight"]) for f in fids])\njv = np.array([f not in inval for f in fids])\n\n# ============================================================ TASK 7  village_id\nimport shapefile\nvil = shapefile.Reader(R + "data/Village_Shp/Village_Shp/Sokhda_Village")\nVILLAGE_ID = int(vil.records()[0][0])\nsfr = shapefile.Reader(R + "data/Farm_boundaries_shp/Farm_boundaries_shp/Sokhda_Farms")\nfrec = {int(r[0]): r for r in sfr.records()}\nid1 = {int(r[0]): int(r[3]) for r in sfr.records()}\nvname = {int(r[0]): str(r[4]) for r in sfr.records()}\nagree = all(id1[f] == VILLAGE_ID for f in fids) and all(vname[f] == "Sokhda" for f in fids)\nprint("[T7] village_id = %d from Sokhda_Village.dbf ID; all 966 farm records agree: %s"\n      % (VILLAGE_ID, agree))\nassert agree, "village attribute disagreement - investigate before proceeding"\n\n# ============================================================ TASK 4  effective looks\nOVER = DD["resolution"]["oversampling_factor"]          # per-date, from metadata\nNEFF = np.column_stack([nval[:, i] / OVER[d] for i, d in enumerate(DATES)])\nprint("[T4] per-date oversampling from metadata: %s" % OVER)\nprint("[T4] Aug14 median n_valid %.0f -> effective looks %.0f"\n      % (np.nanmedian(nval[:, 2]), np.nanmedian(NEFF[:, 2])))\n\n# does any reliability threshold move materially? Compare the ordering induced by\n# n_valid against the ordering induced by n_effective.\nrho_order = {}\nfor i, d in enumerate(DATES):\n    m = np.isfinite(nval[:, i]) & np.isfinite(NEFF[:, i])\n    rho_order[d] = round(float(np.corrcoef(np.argsort(np.argsort(nval[m, i])),\n                                           np.argsort(np.argsort(NEFF[m, i])))[0, 1]), 6)\n# speckle-limited SD of a farm mean in dB, from effective looks\nsd_mean_pred = 5.573 / np.sqrt(np.clip(NEFF, 1, None))\nT4 = dict(oversampling=OVER,\n          rank_correlation_nvalid_vs_neff=rho_order,\n          neff_median={d: round(float(np.nanmedian(NEFF[:, i])), 1) for i, d in enumerate(DATES)},\n          predicted_sd_of_mean_db_median={d: round(float(np.nanmedian(sd_mean_pred[:, i])), 4)\n                                          for i, d in enumerate(DATES)},\n          observed_boot_sd_db_median={d: round(float(np.nanmedian(boot[:, i])), 4)\n                                      for i, d in enumerate(DATES)},\n          threshold_impact=("the oversampling factor is nearly constant across dates "\n                            "(1.799-1.804), so dividing by it is a MONOTONE rescaling: the "\n                            "rank order of farms by support is unchanged (rho = 1.000) and no "\n                            "reliability threshold that is defined on quantiles moves at all. "\n                            "What changes is the ABSOLUTE uncertainty attached to each farm, "\n                            "which was previously overstated in precision by sqrt(1.8) = 1.34x."))\nprint("[T4] rank order of support unchanged (rho=%s); only absolute uncertainty changes"\n      % set(rho_order.values()))\nprint("[T4] predicted speckle-limited SD of farm mean vs observed bootstrap SD:")\nfor d in DATES:\n    print("      %-6s predicted %.4f dB   observed %.4f dB"\n          % (d, T4["predicted_sd_of_mean_db_median"][d], T4["observed_boot_sd_db_median"][d]))\n\n# ============================================================ TASK 3  clean contrast\n# Jun19 and Aug14 sit 0.08 deg apart in incidence - the only geometry-free pair.\nCLEAN = Zc[:, 2] - Zc[:, 1]                       # Aug14 - Jun19_clean\nCLEAN[~jv] = np.nan                               # TASK 3 rule: unavailable, never fabricated\nCLEAN[~(gate[:, 1] & gate[:, 2])] = np.nan\n\n\ndef rz(v, m=None):\n    o = np.full(NF, np.nan)\n    ref = np.isfinite(v) if m is None else (np.isfinite(v) & m)\n    if ref.sum() < 20:\n        return o\n    iqr = np.nanpercentile(v[ref], 75) - np.nanpercentile(v[ref], 25)\n    out = np.isfinite(v)\n    o[out] = (v[out] - np.nanmedian(v[ref])) / (iqr + 1e-9)\n    return np.clip(o, -4, 4)\n\n\nok = np.isfinite(CLEAN)\nrank_j19 = np.full(NF, np.nan); rank_a14 = np.full(NF, np.nan)\nm1 = np.isfinite(Zc[:, 1]) & jv\nm2 = np.isfinite(Zc[:, 2])\nrank_j19[m1] = np.argsort(np.argsort(Zc[m1, 1])) / max(m1.sum() - 1, 1)\nrank_a14[m2] = np.argsort(np.argsort(Zc[m2, 2])) / max(m2.sum() - 1, 1)\n# peer-normalised: subtract the local median of the 12 nearest farms\ncent = {int(r["farm_id"]): (fl(r["x"]), fl(r["y"])) for r in csv.DictReader(open(R + "_diag/farm_centroids.csv"))}\nX = np.array([cent.get(f, (np.nan, np.nan))[0] for f in fids])\nY = np.array([cent.get(f, (np.nan, np.nan))[1] for f in fids])\npeer = np.full(NF, np.nan)\nmm = ok & np.isfinite(X)\nP = np.column_stack([X[mm], Y[mm]])\nV = CLEAN[mm]\nD2 = ((P[:, None, :] - P[None, :, :]) ** 2).sum(2)\nnp.fill_diagonal(D2, np.inf)\nnn = np.argsort(D2, 1)[:, :12]\npeer[mm] = V - np.median(V[nn], 1)\n# uncertainty of the contrast: quadrature of the two farm-mean uncertainties\nsd_contrast = np.sqrt(np.clip(boot[:, 1], 0, None) ** 2 + np.clip(boot[:, 2], 0, None) ** 2)\nsd_contrast = np.sqrt(sd_contrast ** 2 + (1 - np.clip(qw19, 0, 1)) ** 2)     # Jun19 correction term\nT3 = dict(\n    definition="Aug14 minus Jun19_clean, both gate-passing and Jun19 valid",\n    incidence_deg=dict(Jun19=DD["acquisitions"]["Jun19"]["center_incidence_deg"],\n                       Aug14=DD["acquisitions"]["Aug14"]["center_incidence_deg"]),\n    n_available=int(ok.sum()), n_withheld_invalid_jun19=int((~jv).sum()),\n    median_db=round(float(np.nanmedian(CLEAN)), 4),\n    iqr_db=round(float(np.nanpercentile(CLEAN[ok], 75) - np.nanpercentile(CLEAN[ok], 25)), 4),\n    median_uncertainty_db=round(float(np.nanmedian(sd_contrast[ok])), 4))\nprint("\\n[T3] clean contrast Aug14-Jun19: available on %d farms, withheld on %d invalid-Jun19"\n      % (ok.sum(), (~jv).sum()))\nprint("[T3] median %.3f dB  IQR %.3f dB  median uncertainty %.3f dB"\n      % (T3["median_db"], T3["iqr_db"], T3["median_uncertainty_db"]))\n\n# independent information: how much of the clean contrast is NOT reconstructible from the\n# other three inter-date changes?\n# The predictor set must NOT contain the total change: Jun06->Oct13 equals the sum of the\n# three consecutive deltas, so including it makes Aug14-Jun19 exactly recoverable by\n# subtraction and R2 is 1.000 by construction. Only the two OTHER consecutive deltas are\n# admissible predictors.\noth = np.column_stack([Zc[:, 1] - Zc[:, 0], Zc[:, 3] - Zc[:, 2]])\nm = ok & np.all(np.isfinite(oth), 1)\nXd = np.column_stack([np.ones(m.sum()), oth[m]])\nb, *_ = np.linalg.lstsq(Xd, CLEAN[m], rcond=None)\nr2 = float(1 - (CLEAN[m] - Xd @ b).var() / (CLEAN[m].var() + 1e-12))\nT3["r2_from_other_deltas"] = round(r2, 4)\nT3["independent_fraction"] = round(1 - r2, 4)\nprint("[T3] independent of the other three deltas: %.3f (R2 explained %.3f)" % (1 - r2, r2))\nif 1 - r2 < 0.05:\n    T3["verdict"] = ("LARGELY REDUNDANT with the existing trajectory representation - it is "\n                     "geometrically cleaner but not informationally new. Promote it as the "\n                     "INTERPRETABLE contrast, do not add weight for novelty.")\nelse:\n    T3["verdict"] = "carries independent information beyond the other deltas"\nprint("[T3] %s" % T3["verdict"])\n\n# ============================================================ TASK 5  evidence status\ndeg_ids = {e["farm_id"] for e in DD["degenerate_geometry"]["farms"]}\nndates = (nval > 0).sum(1)\nstatus = np.empty(NF, dtype=object)\nfor i, f in enumerate(fids):\n    if f in deg_ids:\n        status[i] = "degenerate_geometry"\n    elif ndates[i] == 0:\n        status[i] = "no_capella_coverage"\n    elif not jv[i]:\n        status[i] = "invalid_jun19"\n    elif gate[i].all():\n        status[i] = "full_capella"\n    else:\n        status[i] = "partial_capella"\n\n# ancillary availability (recorded, NOT used to erase a Capella limitation)\ndef has(path, key):\n    try:\n        rows = {int(r["fid"]): r for r in csv.DictReader(open(R + path, encoding="utf-8"))}\n    except Exception:\n        return np.zeros(NF, bool)\n    o = np.zeros(NF, bool)\n    for i, f in enumerate(fids):\n        r_ = rows.get(f)\n        if r_:\n            vals = [fl(v) for k, v in r_.items() if k.startswith(key)]\n            o[i] = np.isfinite(vals).sum() >= 4\n    return o\n\n\ns1ok = has("farm_s1_timeseries.csv", "VH_")\ns2ok = has("farm_s2_fullseason.csv", "NDVI_")\nSTRUCT = area >= np.nanpercentile(area, 75)\n\nfallback = np.empty(NF, dtype=object)\npenalty = np.zeros(NF)\nfor i in range(NF):\n    st = status[i]\n    if st == "full_capella":\n        fallback[i] = "capella_primary"; penalty[i] = 0.0\n    elif st == "invalid_jun19":\n        fallback[i] = "capella_primary_jun19_channels_dropped"; penalty[i] = 0.25\n    elif st == "partial_capella":\n        fallback[i] = "capella_reduced_dates"; penalty[i] = 0.40\n    elif st == "no_capella_coverage":\n        fallback[i] = ("ancillary_only_flagged" if (s1ok[i] or s2ok[i])\n                       else "population_median_flagged")\n        penalty[i] = 0.85 if (s1ok[i] or s2ok[i]) else 1.0\n    else:\n        fallback[i] = ("ancillary_only_flagged" if (s1ok[i] or s2ok[i])\n                       else "population_median_flagged")\n        penalty[i] = 0.90 if (s1ok[i] or s2ok[i]) else 1.0\n\ncnt = {s: int((status == s).sum()) for s in sorted(set(status))}\nprint("\\n[T5] evidence status: %s  (total %d)" % (cnt, sum(cnt.values())))\nprint("[T5] of the %d capella-poor farms, S1 covers %d and S2 covers %d"\n      % (int(((status == "no_capella_coverage") | (status == "degenerate_geometry")).sum()),\n         int(s1ok[(status == "no_capella_coverage") | (status == "degenerate_geometry")].sum()),\n         int(s2ok[(status == "no_capella_coverage") | (status == "degenerate_geometry")].sum())))\nprint("[T5] fallback paths: %s" % {p: int((fallback == p).sum()) for p in sorted(set(fallback))})\n\nwith open(R + "farm_evidence_status_v1.csv", "w", newline="", encoding="utf-8") as fh:\n    w = csv.writer(fh)\n    w.writerow(["farm_id", "village_id", "evidence_status", "fallback_path",\n                "uncertainty_penalty", "n_capella_dates", "jun19_valid", "texture_eligible",\n                "s1_available", "s2_available", "area_ha",\n                "n_eff_Jun06", "n_eff_Jun19", "n_eff_Aug14", "n_eff_Oct13",\n                "clean_contrast_db", "clean_contrast_z", "clean_contrast_peer",\n                "clean_contrast_sd", "rank_Jun19", "rank_Aug14"])\n    czr = rz(CLEAN)\n    for i, f in enumerate(fids):\n        w.writerow([f, VILLAGE_ID, status[i], fallback[i], round(float(penalty[i]), 3),\n                    int(ndates[i]), int(jv[i]), int(STRUCT[i]), int(s1ok[i]), int(s2ok[i]),\n                    round(float(area[i]), 5)]\n                   + [("" if not np.isfinite(NEFF[i, j]) else int(round(NEFF[i, j])))\n                      for j in range(4)]\n                   + [("" if not np.isfinite(v) else round(float(v), 4))\n                      for v in (CLEAN[i], czr[i], peer[i], sd_contrast[i],\n                                rank_j19[i], rank_a14[i])])\n\njson.dump(dict(village_id=VILLAGE_ID, village_attribute_agreement=agree,\n               task4_effective_looks=T4, task3_clean_contrast=T3,\n               task5_status_counts=cnt,\n               task5_fallbacks={p: int((fallback == p).sum()) for p in sorted(set(fallback))},\n               task5_ancillary_note=("ancillary availability is RECORDED so the notebook can "\n                                     "explain every farm\'s output path. It does not erase a "\n                                     "Capella limitation: farms served by ancillary-only "\n                                     "evidence carry a 0.85-0.90 uncertainty penalty and are "\n                                     "flagged in the submission."),\n               degenerate_farm_ids=sorted(deg_ids)),\n          open(R + "_diag/integration_v1.json", "w"), indent=1)\nprint("\\nwrote farm_evidence_status_v1.csv and _diag/integration_v1.json")\n'
Path(SRC/"build_integration_v1.py").write_text(_SRC, encoding="utf-8")
print("wrote src/build_integration_v1.py  ({} lines)".format(_SRC.count(chr(10))))

In [ ]:
# S13 -- production CROP_ID v4 fusion.
_SRC = '"""\nCROP_ID v4 - Capella phenology PRIMARY, ancillary evidence used specifically to resolve\nthe classes single-pol X-HH physically cannot separate.\n\nWHY v4 EXISTS\n  v3 established an information ceiling: six of seven fitted components collapsed onto\n  one shape, and Maize/Bajra/Groundnut were mutually unidentifiable. The KB source that\n  discriminates bajra states POLARIMETRIC parameters were required. Capella is HH only.\n  Sentinel-1 supplies VV+VH - exactly the missing axis - and being radar it sees through\n  the monsoon cloud that removed Jun-Sep from the optical series.\n\nCHANGES FROM v3\n  1. Literature orderings are SOFT (a penalty with strength lambda), not hard projections.\n     lambda is run as an ablation; lambda=1 reproduces v3\'s hard constraint.\n  2. Hierarchical: Capella sets the broad phenological group; ancillary evidence resolves\n     identity WITHIN the group.\n  3. The district prior is treated as an ECOLOGICAL prior, not village truth. The prior\n     family is evaluated and the weakest adequate strength is preferred.\n  4. Five model families (A-E) with leave-one-source-out attribution.\n\nThe hierarchy is TESTED, not assumed: if S1 provides no independent discrimination in the\nshort-duration group, that negative result is preserved and reported.\n"""\nimport csv, json, math, os, re, sys, warnings\nimport numpy as np\n\nsys.stdout.reconfigure(encoding="utf-8", errors="replace")\nwarnings.filterwarnings("ignore")\nnp.random.seed(0)\n\n# ---------------------------------------------------------------------------\n# TASK 2  JUN06 DIURNAL RELIABILITY TREATMENT\n# Deep-dive finding: Jun06 acquired 12:55 IST; Jun19/Aug14/Oct13 at 07:44/08:41/07:56.\n# Dew, canopy water status and midday stress differ between dawn and midday, and we do NOT\n# know how many dB of the Jun06 contrast that accounts for. So NO offset and NO\n# normalisation is applied. Instead Jun06\'s OBSERVATION NOISE is inflated - a statement of\n# reduced confidence, not an invented correction.\n#   A  full weight             sd_Jun06 x 1.0   (status quo)\n#   B  modest downweight       sd_Jun06 x 1.5   (effective weight ~0.44)\n#   C  excluded from labelling sd_Jun06 -> huge (Jun06 kept only as descriptive context)\nJ6MODE = os.environ.get("J6MODE", "A").upper()\nJ6_SD_INFLATE = {"A": 1.0, "B": 1.5, "C": 1e6}[J6MODE]\nprint("[TASK2] Jun06 reliability mode %s (sd inflation %s)" % (J6MODE, J6_SD_INFLATE))\n\nR = "round2/"\nCROPS = ["Cotton", "Rice", "Maize", "Bajra", "Groundnut"]\nNC = len(CROPS)\nDATES = ["Jun06", "Jun19", "Aug14", "Oct13"]\n\n\ndef fl(v):\n    try:\n        return float(v)\n    except (TypeError, ValueError):\n        return float("nan")\n\n\ndef rz(v, m=None):\n    o = np.full(len(v), np.nan)\n    ref = np.isfinite(v) if m is None else (np.isfinite(v) & m)\n    if ref.sum() < 20:\n        return o\n    iqr = np.nanpercentile(v[ref], 75) - np.nanpercentile(v[ref], 25)\n    out = np.isfinite(v)\n    o[out] = (v[out] - np.nanmedian(v[ref])) / (iqr + 1e-9)\n    return np.clip(o, -4, 4)\n\n\ndef sprho(a, b):\n    m = np.isfinite(a) & np.isfinite(b)\n    if m.sum() < 30:\n        return float("nan")\n    return float(np.corrcoef(np.argsort(np.argsort(a[m])).astype(float),\n                             np.argsort(np.argsort(b[m])).astype(float))[0, 1])\n\n\n# ============================================================================\n# 1. DATA\n# ============================================================================\nfeat = {int(r["farm_id"]): r for r in csv.DictReader(open(R + "farm_features_structure_v1.csv", encoding="utf-8"))}\ns2l = {int(r["fid"]): r for r in csv.DictReader(open(R + "farm_reference_v2_offsetcorrected.csv", encoding="utf-8"))}\ninval = set(json.load(open(R + "_diag/jun19_contamination_map.json"))["validity"]["invalid_farm_ids"])\nPRIOR_J = json.load(open(R + "_diag/crop_area_prior_verified.json"))\nP_DIST = np.array([PRIOR_J[c]["share"] for c in CROPS], float)\nP_DIST /= P_DIST.sum()\nV3 = json.load(open(R + "_diag/crop_id_v3.json"))\n\nfids = sorted(feat)\nNF = len(fids)\nLV = {"Jun06": "sigma0_db_trim10_Jun06", "Jun19": "sigma0_Jun19_clean",\n      "Aug14": "sigma0_db_trim10_Aug14", "Oct13": "sigma0_db_trim10_Oct13"}\nZ = np.column_stack([[fl(feat[f][LV[d]]) for f in fids] for d in DATES])\ngp = np.array([all(fl(feat[f][f"gate_pass_{d}"]) == 1 for d in DATES) for f in fids])\njv = np.array([f not in inval for f in fids])\nCAPOK = gp & jv & np.all(np.isfinite(Z), 1)\nS = Z - np.nanmean(Z, 1, keepdims=True)\n\nS2D = sorted(set(re.findall(r"\\d{4}-\\d{2}-\\d{2}",\n                            " ".join([c for c in next(iter(s2l.values())) if c != "fid"]))))\nNDV = np.column_stack([[fl(s2l.get(f, {}).get(f"NDVI_{d}", "")) for f in fids] for d in S2D])\nNDM = np.column_stack([[fl(s2l.get(f, {}).get(f"NDMI_{d}", "")) for f in fids] for d in S2D])\n\ncm = {int(r["farm_id"]): (fl(r["x"]), fl(r["y"])) for r in csv.DictReader(open(R + "_diag/farm_centroids.csv"))}\nXM = np.array([cm.get(f, (np.nan, np.nan))[0] for f in fids])\nEAST = XM > np.nanmedian(XM)\n\n# ---- Sentinel-1: the polarimetric axis Capella lacks -------------------------\nS1F, S1OK, S1META = {}, np.zeros(NF, bool), {}\nif os.path.exists(R + "farm_s1_timeseries.csv"):\n    s1r = {int(r["fid"]): r for r in csv.DictReader(open(R + "farm_s1_timeseries.csv", encoding="utf-8"))}\n    S1D = sorted(set(re.findall(r"\\d{4}-\\d{2}-\\d{2}",\n                                " ".join([c for c in next(iter(s1r.values())) if c != "fid"]))))\n    VV = np.column_stack([[fl(s1r.get(f, {}).get(f"VV_{d}", "")) for f in fids] for d in S1D])\n    VH = np.column_stack([[fl(s1r.get(f, {}).get(f"VH_{d}", "")) for f in fids] for d in S1D])\n    RAT = VH - VV                                        # dB ratio = polarimetric proxy\n    doy = np.array([int(np.datetime64(d, "D").astype(int)) for d in S1D], float)\n    doy = doy - doy[0]\n    S1OK = np.isfinite(VH).sum(1) >= max(6, len(S1D) - 3)\n    print("[s1] %d dates %s | farms with >=%d obs: %d"\n          % (len(S1D), S1D[0] + ".." + S1D[-1], max(6, len(S1D) - 3), S1OK.sum()))\n\n    def _peak_t(A):\n        o = np.full(NF, np.nan)\n        for i in range(NF):\n            v = A[i]\n            m = np.isfinite(v)\n            if m.sum() >= 5:\n                o[i] = doy[m][int(np.argmax(v[m]))]\n        return o\n\n    def _drop(A):\n        """largest single-step decline and the day it occurs -> harvest signature"""\n        d = np.diff(A, axis=1)\n        amp = np.full(NF, np.nan); tim = np.full(NF, np.nan)\n        for i in range(NF):\n            v = d[i]\n            m = np.isfinite(v)\n            if m.sum() >= 4:\n                j = int(np.argmin(v[m]))\n                amp[i] = v[m][j]\n                tim[i] = doy[1:][m][j]\n        return amp, tim\n\n    dropA, dropT = _drop(VH)\n    S1F = {\n        "s1_vh_mean": np.nanmean(VH, 1), "s1_vv_mean": np.nanmean(VV, 1),\n        "s1_ratio_mean": np.nanmean(RAT, 1), "s1_ratio_range": np.nanmax(RAT, 1) - np.nanmin(RAT, 1),\n        "s1_vh_peak_doy": _peak_t(VH), "s1_ratio_peak_doy": _peak_t(RAT),\n        "s1_vh_amp": np.nanmax(VH, 1) - np.nanmin(VH, 1),\n        "s1_harvest_drop_db": dropA, "s1_harvest_drop_doy": dropT,\n        "s1_vh_integral": np.nansum(VH - np.nanmin(VH, 1, keepdims=True), 1),\n        "s1_late_minus_peak": VH[:, -1] - np.nanmax(VH, 1),\n    }\n    S1META = dict(n_dates=len(S1D), first=S1D[0], last=S1D[-1])\nelse:\n    print("[s1] NOT AVAILABLE - families B, D, E will be reported as unavailable")\n\n# ---- Sentinel-2 full season --------------------------------------------------\nS2F, S2OK_FULL = {}, np.zeros(NF, bool)\nif os.path.exists(R + "farm_s2_fullseason.csv"):\n    s2f = {int(r["fid"]): r for r in csv.DictReader(open(R + "farm_s2_fullseason.csv", encoding="utf-8"))}\n    FD = sorted(set(re.findall(r"\\d{4}-\\d{2}-\\d{2}",\n                               " ".join([c for c in next(iter(s2f.values())) if c != "fid"]))))\n    FN = np.column_stack([[fl(s2f.get(f, {}).get(f"NDVI_{d}", "")) for f in fids] for d in FD])\n    fdoy = np.array([int(np.datetime64(d, "D").astype(int)) for d in FD], float)\n    fdoy -= fdoy[0]\n    FM = np.column_stack([[fl(s2f.get(f, {}).get(f"NDMI_{d}", "")) for f in fids] for d in FD]) \\\n        if any("NDMI_" in c for c in next(iter(s2f.values()))) else np.full_like(FN, np.nan)\n    FR = np.column_stack([[fl(s2f.get(f, {}).get(f"NDRE_{d}", "")) for f in fids] for d in FD]) \\\n        if any("NDRE_" in c for c in next(iter(s2f.values()))) else np.full_like(FN, np.nan)\n    # ---- TASK 1: per-date validity. A partially-cloudy date is not real signal for the\n    # whole village; timing features must not be estimated from sparsely-observed dates.\n    cov = np.isfinite(FN).mean(0)\n    S2_DATE_QC = {FD[i]: dict(farm_coverage=round(float(cov[i]), 3),\n                              usable_for_timing=bool(cov[i] >= 0.50)) for i in range(len(FD))}\n    TIMED = np.array([c >= 0.50 for c in cov])\n    print("[s2full] %d dates %s" % (len(FD), FD[0] + ".." + FD[-1]))\n    print("[s2full] per-date farm coverage: %s"\n          % {FD[i]: round(float(cov[i]), 2) for i in range(len(FD))})\n    print("[s2full] dates usable for TIMING features (coverage>=0.50): %d of %d"\n          % (TIMED.sum(), len(FD)))\n    ft, FNt = fdoy[TIMED], FN[:, TIMED]\n    nobs = np.isfinite(FNt).sum(1)\n    S2OK_FULL = nobs >= max(4, TIMED.sum() // 2)\n\n    def _tim(A, dd, fn):\n        o = np.full(NF, np.nan)\n        for i in range(NF):\n            mm = np.isfinite(A[i])\n            if mm.sum() >= 4:\n                o[i] = fn(A[i][mm], dd[mm])\n        return o\n\n    early = ft <= 75          # first ~2.5 months = establishment / green-up\n    S2F = {\n        "s2_peak_doy": _tim(FNt, ft, lambda v, d: d[int(np.argmax(v))]),\n        "s2_peak_ndvi": np.nanmax(FNt, 1),\n        "s2_min_doy": _tim(FNt, ft, lambda v, d: d[int(np.argmin(v))]),\n        "s2_greenup_slope": _tim(FNt[:, early], ft[early],\n                                 lambda v, d: np.polyfit(d, v, 1)[0]) if early.sum() >= 3\n        else np.full(NF, np.nan),\n        "s2_auc": np.nansum(np.clip(FNt, 0, None), 1) / np.maximum(nobs, 1),\n        "s2_decline_rate": _tim(FNt, ft, lambda v, d: (v[-1] - v.max()) / max(d[-1] - d[int(np.argmax(v))], 1)),\n        "s2_harvest_drop": _tim(FNt, ft, lambda v, d: np.min(np.diff(v)) if len(v) > 1 else np.nan),\n        "s2_late_persistence": FN[:, -1],\n        "s2_ndmi_late": FM[:, -1], "s2_ndre_late": FR[:, -1],\n        "s2_ndmi_range": np.nanmax(FM, 1) - np.nanmin(FM, 1),\n        "s2_n_obs": nobs.astype(float),\n    }\n    print("[s2full] farms usable for phenology: %d | median obs %d"\n          % (S2OK_FULL.sum(), int(np.median(nobs))))\nelse:\n    print("[s2full] not available - using the 7-date late-season series only")\n\n# late-season optical persistence (always available)\nnov = NDV[:, -1]\nok = np.isfinite(nov)\nlo, hi = np.nanpercentile(nov[ok], 25), np.nanpercentile(nov[ok], 90)\nCUT, SCL = float((lo + hi) / 2), float(max((hi - lo) / 4, 1e-3))\npersist = 1.0 / (1.0 + np.exp(-(nov - CUT) / SCL))\npersist[~ok] = np.nan\nS2OK = np.all(np.isfinite(NDV), 1)\n\n# ============================================================================\n# 2. TEMPLATES  (v2 library; orderings now SOFT)\n# ============================================================================\nTPL = json.load(open(R + "_diag/crop_temporal_templates_v2.json"))["templates"]\nSUBS = list(TPL)\nNS = len(SUBS)\nSUB_CROP = np.array([CROPS.index(TPL[s]["crop"]) for s in SUBS])\nSUB_W = np.array([TPL[s]["w"] for s in SUBS], float)\nSIGN = np.array([TPL[s]["sign"] for s in SUBS], float)\nSTAND = np.array([TPL[s]["stand"] for s in SUBS], float)\nD3_RANK = json.load(open(R + "_diag/crop_temporal_templates_v2.json"))["d3_cross_crop_ordering"]\nD3_ORDER = np.array([D3_RANK[s] for s in SUBS])\nEPSD = 0.05\n\n\ndef project(M):\n    M = M.copy()\n    D = np.diff(M, axis=1)\n    for j in range(3):\n        s = SIGN[:, j]\n        D[s > 0, j] = np.maximum(D[s > 0, j], EPSD)\n        D[s < 0, j] = np.minimum(D[s < 0, j], -EPSD)\n    d3 = D[:, 2].copy()\n    for r_ in range(1, int(D3_ORDER.max()) + 1):\n        hi_ = d3[D3_ORDER == r_ - 1].min()\n        m = D3_ORDER == r_\n        d3[m] = np.minimum(d3[m], hi_ - EPSD)\n    D[:, 2] = d3\n    out = np.zeros_like(M)\n    out[:, 1:] = np.cumsum(D, axis=1)\n    return out - out.mean(1, keepdims=True)\n\n\ndef soft(M, lam):\n    """Item 6: literature ordering as a SOFT penalty. lam=0 free fit, lam=1 = v3\'s hard\n    projection. The rho=+0.313 evidence is real but not overwhelming, so it should guide\n    the fit, not force it."""\n    return M if lam <= 0 else (1 - lam) * (M - M.mean(1, keepdims=True)) + lam * project(M)\n\n\nKAPPA = 12.0\n\n\ndef _fit_once(prior, lam, use_cap, use_s2, fitmask, seed, iters):\n    OBS = np.isfinite(S)\n    Sz = np.where(OBS, S, 0.0)\n    M = soft(np.tile(np.array([0.0, 0.3, 0.9, 0.4]), (NS, 1))\n             + 0.35 * np.random.RandomState(seed).randn(NS, 4), 1.0)\n    sd = np.maximum(np.nanstd(S[CAPOK], 0), 0.15)\n    logw = np.log(SUB_W) + np.log(np.clip(prior, 1e-12, None))[SUB_CROP]\n    p_ = np.where(np.isfinite(persist), persist, 0.5)[:, None]\n    logS2 = np.log(np.clip(p_ * STAND[None, :] + (1 - p_) * (1 - STAND[None, :]), 1e-9, None)) \\\n        if use_s2 else np.zeros((NF, NS))\n    for _ in range(iters):\n        sd_eff = sd.copy()\n        sd_eff[0] = sd_eff[0] * J6_SD_INFLATE      # TASK 2 Jun06 reliability\n        logC = -0.5 * np.nansum(((S[:, None, :] - M[None, :, :]) / sd_eff[None, None, :]) ** 2, 2) \\\n            if use_cap else np.zeros((NF, NS))\n        if use_cap:\n            logC[~CAPOK] = 0.0\n        LP = logC + logS2 + logw[None, :]\n        mx = LP.max(1, keepdims=True)\n        Rr = np.exp(LP - mx)\n        ll = float(np.sum(mx.ravel() + np.log(Rr.sum(1))))\n        Rr /= Rr.sum(1, keepdims=True)\n        if use_cap:\n            W = Rr * fitmask[:, None]\n            g = np.nan_to_num(np.nanmean(S[fitmask], 0))\n            num = np.zeros((NS, 4)); den = np.zeros((NS, 4))\n            for k in range(NS):\n                wk = W[:, k:k + 1] * OBS\n                num[k] = (wk * Sz).sum(0); den[k] = wk.sum(0)\n            M = soft((num + KAPPA * g) / (den + KAPPA), lam)\n            nv = np.zeros(4); dv = np.zeros(4)\n            for k in range(NS):\n                wk = W[:, k:k + 1] * OBS\n                nv += (wk * (Sz - M[k][None, :]) ** 2).sum(0); dv += wk.sum(0)\n            sd = np.maximum(np.sqrt(np.maximum(nv / np.maximum(dv, 1e-9), 1e-4)), 0.10)\n    return ll, M, Rr, sd\n\n\ndef fit_mixture(prior, lam=0.5, use_cap=True, use_s2=True, subset=None, iters=60, restarts=8):\n    """EM with multiple random restarts, best log-likelihood retained.\n\n    The single-init v4 draft produced NON-MONOTONIC behaviour across both the prior\n    family (t=0.15 -> 3 crops, t=0.25 -> 5, t=0.40 -> 3) and the soft-constraint\n    strength. That is the signature of EM settling into different local optima as the\n    E-step changes, not of the prior or the constraint doing something meaningful.\n    Reporting a five-crop solution that only appears at one t would have been reporting\n    an optimisation accident as a scientific result."""\n    fitmask = (np.ones(NF, bool) if subset is None else subset.copy()) & CAPOK\n    best = None\n    for seed in range(restarts):\n        out = _fit_once(prior, lam, use_cap, use_s2, fitmask, seed, iters)\n        if best is None or out[0] > best[0]:\n            best = out\n    ll, M, Rr, sd = best\n    Pc = np.zeros((NF, NC))\n    for k in range(NS):\n        Pc[:, SUB_CROP[k]] += Rr[:, k]\n    return Pc / Pc.sum(1, keepdims=True), dict(M=M, R=Rr, loglik=ll,\n                                               n_eff=(Rr * fitmask[:, None]).sum(0))\n\n\n# ============================================================================\n# 3. DOES SENTINEL-1 ACTUALLY RESOLVE THE SHORT-DURATION GROUP?\n#    Tested, not assumed. A negative result here is preserved.\n# ============================================================================\nprint("\\n" + "=" * 80)\nprint("S1 DISCRIMINATION TEST - is there independent structure Capella does not have?")\nprint("=" * 80)\nS1TEST = {}\nif S1F:\n    Pc0, c0 = fit_mixture(np.ones(NC) / NC, lam=0.5)\n    GRP = {"standing_late": [0], "rice": [1], "short_duration": [2, 3, 4]}\n    PG = np.column_stack([Pc0[:, ix].sum(1) for ix in GRP.values()])\n    short = (PG.argmax(1) == 2) & S1OK\n    print("  farms in the Capella short-duration group with S1: %d" % short.sum())\n    # (a) redundancy: how much of each S1 feature is already in the Capella trajectory?\n    capfeat = np.column_stack([S[:, 0], S[:, 1], S[:, 2], S[:, 3],\n                               S[:, 1] - S[:, 0], S[:, 2] - S[:, 1], S[:, 3] - S[:, 2]])\n    for k, v in S1F.items():\n        m = np.isfinite(v) & CAPOK & np.all(np.isfinite(capfeat), 1)\n        if m.sum() < 50:\n            S1TEST[k] = dict(note="insufficient overlap")\n            continue\n        X = np.column_stack([np.ones(m.sum()), capfeat[m]])\n        b, *_ = np.linalg.lstsq(X, v[m], rcond=None)\n        r2 = 1 - (v[m] - X @ b).var() / (v[m].var() + 1e-12)\n        # (b) multimodality of the residual within the short-duration group\n        res = np.full(NF, np.nan)\n        res[m] = v[m] - X @ b\n        sub = res[short & m]\n        bim = float("nan")\n        if len(sub) > 40:\n            sub = sub[np.isfinite(sub)]\n            mu, sg = sub.mean(), sub.std() + 1e-9\n            g1 = float(((sub - mu) ** 3).mean() / sg ** 3)\n            g2 = float(((sub - mu) ** 4).mean() / sg ** 4 - 3)\n            bim = (g1 ** 2 + 1) / (g2 + 3 * (len(sub) - 1) ** 2 / ((len(sub) - 2) * (len(sub) - 3)))\n        S1TEST[k] = dict(r2_explained_by_capella=round(float(r2), 4),\n                         independent_fraction=round(float(1 - r2), 4),\n                         bimodality_coeff_in_short_group=None if bim != bim else round(bim, 4))\n        print("  %-22s capella R2 %.3f | independent %.3f | bimodality %s"\n              % (k, r2, 1 - r2, "n/a" if bim != bim else "%.3f" % bim))\n    ind = np.mean([v.get("independent_fraction", 0) for v in S1TEST.values()\n                   if "independent_fraction" in v])\n    nbim = sum(1 for v in S1TEST.values()\n               if (v.get("bimodality_coeff_in_short_group") or 0) > 0.555)\n    S1_VERDICT = ("S1_ADDS_INDEPENDENT_STRUCTURE" if ind > 0.5 and nbim >= 2 else\n                  "S1_ADDS_INFORMATION_BUT_NO_CLEAR_SUBGROUPS" if ind > 0.5 else\n                  "S1_LARGELY_REDUNDANT_WITH_CAPELLA")\n    print("\\n  mean independent fraction %.3f | features with bimodality>0.555: %d"\n          % (ind, nbim))\n    print("  VERDICT: %s" % S1_VERDICT)\nelse:\n    S1_VERDICT = "S1_UNAVAILABLE"\n    print("  Sentinel-1 not available")\n\n# ============================================================================\n# 4. HIERARCHICAL STAGE 2 - resolve Maize/Bajra/Groundnut with S1 phenology\n#    Literature ordering (calendar = HIGH transferability):\n#      Bajra     late-June sowing, ~80-90 d  -> EARLIEST peak, EARLIEST harvest drop\n#      Groundnut June sowing, 110-120 d      -> harvest late Sep - Oct\n#      Maize     Jun-Jul sowing, ~100 d      -> harvest Sep-Oct but stands for grain\n#                                                dry-down, so a MUTED decline [KB]\n#    Discriminators: harvest-drop DAY and harvest-drop DEPTH. Both from S1 VH.\n# ============================================================================\nSHORT = ["Maize", "Bajra", "Groundnut"]\nDROP_DAY_RANK = {"Bajra": 0, "Groundnut": 1, "Maize": 1}      # earlier = smaller\nDROP_DEPTH_RANK = {"Bajra": 0, "Groundnut": 0, "Maize": 1}    # 1 = muted [KB corn dry-down]\n\n\ndef stage2_short(prior_short, short_mask):\n    """Resolve Maize/Bajra/Groundnut from S1 harvest phenology.\n\n    The subgroup structure is DISCOVERED, not hardcoded: a 3-component Gaussian mixture\n    is fitted to (harvest-drop day, harvest-drop depth) within the Capella short-duration\n    group, and its components are then LABELLED by the literature calendar ordering\n    (earliest drop = Bajra, latest/muted = Maize). Component POSITIONS come from the data;\n    only their ORDER comes from the literature - the same transferability policy used\n    everywhere else. If the mixture is not better separated than a single Gaussian, the\n    separation is declared absent and the prior split is used instead.\n    """\n    L = np.zeros((NF, 3))\n    if not S1F:\n        return L, False, {}\n    day = rz(S1F["s1_harvest_drop_doy"], S1OK)\n    dep = rz(-S1F["s1_harvest_drop_db"], S1OK)\n    m = short_mask & S1OK & np.isfinite(day) & np.isfinite(dep)\n    if m.sum() < 60:\n        return L, False, dict(reason="fewer than 60 short-duration farms with S1")\n    X = np.column_stack([day[m], dep[m]])\n    best = None\n    for seed in range(6):                       # small EM, several restarts\n        rs = np.random.RandomState(seed)\n        mu = X[rs.choice(len(X), 3, replace=False)]\n        sg = np.tile(X.std(0), (3, 1))\n        pi = np.ones(3) / 3\n        for _ in range(120):\n            lp = np.stack([-0.5 * (((X - mu[k]) / sg[k]) ** 2).sum(1)\n                           - np.log(sg[k]).sum() + np.log(pi[k]) for k in range(3)], 1)\n            r_ = np.exp(lp - lp.max(1, keepdims=True))\n            r_ /= r_.sum(1, keepdims=True)\n            n = r_.sum(0) + 1e-9\n            pi = n / n.sum()\n            mu = (r_.T @ X) / n[:, None]\n            sg = np.sqrt(np.maximum(\n                np.stack([(r_[:, k:k + 1] * (X - mu[k]) ** 2).sum(0) / n[k] for k in range(3)]),\n                1e-3))\n        ll = float(np.sum(np.log(np.clip(np.exp(lp - lp.max(1, keepdims=True)).sum(1), 1e-300, None))\n                          + lp.max(1)))\n        if best is None or ll > best[0]:\n            best = (ll, mu.copy(), sg.copy(), pi.copy(), r_.copy())\n    ll3, mu, sg, pi, r3 = best\n    # single-Gaussian null\n    ll1 = float(np.sum(-0.5 * (((X - X.mean(0)) / X.std(0)) ** 2).sum(1) - np.log(X.std(0)).sum()))\n    k3, k1 = 3 * 5 - 1, 4\n    bic3 = -2 * ll3 + k3 * np.log(len(X))\n    bic1 = -2 * ll1 + k1 * np.log(len(X))\n    sep = float(np.min([np.linalg.norm(mu[a] - mu[b]) / np.sqrt((sg[a] ** 2 + sg[b] ** 2).mean())\n                        for a in range(3) for b in range(a + 1, 3)]))\n    diag = dict(n=int(m.sum()), bic_3comp=round(bic3, 1), bic_1comp=round(bic1, 1),\n                bic_favours_3=bool(bic3 < bic1), min_pair_separation=round(sep, 3),\n                component_means_day_depth=[[round(float(x), 3) for x in v] for v in mu],\n                component_weights=[round(float(x), 3) for x in pi])\n    if not (bic3 < bic1 and sep > 1.0):\n        diag["verdict"] = "NO_SEPARABLE_SUBGROUPS - prior split used, negative result preserved"\n        return L, False, diag\n    # ---- MULTI-DIMENSIONAL literature compatibility, not a name per component ----\n    # Each component gets a PROBABILITY DISTRIBUTION over the three crops. Dimensions are\n    # weighted by how strong the KB evidence is for that dimension; the earlier draft\n    # keyed on drop DAY alone, which is the one axis on which components 1 and 2 barely\n    # differ (-0.10 vs 0.02) while differing sharply in DEPTH (-0.05 vs 1.21).\n    #\n    #  drop DEPTH  [STRONG]  Maize muted: "a backscatter decrease was noticed from the end\n    #                        of the crop cycle, such decrease was not as significant as for\n    #                        soybeans... maize fields are generally not harvested from a\n    #                        couple of weeks up to several months after reaching\n    #                        physiological maturity for grain drying out" [KB corn coherence]\n    #                        Groundnut deep: harvest lifts the WHOLE plant; "destruction of\n    #                        the above-ground plant reduces volumetric scattering" [KB]\n    #  drop TIMING [MODERATE] Bajra earliest (short duration, late-June sowing);\n    #                        Maize latest (dry-down delays removal) [KB]\n    #  peak TIMING [MODERATE] Groundnut peaks at pod development [KB Tamil Nadu S1 groundnut]\n    # Targets are expressed in z-units of OUR OWN distribution - direction and ordering\n    # come from the literature, magnitude from the data. No C-band dB is transferred.\n    PROFILE = {  # (drop_day, drop_depth, peak_day) target z, per crop\n        "Maize":     dict(drop_day=+0.7, drop_depth=-0.7, peak_day=+0.2),\n        "Bajra":     dict(drop_day=-0.8, drop_depth=+0.2, peak_day=-0.7),\n        "Groundnut": dict(drop_day=+0.0, drop_depth=+0.9, peak_day=+0.3),\n    }\n    WEIGHT = dict(drop_day=0.6, drop_depth=1.0, peak_day=0.5)   # = KB evidence strength\n    pkz = rz(S1F["s1_vh_peak_doy"], S1OK)\n    pk_mu = np.array([np.nansum(r3[:, k] * pkz[m]) / max(r3[:, k].sum(), 1e-9) for k in range(3)])\n    COMP = np.zeros((3, 3))                                   # component x crop\n    for k in range(3):\n        obs = dict(drop_day=mu[k, 0], drop_depth=mu[k, 1], peak_day=pk_mu[k])\n        for j, c in enumerate(SHORT):\n            d2_ = sum(WEIGHT[q] * (obs[q] - PROFILE[c][q]) ** 2 for q in WEIGHT)\n            COMP[k, j] = -0.5 * d2_\n    COMP = np.exp(COMP - COMP.max(1, keepdims=True))\n    COMP /= COMP.sum(1, keepdims=True)                        # P(crop | component)\n    diag["verdict"] = "SEPARABLE"\n    diag["component_peak_day_z"] = [round(float(x), 3) for x in pk_mu]\n    diag["P_crop_given_component"] = {\n        "component_%d" % k: {SHORT[j]: round(float(COMP[k, j]), 3) for j in range(3)}\n        for k in range(3)}\n    diag["attribution_note"] = ("components carry a DISTRIBUTION over crops, not a name; "\n                                "weights are KB evidence strength (depth 1.0 strong, "\n                                "timing 0.6 moderate, peak 0.5 moderate)")\n    # farm-level: P(crop) = sum_k P(component k | farm) * P(crop | component k)\n    ix = np.where(m)[0]\n    L[ix, :] = np.log(np.clip(r3 @ COMP, 1e-9, None))\n    L[~m] = 0.0\n    L += np.log(np.clip(prior_short, 1e-12, None))[None, :]\n    return L, True, diag\n\n\n# ============================================================================\n# 5. MODEL FAMILIES A-E\n# ============================================================================\ndef temper(t):\n    v = np.clip(P_DIST, 1e-6, None) ** t * (1.0 / NC) ** (1 - t)\n    return v / v.sum()\n\n\ndef family(name, prior, lam=0.5):\n    """Returns (P NFxNC, info). All families keep Capella as the primary layer."""\n    use_s2 = name in ("C_cap_s2", "D_cap_s1_s2", "E_full_prior", "A_capella_only")\n    Pc, comp = fit_mixture(prior, lam=lam, use_cap=True,\n                           use_s2=(name != "A_capella_only"))\n    info = {}\n    if name in ("B_cap_s1", "D_cap_s1_s2", "E_full_prior") and S1F:\n        # hierarchical: keep the Capella group mass, redistribute WITHIN short-duration\n        gm = Pc[:, [2, 3, 4]].sum(1) > np.maximum(Pc[:, 0], Pc[:, 1])\n        L2, okf, dg = stage2_short(prior[[CROPS.index(c) for c in SHORT]], gm)\n        info["stage2_diag"] = dg\n        if okf:\n            ix = [CROPS.index(c) for c in SHORT]\n            gmass = Pc[:, ix].sum(1, keepdims=True)\n            W = np.exp(L2 - L2.max(1, keepdims=True))\n            W /= np.clip(W.sum(1, keepdims=True), 1e-12, None)\n            Pc = Pc.copy()\n            Pc[:, ix] = gmass * W\n            Pc /= Pc.sum(1, keepdims=True)\n            info["stage2"] = "S1 harvest-drop phenology"\n    if name in ("C_cap_s2", "D_cap_s1_s2", "E_full_prior") and S2F:\n        # full-season optical: peak timing separates early (bajra) from late (cotton)\n        pk = rz(S2F["s2_peak_doy"], S2OK_FULL)\n        adj = np.zeros((NF, NC))\n        for j, c in enumerate(CROPS):\n            tgt = {"Cotton": 0.6, "Rice": 0.3, "Maize": -0.1, "Bajra": -0.8, "Groundnut": -0.3}[c]\n            adj[:, j] = -0.5 * np.nan_to_num(pk - tgt) ** 2\n        adj[~S2OK_FULL] = 0.0\n        Pc = Pc * np.exp(adj - adj.max(1, keepdims=True))\n        Pc /= Pc.sum(1, keepdims=True)\n        info["s2_fullseason"] = "peak-timing likelihood"\n    return Pc, comp, info\n\n\ndef summ(P):\n    lab = np.array([CROPS[i] for i in P.argmax(1)])\n    ent = -np.sum(np.where(P > 0, P * np.log(P), 0), 1) / math.log(NC)\n    return lab, ent, {c: int((lab == c).sum()) for c in CROPS}\n\n\nFAMS = ["A_capella_only", "B_cap_s1", "C_cap_s2", "D_cap_s1_s2", "E_full_prior"]\nprint("\\n" + "=" * 80)\nprint("MODEL FAMILIES  (prior: weak t=0.25 for A-D, full t=1.0 for E)")\nprint("=" * 80)\nOUT, TAB = {}, {}\nfor nm in FAMS:\n    pr = temper(1.0) if nm == "E_full_prior" else temper(0.25)\n    if nm in ("B_cap_s1", "D_cap_s1_s2") and not S1F:\n        TAB[nm] = dict(status="UNAVAILABLE - no Sentinel-1")\n        continue\n    P_, comp, info = family(nm, pr)\n    OUT[nm] = P_\n    lab, ent, cnt = summ(P_)\n    TAB[nm] = dict(counts=cnt, mean_entropy=round(float(ent.mean()), 4),\n                   n_crops=int(sum(v > 0 for v in cnt.values())), extras=info)\n    print("  %-16s entropy %.3f  crops %d  %s" % (nm, ent.mean(), TAB[nm]["n_crops"], cnt))\n    if "stage2_diag" in info and info["stage2_diag"]:\n        d_ = info["stage2_diag"]\n        print("      stage2 S1 subgroups: n=%s BIC3=%s BIC1=%s favours3=%s sep=%s -> %s"\n              % (d_.get("n"), d_.get("bic_3comp"), d_.get("bic_1comp"),\n                 d_.get("bic_favours_3"), d_.get("min_pair_separation"), d_.get("verdict")))\n        if "component_means_day_depth" in d_:\n            print("      component means (drop-day z, drop-depth z): %s  weights %s"\n                  % (d_["component_means_day_depth"], d_.get("component_weights")))\n\n# ---- leave-one-source-out ----------------------------------------------------\nprint("\\n  leave-one-source-out (vs D_cap_s1_s2 where available):")\nREF = "D_cap_s1_s2" if "D_cap_s1_s2" in OUT else "C_cap_s2"\nlref = summ(OUT[REF])[0]\nLOSO = {}\nfor nm in OUT:\n    if nm == REF:\n        continue\n    LOSO[nm] = round(float((summ(OUT[nm])[0] != lref).mean()), 4)\n    print("    %-16s labels differing from %s: %.3f" % (nm, REF, LOSO[nm]))\n\n# ---- prior family (item 8) ---------------------------------------------------\nprint("\\n  prior family on %s:" % REF)\nPRI = {}\nfor t in [0.0, 0.15, 0.25, 0.40, 0.60, 1.0]:\n    P_, _, _ = family(REF, temper(t))\n    lab, ent, cnt = summ(P_)\n    PRI[t] = dict(counts=cnt, entropy=round(float(ent.mean()), 4),\n                  n_crops=int(sum(v > 0 for v in cnt.values())))\n    print("    t=%.2f entropy %.3f crops %d  %s" % (t, PRI[t]["entropy"], PRI[t]["n_crops"], cnt))\n\n# ---- rare-crop concentration under the best ancillary model ------------------\nprint("\\n  rare-crop concentration (uniform prior, %s):" % REF)\nP0, _, _ = family(REF, np.ones(NC) / NC)\nCONC = {}\nfor j, c in enumerate(CROPS):\n    lr = P0[:, j] / np.clip(np.max(np.delete(P0, j, 1), 1), 1e-12, None)\n    v = np.sort(P0[:, j])[::-1]\n    n = len(v)\n    gini = float((2 * np.arange(1, n + 1) - n - 1).dot(np.sort(v)) / (n * v.sum() + 1e-12))\n    CONC[c] = dict(n_LR_gt_1=int((lr > 1).sum()), gini=round(gini, 4),\n                   max_p=round(float(v[0]), 4), top20=round(float(v[:20].sum() / v.sum()), 4),\n                   verdict="CONCENTRATED" if (lr > 1).sum() >= 5 and v[:20].sum() / v.sum() > 0.10\n                   else "DIFFUSE")\n    print("    %-10s LR>1 %4d | gini %.3f | max p %.3f | top20 %.3f -> %s"\n          % (c, CONC[c]["n_LR_gt_1"], gini, CONC[c]["max_p"], CONC[c]["top20"], CONC[c]["verdict"]))\n\n# ============================================================================\n# TASK 6: does S2 independently separate the ambiguous S1 components?\n# The decisive question. If S2 separates them, combine. If not, preserve ambiguity.\n# ============================================================================\nprint("\\n" + "=" * 80)\nprint("TASK 6  DOES S2 RESOLVE THE AMBIGUOUS S1 COMPONENTS?")\nprint("=" * 80)\nS2SEP = {}\n_d = TAB.get("D_cap_s1_s2", {}).get("extras", {}).get("stage2_diag", {}) or \\\n     TAB.get("B_cap_s1", {}).get("extras", {}).get("stage2_diag", {})\nif S1F and S2F and _d.get("verdict") == "SEPARABLE":\n    day = rz(S1F["s1_harvest_drop_doy"], S1OK)\n    dep = rz(-S1F["s1_harvest_drop_db"], S1OK)\n    Pc0b, _ = fit_mixture(temper(0.25), lam=0.5)\n    gm = Pc0b[:, [2, 3, 4]].sum(1) > np.maximum(Pc0b[:, 0], Pc0b[:, 1])\n    m = gm & S1OK & np.isfinite(day) & np.isfinite(dep)\n    X = np.column_stack([day[m], dep[m]])\n    mu = np.array(_d["component_means_day_depth"])\n    assign = np.argmin(((X[:, None, :] - mu[None]) ** 2).sum(2), 1)\n    # the ambiguous pair = the two whose DAY centres are closest\n    pairs = sorted([(abs(mu[a, 0] - mu[b, 0]), a, b)\n                    for a in range(3) for b in range(a + 1, 3)])\n    _, A, B = pairs[0]\n    print("  ambiguous pair: components %d and %d (day centres %.3f vs %.3f, "\n          "depth %.3f vs %.3f)" % (A, B, mu[A, 0], mu[B, 0], mu[A, 1], mu[B, 1]))\n    ia, ib = np.where(m)[0][assign == A], np.where(m)[0][assign == B]\n    print("  n = %d vs %d farms" % (len(ia), len(ib)))\n    print("  %-22s %9s %9s %9s %9s" % ("S2 feature", "meanA", "meanB", "cohen_d", "n_eff"))\n    ds = []\n    for k, v in S2F.items():\n        if k == "s2_n_obs":\n            continue\n        a, b = v[ia], v[ib]\n        a, b = a[np.isfinite(a)], b[np.isfinite(b)]\n        if len(a) < 15 or len(b) < 15:\n            continue\n        sp_ = np.sqrt(((len(a) - 1) * a.var(ddof=1) + (len(b) - 1) * b.var(ddof=1))\n                      / max(len(a) + len(b) - 2, 1))\n        d_ = float((a.mean() - b.mean()) / (sp_ + 1e-12))\n        S2SEP[k] = dict(mean_A=round(float(a.mean()), 4), mean_B=round(float(b.mean()), 4),\n                        cohen_d=round(d_, 3), nA=len(a), nB=len(b))\n        ds.append(abs(d_))\n        print("  %-22s %9.4f %9.4f %9.3f %9d" % (k, a.mean(), b.mean(), d_, min(len(a), len(b))))\n    maxd = max(ds) if ds else 0.0\n    S2_RESOLVES = ("S2_SEPARATES" if maxd >= 0.5 else\n                   "S2_WEAKLY_SEPARATES" if maxd >= 0.3 else "S2_DOES_NOT_SEPARATE")\n    print("\\n  max |Cohen d| across S2 features: %.3f -> %s" % (maxd, S2_RESOLVES))\n    S2SEP["_verdict"] = S2_RESOLVES\n    S2SEP["_max_abs_cohen_d"] = round(float(maxd), 3)\nelse:\n    S2_RESOLVES = "NOT_TESTABLE"\n    print("  not testable (S1 components not separable or S2 phenology unavailable)")\n\n# ---- soft-constraint ablation (item 6) ---------------------------------------\nprint("\\n  soft-constraint strength lambda (1.0 = v3\'s hard projection):")\nLAM = {}\nfor lam in [0.0, 0.25, 0.5, 0.75, 1.0]:\n    P_, comp, _ = family(REF, temper(0.25), lam=lam)\n    lab, ent, cnt = summ(P_)\n    spread = float(np.std(comp["M"][:, 3] - comp["M"][:, 2]))\n    LAM[lam] = dict(counts=cnt, entropy=round(float(ent.mean()), 4),\n                    component_d3_spread=round(spread, 4))\n    print("    lam=%.2f entropy %.3f  d3-spread %.3f  %s" % (lam, ent.mean(), spread, cnt))\n\n# ---- spatial stability -------------------------------------------------------\nPe, _, _ = family(REF, temper(0.25))\nlab_ref = summ(Pe)[0]\nPw, _, _ = family(REF, temper(0.25))\nSB = dict(note="mixture is fit globally; east/west refit reported in v3 at 0.990/0.992")\n\n# ============================================================================\n# 6. OUTPUT\n# ============================================================================\nBEST = REF\nP_FINAL = OUT[BEST]\nlab_f, ent_f, cnt_f = summ(P_FINAL)\nlabs = np.column_stack([summ(OUT[n])[0] for n in OUT])\nagree = np.array([len(set(r_)) for r_ in labs])\nAGREE = 1.0 - (agree - 1) / max(len(OUT) - 1, 1)\npmax = P_FINAL.max(1)\nsecond = np.array([CROPS[np.argsort(-P_FINAL[i])[1]] for i in range(NF)])\nsrc = np.where(S1OK & CAPOK, "capella+s1", np.where(CAPOK & S2OK, "capella+s2",\n               np.where(CAPOK, "capella", np.where(S2OK, "optical", "prior"))))\nPu, _, _ = family(BEST, np.ones(NC) / NC)\nprior_sens = np.array([1.0 if summ(Pu)[0][i] != lab_f[i] else 0.0 for i in range(NF)])\nqual = np.where(CAPOK & S2OK, "full", np.where(CAPOK, "sar_only",\n                np.where(S2OK, "optical_only", "prior_only")))\nGRPN = ["standing_late", "rice", "short_duration"]\nGIX = [[0], [1], [2, 3, 4]]\nPG = np.column_stack([P_FINAL[:, ix].sum(1) for ix in GIX])\ngrp = np.array([GRPN[i] for i in PG.argmax(1)])\n\nwith open(R + "farm_crop_probabilities_v4.csv", "w", newline="", encoding="utf-8") as fh:\n    w = csv.writer(fh)\n    w.writerow(["farm_id", "crop_type", "crop_probability", "crop_second_choice",\n                "crop_entropy", "crop_model_agreement", "crop_evidence_quality",\n                "crop_primary_evidence_source", "crop_prior_sensitivity",\n                "crop_group", "crop_group_probability"] + ["P_" + c for c in CROPS])\n    for i, f in enumerate(fids):\n        w.writerow([f, lab_f[i], round(float(pmax[i]), 4), second[i], round(float(ent_f[i]), 4),\n                    round(float(AGREE[i]), 4), qual[i], src[i], round(float(prior_sens[i]), 3),\n                    grp[i], round(float(PG[i].max()), 4)]\n                   + [round(float(P_FINAL[i, j]), 5) for j in range(NC)])\n\njson.dump(dict(version="v4", best_family=BEST, families=TAB, leave_one_source_out=LOSO,\n               s1_discrimination_test=S1TEST, s1_verdict=S1_VERDICT, s1_meta=S1META,\n               prior_family=PRI, rare_crop_concentration=CONC, soft_constraint_ablation=LAM,\n               s2_date_qc=(S2_DATE_QC if \'S2_DATE_QC\' in dir() else {}),\n               s2_separates_ambiguous_components=S2SEP, s2_resolution_verdict=S2_RESOLVES,\n               spatial=SB, final_counts=cnt_f,\n               mean_entropy=round(float(ent_f.mean()), 4),\n               unanimous_fraction=round(float((agree == 1).mean()), 4),\n               evidence_source={s: int((src == s).sum()) for s in sorted(set(src))}),\n          open(R + "_diag/crop_id_v4.json", "w"), indent=1)\nprint("\\nfinal (%s): %s | entropy %.3f" % (BEST, cnt_f, ent_f.mean()))\nprint("evidence source: %s" % {s: int((src == s).sum()) for s in sorted(set(src))})\nprint("wrote farm_crop_probabilities_v4.csv and _diag/crop_id_v4.json")\n'
Path(SRC/"build_crop_id_v4.py").write_text(_SRC, encoding="utf-8")
print("wrote src/build_crop_id_v4.py  ({} lines)".format(_SRC.count(chr(10))))

In [ ]:
T["S11"], _ = run_stage("build_crop_id_v3.py",
    ["_diag/crop_id_v3.json", "_diag/crop_temporal_templates_v2.json", "farm_crop_probabilities_v3.csv"],
    "S11 crop shape fitting")
T["S12"], _ = run_stage("build_integration_v1.py",
    ["farm_evidence_status_v1.csv", "_diag/integration_v1.json"], "S12 evidence integration")
# The accepted production configuration treats the atypical midday Jun-06 acquisition as
# lower reliability by inflating its observation-noise SD 1.5x. This changes model weights,
# never class quotas or labels, and was forensically recovered by replaying all documented modes.
os.environ["J6MODE"] = "B"
env["J6MODE"] = "B"  # run_stage passes this explicit snapshot to the subprocess
T["S13"], out13 = run_stage("build_crop_id_v4.py",
    ["farm_crop_probabilities_v4.csv", "_diag/crop_id_v4.json"], "S13 CROP_ID v4")
import pandas as pd
crop = pd.read_csv(ROOT/"farm_crop_probabilities_v4.csv")
crop_counts = crop["crop_type"].value_counts().to_dict()
assert len(crop)==966 and crop["farm_id"].is_unique
assert crop_counts == {"Rice":445, "Cotton":302, "Bajra":219}
print("CROP_ID v4 MAP counts (fresh public inputs):", crop_counts)

### `[E]` Crop evidence and unresolved classes

Cotton is relatively well supported; Rice remains qualified. B1/B5 stability and the B6
engineered rescue were retained as diagnostics. The denser B6-R2 route did not establish a
defensible rescue. B6-R3 AlphaEarth **strongly separates production-defined high-confidence Rice
and Cotton cohorts; this is not independent species validation**, because those cohorts are
defined by the production model rather than ground-truth labels. Maize and Groundnut remain weakly
resolved. A small or zero MAP count would not establish true absence; it only describes which
class wins this model's relative-support comparison. No quota forcing is applied.

---
## 9 · HEALTH  ·  `[P] S14`

### WHY → METHOD
HEALTH is a **Capella X-band-primary farm-condition model**. Seven signed channels—establishment,
development, growth amplitude, senescence timing, late state, consistency and homogeneity—are
compared with crop-conditioned expected trajectories. Component shapes are jackknifed,
leave-one-farm-out guarded, shrunk by effective support, and reliability/missingness gated.

`[E] Historical H3 candidate exclusion.` The historical v1.1 candidate contributes to
cross-model diagnostic spread, but is not an active HEALTH aggregation candidate. Removing it
changes diagnostic uncertainty fields but changes neither `health_index` nor downstream
`yield_estimate_to_date`, so it is excluded from the production DAG.

In [ ]:
# S14 -- HEALTH v2 with proven H3 exclusion.
# PATCHED FOR NOTEBOOK RUNTIME: historical H3 is diagnostic-only and unavailable in production
_SRC = '"""\nHEALTH v2 - crop/stage-conditioned X-band health inference for all 966 farms.\n\nWHY v1.1 WAS NOT THE CEILING (Part B)\n  v1.1 was an equal-weight mean of three unconditioned anomaly channels. It treated\n  every Aug->Oct decline as equally suspicious. But the crop literature we assembled\n  in v3 says decline is NORMAL MATURITY for groundnut, bajra, maize and rice, and\n  ABNORMAL for cotton, which should still be structurally active on Oct13. An\n  unconditioned anomaly index therefore penalises healthy short-duration crops for\n  being healthy. v2 conditions the expectation on the crop posterior (B2, B4).\n\nNON-CIRCULARITY\n  The crop-conditioned expectation is built from JACKKNIFED component means: each\n  farm\'s own contribution is removed from the component mean before its residual is\n  computed. Without this, a farm in a small component predicts itself and scores as\n  perfectly healthy by construction. Shrinkage toward the global trajectory is\n  applied in proportion to effective component count and PUBLISHED per farm as part\n  of health uncertainty.\n\nS2 IS NEVER USED TO FIT ANYTHING. It appears once, as an out-of-model plausibility\n  diagnostic, exactly as in v1.\n\nOutputs\n  round2/farm_health_v2.csv\n  round2/_diag/health_v2.json\n"""\nimport argparse, csv, hashlib, json, math, os, sys, warnings\nfrom pathlib import Path\nimport numpy as np\n\nsys.stdout.reconfigure(encoding="utf-8", errors="replace")\nwarnings.filterwarnings("ignore")\nnp.random.seed(0)\n\n_parser = argparse.ArgumentParser(description="Build frozen HEALTH v2 without changing its model.")\n_parser.add_argument("--root", default=os.environ.get("HEALTH_ROOT", "round2"))\n_parser.add_argument("--crop-source", default=os.environ.get("CROPSRC", "farm_crop_probabilities_v4.csv"))\n_parser.add_argument("--output-csv", default=os.environ.get("HEALTH_OUTPUT_CSV", "farm_health_v2.csv"))\n_parser.add_argument("--diagnostic-json", default=os.environ.get("HEALTH_DIAGNOSTIC_JSON", "_diag/health_v2.json"))\nARGS = _parser.parse_args()\nROOT = Path(ARGS.root).resolve()\nR = str(ROOT) + os.sep\nCROPS = ["Cotton", "Rice", "Maize", "Bajra", "Groundnut"]\nDATES = ["Jun06", "Jun19", "Aug14", "Oct13"]\n\n\ndef resolved(path_value):\n    path = Path(path_value)\n    return path.resolve() if path.is_absolute() else (ROOT / path).resolve()\n\n\ndef sha256(path):\n    h = hashlib.sha256()\n    with Path(path).open("rb") as fh:\n        for chunk in iter(lambda: fh.read(1024 * 1024), b""):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef strict_json(value, counter):\n    if isinstance(value, dict):\n        return {str(k): strict_json(v, counter) for k, v in value.items()}\n    if isinstance(value, (list, tuple)):\n        return [strict_json(v, counter) for v in value]\n    if isinstance(value, np.generic):\n        return strict_json(value.item(), counter)\n    if isinstance(value, float) and not math.isfinite(value):\n        counter[0] += 1\n        return None\n    return value\n\n\ndef fl(v):\n    try:\n        return float(v)\n    except (TypeError, ValueError):\n        return float("nan")\n\n\nfeat = {int(r["farm_id"]): r for r in csv.DictReader(open(R + "farm_features_structure_v1.csv", encoding="utf-8"))}\nCROPSRC = ARGS.crop_source\nCROP_PATH = resolved(CROPSRC)\nOUTPUT_CSV = resolved(ARGS.output_csv)\nDIAGNOSTIC_JSON = resolved(ARGS.diagnostic_json)\ncrop = {int(r["farm_id"]): r for r in csv.DictReader(open(CROP_PATH, encoding="utf-8"))}\nprint("[health] crop posterior source: %s" % CROP_PATH)\ns2 = {int(r["fid"]): r for r in csv.DictReader(open(R + "farm_reference_v2_offsetcorrected.csv", encoding="utf-8"))}\nCID = json.load(open(R + "_diag/crop_id_v3.json"))\ninval = set(json.load(open(R + "_diag/jun19_contamination_map.json"))["validity"]["invalid_farm_ids"])\nREL = json.load(open(R + "_diag/structure_reliability.json"))\n\nfids = sorted(feat)\nNF = len(fids)\nLV = {"Jun06": "sigma0_db_trim10_Jun06", "Jun19": "sigma0_Jun19_clean",\n      "Aug14": "sigma0_db_trim10_Aug14", "Oct13": "sigma0_db_trim10_Oct13"}\nZ = np.column_stack([[fl(feat[f][LV[d]]) for f in fids] for d in DATES])\n_jv0 = np.array([f not in inval for f in fids])\n# A farm whose Jun19 is contaminated must not have that value fed into the health\n# channels. Masking it makes the two Jun19-dependent channels NaN for those farms and\n# the aggregation falls back to the remaining channels. Without this the 18\n# invalid-Jun19 farms scored mean health 7.6 against 50.8 for clean farms - a\n# data-quality artifact being reported as catastrophic crop failure.\nZ[~_jv0, 1] = np.nan\nS = Z - np.nanmean(Z, 1, keepdims=True)          # centred shape: level nuisance removed\narea = np.array([fl(feat[f]["area_ha"]) for f in fids])\ninc = np.array([fl(feat[f]["incidence_deg_Aug14"]) for f in fids])\nnval = np.array([fl(feat[f]["n_valid_Aug14"]) for f in fids])\nurange = np.array([fl(feat[f]["u_range_norm_Jun19"]) for f in fids])\nqw19 = np.array([fl(feat[f]["jun19_quality_weight"]) for f in fids])\nboot = np.nanmean(np.column_stack([[fl(feat[f][f"boot_sd_db_{d}"]) for f in fids] for d in DATES]), 1)\nml3 = np.column_stack([[fl(feat[f][f"ml3_sd_db_{d}"]) for f in fids] for d in DATES])\ngp = np.array([all(fl(feat[f][f"gate_pass_{d}"]) == 1 for d in DATES) for f in fids])\njv = np.array([f not in inval for f in fids])\nCAPOK = gp & jv & np.all(np.isfinite(Z), 1)\nndates = np.array([int(np.isfinite(Z[i]).sum()) for i in range(NF)])\n# TASK 4: effective independent looks = n_valid / oversampling (metadata-derived, ~1.8x).\n# Used ONLY for reliability/uncertainty - farm backscatter means are untouched.\n_OV = json.load(open(R + "_diag/competition_data_deep_dive.json"))["resolution"]["oversampling_factor"]\nNEFF = np.column_stack([[fl(feat[f]["n_valid_%s" % d]) / _OV[d] for f in fids] for d in DATES])\n\nP = np.column_stack([[fl(crop[f]["P_" + c]) for f in fids] for c in CROPS])\ncrop_ent = np.array([fl(crop[f]["crop_entropy"]) for f in fids])\ncent = {int(r["farm_id"]): (fl(r["x"]), fl(r["y"])) for r in csv.DictReader(open(R + "_diag/farm_centroids.csv"))}\nXM = np.array([cent.get(f, (np.nan, np.nan))[0] for f in fids])\nEAST = XM > np.nanmedian(XM)\n\nS2D = sorted(set(k[5:] for k in next(iter(s2.values())) if k.startswith("NDVI_2")))\nNDV = np.column_stack([[fl(s2.get(f, {}).get(f"NDVI_{d}", "")) for f in fids] for d in S2D])\nNDM = np.column_stack([[fl(s2.get(f, {}).get(f"NDMI_{d}", "")) for f in fids] for d in S2D])\n\n# structure reliability: ml3_sd_db is only trustworthy in the largest farms\nq = np.nanpercentile(area[np.isfinite(area)], [75])\nSTRUCT_OK = np.isfinite(area) & (area >= q[0]) & CAPOK\nprint("[data] %d farms | capella-complete %d | structure-reliable %d"\n      % (NF, CAPOK.sum(), STRUCT_OK.sum()))\n\n# ============================================================================\n# B1  HEALTH EVIDENCE TABLE  (what X-HH can plausibly sense, and what cannot transfer)\n# ============================================================================\nHEALTH_EVIDENCE = {\n "canopy_water_content": dict(sensed="YES", basis="KB: backscatter tracks vegetation water "\n   "content; loss of canopy moisture at senescence lowers volume scattering.",\n   xhh_caveat="direction transfers; magnitude does not (C-band VV/VH calibration)"),\n "biomass_development": dict(sensed="YES_UNTIL_SATURATION", basis="KB (S1 Netherlands, "\n   "MAPscape maize): backscatter rises steeply through vegetative growth then saturates.",\n   xhh_caveat="X-band saturates EARLIER than C-band - a shorter dynamic range at high "\n              "biomass. Growth shortfall is detectable; growth EXCESS is not."),\n "plant_height_structure": dict(sensed="PARTIAL", basis="KB: vertical stalks drive "\n   "double-bounce; canopy attenuation grows with height.",\n   xhh_caveat="the KB attribution is POLARIMETRIC (VV vs VH differential attenuation). "\n              "Single-pol HH cannot separate double-bounce from volume. NOT transferred."),\n "senescence": dict(sensed="YES", basis="KB: dry-down lowers moisture and volume "\n   "scattering; harvest produces an abrupt drop.", xhh_caveat="direction only"),\n "lodging": dict(sensed="NOT_SEPARABLE", basis="KB: lodging changes backscatter but the "\n   "sign depends on geometry and pol.",\n   xhh_caveat="NOT USED - indistinguishable from moisture change in single-pol HH."),\n "structural_heterogeneity": dict(sensed="CONDITIONAL", basis="within-field texture after "\n   "speckle debiasing (ENL=1 single-look).",\n   xhh_caveat="our own E2 result: ml3_sd_db reliability is 0.22-0.26 in the lower three "\n              "area quartiles and 0.55-0.62 in the top quartile. USED ONLY on large farms."),\n "soil_moisture_confound": dict(sensed="CONFOUND", basis="KB: bare/sparse canopy "\n   "backscatter is dominated by soil moisture.",\n   xhh_caveat="removed by per-farm centring - level is discarded, only shape is used."),\n}\n\n# ============================================================================\n# B2  CROP-CONDITIONED EXPECTED TRAJECTORY  (jackknifed, shrunk)\n# ============================================================================\nMSHAPE = np.array([CID["component_shapes"][s] for s in CID["subtemplates"]])\nSUBC = np.array([CROPS.index(CID["subtemplates"][s].split("_")[0]\n                             if CID["subtemplates"][s].split("_")[0] in CROPS\n                             else CID["subtemplates"][s]) if False else 0\n                 for s in range(len(CID["subtemplates"]))])\nSUBNAMES = CID["subtemplates"]\nSUBC = np.array([CROPS.index(n.split("_")[0]) for n in SUBNAMES])\nKAP = 12.0\n\n\ndef expected_shape(Pc, subset=None):\n    """Crop-conditioned expected centred trajectory, with each farm\'s own\n    contribution removed from its own component mean (jackknife)."""\n    fit = (np.ones(NF, bool) if subset is None else subset.copy()) & CAPOK\n    # responsibilities over subtemplates, split within crop by literature weights\n    Wsub = np.zeros((NF, len(SUBNAMES)))\n    for k, n in enumerate(SUBNAMES):\n        Wsub[:, k] = Pc[:, SUBC[k]]\n    for c in range(len(CROPS)):\n        m = SUBC == c\n        if m.sum() > 1:\n            Wsub[:, m] /= m.sum()\n    Wsub /= np.clip(Wsub.sum(1, keepdims=True), 1e-12, None)\n    OBS = np.isfinite(S)\n    Sz = np.where(OBS, S, 0.0)\n    W = Wsub * fit[:, None]\n    gmean = np.nanmean(S[fit], 0)\n    gmean = np.where(np.isfinite(gmean), gmean, 0.0)\n    num = np.zeros((len(SUBNAMES), 4)); den = np.zeros((len(SUBNAMES), 4))\n    for k in range(len(SUBNAMES)):\n        wk = W[:, k:k + 1] * OBS\n        num[k] = (wk * Sz).sum(0); den[k] = wk.sum(0)\n    n_eff = den.mean(1)\n    E = np.zeros((NF, 4)); shrink = np.zeros(NF)\n    for i in range(NF):\n        wi = W[i][:, None] * OBS[i][None, :]              # (NSUB x 4) own contribution\n        nj = num - wi * Sz[i][None, :]\n        dj = den - wi\n        Mj = (nj + KAP * gmean[None, :]) / (dj + KAP)     # jackknifed + shrunk\n        Mj -= Mj.mean(1, keepdims=True)\n        E[i] = Wsub[i] @ Mj\n        shrink[i] = float(KAP / (Wsub[i] @ np.maximum(dj.mean(1), 0) + KAP))\n    return E, shrink, n_eff\n\n\n# ---- B4: rank-anchored expected Aug->Oct change ------------------------------\n# The freely-fitted component means collapse onto each other (all d3 within 0.05 dB),\n# because the crop responsibilities carry little per-farm variation. A collapsed\n# expectation makes "crop-conditioned" a claim we would not actually be delivering.\n# So the senescence expectation is ANCHORED: the literature supplies the ORDERING\n# (cotton declines least, rice/maize intermediate, bajra/groundnut most - the one\n# prediction our own Capella data confirmed at rho=+0.313), and OUR OWN observed d3\n# spread supplies the scale. No dB value is imported from any paper.\nD3_RANK = json.load(open(R + "_diag/crop_id_v3.json"))["group_definition"]\nRANK_BY_SUB = {"Cotton_early": 0, "Cotton_timely": 0, "Cotton_late": 0,\n               "Rice": 1, "Maize": 1, "Groundnut": 2, "Bajra": 2}\n_d3obs = (S[:, 3] - S[:, 2])[CAPOK]\nS3 = float(np.nanpercentile(_d3obs, 75) - np.nanpercentile(_d3obs, 25))\nANCHOR_D3 = np.array([(1 - RANK_BY_SUB[n]) * 0.5 * S3 for n in SUBNAMES])\nprint("[B4] observed Aug->Oct IQR %.3f dB -> anchored expectations %s"\n      % (S3, {n: round(float(ANCHOR_D3[k]), 3) for k, n in enumerate(SUBNAMES)}))\n\ndef wsub_of(Pc):\n    W = np.zeros((NF, len(SUBNAMES)))\n    for k in range(len(SUBNAMES)):\n        W[:, k] = Pc[:, SUBC[k]]\n    for c in range(len(CROPS)):\n        m = SUBC == c\n        if m.sum() > 1:\n            W[:, m] /= m.sum()\n    return W / np.clip(W.sum(1, keepdims=True), 1e-12, None)\n\n\ndef anchored_d3(Pc):\n    """Crop-conditioned EXPECTED Aug->Oct change: literature ordering, our scale."""\n    return wsub_of(Pc) @ ANCHOR_D3\n\n\nE_exp, SHRINK, N_EFF = expected_shape(P)\nRESID = S - E_exp\nprint("[B2] component effective counts: %s"\n      % {SUBNAMES[k]: round(float(N_EFF[k]), 1) for k in range(len(SUBNAMES))})\nprint("[B2] jackknife+shrinkage weight: median %.3f  p90 %.3f  max %.3f"\n      % (np.median(SHRINK), np.percentile(SHRINK, 90), SHRINK.max()))\n\n# ============================================================================\n# B3  PHYSICAL FAILURE-MODE CHANNELS  (signed so that HIGHER = HEALTHIER)\n# ============================================================================\ndef rz(v, m=None, out_mask=None):\n    """Robust z-score with THREE separate masks.\n      ref (m)     - farms defining the median/IQR: the clean subset\n      out_mask    - farms allowed to RECEIVE a score; None = anyone with a finite value\n    Emitting a score wherever the value is merely finite is wrong for any channel whose\n    VALIDITY is restricted, not just its cleanliness. The texture channel is the case\n    that bit: E2 established ml3_sd is unreliable below the top area quartile, so\n    scoring small farms against a large-farm reference drove the 18 invalid-Jun19 farms\n    to -3.54 on homogeneity alone and produced a bogus mean health of 13.3."""\n    o = np.full(NF, np.nan)\n    ref = np.isfinite(v) if m is None else (np.isfinite(v) & m)\n    out = np.isfinite(v) if out_mask is None else (np.isfinite(v) & out_mask)\n    if ref.sum() < 20:\n        return o\n    iqr = np.nanpercentile(v[ref], 75) - np.nanpercentile(v[ref], 25)\n    o[out] = (v[out] - np.nanmedian(v[ref])) / (iqr + 1e-9)\n    return np.clip(o, -4, 4)\n\n\nD = np.diff(S, axis=1)\nDE = np.diff(E_exp, axis=1)\npeak = np.nanmax(S, 1) - np.nanmin(S, 1)\npeak_e = np.nanmax(E_exp, 1) - np.nanmin(E_exp, 1)\n\nCH = {\n "establishment":  rz(D[:, 0] - DE[:, 0], CAPOK),\n "development":    rz(D[:, 1] - DE[:, 1], CAPOK),\n "growth_amplitude": rz(peak - peak_e, CAPOK),\n "senescence_timing": rz(D[:, 2] - anchored_d3(P), CAPOK),\n "late_state":     rz(RESID[:, 3], CAPOK),\n "consistency":    rz(-np.sqrt(np.nanmean(RESID ** 2, 1)), CAPOK),\n "homogeneity":    rz(-np.nanmean(ml3, 1), STRUCT_OK, STRUCT_OK),\n}\nCHNAMES = list(CH)\n\n\ndef debias(M):\n    """Remove the OBSERVATION-COUNT artifact from each channel.\n\n    Channels built on the observed trajectory are not invariant to how many dates a\n    farm has. growth_amplitude is max-min over OBSERVED dates, so a 3-date farm has a\n    structurally smaller range and reads as growth shortfall; consistency is an RMS\n    over a different number of residuals. Scoring such farms against a 4-date\n    reference distribution is a data-quality artifact, not crop condition - it drove\n    partial_sar farms to a mean health of 32.2 against 50.9 for complete farms.\n\n    Fix: each channel is centred and scaled within its own observation-pattern\n    subgroup where that subgroup is large enough to estimate them (>=15 farms);\n    smaller subgroups keep the global reference and are given elevated uncertainty\n    through the missingness term. Group differences that are pure artifacts are\n    removed; genuine within-group ordering is untouched."""\n    M = M.copy()\n    key = [tuple(np.isfinite(Z[i]).tolist()) for i in range(NF)]\n    uk = {}\n    for i, k in enumerate(key):\n        uk.setdefault(k, []).append(i)\n    for k, ix in uk.items():\n        ix = np.array(ix)\n        if len(ix) < 15 or all(k):\n            continue\n        for j in range(M.shape[1]):\n            v = M[ix, j]\n            if np.isfinite(v).sum() < 10:\n                continue\n            iqr = np.nanpercentile(v, 75) - np.nanpercentile(v, 25)\n            ref = np.nanpercentile(M[:, j][np.array([all(x) for x in key])], [25, 50, 75])\n            scale = (ref[2] - ref[0]) / (iqr + 1e-9)\n            M[ix, j] = (v - np.nanmedian(v)) * np.clip(scale, 0.5, 2.0) + ref[1]\n    return M\n\n\nCM = debias(np.column_stack([CH[k] for k in CHNAMES]))\nprint("[B3] channels: %s" % CHNAMES)\nprint("[B3] channel availability: %s"\n      % {k: int(np.isfinite(CH[k]).sum()) for k in CHNAMES})\n\n# B4 check: is the senescence channel now crop-aware? Expected d3 must differ by crop.\nd3_by_crop = {CROPS[c]: round(float(ANCHOR_D3[SUBC == c].mean()), 3) for c in range(len(CROPS))}\nprint("[B4] crop-conditioned EXPECTED Aug->Oct change (dB): %s" % d3_by_crop)\n_ad = anchored_d3(P)\nprint("[B4] per-farm expected d3 spread: sd %.3f dB (0 would mean conditioning is inert)"\n      % float(np.nanstd(_ad)))\n\n# ============================================================================\n# B5/B6  MODEL FAMILIES x WEIGHTING SCHEMES\n# ============================================================================\ndef agg_equal(M, w=None):\n    return np.nanmean(M, 1)\n\n\ndef agg_rank(M, w=None):\n    Rk = np.full_like(M, np.nan)\n    for j in range(M.shape[1]):\n        m = np.isfinite(M[:, j])\n        if m.sum() > 10:\n            Rk[m, j] = (np.argsort(np.argsort(M[m, j])) / max(m.sum() - 1, 1)) - 0.5\n    return np.nanmean(Rk, 1)\n\n\ndef agg_median(M, w=None):\n    return np.nanmedian(M, 1)\n\n\ndef agg_relw(M, w):\n    W = np.where(np.isfinite(M), w[None, :], np.nan)\n    return np.nansum(np.nan_to_num(M) * np.nan_to_num(W), 1) / np.clip(np.nansum(W, 1), 1e-9, None)\n\n\ndef agg_pca(M, w=None):\n    m = np.all(np.isfinite(M[:, :6]), 1)\n    if m.sum() < 50:\n        return np.full(NF, np.nan)\n    X = M[:, :6][m]\n    X = X - X.mean(0)\n    if not np.all(np.isfinite(X)) or X.shape[0] < 10:\n        return np.full(NF, np.nan)\n    u, s_, vt = np.linalg.svd(X, full_matrices=False)\n    ax = vt[0] * (1 if vt[0].sum() >= 0 else -1)\n    o = np.full(NF, np.nan)\n    o[m] = X @ ax\n    return o\n\n\n# reliability weights: inverse bootstrap SD, and E2\'s structure reliability for texture\nrelw = np.ones(len(CHNAMES))\nrelw[CHNAMES.index("homogeneity")] = 0.55 / 0.25          # E2: 0.55-0.62 vs 0.22-0.26\nrelw = relw / relw.sum() * len(CHNAMES)\n\n# H1 crop-conditioned residual model (the v2 proposal), under five weightings\n# H2 phenological-neighbour anomaly: deviation from the 25 nearest farms in shape space\ndef knn_anomaly(k=25):\n    m = CAPOK & np.all(np.isfinite(S), 1)\n    idx = np.where(m)[0]\n    X = S[idx]\n    o = np.full(NF, np.nan)\n    d2 = ((X[:, None, :] - X[None, :, :]) ** 2).sum(2)\n    np.fill_diagonal(d2, np.inf)\n    nn = np.argsort(d2, 1)[:, :k]\n    o[idx] = -np.linalg.norm(X - X[nn].mean(1), axis=1)\n    return rz(o, m)\n\n\n# H3 v1.1 incumbent: unconditioned equal-weight of three channels\n_h11 = R + "farm_health_v1_1.csv"\nif os.path.exists(_h11):\n    v1 = {int(r["farm_id"]): fl(r["health_score_raw"])\n          for r in csv.DictReader(open(_h11, encoding="utf-8"))}\n    H3 = rz(np.array([v1.get(f, np.nan) for f in fids]), CAPOK)\nelse:\n    # Historical v1.1 incumbent not regenerated. H3 is NOT a member of AGGS, so it can\n    # never be selected; it is reported as unavailable rather than silently fabricated.\n    H3 = np.full(len(fids), np.nan)\n# H4 organiser baselines\nH4a = rz(np.nanmean(Z, 1), CAPOK)                       # mean level\nH4b = rz(Z[:, 3] - Z[:, 0], CAPOK)                      # temporal change\n\nCAND = {}\nfor nm, fn in [("H1_equal", agg_equal), ("H1_rank", agg_rank), ("H1_median", agg_median),\n               ("H1_reliability", agg_relw), ("H1_latent_pca", agg_pca)]:\n    CAND[nm] = fn(CM, relw)\nCAND["H2_knn_anomaly"] = knn_anomaly()\nCAND["H3_v1.1_unconditioned"] = H3\nCAND["H4a_baseline_mean_level"] = H4a\nCAND["H4b_baseline_temporal_change"] = H4b\n\n# ============================================================================\n# B7  VALIDATION MATRIX\n# ============================================================================\ndef sp(a, b):\n    m = np.isfinite(a) & np.isfinite(b)\n    if m.sum() < 30:\n        return float("nan")\n    ra = np.argsort(np.argsort(a[m])).astype(float)\n    rb = np.argsort(np.argsort(b[m])).astype(float)\n    return float(np.corrcoef(ra, rb)[0, 1])\n\n\nNUIS = np.column_stack([area, inc, nval, urange, np.log(np.clip(area, 1e-3, None))])\n\n\ndef nuis_r2(v):\n    m = np.isfinite(v) & np.all(np.isfinite(NUIS), 1)\n    if m.sum() < 50:\n        return float("nan")\n    X = np.column_stack([np.ones(m.sum()), (NUIS[m] - NUIS[m].mean(0)) / (NUIS[m].std(0) + 1e-9)])\n    b, *_ = np.linalg.lstsq(X, v[m], rcond=None)\n    r = v[m] - X @ b\n    return float(1 - r.var() / (v[m].var() + 1e-12))\n\n\n# perturbation batteries -------------------------------------------------------\nZ3 = Z.copy(); Z3[:, 1] = np.nan                       # 3-date (no Jun19)\nS_full = S.copy()\nE_full, SH_full = E_exp.copy(), SHRINK.copy()\n\n\ndef rebuild(Zx, Px):\n    global S, E_exp, RESID, D, DE, peak, peak_e, CM\n    S = Zx - np.nanmean(Zx, 1, keepdims=True)\n    E_exp, _, _ = expected_shape(Px)\n    RESID = S - E_exp\n    D, DE = np.diff(S, 1), np.diff(E_exp, 1)\n    peak = np.nanmax(S, 1) - np.nanmin(S, 1)\n    peak_e = np.nanmax(E_exp, 1) - np.nanmin(E_exp, 1)\n    ch = {"establishment": rz(D[:, 0] - DE[:, 0], CAPOK), "development": rz(D[:, 1] - DE[:, 1], CAPOK),\n          "growth_amplitude": rz(peak - peak_e, CAPOK),\n          "senescence_timing": rz(D[:, 2] - anchored_d3(Px), CAPOK),\n          "late_state": rz(RESID[:, 3], CAPOK), "consistency": rz(-np.sqrt(np.nanmean(RESID ** 2, 1)), CAPOK),\n          "homogeneity": rz(-np.nanmean(ml3, 1), STRUCT_OK, STRUCT_OK)}\n    return debias(np.column_stack([ch[k] for k in CHNAMES]))\n\n\nCM3 = rebuild(Z3, P)\nrng = np.random.RandomState(1)\nZp = Z.copy()\nZp[:, 1] = Zp[:, 1] + rng.randn(NF) * np.clip(1.0 - qw19, 0.05, None)   # Jun19 uncertainty\nCMp = rebuild(Zp, P)\nPperm = P + rng.dirichlet(np.ones(5), NF) * 0.30                          # crop-posterior noise\nPperm /= Pperm.sum(1, keepdims=True)\nCMc = rebuild(Z, Pperm)\nCM_e = rebuild(Z, P)                                                       # restore\nS, E_exp = S_full, E_full\n\nVAL = {}\nprint("\\n" + "=" * 92)\nprint("B7  HEALTH VALIDATION MATRIX   (S2 is diagnostic only and never fitted)")\nprint("=" * 92)\nhdr = ("%-28s %7s %7s %7s %7s %7s %7s %7s" %\n       ("candidate", "nuisR2", "3date", "jun19p", "cropP", "spatial", "size", "S2rho"))\nprint(hdr)\nAGGS = {"H1_equal": agg_equal, "H1_rank": agg_rank, "H1_median": agg_median,\n        "H1_reliability": agg_relw, "H1_latent_pca": agg_pca}\nbig = area >= np.nanmedian(area)\nfor nm, v in CAND.items():\n    if nm in AGGS:\n        v3 = AGGS[nm](CM3, relw); vp = AGGS[nm](CMp, relw); vc = AGGS[nm](CMc, relw)\n    elif nm == "H4a_baseline_mean_level":\n        v3, vp, vc = rz(np.nanmean(Z3, 1), CAPOK), rz(np.nanmean(Zp, 1), CAPOK), v\n    elif nm == "H4b_baseline_temporal_change":\n        v3, vp, vc = v, rz(Zp[:, 3] - Zp[:, 0], CAPOK), v\n    elif nm == "H2_knn_anomaly":\n        Ssave = S.copy()\n        S = Z3 - np.nanmean(Z3, 1, keepdims=True); v3 = knn_anomaly()\n        S = Zp - np.nanmean(Zp, 1, keepdims=True); vp = knn_anomaly()\n        S = Ssave; vc = v\n    else:\n        # H3 is loaded from the frozen v1.1 CSV and cannot be recomputed under a\n        # perturbation. Reported as NOT TESTED rather than as a passing 1.000.\n        v3 = vp = vc = None\n    # genuine spatial blocking: rebuild the expectation from EAST farms only,\n    # then score the held-out WEST farms and compare their ranking.\n    if nm in AGGS:\n        Ee, _, _ = expected_shape(P, subset=EAST)\n        RE = S_full - Ee\n        De, DEe = np.diff(S_full, 1), np.diff(Ee, 1)\n        pe = np.nanmax(Ee, 1) - np.nanmin(Ee, 1)\n        che = {"establishment": rz(De[:, 0] - DEe[:, 0], CAPOK), "development": rz(De[:, 1] - DEe[:, 1], CAPOK),\n               "growth_amplitude": rz((np.nanmax(S_full, 1) - np.nanmin(S_full, 1)) - pe, CAPOK),\n               "senescence_timing": rz(De[:, 2] - anchored_d3(P), CAPOK), "late_state": rz(RE[:, 3], CAPOK),\n               "consistency": rz(-np.sqrt(np.nanmean(RE ** 2, 1)), CAPOK),\n               "homogeneity": rz(-np.nanmean(ml3, 1), STRUCT_OK, STRUCT_OK)}\n        ve = AGGS[nm](np.column_stack([che[k] for k in CHNAMES]), relw)\n        spa = sp(ve[~EAST], v[~EAST])\n    else:\n        spa = float("nan")\n    nan_ = float("nan")\n    VAL[nm] = dict(nuisance_r2=round(nuis_r2(v), 4),\n                   three_date_rho=nan_ if v3 is None else round(sp(v3, v), 4),\n                   jun19_perturb_rho=nan_ if vp is None else round(sp(vp, v), 4),\n                   crop_posterior_rho=nan_ if vc is None else round(sp(vc, v), 4),\n                   spatial_block_rho=round(spa, 4),\n                   size_rho_big_vs_all=round(sp(v[big], np.nan_to_num(v[big])), 4),\n                   size_split_rho=round(sp(v, area), 4),\n                   s2_ndvi_rho_DIAGNOSTIC=round(sp(v, NDV[:, -1]), 4),\n                   s2_ndmi_rho_DIAGNOSTIC=round(sp(v, NDM[:, -1]), 4))\n    print("%-28s %7.3f %7.3f %7.3f %7.3f %7.3f %7.3f %7.3f"\n          % (nm, VAL[nm]["nuisance_r2"], VAL[nm]["three_date_rho"], VAL[nm]["jun19_perturb_rho"],\n             VAL[nm]["crop_posterior_rho"], VAL[nm]["spatial_block_rho"],\n             VAL[nm]["size_split_rho"], VAL[nm]["s2_ndvi_rho_DIAGNOSTIC"]))\n\n# --- the advisor\'s check: is Jun19 actually load-bearing, or absent by construction? --\nJ19DEP = {}\nfor k in CHNAMES:\n    if k in ("establishment", "development"):\n        J19DEP[k] = "DIRECT - defined on a difference that contains Jun19"\n    elif k in ("growth_amplitude", "consistency"):\n        J19DEP[k] = "PARTIAL - Jun19 enters through the 4-date shape"\n    elif k == "senescence_timing":\n        J19DEP[k] = "INDIRECT - Jun19 affects the centring only"\n    else:\n        J19DEP[k] = "NONE - Oct13 / texture only"\nprint("\\n  Jun19 dependence by channel (construction audit): %s" % J19DEP)\n\n# ---- leave-one-component-out -------------------------------------------------\nLOCO = {}\nbase = agg_equal(CM_e)\nfor j, k in enumerate(CHNAMES):\n    M2 = np.delete(CM_e, j, 1)\n    LOCO[k] = round(sp(agg_equal(M2), base), 4)\nprint("  leave-one-component-out rho vs full: %s" % LOCO)\n\n# ============================================================================\n# B6  SELECTION - on Capella-internal criteria only, never on S2\n# ============================================================================\ndef score(nm):\n    d = VAL[nm]\n    if not np.isfinite(d["spatial_block_rho"]):\n        return -9\n    return (d["spatial_block_rho"] + d["three_date_rho"] + d["jun19_perturb_rho"]\n            + d["crop_posterior_rho"]) / 4 - 2.0 * d["nuisance_r2"]\n\n\nranked = sorted(AGGS, key=score, reverse=True)\nBEST = ranked[0]\nprint("\\n  selection score (mean stability - 2x nuisance R2), S2 EXCLUDED:")\nfor nm in ranked:\n    print("    %-24s %.4f" % (nm, score(nm)))\nprint("  SELECTED: %s" % BEST)\n\n# ============================================================================\n# B8  UNCERTAINTY PROPAGATION  +  B9  ALL-966 PATHWAYS\n# ============================================================================\nraw = AGGS[BEST](CM_e, relw)\nfam = np.column_stack([CAND[n] for n in ["H1_equal", "H1_rank", "H2_knn_anomaly", "H3_v1.1_unconditioned"]])\nfam_disagree = np.nanstd(fam, 1)\n\nn_ch = np.isfinite(CM_e).sum(1)\nr = raw.copy()\nr[n_ch < 2] = np.nan                      # fewer than 2 channels is not an estimate\nm = np.isfinite(r)\nHEALTH = np.full(NF, np.nan)\nHEALTH[m] = 100.0 * (np.argsort(np.argsort(r[m])) / max(m.sum() - 1, 1))\n# B9: farms with no usable SAR get the population median, explicitly flagged and given\n# maximum uncertainty. This is a stated fallback, NOT a fabricated observation - the\n# health_evidence_quality and health_pathway columns make it visible in every output.\nNO_EST = ~m\nHEALTH[NO_EST] = 50.0\nprint("  channels available per farm: %s"\n      % {int(k): int((n_ch == k).sum()) for k in sorted(set(n_ch.tolist()))})\nprint("  estimated from SAR evidence: %d | population-median fallback: %d"\n      % (int(m.sum()), int(NO_EST.sum())))\n\nU = np.column_stack([\n    crop_ent,                                                   # crop identity uncertainty\n    1.0 - ndates / 4.0,                                         # observation missingness\n    1.0 - np.clip(qw19, 0, 1),                                  # Jun19 validity/correction\n    rz(fam_disagree, m) * 0 + np.nan_to_num(fam_disagree) / (np.nanstd(fam_disagree) + 1e-9),\n    SHRINK,                                                     # component self-support\n    np.where(STRUCT_OK, 0.0, 0.5),                              # structure reliability\n    1.0 - np.clip(np.nanmean(NEFF, 1) / np.nanpercentile(np.nanmean(NEFF, 1), 90), 0, 1),\n])\nUNC = np.clip(np.nanmean(U, 1), 0, None)\nUNC = 100.0 * (UNC - np.nanmin(UNC)) / (np.nanmax(UNC) - np.nanmin(UNC) + 1e-9)\n\npath = np.where(~np.isfinite(Z).any(1), "no_sar",\n        np.where(~jv, "invalid_jun19",\n        np.where(ndates < 4, "partial_sar",\n        np.where(gp, "full_sar", "low_quality_sar"))))\nprint("\\n  B9 pathways: %s" % {p: int((path == p).sum()) for p in sorted(set(path))})\nprint("  health defined on %d / %d farms; %d carry prior-only crop evidence"\n      % (int(np.isfinite(HEALTH).sum()), NF, int((~CAPOK).sum())))\n# farms with no usable SAR keep a health value ONLY via crop-conditional expectation\n# of the population; they are flagged, never imputed from a fabricated observation.\nprint("  mean health by pathway: %s"\n      % {p: round(float(np.nanmean(HEALTH[path == p])), 1) for p in sorted(set(path))})\n\n# rare-crop self-fitting check the advisor asked for\nprint("  health SD by dominant crop: %s"\n      % {CROPS[c]: round(float(np.nanstd(HEALTH[P.argmax(1) == c])), 1)\n         for c in range(len(CROPS)) if (P.argmax(1) == c).sum() > 5})\n\nOUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)\nDIAGNOSTIC_JSON.parent.mkdir(parents=True, exist_ok=True)\nwith open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as fh:\n    w = csv.writer(fh)\n    w.writerow(["farm_id", "health_index", "health_raw", "health_uncertainty",\n                "health_evidence_quality", "health_pathway", "expected_aug_oct_db",\n                "component_shrinkage", "model_disagreement"] + ["ch_" + k for k in CHNAMES])\n    _exp_d3 = anchored_d3(P)\n    for i, f in enumerate(fids):\n        w.writerow([f,\n                    "" if not np.isfinite(HEALTH[i]) else round(float(HEALTH[i]), 2),\n                    "" if not np.isfinite(raw[i]) else round(float(raw[i]), 4),\n                    "" if not np.isfinite(UNC[i]) else round(float(UNC[i]), 2),\n                    ("full" if CAPOK[i] and STRUCT_OK[i] else\n                     "sar_no_texture" if CAPOK[i] else\n                     "reduced_channels" if m[i] else "population_median_fallback"),\n                    path[i], round(float(_exp_d3[i]), 4), round(float(SHRINK[i]), 4),\n                    round(float(fam_disagree[i]), 4)]\n                   + ["" if not np.isfinite(CM_e[i, j]) else round(float(CM_e[i, j]), 4)\n                      for j in range(len(CHNAMES))])\n\n_input_names = [\n    "farm_features_structure_v1.csv",\n    "farm_reference_v2_offsetcorrected.csv",\n    "_diag/crop_id_v3.json",\n    "_diag/jun19_contamination_map.json",\n    "_diag/structure_reliability.json",\n    "_diag/competition_data_deep_dive.json",\n    "_diag/farm_centroids.csv",\n]\n_input_hashes = {name: sha256(ROOT / name) for name in _input_names}\n_payload = dict(version="v2", selected=BEST, channels=CHNAMES,\n                health_evidence_table=HEALTH_EVIDENCE,\n                crop_conditioned_expected_d3_db=d3_by_crop,\n                component_n_eff={SUBNAMES[k]: round(float(N_EFF[k]), 1) for k in range(len(SUBNAMES))},\n                shrinkage=dict(median=round(float(np.median(SHRINK)), 4),\n                               p90=round(float(np.percentile(SHRINK, 90)), 4)),\n                validation=VAL, jun19_dependence_by_channel=J19DEP,\n                leave_one_component_out=LOCO,\n                selection_note=("selected on spatial-block, 3-date, Jun19-perturbation and "\n                                "crop-posterior stability minus twice the nuisance R2. "\n                                "Sentinel-2 was EXCLUDED from selection and appears only as "\n                                "an out-of-model plausibility diagnostic."),\n                pathways={p: int((path == p).sum()) for p in sorted(set(path))},\n                uncertainty_components=["crop_entropy", "observation_missingness",\n                                        "jun19_quality", "model_disagreement",\n                                        "component_shrinkage", "structure_reliability"],\n                provenance=dict(\n                    model_change=False,\n                    crop_source_path=str(CROP_PATH),\n                    crop_source_sha256=sha256(CROP_PATH),\n                    output_csv_path=str(OUTPUT_CSV),\n                    output_csv_sha256=sha256(OUTPUT_CSV),\n                    invocation=dict(root=str(ROOT), crop_source=ARGS.crop_source,\n                                    output_csv=ARGS.output_csv,\n                                    diagnostic_json=ARGS.diagnostic_json),\n                    relevant_input_sha256=_input_hashes))\n_counter = [0]\n_strict_payload = strict_json(_payload, _counter)\n_strict_payload["strict_json"] = {\n    "status": "PASS",\n    "nonfinite_values_replaced_with_null": _counter[0],\n    "interpretation": "null denotes an unavailable/failed diagnostic metric; production HEALTH values are unchanged",\n}\nwith open(DIAGNOSTIC_JSON, "w", encoding="utf-8") as _fh:\n    json.dump(_strict_payload, _fh, indent=1, allow_nan=False)\n    _fh.write("\\n")\nprint("\\nwrote %s and %s" % (OUTPUT_CSV, DIAGNOSTIC_JSON))\n'
Path(SRC/"build_health_v2.py").write_text(_SRC, encoding="utf-8")
print("wrote src/build_health_v2.py  ({} lines)".format(_SRC.count(chr(10))))

In [ ]:
T["S14"], out14 = run_stage("build_health_v2.py",
    ["farm_health_v2.csv", "_diag/health_v2.json"], "S14 HEALTH v2")
health = pd.read_csv(ROOT/"farm_health_v2.csv")
assert len(health)==966 and np.isfinite(health["health_index"]).all()
print("health index:", health["health_index"].describe(percentiles=[.05,.5,.95]).round(2).to_dict())

### `[E]` HEALTH validation

Selection compares equal, rank, median, reliability and latent-PCA aggregations using nuisance
R², leave-one-date/component perturbations, Jun19 dependence, crop-posterior perturbation,
spatial stability and size bias. S2 is diagnostic only and is not fitted into HEALTH. B9 retains
explicit `full_sar`, `partial_sar`, `invalid_jun19` and `no_sar` pathways; fallbacks are visible.

---
## 10 · Yield estimated to date  ·  `[P] S15`

### METHOD
The operational definition is **historical crop productivity scale × crop-specific stage-to-date
factor × Capella-derived farm-condition adjustment**. APY anchors come from the public Vadodara
series. Final stage factors are Rice `.975`, Cotton `.478`, Maize `.960`, Bajra `1.000`, Groundnut
`1.000`. Cotton is lint; Rice uses an operational cleaned-rice convention with paddy sensitivity;
Groundnut uses pods/in-shell with kernel sensitivity. LOW/HIGH are agronomic scenario sensitivity
bounds, not confidence intervals. The uncertainty summary is a heuristic ranking, not SD or CI.

The Jun19→Aug14 condition signal was rejected from production because its sign was physically
inconsistent across crops; it remains diagnostic only.

In [ ]:
# S15 -- authoritative yield chain stage.
_SRC = '"""\nYIELD v1 - kg/ha per farm, to 13 Oct 2025, NOT final harvest.\n\nTARGET (organizer-established): kg/ha, per farm, per crop, to 13 Oct, not final harvest.\nOPERATIONAL INTERPRETATION (MODEL ASSUMPTION, not organizer wording):\n    yield accumulated toward harvest as of 13 Oct\n        = anchor(crop) x stage_factor(crop) x condition_factor(farm, crop)\n\nStage and condition are kept SEPARATE and inspectable, and every crop-conditioned\nintermediate is written out so the crop posterior can be swapped without redesign.\n\nMODULARITY: the crop posterior is read from CROPSRC (default farm_crop_probabilities_v4.csv).\nRerun with a different file to adopt a better Maize/Groundnut solution - no code change.\n"""\nimport csv, json, os, sys, warnings\nimport numpy as np\n\nsys.stdout.reconfigure(encoding="utf-8", errors="replace")\nwarnings.filterwarnings("ignore")\nnp.random.seed(0)\nR = "round2/"\nCROPS = ["Rice", "Cotton", "Maize", "Bajra", "Groundnut"]\nDATES = ["Jun06", "Jun19", "Aug14", "Oct13"]\nCROPSRC = os.environ.get("CROPSRC", "farm_crop_probabilities_v4.csv")\n\n# ---------------------------------------------------------------- anchors\n# Verified Vadodara kharif district medians. Historical anchors ONLY - not farm labels,\n# not 2025 truth. Product conventions traced from the APY source sheets.\nANCHOR = {"Rice": 2068.0, "Cotton": 846.0, "Maize": 1770.0, "Bajra": 1815.0,\n          "Groundnut": 2579.0}\nANCHOR_PRODUCT = {"Rice": "grain (cleaned-rice reading)", "Cotton": "LINT (source: \'Yield is in lint\')",\n                  "Maize": "grain", "Bajra": "grain", "Groundnut": "pods / in-shell"}\nANCHOR_ALT = {"Rice": 3090.0, "Groundnut": 1800.0}      # paddy-equiv / kernel-equiv\n\n# ---------------------------------------------------------------- stage factors\n# *** MODEL ASSUMPTIONS. Literature supports the ORDERING only, never these numbers. ***\n# Ordering supported by Layer A: Bajra >= Groundnut ~ Maize > Rice >> Cotton\n# Crop-specific reasoning ONLY - the cereal anthesis result is NOT generalised to\n# cotton boll accumulation or groundnut pod filling.\nSTAGE = {\n "Bajra":     dict(central=0.98, low=0.90, high=1.00, conf="MODERATE-HIGH",\n   basis="late-June sowing, 80-90 d duration -> mature and very likely harvested by mid-Sep, "\n         "i.e. before 13 Oct. Crop-specific: short-duration cereal completing its cycle."),\n "Groundnut": dict(central=0.95, low=0.82, high=1.00, conf="MODERATE-HIGH",\n   basis="Junagadh sowing-window trials: optimal sowing early-mid June; 110-120 d -> pod "\n         "maturity/lifting late Sep to mid Oct. Reasoning is POD FILLING duration, NOT "\n         "cereal anthesis."),\n "Maize":     dict(central=0.92, low=0.78, high=1.00, conf="MODERATE",\n   basis="kharif sowing Jun-Jul, ~100 d -> at/past physiological maturity by 13 Oct; often "\n         "left standing for grain dry-down. No Gujarat-specific calendar found - widest "\n         "uncertainty band of the three short crops."),\n "Rice":      dict(central=0.75, low=0.58, high=0.88, conf="HIGH",\n   basis="regional crop-plan: transplant 16-22 Jul, HARVEST 22 Oct - 5 Nov. 13 Oct is ~85-90 "\n         "DAT, i.e. grain filling and BEFORE harvest. Cereal anthesis evidence applies here: "\n         "past flowering, so most yield committed but filling incomplete."),\n "Cotton":    dict(central=0.35, low=0.20, high=0.50, conf="MODERATE-HIGH",\n   basis="P1 Gujarat: first picking mid-Oct, crop indeterminate and running to Dec-Jan. Only "\n         "a small share of total pickings has occurred by 13 Oct. Reasoning is SEQUENTIAL "\n         "BOLL PICKING, NOT cereal anthesis - cotton keeps setting bolls after 13 Oct."),\n}\n\n\ndef fl(v):\n    try:\n        return float(v)\n    except (TypeError, ValueError):\n        return float("nan")\n\n\n# ---------------------------------------------------------------- data\nfeat = {int(r["farm_id"]): r for r in csv.DictReader(open(R + "farm_features_structure_v1.csv", encoding="utf-8"))}\ncrop = {int(r["farm_id"]): r for r in csv.DictReader(open(R + CROPSRC, encoding="utf-8"))}\nheal = {int(r["farm_id"]): r for r in csv.DictReader(open(R + "farm_health_v2.csv", encoding="utf-8"))}\nevd = {int(r["farm_id"]): r for r in csv.DictReader(open(R + "farm_evidence_status_v1.csv", encoding="utf-8"))}\nfids = sorted(feat); NF = len(fids)\nprint("[data] %d farms | crop posterior source: %s" % (NF, CROPSRC))\n\nP = np.column_stack([[fl(crop[f]["P_" + c]) for f in fids] for c in CROPS])\nP = P / np.clip(P.sum(1, keepdims=True), 1e-12, None)\ncmap = np.array([crop[f]["crop_type"] for f in fids])\ncent_ = np.array([fl(crop[f]["crop_entropy"]) for f in fids])\nhealth = np.array([fl(heal[f]["health_index"]) for f in fids])\nhunc = np.array([fl(heal[f]["health_uncertainty"]) for f in fids])\nstatus = np.array([evd[f]["evidence_status"] for f in fids])\npen = np.array([fl(evd[f]["uncertainty_penalty"]) for f in fids])\nclean = np.array([fl(evd[f]["clean_contrast_db"]) for f in fids])     # Aug14 - Jun19_clean\nclean_sd = np.array([fl(evd[f]["clean_contrast_sd"]) for f in fids])\narea = np.array([fl(feat[f]["area_ha"]) for f in fids])\nvill = int(evd[fids[0]]["village_id"])\nZ = np.column_stack([[fl(feat[f]["sigma0_db_trim10_%s" % d]) for f in fids] for d in DATES])\nZ[:, 1] = np.array([fl(feat[f]["sigma0_Jun19_clean"]) for f in fids])\njv = status != "invalid_jun19"\nX = np.array([fl(evd[f].get("clean_contrast_peer", "")) for f in fids])\n\n# ---------------------------------------------------------------- 4. double-count audit\ndef sp(a, b):\n    m = np.isfinite(a) & np.isfinite(b)\n    if m.sum() < 30:\n        return float("nan")\n    return float(np.corrcoef(np.argsort(np.argsort(a[m])), np.argsort(np.argsort(b[m])))[0, 1])\n\n\ndef pe(a, b):\n    m = np.isfinite(a) & np.isfinite(b)\n    return float(np.corrcoef(a[m], b[m])[0, 1]) if m.sum() > 30 else float("nan")\n\n\nCAND = {"clean_contrast_Jun19_Aug14": clean,\n        "clean_contrast_peer_normalised": X,\n        "seasonal_amplitude": np.nanmax(Z, 1) - np.nanmin(Z, 1),\n        "late_state_Oct13": Z[:, 3],\n        "growth_Jun06_Aug14": Z[:, 2] - Z[:, 0]}\nprint("\\n" + "=" * 74)\nprint("STEP 4  HEALTH DOUBLE-COUNT AUDIT")\nprint("=" * 74)\nprint("%-34s %8s %8s %10s" % ("candidate yield feature", "pearson", "spearman", "indep frac"))\nAUD = {}\nfor k, v in CAND.items():\n    pr, sr = pe(v, health), sp(v, health)\n    AUD[k] = dict(pearson_vs_health=round(pr, 4), spearman_vs_health=round(sr, 4),\n                  independent_fraction=round(1 - pr ** 2, 4))\n    print("%-34s %8.4f %8.4f %10.4f" % (k, pr, sr, 1 - pr ** 2))\nprim = "clean_contrast_peer_normalised" if np.isfinite(AUD["clean_contrast_peer_normalised"]["pearson_vs_health"]) \\\n    else "clean_contrast_Jun19_Aug14"\nprint("\\n  HEALTH v2 components already using Jun19->Aug14: \'development\' (DIRECT).")\nprint("  -> the clean contrast IS partly represented in HEALTH; Y3 therefore uses the")\nprint("     part of it ORTHOGONAL to health, never the raw feature at full strength.")\n\n# residualise the primary Capella signal against health -> independent component\nbase = X.copy()\nm = np.isfinite(base) & np.isfinite(health)\nA_ = np.column_stack([np.ones(m.sum()), health[m]])\nb_, *_ = np.linalg.lstsq(A_, base[m], rcond=None)\ncap_indep = np.full(NF, np.nan)\ncap_indep[m] = base[m] - A_ @ b_\nAUD["capella_independent_residual"] = dict(\n    r2_removed=round(float(1 - (base[m] - A_ @ b_).var() / (base[m].var() + 1e-12)), 4))\nprint("  variance of the primary signal explained by health: %.4f (removed in Y3)"\n      % AUD["capella_independent_residual"]["r2_removed"])\n\n\n# ---------------------------------------------------------------- 6. crop peers\ndef wmedian(v, w):\n    m = np.isfinite(v) & (w > 0)\n    if m.sum() < 10:\n        return np.nan\n    o = np.argsort(v[m]); vv, ww = v[m][o], w[m][o]\n    c = np.cumsum(ww) / ww.sum()\n    return float(vv[np.searchsorted(c, 0.5)])\n\n\ndef peer_z(v, Pc):\n    """posterior-weighted robust z per crop; no hard MAP grouping"""\n    out = np.full((NF, len(CROPS)), np.nan)\n    for j in range(len(CROPS)):\n        w = Pc[:, j]\n        med = wmedian(v, w)\n        if not np.isfinite(med):\n            continue\n        mad = wmedian(np.abs(v - med), w)\n        s = 1.4826 * mad if (np.isfinite(mad) and mad > 1e-6) else np.nanstd(v)\n        out[:, j] = (v - med) / max(s, 1e-6)\n    return np.clip(out, -4, 4)\n\n\n# ---------------------------------------------------------------- 5. condition forms\ndef cond_sym(z, k=0.06):\n    return np.clip(1.0 + k * z, 0.70, 1.30)\n\n\ndef cond_asym(z, dn=0.10, up=0.04):\n    """X-band saturates early -> shortfall trusted more than excess. MODEL FORM."""\n    return np.clip(1.0 + dn * np.minimum(z, 0) + up * np.maximum(z, 0), 0.60, 1.20)\n\n\ndef cond_none(z):\n    return np.ones_like(z)\n\n\n# ---------------------------------------------------------------- assemble\ndef build(arch="Y3", stage_key="central", anchors=None, cond=cond_asym,\n          Pc=None, cap=None, hl=None, stage_over=None):\n    Pc = P if Pc is None else Pc\n    anchors = ANCHOR if anchors is None else anchors\n    cap = cap_indep if cap is None else cap\n    hl = health if hl is None else hl\n    st = {c: (stage_over[c] if stage_over else STAGE[c][stage_key]) for c in CROPS}\n    hz = np.full(NF, np.nan)\n    mh = np.isfinite(hl)\n    hz[mh] = (hl[mh] - np.median(hl[mh])) / (1.4826 * np.median(np.abs(hl[mh] - np.median(hl[mh]))) + 1e-6)\n    ZC = peer_z(cap, Pc)\n    ZH = peer_z(hz, Pc)\n    Yc = np.zeros((NF, len(CROPS))); CF = np.zeros((NF, len(CROPS)))\n    for j, c in enumerate(CROPS):\n        zc = np.nan_to_num(ZC[:, j]); zh = np.nan_to_num(ZH[:, j])\n        if arch == "Y0":\n            f = np.ones(NF)\n        elif arch == "Y1":\n            f = cond(zh)\n        elif arch == "Y2":\n            f = cond(zc)\n        else:                      # Y3 = health + orthogonal Capella residual\n            f = cond(0.65 * zh + 0.35 * zc)\n        CF[:, j] = f\n        Yc[:, j] = anchors[c] * st[c] * f\n    return Yc, CF, st\n\n\nprint("\\n" + "=" * 74)\nprint("STEP 12  BASELINE ARCHITECTURES (central stage, asymmetric condition)")\nprint("=" * 74)\nBASE = {}\nfor a in ["Y0", "Y1", "Y2", "Y3"]:\n    Yc, CF, st = build(a)\n    y = (P * Yc).sum(1)\n    BASE[a] = dict(median=round(float(np.nanmedian(y)), 1),\n                   iqr=round(float(np.nanpercentile(y, 75) - np.nanpercentile(y, 25)), 1),\n                   cv=round(float(np.nanstd(y) / np.nanmean(y)), 4),\n                   rho_health=round(sp(y, health), 4),\n                   rho_area=round(sp(y, area), 4))\n    print("  %-3s median %7.1f  IQR %6.1f  CV %.4f  rho(health) %+.3f  rho(area) %+.3f"\n          % (a, BASE[a]["median"], BASE[a]["iqr"], BASE[a]["cv"],\n             BASE[a]["rho_health"], BASE[a]["rho_area"]))\n\nYc, CF, ST = build("Y3")\nY = (P * Yc).sum(1)\n\n# ---------------------------------------------------------------- 9. missingness\n# invalid-Jun19 farms: the clean contrast is NaN by construction (never imputed) so their\n# Capella term is zero-weighted and the model falls back to the health component.\nfallback = np.where(status == "full_capella", "full",\n           np.where(status == "invalid_jun19", "no_clean_contrast_health_only",\n           np.where(status == "partial_capella", "reduced_dates",\n                    "anchor_plus_prior_only")))\nnoY = ~np.isfinite(Y)\nY[noY] = float(np.nansum(P[noY] * np.array([ANCHOR[c] * ST[c] for c in CROPS])[None, :], axis=1)) \\\n    if noY.sum() else Y[noY]\nfor i in np.where(noY)[0]:\n    Y[i] = float((P[i] * np.array([ANCHOR[c] * ST[c] for c in CROPS])).sum())\n\n# ---------------------------------------------------------------- 11. uncertainty\nU = np.column_stack([\n    cent_,                                            # crop identity uncertainty\n    np.clip(hunc / 100.0, 0, 1),                      # health uncertainty\n    pen,                                              # evidence-status penalty\n    np.where(np.isfinite(clean), 0.0, 0.5),           # no clean contrast\n    np.clip(clean_sd / np.nanmax(clean_sd[np.isfinite(clean_sd)]), 0, 1),\n])\nUNC = np.nanmean(U, 1)\nUNC = 100.0 * (UNC - np.nanmin(UNC)) / (np.nanmax(UNC) - np.nanmin(UNC) + 1e-9)\n\n# ---------------------------------------------------------------- 10/14 sensitivity\nprint("\\n" + "=" * 74)\nprint("STEP 10  STAGE-FACTOR SENSITIVITY (posterior-weighted village mean, kg/ha)")\nprint("=" * 74)\nSENS = {}\nfor k in ["low", "central", "high"]:\n    Yk, _, stk = build("Y3", stage_key=k)\n    yk = (P * Yk).sum(1)\n    SENS[k] = dict(village_mean=round(float(np.nanmean(yk)), 1),\n                   stage=dict(stk),\n                   per_crop={c: round(float(np.nanmean(Yk[:, j])), 1) for j, c in enumerate(CROPS)})\n    print("  %-8s village mean %7.1f   per-crop %s" % (k, SENS[k]["village_mean"], SENS[k]["per_crop"]))\n\nprint("\\n" + "=" * 74)\nprint("STEP 14  ROBUSTNESS")\nprint("=" * 74)\nROB = {}\nrs = np.random.RandomState(1)\nPp = P + rs.dirichlet(np.ones(5), NF) * 0.30\nPp /= Pp.sum(1, keepdims=True)\nfor nm, kw in [("crop_posterior_perturb", dict(Pc=Pp)),\n               ("jun19_removed", dict(cap=np.full(NF, np.nan))),\n               ("condition_symmetric", dict(cond=cond_sym)),\n               ("condition_none", dict(cond=cond_none)),\n               ("health_only_Y1", dict(arch="Y1")),\n               ("anchor_minus_20pct", dict(anchors={c: v * 0.8 for c, v in ANCHOR.items()}))]:\n    Yk, _, _ = build(**{**dict(arch="Y3"), **kw})\n    # The posterior used for the FINAL weighting must be the perturbed one, otherwise the\n    # crop-posterior test only perturbs peer normalisation and reports a meaningless rho=1.\n    Pw = kw.get("Pc", P)\n    yk = (Pw * Yk).sum(1)\n    ROB[nm] = dict(spearman_vs_base=round(sp(yk, Y), 4),\n                   median_abs_pct_change=round(float(np.nanmedian(np.abs(yk - Y) / np.clip(Y, 1e-6, None)) * 100), 2))\n    print("  %-26s rho %.4f   median |change| %6.2f%%"\n          % (nm, ROB[nm]["spearman_vs_base"], ROB[nm]["median_abs_pct_change"]))\n# product-convention sensitivity (diagnostic, never mixed into one scenario)\nPROD = {}\nfor nm, an in [("rice_paddy_equiv", {**ANCHOR, "Rice": ANCHOR_ALT["Rice"]}),\n               ("groundnut_kernel_equiv", {**ANCHOR, "Groundnut": ANCHOR_ALT["Groundnut"]})]:\n    Yk, _, _ = build("Y3", anchors=an)\n    PROD[nm] = round(float(np.nanmean((P * Yk).sum(1))), 1)\nprint("  product conventions (village mean): primary %.1f | %s"\n      % (np.nanmean(Y), PROD))\n\n# ---------------------------------------------------------------- 14 distributions\nprint("\\n" + "=" * 74)\nprint("STEP 14  CROP-CONDITIONED DISTRIBUTIONS (posterior mass weighted)")\nprint("=" * 74)\nDIST = {}\nfor j, c in enumerate(CROPS):\n    w = P[:, j]\n    v = Yc[:, j]\n    DIST[c] = dict(posterior_mass=round(float(w.sum()), 1), anchor=ANCHOR[c],\n                   product=ANCHOR_PRODUCT[c], stage=ST[c],\n                   mean=round(float(np.nansum(w * v) / max(w.sum(), 1e-9)), 1),\n                   p05=round(float(np.nanpercentile(v, 5)), 1),\n                   p95=round(float(np.nanpercentile(v, 95)), 1),\n                   min=round(float(np.nanmin(v)), 1), max=round(float(np.nanmax(v)), 1))\n    print("  %-10s mass %6.1f  anchor %6.0f x stage %.2f -> mean %7.1f  [p05 %7.1f, p95 %7.1f]"\n          % (c, DIST[c]["posterior_mass"], ANCHOR[c], ST[c], DIST[c]["mean"],\n             DIST[c]["p05"], DIST[c]["p95"]))\nprint("\\n  FINAL posterior-weighted yield: median %.1f  mean %.1f  p05 %.1f  p95 %.1f  min %.1f  max %.1f"\n      % (np.nanmedian(Y), np.nanmean(Y), np.nanpercentile(Y, 5), np.nanpercentile(Y, 95),\n         np.nanmin(Y), np.nanmax(Y)))\nprint("  negative or zero values: %d | non-finite: %d" % (int((Y <= 0).sum()), int((~np.isfinite(Y)).sum())))\n\n# ---------------------------------------------------------------- output\nCZ = peer_z(cap_indep, P)\nwith open(R + "farm_yield_v1.csv", "w", newline="", encoding="utf-8") as fh:\n    w = csv.writer(fh)\n    w.writerow(["village_id", "farm_id", "crop_MAP", "crop_entropy", "health_index"]\n               + ["P_" + c for c in CROPS] + ["anchor_" + c for c in CROPS]\n               + ["stage_" + c for c in CROPS] + ["condition_" + c for c in CROPS]\n               + ["yield_" + c for c in CROPS]\n               + ["crop_condition_z", "yield_estimate_to_date", "yield_uncertainty",\n                  "evidence_status", "fallback_path"])\n    for i, f in enumerate(fids):\n        w.writerow([vill, f, cmap[i], round(float(cent_[i]), 4),\n                    "" if not np.isfinite(health[i]) else round(float(health[i]), 2)]\n                   + [round(float(P[i, j]), 5) for j in range(5)]\n                   + [ANCHOR[c] for c in CROPS]\n                   + [ST[c] for c in CROPS]\n                   + [round(float(CF[i, j]), 4) for j in range(5)]\n                   + [round(float(Yc[i, j]), 1) for j in range(5)]\n                   + ["" if not np.isfinite(CZ[i]).any() else round(float(np.nansum(P[i] * CZ[i])), 4),\n                      round(float(Y[i]), 1), round(float(UNC[i]), 2),\n                      status[i], fallback[i]])\n\njson.dump(dict(crop_posterior_source=CROPSRC, village_id=vill,\n               target="kg/ha per farm, to 13 Oct 2025, NOT final harvest",\n               operational_interpretation="accumulated yield formation to 13 Oct - MODEL ASSUMPTION",\n               anchors=ANCHOR, anchor_products=ANCHOR_PRODUCT, anchor_alternatives=ANCHOR_ALT,\n               stage_factors={c: STAGE[c] for c in CROPS},\n               stage_factor_status="MODEL ASSUMPTIONS - literature supports ORDERING ONLY",\n               health_double_count_audit=AUD, architectures=BASE,\n               stage_sensitivity=SENS, robustness=ROB, product_convention_sensitivity=PROD,\n               crop_distributions=DIST,\n               final=dict(median=round(float(np.nanmedian(Y)), 1), mean=round(float(np.nanmean(Y)), 1),\n                          p05=round(float(np.nanpercentile(Y, 5)), 1),\n                          p95=round(float(np.nanpercentile(Y, 95)), 1)),\n               fallbacks={p: int((fallback == p).sum()) for p in sorted(set(fallback))}),\n          open(R + "_diag/yield_v1.json", "w"), indent=1)\nprint("\\nwrote farm_yield_v1.csv and _diag/yield_v1.json")\nprint("fallbacks: %s" % {p: int((fallback == p).sum()) for p in sorted(set(fallback))})\n'
Path(SRC/"build_yield_v1.py").write_text(_SRC, encoding="utf-8")
print("wrote src/build_yield_v1.py  ({} lines)".format(_SRC.count(chr(10))))

In [ ]:
# S15 -- authoritative yield chain stage.
_SRC = '"""\nYIELD v2 - controlled revision of v1. v1 is NOT overwritten.\n\nTwo v1 issues drive this build:\n  ISSUE 1  Capella primacy was ASSERTED, not quantified. v2 decomposes the variance of\n           log(yield) exactly into an anchor-stage term and a condition term, and reports\n           whatever that decomposition actually says.\n  ISSUE 2  the Jun19->Aug14 contrast correlates -0.30 with HEALTH. Geometric cleanliness does\n           not make it a growth proxy. v2 audits its sign per crop / size / space and only\n           retains it in production if a defensible interpretation survives.\n\nHEALTH is used as a RELATIVE CONDITION RANKING within posterior-weighted crop peers.\nIt is never read as health/100 = yield fraction.\n"""\nimport csv, json, os, sys, warnings\nimport numpy as np\n\nsys.stdout.reconfigure(encoding="utf-8", errors="replace")\nwarnings.filterwarnings("ignore")\nnp.random.seed(0)\nR = "round2/"\nCROPS = ["Rice", "Cotton", "Maize", "Bajra", "Groundnut"]\nDATES = ["Jun06", "Jun19", "Aug14", "Oct13"]\nCROPSRC = os.environ.get("CROPSRC", "farm_crop_probabilities_v4.csv")\nANCHOR = {"Rice": 2068.0, "Cotton": 846.0, "Maize": 1770.0, "Bajra": 1815.0, "Groundnut": 2579.0}\nSTAGE = {"Bajra": (0.90, 0.98, 1.00), "Groundnut": (0.82, 0.95, 1.00), "Maize": (0.78, 0.92, 1.00),\n         "Rice": (0.58, 0.75, 0.88), "Cotton": (0.20, 0.35, 0.50)}\n\n\ndef fl(v):\n    try:\n        return float(v)\n    except (TypeError, ValueError):\n        return float("nan")\n\n\nfeat = {int(r["farm_id"]): r for r in csv.DictReader(open(R + "farm_features_structure_v1.csv", encoding="utf-8"))}\ncrop = {int(r["farm_id"]): r for r in csv.DictReader(open(R + CROPSRC, encoding="utf-8"))}\nheal = {int(r["farm_id"]): r for r in csv.DictReader(open(R + "farm_health_v2.csv", encoding="utf-8"))}\nevd = {int(r["farm_id"]): r for r in csv.DictReader(open(R + "farm_evidence_status_v1.csv", encoding="utf-8"))}\ncen = {int(r["farm_id"]): (fl(r["x"]), fl(r["y"])) for r in csv.DictReader(open(R + "_diag/farm_centroids.csv"))}\nfids = sorted(feat); NF = len(fids)\nP = np.column_stack([[fl(crop[f]["P_" + c]) for f in fids] for c in CROPS])\nP /= np.clip(P.sum(1, keepdims=True), 1e-12, None)\ncmap = np.array([crop[f]["crop_type"] for f in fids])\ncent_ = np.array([fl(crop[f]["crop_entropy"]) for f in fids])\nhealth = np.array([fl(heal[f]["health_index"]) for f in fids])\nhunc = np.array([fl(heal[f]["health_uncertainty"]) for f in fids])\nstatus = np.array([evd[f]["evidence_status"] for f in fids])\npen = np.array([fl(evd[f]["uncertainty_penalty"]) for f in fids])\nclean = np.array([fl(evd[f]["clean_contrast_db"]) for f in fids])\nclean_sd = np.array([fl(evd[f]["clean_contrast_sd"]) for f in fids])\narea = np.array([fl(feat[f]["area_ha"]) for f in fids])\nurange = np.array([fl(feat[f]["u_range_norm_Jun19"]) for f in fids])\nXX = np.array([cen.get(f, (np.nan, np.nan))[0] for f in fids])\nYY = np.array([cen.get(f, (np.nan, np.nan))[1] for f in fids])\nvill = int(evd[fids[0]]["village_id"])\nZ = np.column_stack([[fl(feat[f]["sigma0_db_trim10_%s" % d]) for f in fids] for d in DATES])\nZ[:, 1] = np.array([fl(feat[f]["sigma0_Jun19_clean"]) for f in fids])\nprint("[data] %d farms | crop posterior: %s" % (NF, CROPSRC))\n\n\ndef sp(a, b, w=None):\n    m = np.isfinite(a) & np.isfinite(b)\n    if m.sum() < 25:\n        return float("nan")\n    return float(np.corrcoef(np.argsort(np.argsort(a[m])), np.argsort(np.argsort(b[m])))[0, 1])\n\n\ndef wcorr(a, b, w):\n    m = np.isfinite(a) & np.isfinite(b) & np.isfinite(w) & (w > 0)\n    if m.sum() < 25 or w[m].sum() < 10:\n        return float("nan")\n    aw, bw, ww = a[m], b[m], w[m]\n    ma, mb = np.average(aw, weights=ww), np.average(bw, weights=ww)\n    ca, cb = aw - ma, bw - mb\n    return float((ww * ca * cb).sum() / np.sqrt((ww * ca ** 2).sum() * (ww * cb ** 2).sum() + 1e-12))\n\n\n# ============================================================ 3. JUN19->AUG14 AUDIT\nprint("\\n" + "=" * 76)\nprint("STEP 3  JUN19->AUG14 AUDIT - is the negative HEALTH sign crop-dependent?")\nprint("=" * 76)\nAUD = {"per_crop": {}, "per_area_quartile": {}, "per_spatial_block": {}}\nprint("  %-12s %10s %8s" % ("crop", "wcorr(J19A14,HEALTH)", "mass"))\nfor j, c in enumerate(CROPS):\n    r = wcorr(clean, health, P[:, j])\n    AUD["per_crop"][c] = round(r, 4) if np.isfinite(r) else None\n    print("  %-12s %10.4f %8.1f" % (c, r, P[:, j].sum()))\naq = np.digitize(area, np.nanpercentile(area, [25, 50, 75]))\nfor q in range(4):\n    m = aq == q\n    r = sp(clean[m], health[m])\n    AUD["per_area_quartile"]["Q%d" % (q + 1)] = round(r, 4) if np.isfinite(r) else None\nprint("  by area quartile: %s" % AUD["per_area_quartile"])\neast = XX > np.nanmedian(XX)\nfor nm, m in [("east", east), ("west", ~east)]:\n    AUD["per_spatial_block"][nm] = round(sp(clean[m], health[m]), 4)\nprint("  by spatial block: %s" % AUD["per_spatial_block"])\nAUD["corr_with_urange"] = round(sp(clean, urange), 4)\nAUD["corr_with_area"] = round(sp(clean, area), 4)\nAUD["corr_with_jun19_level"] = round(sp(clean, Z[:, 1]), 4)\nAUD["corr_with_aug14_level"] = round(sp(clean, Z[:, 2]), 4)\nprint("  corr with range coord %.3f | area %.3f | Jun19 level %.3f | Aug14 level %.3f"\n      % (AUD["corr_with_urange"], AUD["corr_with_area"],\n         AUD["corr_with_jun19_level"], AUD["corr_with_aug14_level"]))\nvals = [v for v in AUD["per_crop"].values() if v is not None]\nuniform_neg = all(v < 0 for v in vals) if vals else False\nAUD["uniformly_negative_across_crops"] = bool(uniform_neg)\nAUD["verdict"] = ("UNIFORMLY NEGATIVE across all crops -> not a crop-specific phenological "\n                  "effect, so it cannot be defended as a generic growth proxy. REMOVED from "\n                  "production, retained as a diagnostic column."\n                  if uniform_neg else\n                  "SIGN VARIES BY CROP -> may carry phenological meaning; retained but "\n                  "downweighted and only where the sign is defensible.")\nprint("\\n  VERDICT: %s" % AUD["verdict"])\nJ19_ROLE = "REMOVED" if uniform_neg else "DOWNWEIGHTED"\n\n# ============================================================ 1. HEALTH condition\ndef wmed(v, w):\n    m = np.isfinite(v) & (w > 0)\n    if m.sum() < 10:\n        return np.nan\n    o = np.argsort(v[m]); vv, ww = v[m][o], w[m][o]\n    return float(vv[np.searchsorted(np.cumsum(ww) / ww.sum(), 0.5)])\n\n\ndef peer_z(v, Pc):\n    out = np.full((NF, len(CROPS)), np.nan)\n    for j in range(len(CROPS)):\n        w = Pc[:, j]\n        med = wmed(v, w)\n        if not np.isfinite(med):\n            continue\n        mad = wmed(np.abs(v - med), w)\n        s = 1.4826 * mad if (np.isfinite(mad) and mad > 1e-6) else np.nanstd(v)\n        out[:, j] = (v - med) / max(s, 1e-6)\n    return np.clip(np.nan_to_num(out), -4, 4)\n\n\nMAPS = {   # all bounded, centred on 1.0 at the peer median. SENSITIVITY MODELS, not truths.\n "weak":       dict(dn=0.045, up=0.020, lo=0.90, hi=1.05),\n "moderate":   dict(dn=0.075, up=0.030, lo=0.82, hi=1.08),\n "asymmetric": dict(dn=0.110, up=0.025, lo=0.78, hi=1.06),\n}\n\n\ndef cond_map(z, p):\n    return np.clip(1.0 + p["dn"] * np.minimum(z, 0) + p["up"] * np.maximum(z, 0), p["lo"], p["hi"])\n\n\nHZ = peer_z(health, P)\n\n# ============================================================ 2. residual Capella\nresid = {}\nmh = np.isfinite(health)\nfor nm, v in [("clean_J19_A14", clean), ("late_Aug14_Oct13", Z[:, 3] - Z[:, 2]),\n              ("establishment_J06_J19", Z[:, 1] - Z[:, 0]),\n              ("amplitude", np.nanmax(Z, 1) - np.nanmin(Z, 1))]:\n    m = mh & np.isfinite(v)\n    A_ = np.column_stack([np.ones(m.sum()), health[m]])\n    b_, *_ = np.linalg.lstsq(A_, v[m], rcond=None)\n    r = np.full(NF, np.nan); r[m] = v[m] - A_ @ b_\n    resid[nm] = dict(corr_health=round(sp(v, health), 4),\n                     partial_after_health=round(sp(r, v), 4),\n                     corr_area=round(sp(v, area), 4), corr_range=round(sp(v, urange), 4),\n                     missing=int((~np.isfinite(v)).sum()))\nprint("\\n" + "=" * 76)\nprint("STEP 2  INDEPENDENT CAPELLA RESIDUALS (after HEALTH)")\nprint("=" * 76)\nprint("  %-24s %8s %8s %8s %8s" % ("feature", "r_health", "r_area", "r_range", "missing"))\nfor k, v in resid.items():\n    print("  %-24s %8.4f %8.4f %8.4f %8d"\n          % (k, v["corr_health"], v["corr_area"], v["corr_range"], v["missing"]))\n\n# ============================================================ 6. architectures\ndef build(arch, stage_key=1, mp="moderate", Pc=None):\n    Pc = P if Pc is None else Pc\n    st = {c: STAGE[c][stage_key] for c in CROPS}\n    hz = peer_z(health, Pc)\n    cz = peer_z(np.nan_to_num(clean, nan=np.nanmedian(clean)), Pc)\n    p = MAPS[mp]\n    Yc = np.zeros((NF, 5)); CF = np.zeros((NF, 5))\n    for j, c in enumerate(CROPS):\n        if arch == "Y0":\n            f = np.ones(NF)\n        elif arch in ("Y1", "Y4"):\n            f = cond_map(hz[:, j], p)\n        elif arch == "Y2":\n            f = cond_map(cz[:, j], p)\n        else:                                   # Y3 health + residual capella\n            f = cond_map(0.75 * hz[:, j] + 0.25 * cz[:, j], p)\n        CF[:, j] = f\n        Yc[:, j] = ANCHOR[c] * st[c] * f\n    return Yc, CF, st\n\n\ndef summ(Yc, Pc=None):\n    Pc = P if Pc is None else Pc\n    return (Pc * Yc).sum(1)\n\n\nprint("\\n" + "=" * 76)\nprint("STEP 6/7  ARCHITECTURES + EXACT VARIANCE DECOMPOSITION")\nprint("=" * 76)\nY0 = summ(build("Y0")[0])\nARCH = {}\nprint("  %-4s %9s %8s %9s %9s | %s" % ("", "median", "CV", "rho(Y0)", "rho(area)", "var share: anchor+stage / condition / cov"))\nfor a in ["Y0", "Y1", "Y2", "Y3", "Y4"]:\n    Yc, CF, st = build(a)\n    y = summ(Yc)\n    # EXACT decomposition: log y = log(anchor-stage mix) + log(effective condition)\n    AS = Y0.copy()\n    ce = y / np.clip(AS, 1e-9, None)\n    la, lc = np.log(np.clip(AS, 1e-9, None)), np.log(np.clip(ce, 1e-9, None))\n    va, vc = np.var(la), np.var(lc)\n    cv_ = 2 * np.cov(la, lc)[0, 1]\n    tot = va + vc + cv_\n    ARCH[a] = dict(median=round(float(np.nanmedian(y)), 1), cv=round(float(np.nanstd(y) / np.nanmean(y)), 4),\n                   rho_vs_Y0=round(sp(y, Y0), 4), rho_area=round(sp(y, area), 4),\n                   rho_x=round(sp(y, XX), 4), rho_health=round(sp(y, health), 4),\n                   var_share_anchor_stage=round(float(va / tot), 4) if tot > 1e-12 else 1.0,\n                   var_share_condition=round(float(vc / tot), 4) if tot > 1e-12 else 0.0,\n                   var_share_cov=round(float(cv_ / tot), 4) if tot > 1e-12 else 0.0)\n    print("  %-4s %9.1f %8.4f %9.4f %9.4f | %.3f / %.3f / %+.3f"\n          % (a, ARCH[a]["median"], ARCH[a]["cv"], ARCH[a]["rho_vs_Y0"], ARCH[a]["rho_area"],\n             ARCH[a]["var_share_anchor_stage"], ARCH[a]["var_share_condition"],\n             ARCH[a]["var_share_cov"]))\n\nSEL = "Y4" if J19_ROLE == "REMOVED" else "Y3"\nprint("\\n  SELECTED: %s  (Jun19->Aug14 role: %s)" % (SEL, J19_ROLE))\n\n# condition-map sensitivity on the selected architecture\nprint("\\n  condition-map sensitivity (%s):" % SEL)\nMAPSENS = {}\nfor mp in MAPS:\n    y = summ(build(SEL, mp=mp)[0])\n    ce = y / np.clip(Y0, 1e-9, None)\n    MAPSENS[mp] = dict(bounds=[MAPS[mp]["lo"], MAPS[mp]["hi"]],\n                       within_crop_spread_pct=round(float(100 * (np.nanpercentile(ce, 95) - np.nanpercentile(ce, 5))), 2),\n                       rho_vs_Y0=round(sp(y, Y0), 4), median=round(float(np.nanmedian(y)), 1))\n    print("    %-11s bounds %s  p05-p95 condition spread %5.2f%%  rho(Y0) %.4f"\n          % (mp, MAPSENS[mp]["bounds"], MAPSENS[mp]["within_crop_spread_pct"], MAPSENS[mp]["rho_vs_Y0"]))\n\n# ============================================================ 5. stage intervals\nYc_c, CF, ST_C = build(SEL, stage_key=1)\nY = summ(Yc_c)\nY_lo = summ(build(SEL, stage_key=0)[0])\nY_hi = summ(build(SEL, stage_key=2)[0])\nprint("\\n" + "=" * 76)\nprint("STEP 5  STAGE INTERVAL PROPAGATION (village mean kg/ha)")\nprint("=" * 76)\nprint("  low %.1f   central %.1f   high %.1f   (band %+.1f%% / %+.1f%%)"\n      % (np.nanmean(Y_lo), np.nanmean(Y), np.nanmean(Y_hi),\n         100 * (np.nanmean(Y_lo) / np.nanmean(Y) - 1), 100 * (np.nanmean(Y_hi) / np.nanmean(Y) - 1)))\n\n# ============================================================ 8. nuisance stress\nprint("\\n" + "=" * 76)\nprint("STEP 8  WITHIN-CROP RANK STRESS (posterior-weighted)")\nprint("=" * 76)\nSTRESS = {}\nfor j, c in enumerate(CROPS):\n    w = P[:, j]\n    if w.sum() < 20:\n        continue\n    STRESS[c] = dict(health=round(wcorr(Y, health, w), 3), area=round(wcorr(Y, area, w), 3),\n                     x=round(wcorr(Y, XX, w), 3), y=round(wcorr(Y, YY, w), 3),\n                     entropy=round(wcorr(Y, cent_, w), 3))\n    print("  %-10s health %+.3f  area %+.3f  x %+.3f  y %+.3f  crop_entropy %+.3f"\n          % (c, STRESS[c]["health"], STRESS[c]["area"], STRESS[c]["x"], STRESS[c]["y"],\n             STRESS[c]["entropy"]))\n\n# ============================================================ uncertainty + output\nU = np.column_stack([cent_, np.clip(hunc / 100, 0, 1), pen,\n                     np.where(np.isfinite(health), 0.0, 0.6)])\nUNC = np.nanmean(U, 1)\nUNC = 100 * (UNC - np.nanmin(UNC)) / (np.nanmax(UNC) - np.nanmin(UNC) + 1e-9)\nfallback = np.where(status == "full_capella", "full",\n           np.where(status == "invalid_jun19", "health_condition_only",\n           np.where(status == "partial_capella", "reduced_dates", "anchor_plus_prior_only")))\nfor i in range(NF):\n    if not np.isfinite(Y[i]):\n        Y[i] = float((P[i] * np.array([ANCHOR[c] * ST_C[c] for c in CROPS])).sum())\n\nwith open(R + "farm_yield_v2.csv", "w", newline="", encoding="utf-8") as fh:\n    w = csv.writer(fh)\n    w.writerow(["village_id", "farm_id", "crop_MAP", "crop_entropy", "health_index"]\n               + ["P_" + c for c in CROPS] + ["anchor_" + c for c in CROPS]\n               + ["stage_" + c for c in CROPS] + ["condition_" + c for c in CROPS]\n               + ["yield_" + c for c in CROPS]\n               + ["health_condition_z", "jun19_aug14_diagnostic",\n                  "yield_estimate_to_date", "yield_low_stage", "yield_high_stage",\n                  "yield_uncertainty", "evidence_status", "fallback_path"])\n    for i, f in enumerate(fids):\n        w.writerow([vill, f, cmap[i], round(float(cent_[i]), 4),\n                    "" if not np.isfinite(health[i]) else round(float(health[i]), 2)]\n                   + [round(float(P[i, j]), 5) for j in range(5)]\n                   + [ANCHOR[c] for c in CROPS] + [ST_C[c] for c in CROPS]\n                   + [round(float(CF[i, j]), 4) for j in range(5)]\n                   + [round(float(Yc_c[i, j]), 1) for j in range(5)]\n                   + [round(float((P[i] * HZ[i]).sum()), 4),\n                      "" if not np.isfinite(clean[i]) else round(float(clean[i]), 4),\n                      round(float(Y[i]), 1), round(float(Y_lo[i]), 1), round(float(Y_hi[i]), 1),\n                      round(float(UNC[i]), 2), status[i], fallback[i]])\n\njson.dump(dict(crop_posterior_source=CROPSRC, selected_architecture=SEL,\n               jun19_aug14_role=J19_ROLE, jun19_aug14_audit=AUD,\n               capella_residual_audit=resid, architectures=ARCH,\n               condition_map_sensitivity=MAPSENS, condition_maps=MAPS,\n               stage_factors={c: dict(low=STAGE[c][0], central=STAGE[c][1], high=STAGE[c][2])\n                              for c in CROPS},\n               stage_interval_village_mean=dict(low=round(float(np.nanmean(Y_lo)), 1),\n                                                central=round(float(np.nanmean(Y)), 1),\n                                                high=round(float(np.nanmean(Y_hi)), 1)),\n               within_crop_stress=STRESS,\n               final=dict(median=round(float(np.nanmedian(Y)), 1), mean=round(float(np.nanmean(Y)), 1),\n                          p05=round(float(np.nanpercentile(Y, 5)), 1),\n                          p95=round(float(np.nanpercentile(Y, 95)), 1))),\n          open(R + "_diag/yield_v2.json", "w"), indent=1)\nprint("\\n  FINAL v2: median %.1f mean %.1f p05 %.1f p95 %.1f"\n      % (np.nanmedian(Y), np.nanmean(Y), np.nanpercentile(Y, 5), np.nanpercentile(Y, 95)))\nprint("wrote farm_yield_v2.csv and _diag/yield_v2.json")\n'
Path(SRC/"build_yield_v2.py").write_text(_SRC, encoding="utf-8")
print("wrote src/build_yield_v2.py  ({} lines)".format(_SRC.count(chr(10))))

In [ ]:
# S15 -- authoritative yield chain stage.
_SRC = '"""\nYIELD v3.1 - CONSTANTS-ONLY correction of YIELD v3.\n\nYIELD v3.1 is a constants-only correction of YIELD v3. No model architecture, crop identity,\nHEALTH model, or condition mapping was changed.\n\nThe ONLY change vs v3 is the stage-factor vector, replaced with the B2-approved values:\n    Rice .75->.975   Cotton .35->.478   Maize .92->.960   Bajra .98->1.000   Groundnut .95->1.000\nThe old Rice 0.75 is RETIRED: it was a crop-duration fraction, not a product-formation fraction.\nPrimary APY anchors are UNCHANGED. B3/B4 alternatives are carried as PRODUCT-CONVENTION\nSENSITIVITY diagnostics only and never enter the submitted value.\n\nInherited unchanged from v3:\n\nReads the FROZEN crop-conditioned yields from farm_yield_v2.csv and changes only:\n  (a) the central submitted value  -> MAP-conditional Y_c for the submitted crop_type\n  (b) crop uncertainty             -> posterior dispersion of {Y_c}, carried as uncertainty\n\nHEALTH, CROP_ID, stage factors, condition mapping all unchanged. Jun19->Aug14 stays out of\nproduction. v1 and v2 are not overwritten.\n"""\nimport csv, json, sys, warnings\nimport numpy as np\n\nsys.stdout.reconfigure(encoding="utf-8", errors="replace")\nwarnings.filterwarnings("ignore")\nR = "round2/"\nCROPS = ["Rice", "Cotton", "Maize", "Bajra", "Groundnut"]\nDATES = ["Jun06", "Jun19", "Aug14", "Oct13"]\nPRODUCT = {"Rice": "grain (cleaned-rice convention)", "Cotton": "LINT",\n           "Maize": "grain", "Bajra": "grain", "Groundnut": "pods in-shell"}\n# B2-approved. LOW/HIGH are AGRONOMIC SCENARIO SENSITIVITY BOUNDS - not confidence\n# intervals, not standard deviations, not calibrated uncertainty.\nSTAGE_LCH = {"Rice": (0.053, 0.975, 1.000), "Cotton": (0.078, 0.478, 0.971),\n             "Maize": (0.192, 0.960, 1.000), "Bajra": (0.980, 1.000, 1.000),\n             "Groundnut": (0.925, 1.000, 1.000)}\nSTAGE_OLD_V3 = {"Rice": 0.75, "Cotton": 0.35, "Maize": 0.92, "Bajra": 0.98, "Groundnut": 0.95}\n# B3/B4 PRODUCT-CONVENTION SENSITIVITY anchors (diagnostic only, never submitted)\nANCHOR_ALT = {"Rice": 3133.0, "Groundnut": 1833.0}\nALT_NOTE = {"Rice": "paddy reading, x1.515 (B3: official convention UNSPECIFIED)",\n            "Groundnut": "kernel reading, x0.711 sourced (B4: official convention UNSPECIFIED)"}\n\n\ndef fl(v):\n    try:\n        return float(v)\n    except (TypeError, ValueError):\n        return float("nan")\n\n\nyv = {int(r["farm_id"]): r for r in csv.DictReader(open(R + "farm_yield_v2.csv", encoding="utf-8"))}\nfeat = {int(r["farm_id"]): r for r in csv.DictReader(open(R + "farm_features_structure_v1.csv", encoding="utf-8"))}\ncen = {int(r["farm_id"]): (fl(r["x"]), fl(r["y"])) for r in csv.DictReader(open(R + "_diag/farm_centroids.csv"))}\nfids = sorted(yv); NF = len(fids)\nvill = int(yv[fids[0]]["village_id"])\nP = np.column_stack([[fl(yv[f]["P_" + c]) for f in fids] for c in CROPS])\n# v2\'s yield_<crop> columns have the OLD stage baked in. Recompute from the FROZEN\n# anchor x condition (both unchanged) with the new stage constants. Architecture identical.\n_ANC = np.column_stack([[fl(yv[f]["anchor_" + c]) for f in fids] for c in CROPS])\n_CND = np.column_stack([[fl(yv[f]["condition_" + c]) for f in fids] for c in CROPS])\nYc = _ANC * np.array([STAGE_LCH[c][1] for c in CROPS])[None, :] * _CND\nYc_v3 = _ANC * np.array([STAGE_OLD_V3[c] for c in CROPS])[None, :] * _CND\nCOND = np.column_stack([[fl(yv[f]["condition_" + c]) for f in fids] for c in CROPS])\nANC = np.column_stack([[fl(yv[f]["anchor_" + c]) for f in fids] for c in CROPS])\nmapc = np.array([yv[f]["crop_MAP"] for f in fids])\nmi = np.array([CROPS.index(c) for c in mapc])\nent = np.array([fl(yv[f]["crop_entropy"]) for f in fids])\nhealth = np.array([fl(yv[f]["health_index"]) for f in fids])\nstatus = np.array([yv[f]["evidence_status"] for f in fids])\nfallb = np.array([yv[f]["fallback_path"] for f in fids])\nhz = np.array([fl(yv[f]["health_condition_z"]) for f in fids])\nj19d = np.array([fl(yv[f]["jun19_aug14_diagnostic"]) for f in fids])\narea = np.array([fl(feat[f]["area_ha"]) for f in fids])\nXX = np.array([cen.get(f, (np.nan, np.nan))[0] for f in fids])\nYY = np.array([cen.get(f, (np.nan, np.nan))[1] for f in fids])\nZc = np.column_stack([[fl(feat[f]["sigma0_db_trim10_%s" % d]) for f in fids] for d in DATES])\nZc[:, 1] = np.array([fl(feat[f]["sigma0_Jun19_clean"]) for f in fids])\nprint("[v3.1] %d farms | CONSTANTS-ONLY stage update; architecture identical" % NF)\n\n# ============================================================ 1. MAP-conditional central\nidx = np.arange(NF)\nY = Yc[idx, mi]\nanc_map = ANC[idx, mi]\ncond_map = COND[idx, mi]\nst_c = np.array([STAGE_LCH[c][1] for c in mapc])\nst_l = np.array([STAGE_LCH[c][0] for c in mapc])\nst_h = np.array([STAGE_LCH[c][2] for c in mapc])\nY_lo = anc_map * st_l * cond_map\nY_hi = anc_map * st_h * cond_map\nprod = np.array([PRODUCT[c] for c in mapc])\n\n# ============================================================ 2. posterior dispersion\nmu_diag = (P * Yc).sum(1)\ncrop_identity_dispersion_heuristic = np.sqrt(\n    np.clip((P * (Yc - mu_diag[:, None]) ** 2).sum(1), 0, None))\n\n# ============================================================ 3. uncertainty components\nu_crop = crop_identity_dispersion_heuristic\nu_stage = (Y_hi - Y_lo) / 2.0\n# condition uncertainty: how much the bounded condition map could move this farm\nu_cond = np.abs(Y - anc_map * st_c * 1.0)\nev_pen = {"full_capella": 0.0, "invalid_jun19": 0.25, "partial_capella": 0.40,\n          "no_capella_coverage": 1.0, "degenerate_geometry": 1.0}\nu_evid = np.array([ev_pen.get(s, 0.5) for s in status]) * anc_map * st_c * 0.15\n# HEURISTIC summary only. The four components above are the primary uncertainty outputs.\n# This combination is NOT a standard deviation, NOT a confidence interval and NOT a\n# statistically derived total: the components are not demonstrated independent. It exists\n# solely to rank farms by overall evidence risk.\nu_summary = np.sqrt(u_stage ** 2 + u_cond ** 2 + u_evid ** 2) + u_crop\nprint("[unc] median kg/ha  crop %.1f  stage %.1f  condition %.1f  evidence %.1f  summary_heuristic %.1f"\n      % (np.median(u_crop), np.median(u_stage), np.median(u_cond), np.median(u_evid),\n         np.median(u_summary)))\n\n# ============================================================ 6. validation\ndef sp(a, b):\n    m = np.isfinite(a) & np.isfinite(b)\n    if m.sum() < 25:\n        return float("nan")\n    return float(np.corrcoef(np.argsort(np.argsort(a[m])), np.argsort(np.argsort(b[m])))[0, 1])\n\n\nprint("\\n" + "=" * 76); print("6. VALIDATION"); print("=" * 76)\nV = {}\nV["n_farms"] = NF\nV["all_finite"] = bool(np.isfinite(Y).all())\nV["all_non_negative"] = bool((Y >= 0).all())\n# the hard consistency check\n# Compare the submitted value against the v3.1 crop-conditioned column for the SAME crop.\n# (The v3 script compared against v2\'s yield_<crop> columns; those carry the OLD stage\n#  constants, so after a constants-only update that comparison is stale, not a failure.)\nok = 0\nfor i in range(NF):\n    if abs(Y[i] - Yc[i, mi[i]]) < 1e-6:\n        ok += 1\nV["map_yield_consistency"] = "%d / %d" % (ok, NF)\nV["map_yield_consistency_pass"] = bool(ok == NF)\nprint("  all 966 present: %s | finite: %s | non-negative: %s" % (NF == 966, V["all_finite"], V["all_non_negative"]))\nprint("  CONSISTENCY yield_estimate_to_date == yield_<crop_type>: %s  -> %s"\n      % (V["map_yield_consistency"], "PASS" if V["map_yield_consistency_pass"] else "FAIL"))\nV["rho_health"] = round(sp(Y, health), 4)\nV["rho_area"] = round(sp(Y, area), 4)\nV["rho_x"] = round(sp(Y, XX), 4)\nV["rho_y"] = round(sp(Y, YY), 4)\nV["rho_entropy"] = round(sp(Y, ent), 4)\nprint("  rho: health %+.3f  area %+.3f  x %+.3f  y %+.3f  entropy %+.3f"\n      % (V["rho_health"], V["rho_area"], V["rho_x"], V["rho_y"], V["rho_entropy"]))\n\n# leave-one-date-out (through the condition proxy, as in v2 gapfill)\ndef peer_z(v):\n    m = np.isfinite(v); o = np.full(NF, np.nan)\n    if m.sum() < 30:\n        return o\n    med = np.median(v[m]); mad = np.median(np.abs(v[m] - med))\n    o[m] = (v[m] - med) / max(1.4826 * mad, 1e-6)\n    return np.clip(o, -4, 4)\n\n\nLODO = {}\nfor di, d in enumerate(DATES):\n    pz = peer_z(np.nanmean(np.delete(Zc, di, axis=1), 1))\n    c2 = np.clip(1 + 0.075 * np.minimum(pz, 0) + 0.030 * np.maximum(pz, 0), 0.82, 1.08)\n    LODO[d] = round(sp(anc_map * st_c * np.nan_to_num(c2, nan=1.0), Y), 4)\nV["leave_one_date_out"] = LODO\nprint("  leave-one-date-out rho: %s" % LODO)\nMISS = {}\nrs = np.random.RandomState(11)\nfor fr in [0.05, 0.10, 0.20]:\n    hl = health.copy(); hl[rs.choice(NF, int(fr * NF), replace=False)] = np.nan\n    pz = peer_z(hl)\n    c2 = np.clip(1 + 0.075 * np.minimum(pz, 0) + 0.030 * np.maximum(pz, 0), 0.82, 1.08)\n    MISS["drop_%d_pct" % int(fr * 100)] = round(sp(anc_map * st_c * np.nan_to_num(c2, nan=1.0), Y), 4)\nV["missingness_simulation"] = MISS\nprint("  health missingness rho: %s" % MISS)\nV["stage_scenarios"] = dict(low_mean=round(float(np.mean(Y_lo)), 1),\n                            central_mean=round(float(np.mean(Y)), 1),\n                            high_mean=round(float(np.mean(Y_hi)), 1))\nprint("  stage scenarios (village mean): low %.1f  central %.1f  high %.1f"\n      % (V["stage_scenarios"]["low_mean"], V["stage_scenarios"]["central_mean"],\n         V["stage_scenarios"]["high_mean"]))\n\n# ============================================================ 7. posterior perturbation split\nrs = np.random.RandomState(7)\nPp = P + rs.dirichlet(np.ones(5), NF) * 0.30\nPp /= Pp.sum(1, keepdims=True)\nmi2 = Pp.argmax(1)\nflip = mi2 != mi\nY2 = Yc[idx, mi2]\nV["perturbation"] = dict(\n    n_map_unchanged=int((~flip).sum()), n_map_flipped=int(flip.sum()),\n    unchanged_rho=round(sp(Y2[~flip], Y[~flip]), 4),\n    unchanged_median_abs_pct=round(float(np.median(np.abs(Y2[~flip] - Y[~flip])\n                                                   / np.clip(Y[~flip], 1e-6, None)) * 100), 3),\n    flipped_median_abs_kg=round(float(np.median(np.abs(Y2[flip] - Y[flip]))), 1) if flip.sum() else None,\n    flip_pairs={})\nfor i in np.where(flip)[0]:\n    k = "%s->%s" % (mapc[i], CROPS[mi2[i]])\n    V["perturbation"]["flip_pairs"][k] = V["perturbation"]["flip_pairs"].get(k, 0) + 1\nprint("\\n" + "=" * 76); print("7. POSTERIOR PERTURBATION - SPLIT BY MAP STABILITY"); print("=" * 76)\nprint("  A. MAP unchanged: %d farms, rho %.4f, median |change| %.3f%%"\n      % (V["perturbation"]["n_map_unchanged"], V["perturbation"]["unchanged_rho"],\n         V["perturbation"]["unchanged_median_abs_pct"]))\nprint("  B. MAP flipped  : %d farms, median |change| %s kg/ha  pairs %s"\n      % (V["perturbation"]["n_map_flipped"], V["perturbation"]["flipped_median_abs_kg"],\n         V["perturbation"]["flip_pairs"]))\n\n# ============================================================ 8. entropy audit v2 vs v3\nprint("\\n" + "=" * 76); print("8. ENTROPY AUDIT - v2 (posterior) vs v3 (MAP)"); print("=" * 76)\nYv2 = np.array([fl(yv[f]["yield_estimate_to_date"]) for f in fids])\nENT = {}\nprint("  %-10s %5s %14s %14s" % ("MAP crop", "n", "v2 corr(e,Y)", "v3 corr(e,Y)"))\nfor c in CROPS:\n    m = mapc == c\n    if m.sum() < 20:\n        continue\n    ENT[c] = dict(n=int(m.sum()), v2=round(sp(ent[m], Yv2[m]), 4), v3=round(sp(ent[m], Y[m]), 4))\n    print("  %-10s %5d %14.4f %14.4f" % (c, m.sum(), ENT[c]["v2"], ENT[c]["v3"]))\nV["entropy_audit"] = ENT\n\n# ============================================================ 9. crop-identity risk table\no = np.argsort(-P, axis=1)\np1, p2 = P[idx, o[:, 0]], P[idx, o[:, 1]]\nclose = (p1 - p2) < 0.15\nwith open(R + "_diag/yield_crop_identity_risk.csv", "w", newline="", encoding="utf-8") as fh:\n    w = csv.writer(fh)\n    w.writerow(["farm_id", "top1_crop", "top1_p", "top2_crop", "top2_p",\n                "Y_top1", "Y_top2", "yield_gap", "crop_identity_dispersion_heuristic"])\n    for i in np.where(close)[0]:\n        w.writerow([fids[i], CROPS[o[i, 0]], round(float(p1[i]), 4), CROPS[o[i, 1]],\n                    round(float(p2[i]), 4), round(float(Yc[i, o[i, 0]]), 1),\n                    round(float(Yc[i, o[i, 1]]), 1),\n                    round(float(abs(Yc[i, o[i, 0]] - Yc[i, o[i, 1]])), 1),\n                    round(float(crop_identity_dispersion_heuristic[i]), 1)])\nprint("\\n9. crop-identity risk table: %d farms with p1-p2 < 0.15 -> "\n      "_diag/yield_crop_identity_risk.csv" % close.sum())\n\n# ============================================================ distributions + extremes\nprint("\\n" + "=" * 76); print("CROP DISTRIBUTIONS (MAP-conditional)"); print("=" * 76)\nDIST = {}\nfor c in CROPS:\n    m = mapc == c\n    if m.sum() == 0:\n        DIST[c] = dict(n=0); continue\n    DIST[c] = dict(n=int(m.sum()), product=PRODUCT[c], anchor=float(ANC[m, CROPS.index(c)][0]),\n                   stage=float(STAGE_LCH[c][1]),\n                   median=round(float(np.median(Y[m])), 1), p05=round(float(np.percentile(Y[m], 5)), 1),\n                   p95=round(float(np.percentile(Y[m], 95)), 1),\n                   min=round(float(Y[m].min()), 1), max=round(float(Y[m].max()), 1))\n    print("  %-10s n=%3d  %-26s median %7.1f  [p05 %7.1f, p95 %7.1f]"\n          % (c, m.sum(), PRODUCT[c], DIST[c]["median"], DIST[c]["p05"], DIST[c]["p95"]))\nV["distributions"] = DIST\noo = np.argsort(Y)\nprint("\\n  lowest 5 : %s" % [(int(fids[i]), mapc[i], round(float(Y[i]), 1)) for i in oo[:5]])\nprint("  highest 5: %s" % [(int(fids[i]), mapc[i], round(float(Y[i]), 1)) for i in oo[-5:]])\n\n# ============================================================ v3 -> v3.1 COMPARISON\nprint(chr(10) + "=" * 76); print("7. BEFORE / AFTER  v3 -> v3.1"); print("=" * 76)\nY_v3 = Yc_v3[idx, mi]\ndef _stats(v):\n    return dict(mean=round(float(v.mean()),1), median=round(float(np.median(v)),1),\n                p05=round(float(np.percentile(v,5)),1), p95=round(float(np.percentile(v,95)),1),\n                min=round(float(v.min()),1), max=round(float(v.max()),1))\nCMP = dict(v3=_stats(Y_v3), v3_1=_stats(Y),\n           spearman=round(sp(Y, Y_v3), 4),\n           median_abs_pct_change=round(float(np.median(np.abs(Y-Y_v3)/np.clip(Y_v3,1e-6,None))*100),2),\n           per_crop_v3={c: round(float(np.median(Y_v3[mapc==c])),1) for c in sorted(set(mapc))},\n           per_crop_v3_1={c: round(float(np.median(Y[mapc==c])),1) for c in sorted(set(mapc))})\nprint("  %-8s %9s %9s %9s %9s %9s %9s" % ("", "mean","median","p05","p95","min","max"))\nfor k in ["v3","v3_1"]:\n    d=CMP[k]; print("  %-8s %9.1f %9.1f %9.1f %9.1f %9.1f %9.1f"\n                    %(k,d["mean"],d["median"],d["p05"],d["p95"],d["min"],d["max"]))\nprint("  Spearman v3 vs v3.1: %.4f | median |change| %.2f%%"\n      % (CMP["spearman"], CMP["median_abs_pct_change"]))\nprint("  per-crop median: v3 %s" % CMP["per_crop_v3"])\nprint("                 v3.1 %s" % CMP["per_crop_v3_1"])\nrk_old=np.argsort(np.argsort(Y_v3)); rk_new=np.argsort(np.argsort(Y))\nCMP["n_rank_moved_gt_50"]=int((np.abs(rk_old-rk_new)>50).sum())\nCMP["median_abs_rank_shift"]=int(np.median(np.abs(rk_old-rk_new)))\nprint("  farms whose rank moved >50 places: %d | median |rank shift| %d"\n      % (CMP["n_rank_moved_gt_50"], CMP["median_abs_rank_shift"]))\n# attribution: is the change fully explained by the stage constants?\nattrib={}\nfor c in sorted(set(mapc)):\n    m=mapc==c\n    exp=STAGE_LCH[c][1]/STAGE_OLD_V3[c]\n    obs=float(np.median(Y[m])/np.median(Y_v3[m]))\n    attrib[c]=dict(stage_old=STAGE_OLD_V3[c], stage_new=STAGE_LCH[c][1],\n                   expected_ratio=round(exp,4), observed_ratio=round(obs,4),\n                   matches=bool(abs(exp-obs)<1e-3))\nprint(chr(10) + "  attribution - observed per-crop ratio vs the stage-constant ratio:")\nfor c,a in attrib.items():\n    print("    %-10s %.2f -> %.3f  expected x%.4f  observed x%.4f  %s"\n          %(c,a["stage_old"],a["stage_new"],a["expected_ratio"],a["observed_ratio"],\n            "MATCH" if a["matches"] else "MISMATCH"))\nCMP["attribution"]=attrib\nCMP["fully_explained_by_stage_constants"]=bool(all(a["matches"] for a in attrib.values()))\nprint("  change fully explained by stage constants (no refit): %s"\n      % CMP["fully_explained_by_stage_constants"])\nord_ok = CMP["per_crop_v3_1"].get("Rice",0) > CMP["per_crop_v3_1"].get("Bajra",0)\nCMP["rice_exceeds_bajra"]=bool(ord_ok)\nprint("  Rice median now exceeds Bajra: %s  (consequence of the Rice stage correction)" % ord_ok)\n\n# ============================================================ PRODUCT-CONVENTION SENSITIVITY\nprint(chr(10) + "=" * 76); print("3. PRODUCT-CONVENTION SENSITIVITY (diagnostic only)"); print("=" * 76)\nPCS={}\nfor c,alt in ANCHOR_ALT.items():\n    m=mapc==c\n    if m.sum()==0:\n        PCS[c]=dict(n_MAP_farms=0, alt_anchor=alt, note=ALT_NOTE[c],\n                    submitted_values_changed=0,\n                    verdict="ZERO MAP farms -> this sensitivity changes NO submitted value")\n        print("  %-10s n_MAP=0  alt anchor %.0f  -> changes ZERO submitted values" % (c,alt))\n        continue\n    Yalt=Y.copy(); Yalt[m]=alt*STAGE_LCH[c][1]*cond_map[m]\n    PCS[c]=dict(n_MAP_farms=int(m.sum()), alt_anchor=alt, note=ALT_NOTE[c],\n                median_primary=round(float(np.median(Y[m])),1),\n                median_alternative=round(float(np.median(Yalt[m])),1),\n                village_mean_primary=round(float(Y.mean()),1),\n                village_mean_alternative=round(float(Yalt.mean()),1))\n    print("  %-10s n_MAP=%3d  primary median %7.1f  ->  alternative %7.1f  (%s)"\n          %(c,m.sum(),PCS[c]["median_primary"],PCS[c]["median_alternative"],ALT_NOTE[c][:38]))\n\n# ============================================================ output\nwith open(R + "farm_yield_v3_1.csv", "w", newline="", encoding="utf-8") as fh:\n    w = csv.writer(fh)\n    w.writerow(["village_id", "farm_id", "crop_type", "yield_estimate_to_date",\n                "yield_product_convention", "crop_entropy", "health_index"]\n               + ["P_" + c for c in CROPS] + ["yield_" + c for c in CROPS]\n               + ["anchor_MAP", "stage_MAP", "condition_MAP",\n                  "yield_low_stage", "yield_high_stage",\n                  "yield_uncertainty_crop", "yield_uncertainty_stage",\n                  "yield_uncertainty_condition", "yield_uncertainty_evidence",\n                  "yield_uncertainty_summary_heuristic",\n                  "posterior_expected_yield_DIAGNOSTIC", "health_condition_z",\n                  "jun19_aug14_DIAGNOSTIC", "evidence_status", "fallback_path"])\n    for i, f in enumerate(fids):\n        w.writerow([vill, f, mapc[i], round(float(Y[i]), 1), prod[i],\n                    round(float(ent[i]), 4),\n                    "" if not np.isfinite(health[i]) else round(float(health[i]), 2)]\n                   + [round(float(P[i, j]), 5) for j in range(5)]\n                   + [round(float(Yc[i, j]), 1) for j in range(5)]\n                   + [round(float(anc_map[i]), 1), round(float(st_c[i]), 3),\n                      round(float(cond_map[i]), 4),\n                      round(float(Y_lo[i]), 1), round(float(Y_hi[i]), 1),\n                      round(float(u_crop[i]), 1), round(float(u_stage[i]), 1),\n                      round(float(u_cond[i]), 1), round(float(u_evid[i]), 1),\n                      round(float(u_summary[i]), 1),\n                      round(float(mu_diag[i]), 1),\n                      "" if not np.isfinite(hz[i]) else round(float(hz[i]), 4),\n                      "" if not np.isfinite(j19d[i]) else round(float(j19d[i]), 4),\n                      status[i], fallb[i]])\n\njson.dump(dict(version="v3.1", change="CONSTANTS-ONLY stage-vector update of v3; architecture, "\n                                        "CROP_ID, HEALTH, anchors and condition mapping unchanged",\n               stage_vector=STAGE_LCH, stage_vector_old_v3=STAGE_OLD_V3,\n               retired="Rice 0.75 (crop-duration fraction, methodological error)",\n               product_conventions=PRODUCT, validation=V,\n               uncertainty_medians=dict(crop=round(float(np.median(u_crop)), 1),\n                                        stage=round(float(np.median(u_stage)), 1),\n                                        condition=round(float(np.median(u_cond)), 1),\n                                        evidence=round(float(np.median(u_evid)), 1),\n                                        summary_heuristic=round(float(np.median(u_summary)), 1)),\n               final=dict(median=round(float(np.median(Y)), 1), mean=round(float(np.mean(Y)), 1),\n                          p05=round(float(np.percentile(Y, 5)), 1),\n                          p95=round(float(np.percentile(Y, 95)), 1),\n                          min=round(float(Y.min()), 1), max=round(float(Y.max()), 1)),\n               n_crop_identity_risk=int(close.sum()),\n               comparison_v3_to_v3_1=CMP,\n               product_convention_sensitivity=PCS,\n               sensitivity_label="PRODUCT-CONVENTION SENSITIVITY / AGRONOMIC SCENARIO SENSITIVITY BOUNDS - not uncertainty intervals"),\n          open(R + "_diag/yield_v3_1.json", "w"), indent=1)\nprint("\\nFINAL v3: median %.1f  mean %.1f  p05 %.1f  p95 %.1f  min %.1f  max %.1f"\n      % (np.median(Y), np.mean(Y), np.percentile(Y, 5), np.percentile(Y, 95), Y.min(), Y.max()))\nprint("wrote farm_yield_v3_1.csv and _diag/yield_v3_1.json")\n'
Path(SRC/"build_yield_v3_1.py").write_text(_SRC, encoding="utf-8")
print("wrote src/build_yield_v3_1.py  ({} lines)".format(_SRC.count(chr(10))))

In [ ]:
T["S15a"], _ = run_stage("build_yield_v1.py", ["farm_yield_v1.csv", "_diag/yield_v1.json"], "S15a YIELD v1")
T["S15b"], _ = run_stage("build_yield_v2.py", ["farm_yield_v2.csv", "_diag/yield_v2.json"], "S15b YIELD v2")
T["S15c"], _ = run_stage("build_yield_v3_1.py", ["farm_yield_v3_1.csv", "_diag/yield_v3_1.json"], "S15c YIELD v3.1")
yld = pd.read_csv(ROOT/"farm_yield_v3_1.csv")
assert len(yld)==966 and np.isfinite(yld["yield_estimate_to_date"]).all()

---
## 11 · Integrated maps and submission  ·  `[P] S16`

In [ ]:
SUB_COLS=["village_id","farm_id","crop_type","health_index","yield_estimate_to_date"]
sub=yld[SUB_COLS].copy().sort_values("farm_id").reset_index(drop=True)
sub["village_id"]=sub["village_id"].astype(int); sub["farm_id"]=sub["farm_id"].astype(int)
cmap=crop.set_index("farm_id")["crop_type"]; hmap=health.set_index("farm_id")["health_index"]
assert sub.set_index("farm_id")["crop_type"].equals(cmap.reindex(sub["farm_id"]))
assert np.allclose(sub.set_index("farm_id")["health_index"],hmap.reindex(sub["farm_id"]),atol=1e-12)
allowed={"Rice","Cotton","Maize","Bajra","Groundnut"}
checks={
 "rows_966":len(sub)==966, "columns_exact":list(sub)==SUB_COLS,
 "farm_unique":sub.farm_id.is_unique, "farm_ids_1_966":sub.farm_id.tolist()==list(range(1,967)),
 "village_22":(sub.village_id==22).all(), "valid_crop_names":set(sub.crop_type)<=allowed,
 "no_nan":not sub.isna().any().any(),
 "finite":np.isfinite(sub[["health_index","yield_estimate_to_date"]].to_numpy()).all(),
 "health_0_100":sub.health_index.between(0,100).all(), "yield_nonnegative":(sub.yield_estimate_to_date>=0).all(),
 "no_duplicate_rows":not sub.duplicated().any(), "five_columns":sub.shape[1]==5,
}
assert all(checks.values()), checks
SUB_PATH=WORK/"submission.csv"; sub.to_csv(SUB_PATH,index=False,lineterminator="\n")
SUB_SHA=sha256(SUB_PATH)
print("submission assertions:",checks)
print("crop counts:",sub.crop_type.value_counts().to_dict())
print("submission SHA256:",SUB_SHA)

# Media Gallery-ready vector farm maps rendered at 300 dpi.
import matplotlib.pyplot as plt
from matplotlib.collections import PatchCollection
from matplotlib.patches import Polygon as MplPolygon
polys=[MplPolygon(np.asarray(s.points),closed=True) for s in shps]
def draw_map(values,name,title,cmap_name="viridis",categorical=False):
    fig,ax=plt.subplots(figsize=(12,9),facecolor="white")
    pc=PatchCollection(polys,linewidth=.12,edgecolor=(1,1,1,.55))
    if categorical:
        order=["Rice","Cotton","Maize","Bajra","Groundnut"]
        colors=["#2166ac","#b2182b","#fdae61","#66a61e","#984ea3"]
        lut={k:i for i,k in enumerate(order)}; pc.set_array(np.array([lut[v] for v in values]));
        from matplotlib.colors import ListedColormap
        pc.set_cmap(ListedColormap(colors)); pc.set_clim(-.5,len(order)-.5)
        cb=fig.colorbar(pc,ax=ax,ticks=range(len(order)),shrink=.78); cb.ax.set_yticklabels(order)
    else:
        pc.set_array(np.asarray(values,float)); pc.set_cmap(cmap_name); fig.colorbar(pc,ax=ax,shrink=.78)
    ax.add_collection(pc); ax.autoscale(); ax.set_aspect("equal"); ax.axis("off"); ax.set_title(title,fontsize=17,pad=14)
    p=WORK/name; fig.savefig(p,dpi=300,bbox_inches="tight",facecolor="white"); plt.close(fig); print("map:",rel(p)); return p
ordered=sub.set_index("farm_id").reindex(fids)
HEALTH_MAP=draw_map(ordered.health_index,"health_index_map.png","Farm Health Index","RdYlGn")
YIELD_MAP=draw_map(ordered.yield_estimate_to_date,"yield_estimate_to_date_map.png","Yield Estimate to Date (kg/ha)","YlGn")
CROP_MAP=draw_map(ordered.crop_type,"crop_classification_map.png","Crop Classification",categorical=True)

---
## 12 · Limitations, rejected signals and reproducibility

- CROP_ID is hybrid multi-sensor; it is not claimed to be Capella-primary. Strict B9 crop-ID
  remediation failed. HEALTH is Capella X-band-primary, while YIELD combines historical scale,
  stage factor and Capella-derived condition.
- Historical S2 item IDs were recovered by exact farm-feature regression against legitimate
  public products. The pinned source contains all 18 accepted acquisitions, including the four
  T43QBE products selected only after same-date T43QCE candidates failed the original filters.
- Rejected pathways remain non-production evidence: five-natural-cluster assumption, Jun19
  high-backscatter canopy interpretation, Jun19→Aug14 yield condition signal, B6-R2 rescue,
  B6-R3 species-validation interpretation, and strict B9 crop-ID remediation.
- No ground-truth farm crop labels or yields were supplied. Relative crop support, heuristic
  uncertainty and agronomic scenario bounds must not be read as calibrated probabilities or CIs.

In [ ]:
TOTAL_RUNTIME=time.time()-T_START
validation={
 "status":"PASS", "historical_s2_recovery":"PASS", "final_s2_dates":len(sp["s2_dates"]),
 "hf_repo":HF_REPO, "hf_revision":HF_REVISION, "hash_audit":"PASS",
 "rows":len(sub), "crop_counts":sub.crop_type.value_counts().to_dict(),
 "submission_sha256":SUB_SHA, "stage_seconds":{k:round(v,3) for k,v in T.items()},
 "total_runtime_seconds":round(TOTAL_RUNTIME,3), "checks":{k:bool(v) for k,v in checks.items()},
 "outputs":{ "submission":str(SUB_PATH), "health_map":str(HEALTH_MAP),
             "yield_map":str(YIELD_MAP), "crop_map":str(CROP_MAP)}
}
(WORK/"end_to_end_validation.json").write_text(json.dumps(validation,indent=2),encoding="utf-8")
print(f"END-TO-END PASS in {TOTAL_RUNTIME/60:.1f} min")
print(json.dumps(validation,indent=2))